# GLM-5.3-Flash on a free Kaggle TPU v5e-8

This notebook serves **GLM-5.3-Flash**, a 320B-parameter mixture-of-experts model (18B active), on Kaggle's free
TPU and gives you a public endpoint that speaks both the OpenAI and the Anthropic API. Use it from your laptop,
Claude Code, Codex CLI, opencode, or plain `curl`.

As far as we know this is the first inference engine to run GLM-5.3-Flash on a TPU, and it runs on the free one.
We wrote it in JAX for this model: the experts sit in HBM as 3-bit weights that custom TPU kernels read straight
into the matrix multiply, and a small draft model proposes blocks of tokens that the big model verifies in one step.
Measured here: about **165 tok/s** for one stream, about **170 tok/s** aggregate over three, the model's full
**262,144-token context**, prefix caching that resumes an agent session in about a second, and image inputs.

## Before you run: three clicks in the right sidebar
1. **Accelerator → TPU VM v5e-8**
2. **Internet → ON** (the tunnel needs it)
3. **Add Input** → attach these three datasets:
   - `rahim3/glm53-flash-iq3xxs-1` and `rahim3/glm53-flash-iq3xxs-2`: the experts (Unsloth's UD-IQ3_XXS GGUF)
   - `rahim3/glm53-flash-serve-v3`: the rest of the weights, the vision tower, the tokenizer and the programs

Then run all cells. The last cell is the server and keeps running on purpose; the endpoint URL and the API key
appear in its `READY` banner after about 6–13 minutes (the weights are most of that). The draft model
(`incoai/GLM-5.3-Flash-DFlash2`, CC BY-NC-ND 4.0, research use) comes from Hugging Face when the server starts;
set `dflash` to `""` in the config cell to run without it, at about 79 tok/s.

If the last cell stops in its first minute, its message says why: no TPU in this session (Kaggle does that
sometimes; stop and start the session again), Internet off, or a dataset not attached.

The code, the engine and a guide for the same recipe on a rented TPU VM are in the GitHub repo:
https://github.com/ARahim3/kaggle-tpu-lab. Keep an eye on it for more models in the future!


In [ ]:
%%writefile serve_config.json
{
  "streams": 3,
  "max_len": 262144,
  "reasoning_effort_default": "low",
  "vision": true,
  "keepalive_min": 90
}


### Configuration
The defaults above are what we serve. Things you might change:
- `streams`: how many requests decode together. Three fit in HBM at the full 262k context; a fourth request waits
  for a free slot. One stream runs at about 165 tok/s, three together at about 170 tok/s aggregate.
- `dflash`: where the draft model comes from: `hf:<repo>@<revision>` (the default, fetched from Hugging Face), an
  attached dataset or directory holding its `config.json` and `model.safetensors`, or `""` for no speculative
  decoding.
- `reasoning_effort_default`: `"low"`, `"medium"` or `"high"`; clients can still set it per request.
- `vision`: `false` skips the vision tower (a minute and a little HBM saved; image inputs then error out).
- `keepalive_min`: the server shuts itself down after this long. Raise it (up to about 480) for a long-lived
  endpoint; the default keeps a test run cheap on your TPU quota.
- `api_key`: set your own; otherwise one is generated and printed in the banner.
- `ntfy_topic`: optional; the script posts its progress to `ntfy.sh/<topic>` so you can follow it from your phone.


### The engine
One cell that writes the engine package next to the script. It is generated from the repo, so there is nothing to edit here; scroll past it.


In [ ]:
# The engine: the `glm53` package (19 files), generated from the repo's glm53-flash/engine/glm53/ — nothing to edit here.
import os
os.makedirs("glm53", exist_ok=True)
FILES = {}
FILES["glm53/__init__.py"] = r'''"""GLM-5.3-Flash (Glm5Next) in JAX for TPU v5e-8 with host-resident experts."""
'''
FILES["glm53/aot.py"] = r'''"""Ahead-of-time export of the engine's programs (`jax.export`).

The first call of a program pays three things: JAX traces the Python (the model code and every Pallas kernel body),
lowers the trace to StableHLO, and XLA compiles that. The persistent compile cache removes the third. This module
removes the first two: a program that had to be traced is exported and written to disk (StableHLO with the Mosaic
kernels as `tpu_custom_call`s; the weights are arguments, so a file holds code only), and a later process
deserializes the file and calls it (`Exported.call`), whose lowering is one module merge.

`aot.jit(key, fn, **jit_kw)` stands in for `jax.jit` at the engine's program sites. Until `configure` names a
directory it IS `jax.jit`. With one, the program is resolved at its first call:

  * a file for (code, versions, engine configuration, program key, argument signature) exists -> `jit(Exported.call)`;
  * otherwise the function is traced once for the export, the file is written, and the program that runs is the
    export read back — the module a later run rebuilds from the file, so both runs ask the compile cache for the
    same executable;
  * an export that fails for any reason leaves the plain `jax.jit` program (logged once per program).

A file's name hashes everything the trace depends on besides the function's own arguments: the package's source, the
jax / jaxlib versions, the device kind and jax's tracing flags (`configure`), the engine configuration the programs
close over (`engine_context`), the program's key and the shapes, dtypes and shardings of its arguments. A file from
any other build is simply not found.
"""
import dataclasses
import hashlib
import importlib
import os
import sys
import time

import numpy as np
import jax

EXT = ".jaxexp"
MODULES = ("model", "engine", "resident", "dflash", "pallas_moe", "pallas_mhc", "pallas_kda", "quant8", "planes",
           "iqquant", "iq_grids_data", "vision", "aot")       # the code a program's trace can reach


class _Store:
    def __init__(self):
        self.write_dir = None          # traced programs are exported here (None: never export)
        self.read_dirs = ()            # searched in order for a program's file
        self.tag = ""                  # code + versions + device kind
        self.context = None            # () -> str: the configuration the programs close over
        self.log = print
        self.stats = {"loaded": 0, "exported": 0, "traced": 0, "failed": 0, "load_s": 0.0, "export_s": 0.0,
                      "first_loaded_s": 0.0, "first_exported_s": 0.0, "first_traced_s": 0.0}   # first calls, by origin

    @property
    def active(self):
        return bool(self.write_dir or self.read_dirs)


STORE = _Store()


def _stable(x):
    """A repr without addresses or array contents (configuration values only)."""
    if x is None or isinstance(x, (bool, int, float, str, bytes)):
        return repr(x)
    if isinstance(x, (tuple, list)):
        return "[" + ",".join(_stable(v) for v in x) + "]"
    if isinstance(x, (set, frozenset)):
        return "{" + ",".join(sorted(_stable(v) for v in x)) + "}"
    if isinstance(x, dict):
        items = sorted(x.items(), key=lambda kv: repr(kv[0]))
        return "{" + ",".join(f"{_stable(k)}:{_stable(v)}" for k, v in items) + "}"
    if dataclasses.is_dataclass(x) and not isinstance(x, type):
        return type(x).__name__ + _stable({f.name: getattr(x, f.name) for f in dataclasses.fields(x)})
    if isinstance(x, jax.sharding.PartitionSpec):
        return repr(x)
    try:
        return "dtype:" + np.dtype(x).name
    except Exception:  # noqa: BLE001
        pass
    if callable(x):
        return f"{getattr(x, '__module__', '')}.{getattr(x, '__qualname__', type(x).__name__)}"
    return f"<{type(x).__name__}>"


def _flags():
    """The modules' upper-case settings (MHC_KERNEL, KDA_KERNEL, SAMPLE_IMPL, ...): options a build may switch."""
    out = {}
    for m in MODULES:
        mod = sys.modules.get("glm53." + m)             # (`configure` imported them all: see there)
        for k, v in (vars(mod).items() if mod is not None else ()):
            if k.isupper() and (v is None or isinstance(v, (bool, int, float, str, tuple))):
                out[f"{m}.{k}"] = v
    return out


def engine_context(eng):
    """What `eng`'s programs close over besides their own key and arguments, as a string: the model and drafter
    configurations, the capacity, the layer grouping, the expert kernels' options and the modules' settings."""
    f, d = eng.expert_fetch, getattr(eng, "dflash", None)
    simple = lambda o: {k: v for k, v in vars(o).items() if not k.startswith("_")}   # noqa: E731
    return _stable({
        "cfg": eng.cfg, "lcfg": eng.lcfg, "max_len": eng.max_len, "n": eng.n, "axes": tuple(eng.mesh.axis_names),
        "lpp": getattr(eng, "lpp", None), "piece": getattr(eng, "prefill_piece", None), "class": type(eng).__name__,
        "fetch": None if f is None else (type(f).__name__, simple(f)),
        "dflash": None if d is None else (d.cfg, d.lcfg, d.temp_scale, d.embed_spec, d.lm_head_spec),
        "mtp": None if getattr(eng, "mtp", None) is None else getattr(eng, "mtp_k", None),
        "flags": _flags()})


def configure(write_dir=None, read_dirs=(), context=None, log=None):
    """Turn the export on: traced programs are written to `write_dir`, and a program's file is looked up there and
    in `read_dirs` (e.g. the serve dataset's `exported/`). `context` () -> str names the engine configuration
    (`lambda: aot.engine_context(eng)`; evaluated when a program is resolved, so the engine may be built later).
    Without the `flatbuffers` package (jax serializes an export with it) the export stays off."""
    if log is not None:
        STORE.log = log
    try:
        import flatbuffers  # noqa: F401
    except ImportError:
        STORE.write_dir, STORE.read_dirs = None, ()
        STORE.log("   aot: no flatbuffers package (pip install flatbuffers) -> programs are traced as usual")
        return STORE
    h = hashlib.sha256()
    for m in MODULES:                                   # the package's source as it is on disk now
        with open(os.path.join(os.path.dirname(os.path.abspath(__file__)), m + ".py"), "rb") as fh:
            h.update(fh.read())
        # Import every module now. Some are first imported while a program is traced (pallas_mhc: by the first decode
        # trace), and their settings are part of the files' names (`_flags`): a run that loads its programs traces
        # nothing, so without this it would name the programs after that point differently from the run that
        # exported them (v6e 2026-10-06: 98 of 99 loaded, the first program after the import exported again).
        try:
            importlib.import_module("glm53." + m)
        except Exception as e:  # noqa: BLE001
            STORE.log(f"   aot: glm53.{m} not importable ({type(e).__name__}: {str(e)[:120]}); its settings are left out")
    import jaxlib
    dev = jax.devices()[0]
    flags = [str(getattr(jax.config, k, "")) for k in ("jax_enable_x64", "jax_default_matmul_precision",
                                                       "jax_use_shardy_partitioner")]      # what a trace reads
    STORE.tag = "|".join((h.hexdigest(), jax.__version__, jaxlib.__version__, dev.platform,
                          str(getattr(dev, "device_kind", "")), str(jax.device_count()), *flags))
    STORE.write_dir = str(write_dir) if write_dir else None
    dirs = ([STORE.write_dir] if STORE.write_dir else []) + [str(d) for d in read_dirs if d]
    STORE.read_dirs = tuple(dict.fromkeys(dirs))
    STORE.context = context
    if STORE.write_dir:
        os.makedirs(STORE.write_dir, exist_ok=True)
    return STORE


def _signature(args):
    """Shapes, dtypes and shardings of a call's arguments (the pytree structure included)."""
    leaves, tree = jax.tree.flatten(args)

    def one(a):
        sh = getattr(a, "sharding", None)
        where = "host" if sh is None else str(getattr(sh, "spec", type(sh).__name__))
        return f"{tuple(np.shape(a))}:{getattr(a, 'dtype', type(a).__name__)}:{where}:{getattr(a, '_committed', '')}"
    return str(tree) + "|" + ";".join(one(a) for a in leaves)


def _label(key):
    k = key
    while isinstance(k, tuple) and k:
        k = k[0]
    return "".join(c if c.isalnum() or c == "_" else "_" for c in str(k))[:32] or "prog"


def _call(exp):
    """`exp.call` under the exported function's own name (the jit program keeps the name it had: logs, profiles)."""
    def call(*args):
        return exp.call(*args)
    call.__name__ = call.__qualname__ = exp.fun_name
    return call


class _Program:
    """A program resolved at its first call (see the module docstring); afterwards one extra Python call."""

    def __init__(self, fn, key, jit_kw):
        self._fn, self._key, self._kw = fn, key, jit_kw
        self._f = self._sig = None
        self.origin = None                          # "loaded" | "exported" | "traced" once resolved

    def __call__(self, *args):
        f = self._f
        if f is None:
            return self._first(args)
        try:
            return f(*args)
        except Exception:
            # jax.jit retraces for a new argument signature; an export holds one. The error of a mismatch is raised
            # while the call is traced (nothing ran, no donated buffer consumed): fall back to the plain program.
            try:
                same = self._sig is None or _signature(args) == self._sig
            except Exception:  # noqa: BLE001
                same = True
            if same:
                raise
            STORE.log(f"   aot: {_label(self._key)} called with a second signature -> traced")
            self._plain()
            return self._f(*args)

    def _plain(self):
        STORE.stats["traced"] += 1
        self._f, self._sig, self.origin = jax.jit(self._fn, **self._kw), None, "traced"

    def _first(self, args):
        """Resolve and make the first call (the lowering and the compile, or the compile cache's read, happen here).
        An export that cannot be called after all (another platform or device count, a compile error) is replaced by
        the plain program, once."""
        f = self._resolve(args)
        t = time.time()
        try:
            out = f(*args)
        except Exception as e:  # noqa: BLE001
            if self.origin == "traced":
                raise
            STORE.stats["failed"] += 1
            STORE.log(f"   aot: {_label(self._key)} ({self.origin}) failed at its first call ({type(e).__name__}: "
                      f"{str(e)[:300]}) -> traced")
            self._plain()
            f, out = self._f, self._f(*args)
        STORE.stats[f"first_{self.origin}_s"] += time.time() - t
        self._f = f
        return out

    def _cache_size(self):
        return 0 if self._f is None else self._f._cache_size()

    def _resolve(self, args):
        from jax import export as jex
        st = STORE
        sig = _signature(args)
        ctx = st.context() if st.context is not None else ""
        h = hashlib.sha256("\n".join((st.tag, ctx, _stable(self._key), _stable(self._kw), sig)).encode()).hexdigest()
        name = f"{_label(self._key)}-{h[:24]}{EXT}"
        for d in st.read_dirs:
            p = os.path.join(d, name)
            if not os.path.isfile(p):
                continue
            try:
                t = time.time()
                with open(p, "rb") as fh:
                    exp = jex.deserialize(bytearray(fh.read()))
                f = jax.jit(_call(exp), **self._kw)
                st.stats["loaded"] += 1
                st.stats["load_s"] += time.time() - t
                self._sig, self.origin = sig, "loaded"
                return f
            except Exception as e:  # noqa: BLE001
                st.log(f"   aot: {p} unreadable ({type(e).__name__}: {str(e)[:200]}) -> traced")
                break
        jf = jax.jit(self._fn, **self._kw)
        if st.write_dir:
            try:
                t = time.time()
                blob = jex.export(jf)(*args).serialize()
                tmp = os.path.join(st.write_dir, f".{name}.{os.getpid()}.tmp")
                with open(tmp, "wb") as fh:
                    fh.write(blob)
                os.replace(tmp, os.path.join(st.write_dir, name))
                f = jax.jit(_call(jex.deserialize(bytearray(blob))), **self._kw)
                st.stats["exported"] += 1
                st.stats["export_s"] += time.time() - t
                self._sig, self.origin = sig, "exported"
                return f
            except Exception as e:  # noqa: BLE001
                st.stats["failed"] += 1
                st.log(f"   aot: export of {name} failed ({type(e).__name__}: {str(e)[:400]}) -> traced")
        st.stats["traced"] += 1
        self.origin = "traced"
        return jf


def jit(key, fn, **jit_kw):
    """`jax.jit(fn, **jit_kw)` for a program site; `key` identifies the program within its engine (the `_progs` key)."""
    if not STORE.active:
        return jax.jit(fn, **jit_kw)
    return _Program(fn, key, jit_kw)


def summary():
    s = STORE.stats
    return (f"{s['loaded']} programs loaded from exports (read {s['load_s']:.0f}s, first calls "
            f"{s['first_loaded_s']:.0f}s), {s['exported']} traced and exported (export {s['export_s']:.0f}s, first "
            f"calls {s['first_exported_s']:.0f}s), {s['traced']} traced only (first calls {s['first_traced_s']:.0f}s), "
            f"{s['failed']} export failures")
'''
FILES["glm53/basestore.py"] = r'''"""Base store: everything a serving build needs besides the routed experts — the non-expert parameters exactly as
the engine holds them (int8 nodes included), the vision tower and the tokenizer / config files — written once to a
directory (a Kaggle kernel's output, turned into a dataset) and read back in seconds instead of from the 265 GB FP8
checkpoint.

Layout: `manifest.json`, `top.npz` (embed / norm / lm_head), `layer_NN.npz` (one per layer, the expert tables left
out), `vision.npz`, and the tokenizer files. Trees are flattened to "a/b/0/c" keys (lists as numbered keys); bf16
arrays are stored as uint16 bit patterns with the dtype recorded; non-array leaves go into the manifest."""
import json
import os
import shutil

import numpy as np
import jax

EXPERT_KEYS = ("gate_q", "up_q", "down_q")                     # resident expert tables (glm53.resident), never stored
HF_FILES = ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja",
            "special_tokens_map.json", "preprocessor_config.json", "video_preprocessor_config.json")


def flatten(tree, prefix=""):
    """-> {"a/b/0/c": leaf}; dicts and lists are containers, everything else is a leaf."""
    out = {}
    if isinstance(tree, dict):
        for k, v in tree.items():
            out.update(flatten(v, f"{prefix}{k}/"))
    elif isinstance(tree, (list, tuple)):
        for i, v in enumerate(tree):
            out.update(flatten(v, f"{prefix}{i}/"))
    else:
        out[prefix[:-1]] = tree
    return out


def unflatten(flat):
    """Inverse of `flatten`: a container whose keys are all integers becomes a list (in index order)."""
    root = {}
    for key, leaf in flat.items():
        parts = key.split("/")
        d = root
        for p in parts[:-1]:
            d = d.setdefault(p, {})
        d[parts[-1]] = leaf

    def fix(d):
        if not isinstance(d, dict):
            return d
        d = {k: fix(v) for k, v in d.items()}
        if d and all(k.isdigit() for k in d):
            return [d[str(i)] for i in range(len(d))]
        return d
    return fix(root)


def _host(x):
    a = np.asarray(jax.device_get(x)) if hasattr(x, "shape") else x
    if getattr(a, "dtype", None) is not None and a.dtype.name == "bfloat16":
        return a.view(np.uint16), "bfloat16"
    return a, str(a.dtype)


def save_tree(path, tree):
    """One npz for a tree: arrays stored natively (bf16 as uint16), other leaves in the "__meta__" entry."""
    arrays, dtypes, others = {}, {}, {}
    for k, v in flatten(tree).items():
        if hasattr(v, "shape") and hasattr(v, "dtype"):
            a, dt = _host(v)
            arrays[k] = a
            dtypes[k] = dt
        else:
            others[k] = v
    meta = json.dumps({"dtypes": dtypes, "others": others})
    np.savez(path, __meta__=np.frombuffer(meta.encode(), np.uint8), **arrays)


def load_tree(path):
    import ml_dtypes
    with np.load(path) as z:
        meta = json.loads(bytes(z["__meta__"]).decode())
        flat = {}
        for k, dt in meta["dtypes"].items():
            a = z[k]
            flat[k] = a.view(ml_dtypes.bfloat16) if dt == "bfloat16" else a
    flat.update(meta["others"])
    return unflatten(flat)


def dump(out_dir, params, vision=None, hf_dir=None, meta=None, log=print):
    """Write the store: `params` = the engine's parameter tree (device or host arrays; expert tables skipped),
    `vision` = the vision tower's host parameter tree (as `glm53.vision.load_vision` returns it, any dtype),
    `hf_dir` = where the tokenizer / config files are."""
    os.makedirs(out_dir, exist_ok=True)
    save_tree(os.path.join(out_dir, "top.npz"), {k: params[k] for k in ("embed", "norm", "lm_head")})
    n = len(params["layers"])
    for i, L in enumerate(params["layers"]):
        L = {**L, "mlp": {k: v for k, v in L["mlp"].items() if k not in EXPERT_KEYS}}
        save_tree(os.path.join(out_dir, f"layer_{i:02d}.npz"), L)
        if i % 10 == 0 or i == n - 1:
            log(f"   base store: layer {i + 1}/{n}")
    if vision is not None:
        save_tree(os.path.join(out_dir, "vision.npz"), vision)
    files = []
    if hf_dir:
        for f in HF_FILES:
            src = os.path.join(hf_dir, f)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(out_dir, f)); files.append(f)
    manifest = {"n_layers": n, "vision": vision is not None, "files": files, **(meta or {})}
    json.dump(manifest, open(os.path.join(out_dir, "manifest.json"), "w"), indent=1)
    size = sum(os.path.getsize(os.path.join(out_dir, f)) for f in os.listdir(out_dir)) / 1e9
    log(f"   base store written: {out_dir} ({size:.2f} GB, {n} layers, vision {vision is not None})")
    return manifest


def load(in_dir):
    """-> {"top": {embed, norm, lm_head}, "layers": [layer trees without expert tables], "vision": tree or None,
    "manifest": dict}. Host arrays; bf16 restored."""
    manifest = json.load(open(os.path.join(in_dir, "manifest.json")))
    top = load_tree(os.path.join(in_dir, "top.npz"))
    layers = [load_tree(os.path.join(in_dir, f"layer_{i:02d}.npz")) for i in range(manifest["n_layers"])]
    vision = load_tree(os.path.join(in_dir, "vision.npz")) if manifest.get("vision") else None
    return {"top": top, "layers": layers, "vision": vision, "manifest": manifest}
'''
FILES["glm53/checkpoint.py"] = r'''"""Read the real GLM-5.3-Flash safetensors checkpoint into the `glm53.model` param layout.

Checkpoint facts (verified 2026-09-05): fp8 e4m3 tensors carry `<name>_scale_inv` float32 with 128x128
blocks; KDA projections / conv / gates / indexer / kv_b / router / mHC are bf16 or fp32. Expert tensors
are per-expert (`mlp.experts.N.{gate,up,down}_proj.weight`). All Linear weights are torch [out, in].
"""
from __future__ import annotations

import json
import os
import re
from functools import lru_cache

import numpy as np

PREFIX = "model.language_model."
BLOCK = 128


@lru_cache(maxsize=None)
def weight_map(model_dir: str) -> dict[str, str]:
    """model_dir may be one directory or several joined by ':' (e.g. the 4 Kaggle dataset mounts); the index is
    taken from the first dir that has it and each shard is resolved to whichever dir contains it."""
    for d in model_dir.split(":"):
        p = os.path.join(d, "model.safetensors.index.json")
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)["weight_map"]
    for d in model_dir.split(":"):                    # a single-file model (e.g. the DFlash drafter): no index
        p = os.path.join(d, "model.safetensors")
        if os.path.exists(p):
            with open(p, "rb") as f:
                hdr = json.loads(f.read(int.from_bytes(f.read(8), "little")))
            return {k: "model.safetensors" for k in hdr if k != "__metadata__"}
    raise FileNotFoundError("model.safetensors.index.json not found in " + model_dir)


def resolve_shard(model_dir: str, fname: str) -> str:
    for d in model_dir.split(":"):
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{fname} not found in {model_dir}")


class ShardReader:
    """Caches open safetensors handles (torch framework, so fp8 loads natively). `model_dir` may be ':'-joined."""

    def __init__(self, model_dir: str):
        self.model_dir = model_dir
        self.wm = weight_map(model_dir)
        self._open: dict[str, object] = {}

    def _handle(self, fname):
        from safetensors import safe_open
        if fname not in self._open:
            self._open[fname] = safe_open(resolve_shard(self.model_dir, fname), framework="pt")
        return self._open[fname]

    def release(self, fname: str):
        """Close a shard and drop its pages from the page cache (page cache counts toward the Kaggle cgroup)."""
        self._open.pop(fname, None)
        try:
            fd = os.open(resolve_shard(self.model_dir, fname), os.O_RDONLY)
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            os.close(fd)
        except Exception:  # noqa: BLE001
            pass

    def has(self, name: str) -> bool:
        return name in self.wm

    def raw(self, name: str):
        """torch tensor as stored (bf16 / fp8 / f32)."""
        return self._handle(self.wm[name]).get_tensor(name)

    def get(self, name: str, dtype=np.float32) -> np.ndarray:
        """Dequantized numpy array. fp8 tensors are multiplied by their blockwise scale_inv."""
        import torch
        t = self.raw(name)
        if t.dtype == torch.float8_e4m3fn:
            s = self.raw(name + "_scale_inv").float().numpy()
            return dequant_fp8(t.float().numpy(), s).astype(dtype)
        return t.float().numpy().astype(dtype)

    def get_fp8_raw(self, name: str):
        """(fp8 bytes as int8-viewed uint8 array [O,I], scale_inv f32 [O/128, I/128]) for host expert tables."""
        import torch
        t = self.raw(name)
        assert t.dtype == torch.float8_e4m3fn, name
        s = self.raw(name + "_scale_inv").float().numpy()
        return t.view(torch.uint8).numpy(), s


def dequant_fp8(w: np.ndarray, scale_inv: np.ndarray, block: int = BLOCK) -> np.ndarray:
    O, I = w.shape
    s = np.repeat(np.repeat(scale_inv, block, axis=0), block, axis=1)[:O, :I]
    return w.astype(np.float32) * s


def _lin(r: ShardReader, name: str, dtype):
    return np.ascontiguousarray(r.get(name, dtype).T)


def load_top(r: ShardReader, dtype=np.float32):
    return {
        "embed": r.get(PREFIX + "embed_tokens.weight", dtype),
        "norm": r.get(PREFIX + "norm.weight", dtype),
        "lm_head": _lin(r, "lm_head.weight", dtype),
    }


def load_hc(r: ShardReader, pfx: str, site: str, dtype):
    return {"fn": _lin(r, f"{pfx}hc_{site}_fn", dtype), "base": r.get(f"{pfx}hc_{site}_base"),
            "scale": r.get(f"{pfx}hc_{site}_scale")}


def load_kda(r: ShardReader, a: str, dtype):
    return {
        "q": _lin(r, a + "q_proj.weight", dtype), "k": _lin(r, a + "k_proj.weight", dtype),
        "v": _lin(r, a + "v_proj.weight", dtype),
        "conv_q": r.get(a + "q_conv1d.weight")[:, 0, :], "conv_k": r.get(a + "k_conv1d.weight")[:, 0, :],
        "conv_v": r.get(a + "v_conv1d.weight")[:, 0, :],
        "f_a": _lin(r, a + "f_a_proj.weight", dtype), "f_b": _lin(r, a + "f_b_proj.weight", dtype),
        "dt_bias": r.get(a + "dt_bias"), "A_log": r.get(a + "A_log"),
        "b": _lin(r, a + "b_proj.weight", dtype),
        "g_a": _lin(r, a + "g_a_proj.weight", dtype), "g_b": _lin(r, a + "g_b_proj.weight", dtype),
        "o_norm": r.get(a + "o_norm.weight"), "o": _lin(r, a + "o_proj.weight", dtype),
    }


def load_mla(r: ShardReader, a: str, dtype):
    p = {
        "q_a": _lin(r, a + "q_a_proj.weight", dtype), "q_a_norm": r.get(a + "q_a_layernorm.weight"),
        "q_b": _lin(r, a + "q_b_proj.weight", dtype),
        "kv_a": _lin(r, a + "kv_a_proj_with_mqa.weight", dtype), "kv_a_norm": r.get(a + "kv_a_layernorm.weight"),
        "kv_b": _lin(r, a + "kv_b_proj.weight", dtype), "o": _lin(r, a + "o_proj.weight", dtype),
    }
    ix = a + "indexer."
    if r.has(ix + "wk.weight"):
        p["indexer"] = {
            "wq_b": _lin(r, ix + "wq_b.weight", dtype), "wk": _lin(r, ix + "wk.weight", dtype),
            "k_norm_w": r.get(ix + "k_norm.weight"), "k_norm_b": r.get(ix + "k_norm.bias"),
            "weights_proj": _lin(r, ix + "weights_proj.weight", dtype),
            "ape": r.get(ix + "index_kpool_compress_ape"), "gate": _lin(r, ix + "index_kpool_compress_gate", dtype),
        }
    return p


def load_mlp(r: ShardReader, m: str, dtype):
    return {"gate": _lin(r, m + "gate_proj.weight", dtype), "up": _lin(r, m + "up_proj.weight", dtype),
            "down": _lin(r, m + "down_proj.weight", dtype)}


def load_moe_nonexpert(r: ShardReader, m: str, dtype):
    """Router + shared expert only. Routed experts are loaded separately (host tables)."""
    return {"router_w": _lin(r, m + "gate.weight", dtype), "router_bias": r.get(m + "gate.e_score_correction_bias"),
            "shared": load_mlp(r, m + "shared_experts.", dtype)}


def load_expert(r: ShardReader, layer: int, e: int, dtype=np.float32):
    """Dequantized (gate_up [D, 2mi], down [mi, D]) for one routed expert."""
    m = f"{PREFIX}layers.{layer}.mlp.experts.{e}."
    gate = _lin(r, m + "gate_proj.weight", dtype)
    up = _lin(r, m + "up_proj.weight", dtype)
    return np.concatenate([gate, up], axis=1), _lin(r, m + "down_proj.weight", dtype)


def load_layer(r: ShardReader, i: int, layer_type: str, mlp_type: str, dtype=np.float32, with_experts=False):
    pfx = f"{PREFIX}layers.{i}."
    p = {
        "ln1": r.get(pfx + "input_layernorm.weight"), "ln2": r.get(pfx + "post_attention_layernorm.weight"),
        "hc_attn": load_hc(r, pfx, "attn", dtype), "hc_ffn": load_hc(r, pfx, "ffn", dtype),
    }
    a = pfx + "self_attn."
    p["attn"] = load_kda(r, a, dtype) if layer_type == "linear_attention" else load_mla(r, a, dtype)
    m = pfx + "mlp."
    if mlp_type == "sparse":
        p["mlp"] = load_moe_nonexpert(r, m, dtype)
        if with_experts:
            n_exp = len({k for k in r.wm if k.startswith(m + "experts.") and k.endswith("gate_proj.weight")})
            gu, dn = zip(*(load_expert(r, i, e, dtype) for e in range(n_exp)))
            p["mlp"]["gate_up"] = np.stack(gu)
            p["mlp"]["down"] = np.stack(dn)
    else:
        p["mlp"] = load_mlp(r, m, dtype)
    return p


def load_mtp_layer(r: ShardReader, i: int, dtype=np.float32):
    """The MTP (NextN) layer: a plain pre-norm MLA (+ indexer) + MoE block without hyper-connections, fed with
    eh_proj(cat(enorm(embed(next token)), hnorm(previous hidden))); `head_norm` precedes the shared lm_head.
    Routed experts come from the GGUF (Q2_K / Q3_K planes), like every other sparse layer."""
    pfx = f"{PREFIX}layers.{i}."
    return {
        "ln1": r.get(pfx + "input_layernorm.weight"), "ln2": r.get(pfx + "post_attention_layernorm.weight"),
        "attn": load_mla(r, pfx + "self_attn.", dtype), "mlp": load_moe_nonexpert(r, pfx + "mlp.", dtype),
        "enorm": r.get(pfx + "enorm.weight"), "hnorm": r.get(pfx + "hnorm.weight"),
        "eh_proj": _lin(r, pfx + "eh_proj.weight", dtype), "head_norm": r.get(pfx + "shared_head.norm.weight"),
    }


def layer_shards(model_dir: str, i: int) -> set[str]:
    return {v for k, v in weight_map(model_dir).items() if re.search(rf"layers\.{i}\.", k)}


# ----------------------------------------------------------------------------- device-side fp8 dequant
def dequant_fp8_device(w_u8, scale_inv, out_dtype=None, block: int = BLOCK):
    """JAX: fp8-e4m3 bytes viewed as uint8 [..., O, I] + scale_inv [..., O/b, I/b] -> float [..., O, I].
    Used on TPU after streaming raw expert bytes from the host (halves PCIe traffic vs bf16)."""
    import jax
    import jax.numpy as jnp
    f8 = jax.lax.bitcast_convert_type(w_u8, jnp.float8_e4m3fn)
    w = f8.astype(jnp.float32)
    O, I = w.shape[-2:]
    s = jnp.repeat(jnp.repeat(scale_inv.astype(jnp.float32), block, axis=-2), block, axis=-1)[..., :O, :I]
    out = w * s
    return out if out_dtype is None else out.astype(out_dtype)


# ----------------------------------------------------------------------------- int4 (RTN, blockwise) pack/unpack
def quant_int4_blockwise(w: np.ndarray, block: int = BLOCK):
    """float [O, I] -> (packed uint8 [O, I//2] (low nibble = even column), scale f32 [O/b, I/b]); symmetric RTN
    to [-8, 7] per block. Requires O, I multiples of `block` and I even."""
    O, I = w.shape
    blk = w.reshape(O // block, block, I // block, block)
    amax = np.abs(blk).max(axis=(1, 3), keepdims=True)
    scale = np.where(amax > 0, amax / 7.0, 1.0).astype(np.float32)
    q = np.clip(np.round(blk / scale), -8, 7).astype(np.int8).reshape(O, I)
    u = (q & 0xF).astype(np.uint8)
    packed = (u[:, 0::2] | (u[:, 1::2] << 4)).astype(np.uint8)
    return packed, scale.reshape(O // block, I // block)


def dequant_int4_device(packed_u8, scale, out_dtype=None, block: int = BLOCK):
    """JAX: packed [..., O, I//2] uint8 + scale [..., O/b, I/b] -> float [..., O, I]."""
    import jax.numpy as jnp
    lo = (packed_u8 & 0xF).astype(jnp.int8)
    hi = (packed_u8 >> 4).astype(jnp.int8)
    lo = jnp.where(lo > 7, lo - 16, lo)
    hi = jnp.where(hi > 7, hi - 16, hi)
    q = jnp.stack([lo, hi], axis=-1).reshape(packed_u8.shape[:-1] + (packed_u8.shape[-1] * 2,)).astype(jnp.float32)
    O, I = q.shape[-2:]
    s = jnp.repeat(jnp.repeat(scale.astype(jnp.float32), block, axis=-2), block, axis=-1)[..., :O, :I]
    out = q * s
    return out if out_dtype is None else out.astype(out_dtype)


def dequant_int4_numpy(packed_u8, scale, block: int = BLOCK):
    lo = (packed_u8 & 0xF).astype(np.int8); hi = (packed_u8 >> 4).astype(np.int8)
    lo[lo > 7] -= 16; hi[hi > 7] -= 16
    q = np.stack([lo, hi], -1).reshape(packed_u8.shape[:-1] + (packed_u8.shape[-1] * 2,)).astype(np.float32)
    O, I = q.shape[-2:]
    s = np.repeat(np.repeat(scale, block, axis=-2), block, axis=-1)[..., :O, :I]
    return q * s


# ----------------------------------------------------------------------------- mmap-free shard reader
class RawShardReader(ShardReader):
    """Reads tensors with explicit pread instead of safetensors' mmap. With mmap, every touched page of every open
    shard counts toward the process RSS (and the Kaggle cgroup), which is what pushed the 2026-09-06 full build over
    the memory guard and forced int4 for half the layers. Here only our own arrays occupy memory."""

    _DT = {"F32": np.float32, "F16": np.float16, "BF16": "bf16", "F8_E4M3": "fp8", "I8": np.int8, "U8": np.uint8,
           "I32": np.int32, "I64": np.int64}

    def _header(self, fname):
        if fname not in self._open:
            path = resolve_shard(self.model_dir, fname)
            with open(path, "rb") as f:
                n = int.from_bytes(f.read(8), "little")
                hdr = json.loads(f.read(n))
            hdr.pop("__metadata__", None)
            self._open[fname] = (path, 8 + n, hdr)
        return self._open[fname]

    def _read(self, name):
        path, base, hdr = self._header(self.wm[name])
        info = hdr[name]
        b0, b1 = info["data_offsets"]
        buf = np.empty(b1 - b0, np.uint8)
        fd = os.open(path, os.O_RDONLY)
        try:
            view = memoryview(buf)
            off = 0
            while off < len(buf):
                k = os.preadv(fd, [view[off:]], base + b0 + off)
                if k <= 0:
                    raise IOError(f"short read on {name}")
                off += k
        finally:
            os.close(fd)
        return buf, info["dtype"], tuple(info["shape"])

    def raw(self, name):  # torch tensor for API compatibility (rarely needed)
        import torch
        buf, dt, shape = self._read(name)
        t = torch.frombuffer(buf.tobytes(), dtype={"F32": torch.float32, "BF16": torch.bfloat16,
                                                   "F8_E4M3": torch.float8_e4m3fn}[dt])
        return t.reshape(shape)

    def get(self, name, dtype=np.float32):
        buf, dt, shape = self._read(name)
        if dt == "F32":
            return buf.view(np.float32).reshape(shape).astype(dtype, copy=False)
        if dt == "BF16":
            u16 = buf.view(np.uint16).astype(np.uint32) << 16
            return u16.view(np.float32).reshape(shape).astype(dtype, copy=False)
        if dt == "F8_E4M3":
            f = _fp8_to_f32(buf).reshape(shape)
            s = self.get(name + "_scale_inv")
            return dequant_fp8(f, s).astype(dtype, copy=False)
        raise ValueError(f"unsupported dtype {dt} for {name}")

    def get_fp8_raw(self, name):
        buf, dt, shape = self._read(name)
        assert dt == "F8_E4M3", (name, dt)
        return buf.reshape(shape), self.get(name + "_scale_inv")

    def release(self, fname):
        self._open.pop(fname, None)
        try:
            fd = os.open(resolve_shard(self.model_dir, fname), os.O_RDONLY)
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            os.close(fd)
        except Exception:  # noqa: BLE001
            pass


_FP8_LUT = None


def _fp8_to_f32(u8: np.ndarray) -> np.ndarray:
    """e4m3fn bytes -> float32 via a 256-entry lookup table (pure numpy)."""
    global _FP8_LUT
    if _FP8_LUT is None:
        lut = np.empty(256, np.float32)
        for i in range(256):
            s = -1.0 if i & 0x80 else 1.0
            e = (i >> 3) & 0xF
            m = i & 0x7
            if e == 0:
                v = s * (m / 8.0) * 2.0 ** -6
            elif e == 15 and m == 7:
                v = np.nan
            else:
                v = s * (1 + m / 8.0) * 2.0 ** (e - 7)
            lut[i] = v
        _FP8_LUT = lut
    return _FP8_LUT[u8]
'''
FILES["glm53/dflash.py"] = r'''"""DFlash 2 drafter for GLM-5.3-Flash (block-diffusion speculative decoding) in pure JAX.

Reference: z-lab/dflash `dflash/model.py` (MIT; `DFlash2DraftModel`, vendored as tests/dflash_ref.py), checkpoint
`incoai/GLM-5.3-Flash-DFlash2` (5 Qwen3-style layers, hidden 4096, 32 q / 8 kv heads of 128, MLP 12288, block 8,
sliding window 2048). The drafter predicts the block-1 tokens after an anchor in ONE forward: the block is the
target's embedding rows of [anchor, MASK x (block-1)]; every layer attends bidirectionally to the block and to the K/V
of the committed context positions within the window. The context comes from the target's own hidden states:
feats = the mean of the 4 mHC streams after layers `target_layers`, concatenated -> fc -> hidden_norm, and that one
tensor goes through every layer's k/v projections (no separate target projections). DFlash 2 adds a grouped dynamic
causal conv (2 taps over the block) around each attention and MLP sublayer, and a selector that walks one path
through the top-16 candidates of every draft position.

Parameter layout ([in, out], y = x @ W, as glm53.model):
  fc [nF*D, D], hidden_norm [D], norm [D]
  layers[i]: ln1, ln2 [D]; q [D, H*hd], k, v [D, KV*hd], o [H*hd, D], q_norm, k_norm [hd];
             gate, up [D, I], down [I, D]; conv_a, conv_m: {base [2, K, D], proj [D, 2*K*G]}
  sel: pred, succ [V, r] (codebooks), h [D, r]

Context cache ("ring"): k, v [L, B, W, KV, hd] = post-k_norm, post-RoPE K and V of context position p in slot p % W
(exact: a block query never sees further back than W - 1 positions), p [B, W] int32 = the position each slot holds
+ 1 (0 = empty): a slot is read only when that position is inside the window, so a position that never reached the
ring is a hole, not stale context (the engine writes every fed position: prefill, the verify's accepted positions,
batched decode steps). Positions are absolute; the drafter has its own RoPE (theta 10000) although GLM is NoPE.

Tensor parallel inside shard_map (axis engine.AXIS): q/k/v heads and MLP columns split over the chips (one KV head per
chip on 8 chips; the ring is sharded with them), o and down row-parallel (psum), fc and the conv kernel projections
column-parallel (all-gather), the codebooks, the embedding and the lm_head ([V, D] rows) vocab-sharded like the target's.
"""
from __future__ import annotations

import dataclasses
import json
import os
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
from jax import lax
from jax.sharding import NamedSharding, PartitionSpec as P

from glm53 import aot
from glm53 import model as M
from glm53 import quant8 as Q8
from glm53.engine import AXIS, R, shard_map


@dataclasses.dataclass(frozen=True)
class DCfg:
    hidden: int
    heads: int
    kv_heads: int
    hd: int
    inter: int
    n_layers: int
    eps: float
    theta: float
    window: int
    block: int
    mask_id: int
    target_layers: tuple[int, ...]
    vocab: int
    sel_rank: int
    sel_topk: int
    conv_k: int
    conv_group: int
    out_mult: float = 1.0
    softcap: float | None = None
    emb_scale: float = 1.0
    dtype: Any = jnp.float32
    tp_axis: Any = None               # shard_map axis name (None = one device, no collectives)

    def reduce(self, y):
        return y if self.tp_axis is None else lax.psum(y, self.tp_axis)

    def gather(self, y):
        return y if self.tp_axis is None else lax.all_gather(y, self.tp_axis, axis=-1, tiled=True)

    @classmethod
    def from_hf(cls, c: dict, dtype=jnp.float32) -> "DCfg":
        """`c` = the drafter's config.json (a Qwen3 config with a `dflash_config` dict)."""
        dc = c.get("dflash_config", {})

        def val(name, default=None):          # dflash_config first, then the top level (the reference's _draft_value)
            return dc.get(name, c.get(name, default))
        assert not c.get("is_causal", False) and set(c["layer_types"]) == {"sliding_attention"}, "bidirectional SWA only"
        rope = c.get("rope_parameters") or {"rope_theta": c.get("rope_theta", 10000.0), "rope_type": "default"}
        assert rope.get("rope_type", "default") == "default", rope
        return cls(hidden=c["hidden_size"], heads=c["num_attention_heads"], kv_heads=c["num_key_value_heads"],
                   hd=c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"], inter=c["intermediate_size"],
                   n_layers=c["num_hidden_layers"], eps=c["rms_norm_eps"], theta=float(rope["rope_theta"]),
                   window=c["sliding_window"], block=int(val("block_size", 16)), mask_id=int(val("mask_token_id")),
                   target_layers=tuple(val("target_layer_ids")), vocab=c["vocab_size"],
                   sel_rank=int(val("selector_rank")), sel_topk=int(val("selector_top_k")),
                   conv_k=int(val("conv_kernel_size")), conv_group=int(val("conv_group_size")),
                   out_mult=float(val("output_multiplier", 1.0)), softcap=val("final_logit_softcapping"),
                   emb_scale=float(val("input_embedding_scale", 1.0)), dtype=dtype)


# ----------------------------------------------------------------------------- weights
def load_params(model_dir: str, dtype=np.float32):
    """incoai/GLM-5.3-Flash-DFlash2 (config.json + model.safetensors) -> (DCfg, params as NumPy arrays)."""
    from glm53 import checkpoint as C
    with open(os.path.join(model_dir, "config.json")) as f:
        cfg = DCfg.from_hf(json.load(f))
    r = C.RawShardReader(model_dir)

    def lin(name):                                            # torch Linear [out, in] -> [in, out]
        return np.ascontiguousarray(r.get(name + ".weight", dtype).T)

    def conv(pfx):
        return {"base": r.get(pfx + ".base_kernel", dtype), "proj": lin(pfx + ".kernel_projection")}
    layers = []
    for i in range(cfg.n_layers):
        a, m = f"layers.{i}.self_attn.", f"layers.{i}.mlp."
        layers.append({"ln1": r.get(f"layers.{i}.input_layernorm.weight", dtype),
                       "ln2": r.get(f"layers.{i}.post_attention_layernorm.weight", dtype),
                       "q": lin(a + "q_proj"), "k": lin(a + "k_proj"), "v": lin(a + "v_proj"), "o": lin(a + "o_proj"),
                       "q_norm": r.get(a + "q_norm.weight", dtype), "k_norm": r.get(a + "k_norm.weight", dtype),
                       "gate": lin(m + "gate_proj"), "up": lin(m + "up_proj"), "down": lin(m + "down_proj"),
                       "conv_a": conv(f"layers.{i}.attention_conv"), "conv_m": conv(f"layers.{i}.mlp_conv")})
    s = "candidate_selector."
    params = {"fc": lin("fc"), "hidden_norm": r.get("hidden_norm.weight", dtype), "norm": r.get("norm.weight", dtype),
              "layers": layers,
              "sel": {"pred": r.get(s + "predecessor_codebook", dtype), "succ": r.get(s + "successor_codebook", dtype),
                      "h": lin(s + "hidden_projection")}}
    return cfg, params


def param_specs(cfg: DCfg) -> dict:
    col, row = P(None, AXIS), P(AXIS, None)
    conv = {"base": R, "proj": col}
    layer = {"ln1": R, "ln2": R, "q": col, "k": col, "v": col, "o": row, "q_norm": R, "k_norm": R,
             "gate": col, "up": col, "down": row, "conv_a": conv, "conv_m": conv}
    return {"fc": col, "hidden_norm": R, "norm": R, "layers": [layer] * cfg.n_layers,
            "sel": {"pred": row, "succ": row, "h": R}}


RING_SPEC = P(None, None, None, AXIS)             # [L, B, W, KV, hd]: KV heads over the chips
RING_SPECS = {"k": RING_SPEC, "v": RING_SPEC, "p": R}


# ----------------------------------------------------------------------------- the model
def rope(x, pos, theta):
    """Rotate-half RoPE (Qwen3): x [B, T, h, hd], pos [B, T] absolute positions; angles in f32."""
    hd = x.shape[-1]
    inv = 1.0 / (theta ** (jnp.arange(0, hd, 2, dtype=jnp.float32) / hd))
    ang = pos.astype(jnp.float32)[..., None] * inv
    ang = jnp.concatenate([ang, ang], -1)[:, :, None, :]
    xf = x.astype(jnp.float32)
    rot = jnp.concatenate([-xf[..., hd // 2:], xf[..., :hd // 2]], -1)
    return (xf * jnp.cos(ang) + rot * jnp.sin(ang)).astype(x.dtype)


def dyn_conv(x, dyn, base, group):
    """Grouped dynamic causal conv over the block: out_t = sum_o (base[o] + dyn_t[o]) * x_{t-o}, x before the block's
    first position = 0. x [B, T, D], dyn [B, T, K, G] (one kernel per group of `group` channels), base [K, D]."""
    out = jnp.zeros_like(x)
    for o in range(base.shape[0]):
        xs = x if o == 0 else jnp.pad(x[:, :-o], ((0, 0), (o, 0), (0, 0)))
        out = out + base[o].astype(x.dtype) * xs + jnp.repeat(dyn[:, :, o], group, axis=-1) * xs
    return out


def conv_kernels(p, x, cfg: DCfg):
    """The sublayer's (normed) input [B, T, D] -> dynamic kernels [B, T, 2, K, G]: [:, :, 0] for the input conv
    (`prepare`), [:, :, 1] for the output conv (`finish`)."""
    d = cfg.gather(x @ p["proj"])
    return d.reshape(x.shape[:2] + (2, cfg.conv_k, cfg.hidden // cfg.conv_group))


def _rows(feats):
    """(B, N) of target features given as one array or as the per-layer list."""
    return (feats[0] if isinstance(feats, (list, tuple)) else feats).shape[:2]


def fuse_features(params, feats, cfg: DCfg):
    """Target features [B, N, nF*D] (or the list of nF [B, N, D] per target layer, as the engine captures them) ->
    the drafter's context hidden hidden_norm(fc(feats)) [B, N, D]."""
    if isinstance(feats, (list, tuple)):
        feats = jnp.concatenate(feats, -1)
    return M.rmsnorm(cfg.gather(feats.astype(cfg.dtype) @ params["fc"]), params["hidden_norm"], cfg.eps)


def context_kv(params, ctx, pos, cfg: DCfg):
    """ctx [B, N, D] (fuse_features) at positions pos [B, N] -> (k, v) [L, B, N, kv_local, hd] for every layer."""
    B, N, _ = ctx.shape
    ks, vs = [], []
    for p in params["layers"]:
        k = M.rmsnorm((ctx @ p["k"]).reshape(B, N, -1, cfg.hd), p["k_norm"], cfg.eps)
        ks.append(rope(k, pos, cfg.theta))
        vs.append((ctx @ p["v"]).reshape(B, N, -1, cfg.hd))
    return jnp.stack(ks), jnp.stack(vs)


def ring_write(ring, k, v, pos0, length):
    """Write k, v [L, B, N, kvl, hd] of positions pos0 + i (i < length; pos0, length [B]) into slots position % W.
    Rows more than W before the end are skipped (a later row owns their slot)."""
    W = ring["k"].shape[2]
    B, N = k.shape[1], k.shape[2]
    i = jnp.arange(N)[None, :]
    keep = (i < length[:, None]) & (i >= length[:, None] - W)
    slot = jnp.where(keep, jnp.mod(pos0[:, None] + i, W), W)        # W = out of range -> dropped
    b = jnp.arange(B)[:, None]
    return {"k": ring["k"].at[:, b, slot].set(k.astype(ring["k"].dtype), mode="drop"),
            "v": ring["v"].at[:, b, slot].set(v.astype(ring["v"].dtype), mode="drop"),
            "p": ring["p"].at[b, slot].set(pos0[:, None] + i + 1, mode="drop")}


def ingest(params, ring, feats, pos0, length, cfg: DCfg):
    """Context features of positions pos0 + i, i < length -> written into the ring (prefill pieces, accepted tokens)."""
    N = _rows(feats)[1]
    pos = pos0[:, None] + jnp.arange(N)[None, :]
    k, v = context_kv(params, fuse_features(params, feats, cfg), pos, cfg)
    return ring_write(ring, k, v, pos0, length)


def block_attention(p, x, ring_k, ring_v, ring_p, start, cfg: DCfg):
    """x [B, T, D] (the block after norm + conv) at positions start + t; ring_k/v [B, W, kvl, hd] this layer's context
    (positions < start), ring_p [B, W] their positions + 1 — or lists of B per-row arrays (batch 1 each: every stream
    its own ring; the projections stay batched). Bidirectional inside the block, window |q - k| < cfg.window to the
    context. -> [B, T, D]."""
    B, T, _ = x.shape
    hd = cfg.hd
    pos = start[:, None] + jnp.arange(T)[None, :]
    q = rope(M.rmsnorm((x @ p["q"]).reshape(B, T, -1, hd), p["q_norm"], cfg.eps), pos, cfg.theta)
    k = rope(M.rmsnorm((x @ p["k"]).reshape(B, T, -1, hd), p["k_norm"], cfg.eps), pos, cfg.theta)
    v = (x @ p["v"]).reshape(B, T, -1, hd)
    if isinstance(ring_k, (list, tuple)):
        o = jnp.concatenate([_block_attend(q[b:b + 1], k[b:b + 1], v[b:b + 1], ring_k[b], ring_v[b], ring_p[b],
                                           pos[b:b + 1], cfg) for b in range(B)], 0)
    else:
        o = _block_attend(q, k, v, ring_k, ring_v, ring_p, pos, cfg)
    return cfg.reduce(o @ p["o"])


def _block_attend(q, k, v, ring_k, ring_v, ring_p, pos, cfg: DCfg):
    """The block's queries against [the ring's context, the block] -> [B, T, H_local * hd] (before o)."""
    B, T = q.shape[:2]
    hd = cfg.hd
    kvl = k.shape[2]
    keys = jnp.concatenate([ring_k.astype(k.dtype), k], 1)
    vals = jnp.concatenate([ring_v.astype(v.dtype), v], 1)
    kpos = ring_p - 1                                                    # -1: an empty slot
    ctx_ok = (kpos[:, None, :] >= 0) & (pos[:, :, None] - kpos[:, None, :] < cfg.window)      # [B, T, W]
    mask = jnp.concatenate([ctx_ok, jnp.ones((B, T, T), bool)], -1)
    qg = q.reshape(B, T, kvl, -1, hd)                                    # q head g*rep + r reads kv head g
    s = jnp.einsum("btgrd,bsgd->bgrts", qg, keys, preferred_element_type=jnp.float32) * hd ** -0.5
    s = jnp.where(mask[:, None, None], s, -jnp.inf)
    a = jax.nn.softmax(s, -1).astype(vals.dtype)
    return jnp.einsum("bgrts,bsgd->btgrd", a, vals).reshape(B, T, -1)


def draft_hidden(params, emb, ring, start, cfg: DCfg):
    """emb [B, T, D] = embedding rows of [anchor, MASK...] (x emb_scale) at positions start + t -> final-norm hidden
    [B, T, D] (the reference's `DFlashDraftModel.forward` over the block). `ring`: one ring of batch B or a list of B
    per-row rings."""
    x = emb.astype(cfg.dtype)
    g = cfg.conv_group
    rows = isinstance(ring, (list, tuple))
    for li, p in enumerate(params["layers"]):
        ca, cm = p["conv_a"], p["conv_m"]
        h = M.rmsnorm(x, p["ln1"], cfg.eps)
        dk = conv_kernels(ca, h, cfg)
        if rows:
            rk, rv, rp = [r["k"][li] for r in ring], [r["v"][li] for r in ring], [r["p"] for r in ring]
        else:
            rk, rv, rp = ring["k"][li], ring["v"][li], ring["p"]
        a = block_attention(p, dyn_conv(h, dk[:, :, 0], ca["base"][0], g), rk, rv, rp, start, cfg)
        x = x + dyn_conv(a, dk[:, :, 1], ca["base"][1], g)
        h = M.rmsnorm(x, p["ln2"], cfg.eps)
        dk = conv_kernels(cm, h, cfg)
        h = dyn_conv(h, dk[:, :, 0], cm["base"][0], g)
        m = cfg.reduce((M.silu(h @ p["gate"]) * (h @ p["up"])) @ p["down"])
        x = x + dyn_conv(m, dk[:, :, 1], cm["base"][1], g)
    return M.rmsnorm(x, params["norm"], cfg.eps)


# ----------------------------------------------------------------------------- vocab-sharded pieces
def vocab_rows(table_local, ids, cfg: DCfg):
    """Rows `ids` (any shape) of a table whose rows are split over the chips in consecutive blocks (the target's
    embedding, the selector's codebooks); an int8 q8 table is dequantized row by row (Engine._embed for any table)."""
    q8 = Q8.is_q8(table_local)
    t = table_local["q"] if q8 else table_local
    rows = t.shape[0]
    local = ids if cfg.tp_axis is None else ids - lax.axis_index(cfg.tp_axis) * rows
    idx = jnp.clip(local, 0, rows - 1)
    e = jnp.take(t, idx, axis=0)
    if q8:
        e = e.astype(jnp.float32) * jnp.take(table_local["s"], idx, axis=0)
    if cfg.tp_axis is None:
        return e
    return lax.psum(jnp.where(((local >= 0) & (local < rows))[..., None], e, 0), cfg.tp_axis)


def logits_local(lm_head_local, h, cfg: DCfg):
    """This chip's vocab columns of the draft logits (the target's lm_head, bf16 or int8 q8), f32, with the
    drafter's output multiplier and soft cap."""
    from glm53.engine import dot_t                     # the lm_head is stored [V, D] (engine.lm_head_t)
    if Q8.is_q8(lm_head_local):
        z = dot_t(h, lm_head_local["q"].astype(h.dtype)) * lm_head_local["s"].reshape(-1)
    else:
        z = dot_t(h, lm_head_local)
    z = z * cfg.out_mult
    if cfg.softcap:
        z = jnp.tanh(z / cfg.softcap) * cfg.softcap
    return z


def topk_vocab(z_local, k, cfg: DCfg):
    """Exact top-k over vocab-sharded logits: (values, global ids) [..., k], descending (each chip's top-k,
    all-gathered, then the top-k of those)."""
    v, i = M.top_k(z_local, k)                     # (chunked: one XLA sort over the vocab shard cost ~0.17 ms)
    if cfg.tp_axis is None:
        return v, i
    i = i + lax.axis_index(cfg.tp_axis) * z_local.shape[-1]
    v, i = cfg.gather(v), cfg.gather(i)
    v, j = lax.top_k(v, k)
    return v, jnp.take_along_axis(i, j, -1)


def select(sel, hidden, z_local, anchor, cfg: DCfg, temperature=None, key=None):
    """DFlash 2 candidate selector over the draft positions. hidden [B, T1, D], z_local their logits (this chip's
    vocab columns), anchor [B]. score_t(a, b) = logit_t(b) + <P[a] * (h_t @ H), S[b]> for the top-k candidates b of
    position t, a = the token chosen at t-1 (the anchor at t = 0); greedy argmax, or (temperature given: a float or a
    traced scalar / [B]) a draw from softmax(score / temperature) with `key`, the argmax where temperature <= 0 (q is
    then one-hot). -> (path [B, T1], candidates [B, T1, k], q [B, T1, k] or None, scores [B, T1, k])."""
    unary, cand = topk_vocab(z_local, cfg.sel_topk, cfg)
    B, T1, k = cand.shape
    hs = jnp.dot(hidden, sel["h"], preferred_element_type=jnp.float32)
    ids = jnp.concatenate([anchor[:, None], cand.reshape(B, -1)], 1)
    pr = vocab_rows(sel["pred"], ids, cfg).astype(jnp.float32)          # P rows of the anchor and every candidate
    pa, pc = pr[:, 0], pr[:, 1:].reshape(B, T1, k, -1)
    sc = vocab_rows(sel["succ"], cand, cfg).astype(jnp.float32)          # [B, T1, k, r]
    path, qs, scores = [], [], []
    if temperature is not None:              # a host float or a traced scalar / [B] (rows at <= 0 pick the argmax)
        tt = jnp.reshape(jnp.asarray(temperature, jnp.float32), (-1, 1))
    for t in range(T1):
        s = unary[:, t] + jnp.einsum("br,bkr->bk", pa * hs[:, t], sc[:, t])
        if temperature is None:
            j = jnp.argmax(s, -1)
        else:
            key, sub = jax.random.split(key)
            z = s / jnp.where(tt > 0, tt, 1.0)
            g = jnp.argmax(s, -1)
            j = jnp.where(tt[:, 0] > 0, jax.random.categorical(sub, z, -1), g)
            qs.append(jnp.where(tt > 0, jax.nn.softmax(z, -1), jax.nn.one_hot(g, k, dtype=jnp.float32)))
        path.append(jnp.take_along_axis(cand[:, t], j[:, None], -1)[:, 0])
        pa = jnp.take_along_axis(pc[:, t], j[:, None, None], 1)[:, 0]
        scores.append(s)
    return jnp.stack(path, 1), cand, (jnp.stack(qs, 1) if qs else None), jnp.stack(scores, 1)


def draft(params, ring, feats, pos0, n_new, anchor, embed, lm_head, cfg: DCfg, T=None, temperature=None, key=None):
    """One draft: write the n_new new context positions' features (feats [B, N, nF*D] at pos0 + i, i < n_new) into
    the ring, then draft the block [anchor, MASK x (T-1)] at position start = pos0 + n_new. `ring` = one ring of batch
    B, or a list of B per-row rings (batch 1 each; returned as a list).
    -> (path [B, T-1], candidates, q, scores, ring, hidden [B, T, D])."""
    T = cfg.block if T is None else T
    if isinstance(ring, (list, tuple)):              # per-row rings: the context K/V batched, written row by row
        N = _rows(feats)[1]
        k, v = context_kv(params, fuse_features(params, feats, cfg), pos0[:, None] + jnp.arange(N)[None, :], cfg)
        ring = [ring_write(r, k[:, b:b + 1], v[:, b:b + 1], pos0[b:b + 1], n_new[b:b + 1]) for b, r in enumerate(ring)]
    else:
        ring = ingest(params, ring, feats, pos0, n_new, cfg)
    start = pos0 + n_new
    B = anchor.shape[0]
    ids = jnp.concatenate([anchor[:, None], jnp.full((B, T - 1), cfg.mask_id, anchor.dtype)], 1)
    emb = vocab_rows(embed, ids, cfg).astype(cfg.dtype) * cfg.emb_scale
    h = draft_hidden(params, emb, ring, start, cfg)
    z = logits_local(lm_head, h[:, 1:], cfg)
    path, cand, q, scores = select(params["sel"], h[:, 1:], z, anchor, cfg, temperature, key)
    return path, cand, q, scores, ring, h


# ----------------------------------------------------------------------------- on the chips
class Drafter:
    """The drafter's params on a mesh (axis engine.AXIS) and its jitted programs. `embed_spec` / `lm_head_spec` =
    the target's (engine `specs["embed"]`, `specs["lm_head"]`: vocab-sharded, possibly q8 dicts); the programs take
    the target's arrays as arguments (the lm_head [V, D], `engine.lm_head_t`). int8=True stores the big matrices (layers, fc) as q8 (glm53.quant8); the
    codebooks stay in their dtype. Ring buffers are donated to `ingest` / `draft` (never reuse a ring passed in).
    temp_scale: a sampled draft draws its path at temp_scale x the given temperature (q is that distribution)."""

    def __init__(self, cfg: DCfg, params, mesh, embed_spec=P(AXIS, None), lm_head_spec=P(AXIS, None), int8=False,
                 temp_scale=1.0):
        assert cfg.kv_heads % mesh.size == 0 and cfg.vocab % mesh.size == 0, (cfg.kv_heads, cfg.vocab, mesh.size)
        self.cfg, self.mesh = cfg, mesh
        self.lcfg = dataclasses.replace(cfg, tp_axis=AXIS)
        specs = param_specs(cfg)
        if int8:
            layers, lspecs = [], []
            for L, sp in zip(params["layers"], specs["layers"]):
                qp, qs = Q8.quantize_layer(L, sp, lambda w, ax: Q8.quantize_array_host(w, ax))
                layers.append(qp); lspecs.append(qs)
            params = {**params, "layers": layers, "fc": Q8.quantize_array_host(params["fc"], 1)}
            specs = {**specs, "layers": lspecs, "fc": {"q": specs["fc"], "s": specs["fc"]}}
        self.specs = specs
        self.params = jax.tree.map(lambda a, s: jax.device_put(a, NamedSharding(mesh, s)), params, specs,
                                   is_leaf=lambda x: isinstance(x, P))
        self.embed_spec, self.lm_head_spec = embed_spec, lm_head_spec
        self.temp_scale = float(temp_scale)      # the sampled path is drawn at temp_scale x the given temperature
        self._progs = {}

    def nbytes_per_chip(self):
        return sum(s.data.nbytes for a in jax.tree.leaves(self.params) for s in a.addressable_shards[:1])

    def alloc_ring(self, B):
        shape = (self.cfg.n_layers, B, self.cfg.window, self.cfg.kv_heads, self.cfg.hd)
        z = jax.jit(lambda: jnp.zeros(shape, self.cfg.dtype), out_shardings=NamedSharding(self.mesh, RING_SPEC))
        zp = jax.jit(lambda: jnp.zeros((B, self.cfg.window), jnp.int32), out_shardings=NamedSharding(self.mesh, R))
        return {"k": z(), "v": z(), "p": zp()}

    def _prog_ingest(self, B, N):
        key = ("ingest", B, N)
        if key not in self._progs:
            def prog(params, ring, feats, pos0, length):
                return ingest(Q8.dequant_tree(params, self.cfg.dtype), ring, feats, pos0, length, self.lcfg)
            rs = RING_SPECS
            sm = shard_map(prog, mesh=self.mesh, in_specs=(self.specs, rs, R, R, R), out_specs=rs, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(1,))
        return self._progs[key]

    def _prog_draft(self, B, N, T, sample, feat_index=None, rows=False):
        """rows=True: B per-row rings (a list, each batch 1, donated) instead of one ring of batch B."""
        key = ("draft", B, N, T, sample) + ((feat_index,) if feat_index is not None else ()) + (("rows",) if rows else ())
        if key not in self._progs:
            def prog(params, ring, feats, pos0, n_new, anchor, embed, lm_head, *samp):
                if feat_index is not None:      # stacked per layer group (the engine's verify): pick the target layers
                    feats = [f[j] for f, js in zip(feats, feat_index) for j in js]
                params = Q8.dequant_tree(params, self.cfg.dtype)
                temp, k = samp if sample else (None, None)
                if sample:                      # the engine passes the sampler's key, which its acceptance also splits
                    k = jax.random.fold_in(k, 0xDF)
                    temp = temp * self.temp_scale   # (a sharper q than the target's accepts more: v6e 2026-10-04)
                path, cand, q, scores, ring, _ = draft(params, ring, feats, pos0, n_new, anchor, embed, lm_head,
                                                       self.lcfg, T, temp, k)
                out = {"path": path, "cand": cand, "scores": scores, "rings" if rows else "ring": ring,
                       "seq": jnp.concatenate([anchor[:, None].astype(path.dtype), path], 1)}
                return {**out, "q": q} if sample else out
            rs = [RING_SPECS] * B if rows else RING_SPECS
            in_specs = (self.specs, rs, R, R, R, R, self.embed_spec, self.lm_head_spec) + ((R, R) if sample else ())
            out_specs = {"path": R, "cand": R, "scores": R, "rings" if rows else "ring": rs, "seq": R,
                         **({"q": R} if sample else {})}
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(1,))
        return self._progs[key]

    def ingest(self, ring, feats, pos0, length):
        """feats [B, N, nF*D] (or the per-layer list) of positions pos0 + i, i < length (pos0, length [B] int32)
        -> new ring."""
        B, N = _rows(feats)
        return self._prog_ingest(B, N)(self.params, ring, feats, jnp.asarray(pos0, jnp.int32),
                                       jnp.asarray(length, jnp.int32))

    def draft(self, ring, feats, pos0, n_new, anchor, embed, lm_head, T=None, temperature=None, key=None,
              feat_index=None):
        """-> dict: path [B, T-1] (the drafted tokens), seq [B, T] (anchor + path: the verify's input), cand / scores
        [B, T-1, k], ring (the new one), q [B, T-1, k] when a temperature is given. pos0, n_new, anchor [B] int32 (host
        values or device arrays). `feat_index`: feats is a list of stacked [n_g, B, N, D] arrays and feat_index[g] the
        target positions to take from each (the engine's per-group capture)."""
        B, N = _rows(feats) if feat_index is None else feats[0].shape[1:3]
        T = self.cfg.block if T is None else T
        args = (self.params, ring, feats, jnp.asarray(pos0, jnp.int32), jnp.asarray(n_new, jnp.int32),
                jnp.asarray(anchor, jnp.int32), embed, lm_head)
        if temperature is not None:              # (a device scalar, e.g. DeviceSampler.bind's, passes through)
            args += (temperature if isinstance(temperature, jax.Array) else jnp.float32(temperature), key)
        return self._prog_draft(B, N, T, temperature is not None, feat_index)(*args)

    def draft_rows(self, rings, feats, pos0, n_new, anchor, embed, lm_head, T=None, temperature=None, key=None,
                   feat_index=None):
        """`draft` for B streams that each keep their own ring (`rings`: a list of B rings of batch 1, donated; the
        engine's per-stream cache sets) -> the same dict with "rings" (the new list) instead of "ring"."""
        B = len(rings)
        N = (_rows(feats) if feat_index is None else feats[0].shape[1:3])[1]
        T = self.cfg.block if T is None else T
        args = (self.params, list(rings), feats, jnp.asarray(pos0, jnp.int32), jnp.asarray(n_new, jnp.int32),
                jnp.asarray(anchor, jnp.int32), embed, lm_head)
        if temperature is not None:
            args += (temperature if isinstance(temperature, jax.Array) else jnp.float32(temperature), key)
        return self._prog_draft(B, N, T, temperature is not None, feat_index, rows=True)(*args)
'''
FILES["glm53/engine.py"] = r'''"""Tensor-parallel (TP = #devices) engine for `glm53.model` with host-resident routed experts.

Sharding (mesh axis "x"):
  * attention heads (KDA 64, MLA 64) and MLP/expert intermediate columns are split across devices;
    every row-parallel matmul output is `psum`'d via `Cfg.tp_reduce`, so activations/streams stay replicated.
  * embed rows and lm_head rows (stored transposed, [V, D]: `lm_head_t`) are vocab-sharded.
  * routed experts are NOT on device: `HostExperts.fetch` gathers each device's slice of the selected experts on the
    host (NumPy) and ships it through `jax.pure_callback` inside the jitted `shard_map` program.

Everything runs on CPU with `XLA_FLAGS=--xla_force_host_platform_device_count=8` for tests.
"""
from __future__ import annotations

import dataclasses
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
from jax import lax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

from glm53 import model as M

try:
    from jax import shard_map
except ImportError:  # older jax
    from jax.experimental.shard_map import shard_map

AXIS = "x"
R = P()  # replicated


# ----------------------------------------------------------------------------- device-side sampling
SAMPLE_IMPL = "approx"          # "approx": chunked exact top-k candidates + bisection top-p (no full sorts); "sort": top_k + sort


def _candidates(z_local, temperature, top_p, n_cand, impl):
    """The candidate set of `device_sample`: (gidx [N, C] global ids, lp [N, C] f32 log-weights after temperature with
    -inf outside the top-p set, greedy [N] the global argmax, gi [N] its index in gidx, temperature [1 or N, 1])."""
    impl = SAMPLE_IMPL if impl is None else impl
    N, Vl = z_local.shape
    n = lax.axis_size(AXIS)
    kc = min(max(1, n_cand // n), Vl)
    z = z_local.astype(jnp.float32)
    if impl == "sort" or kc >= Vl:
        vals, idx = lax.top_k(z, kc)                                             # [N, kc] per chip
    else:
        # the exact chunked top-k (model.top_k): v6e 2026-10-05 (j151, [4, 19360] rows, k = 256) 74 us against 122 for
        # lax.approx_max_k (recall 0.99 + the argmax forced in) and 122 for the plain lax.top_k
        vals, idx = M.top_k(z, kc)
    gidx = idx.astype(jnp.int32) + lax.axis_index(AXIS).astype(jnp.int32) * Vl
    vals = lax.all_gather(vals, AXIS, axis=1, tiled=True)                        # [N, n*kc], replicated
    gidx = lax.all_gather(gidx, AXIS, axis=1, tiled=True)
    gi = jnp.argmax(vals, axis=1)
    greedy = jnp.take_along_axis(gidx, gi[:, None], axis=1)[:, 0]
    temperature = jnp.reshape(jnp.asarray(temperature, jnp.float32), (-1, 1))     # [] or [N] -> [1 or N, 1] per row
    top_p = jnp.reshape(jnp.asarray(top_p, jnp.float32), (-1, 1))
    t = jnp.where(temperature > 0, temperature, 1.0)
    lp = vals / t
    lp = lp - jnp.max(lp, axis=1, keepdims=True)
    p = jnp.exp(lp)
    p = p / jnp.sum(p, axis=1, keepdims=True)
    if impl == "sort":
        ps = -jnp.sort(-p, axis=1)                                               # descending
        cum = jnp.cumsum(ps, axis=1)
        n_keep = jnp.minimum(jnp.sum(cum <= top_p, axis=1) + 1, ps.shape[1])    # host rule: count(cum <= top_p) + 1
        thr = jnp.take_along_axis(ps, (n_keep - 1)[:, None], axis=1)
    else:                                                                        # largest t with mass(p >= t) >= top_p
        lo = jnp.zeros((N, 1), jnp.float32)
        hi = jnp.max(p, axis=1, keepdims=True)
        for _ in range(24):
            mid = 0.5 * (lo + hi)
            ok = jnp.sum(jnp.where(p >= mid, p, 0.0), axis=1, keepdims=True) >= top_p
            lo = jnp.where(ok, mid, lo)
            hi = jnp.where(ok, hi, mid)
        thr = lo
    return gidx, jnp.where(p >= thr, lp, -jnp.inf), greedy, gi, temperature


def device_sample(z_local, temperature, top_p, key, n_cand=2048, impl=None):
    """Inside a shard_map program: one token per row of the logits, whose vocab columns are sharded over AXIS
    (`z_local` [N, V/n], any float dtype). Candidates = the top n_cand/n logits of every chip, all-gathered (2048 on
    8 chips, the same truncation as the host sampler's partial sort); temperature and top-p in f32 over the
    candidates, then a categorical draw with `key` (identical on every chip -> the same token everywhere).
    temperature <= 0 -> the argmax (exact over the whole vocab). impl "approx" (default): `lax.approx_max_k`
    picks the candidates (recall 0.99; the exact per-chip argmax is forced into the set, so greedy stays exact and
    the mode is never missing) and the top-p set {p >= t} is found by bisection on t (24 steps) instead of a sort —
    XLA sorts are slow on the TPU (the "sort" variant cost ~2 ms per token, as much as the host sampler).
    Returns int32 [N] token ids."""
    gidx, lp, greedy, _, temperature = _candidates(z_local, temperature, top_p, n_cand, impl)
    draw = jax.random.categorical(key, lp, axis=1)
    sampled = jnp.take_along_axis(gidx, draw[:, None], axis=1)[:, 0]
    return jnp.where(temperature[:, 0] > 0, sampled, greedy).astype(jnp.int32)


def device_probs(z_local, temperature, top_p, n_cand=2048, impl=None):
    """`device_sample`'s distribution instead of a draw: (gidx [N, C] global token ids, p [N, C] f32 their
    probabilities — the truncated softmax a draw would use; one-hot at the argmax for temperature <= 0). Tokens
    outside gidx have probability 0. For speculative acceptance (`spec_accept`)."""
    gidx, lp, _, gi, temperature = _candidates(z_local, temperature, top_p, n_cand, impl)
    p = jax.nn.softmax(lp, axis=1)
    onehot = (jnp.arange(gidx.shape[1])[None, :] == gi[:, None]).astype(jnp.float32)
    return gidx, jnp.where(temperature > 0, p, onehot)


def spec_accept(gidx, p, drafts, cand, q, key, stop_ids=()):
    """Speculative acceptance by rejection sampling (Leviathan et al. 2023), replicated arrays (no collectives).
    Row b verified the block [anchor, drafts[b]]: position t's target distribution (gidx, p: [B, T, C], `device_probs`)
    predicts drafts[b, t] (t < T-1) and the drafter drew drafts[b, t] from q[b, t] over its candidates cand[b, t]
    ([B, T-1, k]). Draft t is accepted with probability min(1, p(d)/q(d)); at the first rejection the emitted token
    is drawn from norm(max(0, p - q)), after T-1 acceptances from p of the last position — so every emitted token
    is distributed exactly as plain sampling from the target. A draft in `stop_ids` ends the run: if accepted it is
    the emitted (unfed) token. With one-hot p and q (temperature 0) this is "accepted iff equal to the argmax, then
    the argmax". -> (a [B] accepted drafts, bonus [B] the token emitted after them)."""
    B, T, C = p.shape
    d = drafts.astype(jnp.int32)
    k_u, k_r = jax.random.split(key)
    p_d = jnp.sum(jnp.where(gidx[:, :-1] == d[..., None], p[:, :-1], 0.0), -1)            # [B, T-1]
    q_d = jnp.sum(jnp.where(cand == d[..., None], q, 0.0), -1)
    u = jax.random.uniform(k_u, (B, T - 1), jnp.float32)
    acc = u * q_d < p_d
    ok = acc
    for sid in stop_ids:
        ok &= d != sid
    a = jnp.cumprod(ok.astype(jnp.int32), 1).sum(1)
    q_on = jnp.sum(jnp.where(gidx[:, :-1, :, None] == cand[:, :, None, :], q[:, :, None, :], 0.0), -1)   # [B, T-1, C]
    q_on = jnp.concatenate([q_on, jnp.zeros((B, 1, C), q_on.dtype)], 1)
    r = jnp.maximum(p - q_on, 0.0)
    r = jnp.where(jnp.sum(r, -1, keepdims=True) > 1e-12, r, p)                  # (no mass left: only on rounding)
    x = jnp.take_along_axis(gidx, jax.random.categorical(k_r, jnp.log(r), axis=-1)[..., None], -1)[..., 0]   # [B, T]
    x_a = jnp.take_along_axis(x, a[:, None], 1)[:, 0]
    dpad = jnp.concatenate([d, jnp.zeros((B, 1), jnp.int32)], 1)
    accpad = jnp.concatenate([acc, jnp.zeros((B, 1), bool)], 1)
    stopped = jnp.take_along_axis(accpad, a[:, None], 1)[:, 0]                   # an accepted stop draft ended the run
    return a, jnp.where(stopped, jnp.take_along_axis(dpad, a[:, None], 1)[:, 0], x_a).astype(jnp.int32)


class DeviceSampler:
    """Host-side handle of the device sampler: temperature, top_p and the PRNG key live on the chips as REPLICATED
    arrays (placed once per mesh by `bind`); the sampling programs return the next key, so a decode step moves no
    sampling state between host and device (per-step scalar transfers cost ~2 ms on the Kaggle TPU VM)."""

    def __init__(self, temperature=1.0, top_p=1.0, seed=None):
        self.seed = int(np.random.SeedSequence(seed).generate_state(1)[0]) if seed is None else int(seed)
        self._mesh = None
        self.set(temperature, top_p)

    def set(self, temperature, top_p):
        """Scalars (every row) or one value per row of the batch ([B] sequences, `decode_rows`); re-placed on the
        chips at the next `bind` only when they changed (a batch membership change, not every step)."""
        self.temperature = np.asarray(temperature, np.float32)
        self.top_p = np.asarray(top_p, np.float32)
        self._placed = None

    def bind(self, mesh):
        """-> (temperature, top_p, key) replicated on `mesh` (placed once per mesh / per `set`)."""
        sh = NamedSharding(mesh, P())
        if self._mesh is not mesh:
            self.key = jax.device_put(jax.random.PRNGKey(self.seed), sh)
            self._mesh, self._placed = mesh, None
        if self._placed is None:
            self.t = jax.device_put(self.temperature, sh)
            self.p = jax.device_put(self.top_p, sh)
            self._placed = True
        return self.t, self.p, self.key

    def advance(self, key):
        self.key = key


# ----------------------------------------------------------------------------- specs
def layer_specs(cfg: M.Cfg, i: int, p: dict) -> dict:
    col, row = P(None, AXIS), P(AXIS, None)
    s = {"ln1": R, "ln2": R,
         "hc_attn": {k: R for k in p["hc_attn"]}, "hc_ffn": {k: R for k in p["hc_ffn"]}}   # fn / fnT, base, scale
    if cfg.layer_types[i] == "linear_attention":
        s["attn"] = {"q": col, "k": col, "v": col, "conv_q": row, "conv_k": row, "conv_v": row,
                     "f_a": R, "f_b": col, "dt_bias": P(AXIS), "A_log": P(AXIS), "b": col,
                     "g_a": R, "g_b": col, "o_norm": R, "o": row}
    else:
        s["attn"] = {"q_a": R, "q_a_norm": R, "q_b": col, "kv_a": R, "kv_a_norm": R, "kv_b": col, "o": row}
        if "indexer" in p["attn"]:
            s["attn"]["indexer"] = {k: R for k in p["attn"]["indexer"]}
    if cfg.mlp_types[i] == "sparse":
        s["mlp"] = {"router_w": R, "router_bias": R, "shared": {"gate": col, "up": col, "down": row}}
        if "gate_up" in p["mlp"]:            # on-device experts (tests only): [E, D, 2, mi] / [E, mi, D]
            s["mlp"]["gate_up"] = P(None, None, None, AXIS)
            s["mlp"]["down"] = P(None, AXIS, None)
        for k in ("gate_q", "up_q", "down_q"):   # resident codebook-quantized experts, chip-major (glm53.resident)
            if k in p["mlp"]:                    # dict of planar arrays [n_dev, E*rows_p, R] (or one array, legacy)
                s["mlp"][k] = {pk: P(AXIS) for pk in p["mlp"][k]} if isinstance(p["mlp"][k], dict) else P(AXIS)
    else:
        s["mlp"] = {"gate": col, "up": col, "down": row}
    return s


def lm_head_t(w):
    """The lm_head as the engine stores it: [V, D] (vocab rows) instead of the checkpoint's [D, V]. The TPU dot wants
    the contracted dim minor in its weight, so a [D, V] weight was copied into that layout on EVERY call (the head
    and the draft programs: 50 us each per step for the 79 MB int8 shard, v6e 2026-10-05 j152); stored transposed
    and read with `dot_t` nothing is copied. Accepts an array or a q8 dict {q, s [V]} (left alone when q already has
    V rows)."""
    from glm53 import quant8 as Q8
    if Q8.is_q8(w):                                       # the base store's {q [D, V] int8, s [1, V]} (per-column scales)
        q, sc = np.asarray(w["q"]), np.asarray(w["s"])
        if q.shape[0] == sc.size and q.shape[0] != q.shape[1]:   # already [V, D] with s [V, 1]
            return {"q": q, "s": sc.reshape(-1, 1)}
        return {"q": np.ascontiguousarray(q.T), "s": np.ascontiguousarray(sc.reshape(-1, 1))}   # -> q [V, D], s [V, 1]
    return np.ascontiguousarray(np.asarray(w).T)


def dot_t(h, w_t, preferred_element_type=jnp.float32):
    """h [..., D] x w_t [N, D] -> [..., N] f32: the dot with its weight given transposed (the MXU's natural weight
    layout: no relayout copy of the weight)."""
    return lax.dot_general(h, w_t, (((h.ndim - 1,), (1,)), ((), ())), preferred_element_type=preferred_element_type)


def param_specs(cfg: M.Cfg, params: dict) -> dict:
    return {"embed": P(AXIS, None), "norm": R, "lm_head": P(AXIS, None),      # lm_head stored [V, D] (`lm_head_t`)
            "layers": [layer_specs(cfg, i, params["layers"][i]) for i in range(cfg.n_layers)]}


def cache_specs(cfg: M.Cfg, caches: list, seq_shard: int = 1) -> list:
    """KDA caches are head-sharded; MLA/indexer caches are replicated, or sequence-sharded (axis 1) when the
    engine was built with seq_shard (glm53.model cache layout notes)."""
    out = []
    mla = P(None, AXIS) if seq_shard > 1 else R
    for i, c in enumerate(caches):
        if cfg.layer_types[i] == "linear_attention":
            out.append({"conv": P(None, None, AXIS), "state": P(None, AXIS)})
        else:
            out.append({k: (mla if k in M.ROW_KEYS else R) for k in c})       # tail buffers are replicated
    return out


def local_cfg(cfg: M.Cfg, n: int, seq_shard: bool = False, q_block: int = 128, cache_q8: bool = False) -> M.Cfg:
    assert cfg.kda_heads % n == 0 and cfg.n_heads % n == 0 and cfg.inter % n == 0 and cfg.moe_inter % n == 0
    return dataclasses.replace(cfg, kda_heads=cfg.kda_heads // n, n_heads=cfg.n_heads // n, inter=cfg.inter // n,
                               moe_inter=cfg.moe_inter // n, tp_reduce=lambda y: lax.psum(y, AXIS), tp_axis=AXIS,
                               seq_shard=n if seq_shard else 1, q_block=q_block, cache_q8=cache_q8 or cfg.cache_q8)


# ----------------------------------------------------------------------------- host experts
class HostExperts:
    """Routed expert tables kept in host RAM, pre-sliced per device.

    Layout is CHIP-MAJOR: every table is [n_dev, E, ...] so a device's expert slice is one contiguous block
    (measured on Kaggle: [E, chip] layout + strided take = 912 ms/layer; chip-major + threaded memcpy = 7 ms).
    mode "float": tables[layer] = (gate_up [n_dev, E, D, 2*ml], down [n_dev, E, ml, D]) float arrays.
    mode "fp8":   tables[layer] = (gu_bytes [n_dev, E, D, 2*ml] uint8, gu_scale [n_dev, E, D/128, 2*ml/128] f32,
                                   dn_bytes [n_dev, E, ml, D] uint8,   dn_scale [n_dev, E, ml/128, D/128] f32)
                  — raw e4m3 bytes streamed to the device and dequantized there (`checkpoint.dequant_fp8_device`).
    """

    def __init__(self, tables: dict, mode="float", out_dtype=jnp.float32):
        """mode: "float" | "fp8" | "int4", or a dict {layer: mode} for mixed tables (e.g. fp8 until host RAM ran
        out, int4 after). int4 tables: (gu_packed [n,E,D,ml] u8, gu_scale, dn_packed [n,E,ml,D/2] u8, dn_scale)."""
        self.tables = tables
        self.modes = dict(mode) if isinstance(mode, dict) else {L: mode for L in tables}
        assert all(m in ("float", "fp8", "int4") for m in self.modes.values())
        self.out_dtype = out_dtype

    @property
    def mode(self):
        ms = set(self.modes.values())
        return ms.pop() if len(ms) == 1 else "mixed"

    def mode_for(self, layer):
        return self.modes[layer]

    @classmethod
    def from_dense(cls, params: dict, cfg: M.Cfg, n_dev: int, out_dtype=jnp.float32) -> "HostExperts":
        """Split on-device style tables gate_up [E,D,2mi], down [E,mi,D] into per-device slices (float mode)."""
        tables = {}
        ml = cfg.moe_inter // n_dev
        for i in range(cfg.n_layers):
            if cfg.mlp_types[i] != "sparse":
                continue
            gu = np.asarray(params["layers"][i]["mlp"]["gate_up"])           # [E, D, 2mi]
            dn = np.asarray(params["layers"][i]["mlp"]["down"])              # [E, mi, D]
            E, D, _ = gu.shape
            tables[i] = (chip_major(split_gate_up(gu, n_dev)), chip_major(dn.reshape(E, n_dev, ml, D)))
        return cls(tables, "float", out_dtype)

    def gather(self, layer: int, idx: np.ndarray, chip: int):
        return tuple(t[chip][idx] for t in self.tables[layer])

    def make_fetch(self, cfg_local: M.Cfg, n_tokens: int):
        """Returns fetch(layer, idx) usable inside shard_map (per-device callback)."""
        from glm53.checkpoint import BLOCK, dequant_fp8_device, dequant_int4_device
        k, D, ml = cfg_local.topk, cfg_local.hidden, cfg_local.moe_inter

        def shapes_for(mode):
            if mode == "float":
                return (jax.ShapeDtypeStruct((n_tokens, k, D, 2 * ml), self.out_dtype),
                        jax.ShapeDtypeStruct((n_tokens, k, ml, D), self.out_dtype))
            if mode == "int4":
                return (jax.ShapeDtypeStruct((n_tokens, k, D, ml), jnp.uint8),
                        jax.ShapeDtypeStruct((n_tokens, k, D // BLOCK, 2 * ml // BLOCK), jnp.float32),
                        jax.ShapeDtypeStruct((n_tokens, k, ml, D // 2), jnp.uint8),
                        jax.ShapeDtypeStruct((n_tokens, k, ml // BLOCK, D // BLOCK), jnp.float32))
            return (jax.ShapeDtypeStruct((n_tokens, k, D, 2 * ml), jnp.uint8),
                    jax.ShapeDtypeStruct((n_tokens, k, D // BLOCK, 2 * ml // BLOCK), jnp.float32),
                    jax.ShapeDtypeStruct((n_tokens, k, ml, D), jnp.uint8),
                    jax.ShapeDtypeStruct((n_tokens, k, ml // BLOCK, D // BLOCK), jnp.float32))

        def fetch(layer, idx):
            chip = lax.axis_index(AXIS)
            mode = self.mode_for(layer)
            out_shapes = shapes_for(mode)

            def host_fn(idx_np, chip_np):
                outs = self.gather(layer, np.asarray(idx_np), int(chip_np))
                return tuple(np.ascontiguousarray(o, dtype=np.dtype(sh.dtype)) for o, sh in zip(outs, out_shapes))
            outs = jax.pure_callback(host_fn, out_shapes, idx, chip)
            if mode == "float":
                return outs
            gu_b, gu_s, dn_b, dn_s = outs
            if mode == "int4":
                return (dequant_int4_device(gu_b, gu_s, self.out_dtype), dequant_int4_device(dn_b, dn_s, self.out_dtype))
            return (dequant_fp8_device(gu_b, gu_s, self.out_dtype), dequant_fp8_device(dn_b, dn_s, self.out_dtype))
        return fetch


def split_gate_up(gu: np.ndarray, n_dev: int) -> np.ndarray:
    """[E, D, 2mi] (gate|up) -> [E, n_dev, D, 2*ml] with each device holding (gate_slice|up_slice)."""
    E, D, two_mi = gu.shape
    ml = two_mi // 2 // n_dev
    return np.ascontiguousarray(gu.reshape(E, D, 2, n_dev, ml).transpose(0, 3, 1, 2, 4).reshape(E, n_dev, D, 2 * ml))


def split_gate_up_scales(gs: np.ndarray, n_dev: int) -> np.ndarray:
    """Same split for blockwise scales [E, D/128, 2mi/128] -> [E, n_dev, D/128, 2*ml/128] (ml multiple of 128)."""
    return split_gate_up(gs, n_dev)


def chip_major(t: np.ndarray) -> np.ndarray:
    """[E, n_dev, ...] -> contiguous [n_dev, E, ...]."""
    return np.ascontiguousarray(np.moveaxis(t, 1, 0))


# ----------------------------------------------------------------------------- engine
class Engine:
    def __init__(self, cfg: M.Cfg, params: dict, host_experts: HostExperts | None = None, devices=None,
                 max_len: int = 4096, expert_fetch=None, int8_nonexpert: bool = False, seq_shard: bool = False,
                 q_block: int = 128, cache_q8: bool = False):
        devices = devices if devices is not None else jax.devices()
        cfg = dataclasses.replace(cfg, cache_q8=cache_q8 or cfg.cache_q8)     # int8 latent cache (half the HBM per set)
        self.max_len = max_len
        self.expert_fetch = expert_fetch      # e.g. glm53.resident.ResidentFetch (tables live inside params)
        self.n = len(devices)
        self.mesh = Mesh(np.array(devices), (AXIS,))
        self.cfg = cfg
        # seq_shard: MLA latent + indexer caches sharded over the chips (16.9 KB/token/chip replicated -> 2.1 KB;
        # needed beyond ~32k tokens); the capacity must be a multiple of kpool * n_dev
        assert not seq_shard or max_len % (cfg.idx_kpool * self.n) == 0, (max_len, cfg.idx_kpool, self.n)
        self.lcfg = local_cfg(cfg, self.n, seq_shard, q_block, cfg.cache_q8)
        self.host_experts = host_experts
        params = dict(params)
        params["lm_head"] = lm_head_t(params["lm_head"])
        if host_experts is not None:   # drop on-device expert tables
            params["layers"] = [{**L, "mlp": {k: v for k, v in L["mlp"].items() if k not in ("gate_up", "down")}}
                                if cfg.mlp_types[i] == "sparse" else L for i, L in enumerate(params["layers"])]
        else:
            params["layers"] = [{**L, "mlp": {**L["mlp"], "gate_up": np.asarray(L["mlp"]["gate_up"]).reshape(
                cfg.n_exp, cfg.hidden, 2, cfg.moe_inter)}} if cfg.mlp_types[i] == "sparse" and "gate_up" in L["mlp"]
                else L for i, L in enumerate(params["layers"])]
        if M.mhc_active():                    # the mHC kernels read the site projections transposed (same bytes)
            params["layers"] = [{**L, "hc_attn": M.mhc_params(L["hc_attn"]), "hc_ffn": M.mhc_params(L["hc_ffn"])}
                                for L in params["layers"]]
        self.specs = param_specs(cfg, params)
        from glm53 import quant8 as Q8
        if int8_nonexpert:                    # quantize the big non-expert matrices on the host before device_put
            layers, lspecs = [], []
            for L, sp in zip(params["layers"], self.specs["layers"]):
                qp, qs = Q8.quantize_layer(L, sp, lambda w, ax: jax.tree.map(np.asarray, Q8.quantize_array(w, ax)))
                layers.append(qp); lspecs.append(qs)
            params = {**params, "layers": layers}
            self.specs = {**self.specs, "layers": lspecs}
            if int8_nonexpert == "all":           # + the vocab-sharded embedding (per-row scales) and lm_head (per-column),
                q = Q8.quantize_array_host             # quantized on the host (a device pass OOMed at 262k)
                params = {**params, "embed": q(params["embed"], 0),
                          "lm_head": params["lm_head"] if Q8.is_q8(params["lm_head"]) else q(params["lm_head"], 0)}
        self.specs = Q8.expand_specs(params, self.specs)   # params already holding q8 nodes (hot-swapped on device)
        self.params = jax.tree.map(lambda a, s: jax.device_put(a, NamedSharding(self.mesh, s)), params, self.specs,
                                   is_leaf=lambda x: isinstance(x, P))
        self._prefill = {}
        self._decode = {}

    def _cache_spec(self, i):
        if self.cfg.layer_types[i] == "linear_attention":
            return {"conv": P(None, None, AXIS), "state": P(None, AXIS)}
        mla = P(None, AXIS) if self.lcfg.seq_shard > 1 else R
        spec = {"c": mla, "pk": mla, "tk": R, "tg": R} if "indexer" in self.params["layers"][i]["attn"] else {"c": mla}
        if self.cfg.cache_q8:
            spec["cs"] = mla
        return spec

    # ---- per-device program pieces
    def _embed(self, embed_local, tokens):
        from glm53 import quant8 as Q8
        chip = lax.axis_index(AXIS)
        q8 = Q8.is_q8(embed_local)
        table = embed_local["q"] if q8 else embed_local
        rows = table.shape[0]
        local = tokens - chip * rows
        ok = (local >= 0) & (local < rows)
        idx = jnp.clip(local, 0, rows - 1)
        e = jnp.take(table, idx, axis=0)
        if q8:                                              # dequantize only the gathered rows
            e = e.astype(jnp.float32) * jnp.take(embed_local["s"], idx, axis=0)
        return lax.psum(jnp.where(ok[..., None], e, 0), AXIS)

    def _logits_local(self, lm_head_local, h):
        """This chip's vocab columns of the logits [..., V/n] (the shard `_logits` all-gathers). The weight is stored
        [V/n, D] (`lm_head_t`) and read as the dot's transposed operand."""
        from glm53 import quant8 as Q8
        if Q8.is_q8(lm_head_local):                         # int8 rows: the convert fuses into the dot, scales after
            z = dot_t(h, lm_head_local["q"].astype(h.dtype)) * lm_head_local["s"].reshape(-1)    # s [V, 1] -> [V]
            return z.astype(h.dtype)
        return dot_t(h, lm_head_local).astype(h.dtype)

    def _logits(self, lm_head_local, h):
        return lax.all_gather(self._logits_local(lm_head_local, h), AXIS, axis=-1, tiled=True)

    PREFILL_BUCKETS = (32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)

    def _bucket(self, T):
        for b in self.PREFILL_BUCKETS:
            if b >= T and b <= self.max_len:
                return b
        return T

    def _program(self, params, tokens, caches, pos0, use_recurrent, n_tokens, length=None):
        # pos0 / length are traced int32 scalars; caches have fixed capacity self.max_len
        fetch = None if self.host_experts is None else self.host_experts.make_fetch(self.lcfg, n_tokens)
        if self.expert_fetch is not None:
            fetch = self.expert_fetch
        if self.host_experts is None:   # local on-device tables arrive as [E, D, 2, ml] -> [E, D, 2*ml]
            params = {**params, "layers": [
                {**L, "mlp": {**L["mlp"], "gate_up": L["mlp"]["gate_up"].reshape(L["mlp"]["gate_up"].shape[0],
                                                                                L["mlp"]["gate_up"].shape[1], -1)}}
                if "gate_up" in L["mlp"] else L for L in params["layers"]]}
        from glm53 import quant8 as Q8
        params = {**params, "layers": Q8.dequant_tree(params["layers"], self.lcfg.dtype)}
        emb = self._embed(params["embed"], tokens)
        h, new_caches = M.forward(params, tokens, self.lcfg, caches=caches, pos0=pos0, use_recurrent=use_recurrent,
                                  fetch=fetch, sparse="auto", embeds=emb, cap=self.max_len, length=length)
        last = h[:, -1] if length is None else lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
        return self._logits(params["lm_head"], last), new_caches

    def _build(self, B, T, caches, use_recurrent):
        # caches may be None (prefill) — we need their specs for in/out
        def prog(params, tokens, pos0, length, caches_in):
            return self._program(params, tokens, caches_in, pos0, use_recurrent, B * T, None if use_recurrent else length)

        cs = None if caches is None else cache_specs(self.cfg, caches, self.lcfg.seq_shard)
        # discover output cache structure by abstract evaluation is awkward under shard_map; build specs from a
        # dry structure: same layout as inputs, or generated for prefill
        cs_out = [self._cache_spec(i) for i in range(self.cfg.n_layers)] if cs is None else cs
        sm = shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R, R, R, cs), out_specs=(R, cs_out), check_vma=False)
        return jax.jit(sm)

    def prefill(self, tokens: np.ndarray):
        B, T = tokens.shape
        assert T <= self.max_len
        Tp = self._bucket(T)
        padded = np.zeros((B, Tp), np.int32)
        padded[:, :T] = tokens
        key = (B, Tp)
        if key not in self._prefill:
            self._prefill[key] = self._build(B, Tp, None, False)
        logits, caches = self._prefill[key](self.params, jnp.asarray(padded), jnp.int32(0), jnp.int32(T), None)
        return logits, caches, T

    def decode(self, token: np.ndarray, caches, pos: int):
        B = token.shape[0]
        key = B
        if key not in self._decode:
            self._decode[key] = self._build(B, 1, caches, True)
        logits, caches = self._decode[key](self.params, jnp.asarray(token).reshape(B, 1), jnp.int32(pos), jnp.int32(1),
                                           caches)
        return logits, caches, pos + 1

    def generate(self, tokens: np.ndarray, max_new: int, eos=(), greedy=True):
        logits, caches, pos = self.prefill(tokens)
        out = []
        for _ in range(max_new):
            nxt = int(jnp.argmax(logits[0]))
            out.append(nxt)
            if nxt in eos:
                break
            logits, caches, pos = self.decode(np.array([nxt]), caches, pos)
        return out


# ----------------------------------------------------------------------------- segmented engine (no host callbacks)
class SegmentedEngine(Engine):
    """Same sharding as `Engine`, but each layer is its own jitted shard_map program and routed-expert weights are
    gathered on the host *between* programs and shipped with `device_put` (works on TPU runtimes where host
    callbacks inside jit are unavailable). Prefill and decode use the same per-layer programs.

    `transport(arrays: tuple[np.ndarray])` -> tuple of device arrays sharded P('x') on axis 0 (per-chip slices);
    default = numpy device_put. Override for pinned-host tables."""

    def __init__(self, cfg, params, host_experts: HostExperts, devices=None, transport=None, max_len: int = 4096):
        assert host_experts is not None
        super().__init__(cfg, params, host_experts, devices, max_len)
        self.transport = transport or self._default_transport
        self._progs = {}
        from concurrent.futures import ThreadPoolExecutor
        self._pool = ThreadPoolExecutor(max_workers=32)
        from glm53.checkpoint import BLOCK
        self.BLOCK = BLOCK

    CHUNK_BYTES = 320 * 2 ** 20   # GB-sized puts run at ~5 GB/s, ~300 MB strided chunks at ~22 GB/s (measured)

    def _default_transport(self, arrays):
        """arrays: tuple of host [n_dev, Eh, ...] -> tuple of LISTS of device arrays (chunks along axis 1),
        each sharded P('x') on axis 0. Chunks are re-joined on device inside the post program."""
        sh = NamedSharding(self.mesh, P(AXIS))
        out = []
        for a in arrays:
            per_e = a[:, :1].nbytes
            step = max(1, self.CHUNK_BYTES // max(per_e, 1))
            out.append([jax.device_put(a[:, i:i + step], sh) for i in range(0, a.shape[1], step)])
        return tuple(out)

    def _dequant(self, outs, layer):
        from glm53.checkpoint import dequant_fp8_device, dequant_int4_device
        he = self.host_experts
        mode = he.mode_for(layer)
        if mode == "float":
            return outs
        gu_b, gu_s, dn_b, dn_s = outs
        f = dequant_int4_device if mode == "int4" else dequant_fp8_device
        return f(gu_b, gu_s, he.out_dtype), f(dn_b, dn_s, he.out_dtype)

    # ---- per-layer programs (built lazily per (layer, B, T, has_cache))
    def _prog_embed(self, B, T):
        key = ("embed", B, T)
        if key not in self._progs:
            def prog(params, tokens):
                x = self._embed(params["embed"], tokens).astype(self.lcfg.dtype)
                return jnp.broadcast_to(x[:, :, None, :], (B, T, self.cfg.hc, x.shape[-1]))
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R), out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_pre(self, i, B, T, has_cache, use_recurrent):
        key = ("pre", i, B, T, has_cache, use_recurrent)
        if key not in self._progs:
            lt, mt = self.cfg.layer_types[i], self.cfg.mlp_types[i]
            cs = self._cache_spec(i)
            def prog(params, streams, pos0, length, cache):
                p = params["layers"][i]
                streams, h, post, comb, new_cache, idx, w = M.layer_pre(
                    p, streams, self.lcfg, lt, mt, cache, pos0, use_recurrent, "auto", self.max_len,
                    None if use_recurrent else length)
                if idx is None:
                    idx = jnp.zeros((B * T, 1), jnp.int32); w = jnp.zeros((B * T, 1), jnp.float32)
                return streams, h, post, comb, new_cache, idx, w
            in_specs = (self.specs, R, R, R, cs if has_cache else None)
            out_specs = (R, R, R, R, cs, R, R)
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_post(self, i, B, T, Eh=None):
        """Eh: None (dense layer) or (Ep, (n_chunks per table...)) — both fix the program's input structure."""
        key = ("post", i, B, T, Eh)
        if key not in self._progs:
            mt = self.cfg.mlp_types[i]
            n_in = (2 if self.host_experts.mode_for(i) == "float" else 4) if mt == "sparse" else 0
            def prog(params, streams, h, post, comb, w, idx_local, *outs):
                p = params["layers"][i]
                if mt == "sparse":
                    # each table arrives as a list of chunks [1, e_i, ...]; join along the expert axis
                    outs = tuple(jnp.concatenate([c[0] for c in chunks], axis=0) for chunks in outs)
                    gu_w, dn_w = self._dequant(outs, i)
                    return M.layer_post(p, streams, h, post, comb, self.lcfg, mt, gu_w, dn_w, w, idx_local)
                return M.layer_post(p, streams, h, post, comb, self.lcfg, mt)
            in_specs = (self.specs, R, R, R, R, R, R) + (tuple([P(AXIS)] * nc for nc in Eh[1]) if mt == "sparse" else ())
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_head(self, B, T):
        key = ("head", B, T)
        if key not in self._progs:
            def prog(params, streams, length):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), params["norm"], self.cfg.eps)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                return self._logits(params["lm_head"], last)
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R, R), out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    # ---- host side
    BUCKETS = (8, 16, 32, 64, 128, 192, 288)

    def _gather_unique(self, layer, idx_np):
        """Unique experts hit by idx [N,k] -> (host arrays [n_dev, Eh_pad, ...], idx_local [N,k] int32).
        Eh is padded to a bucket so the post program compiles once per bucket; padding experts are unused."""
        tables = self.host_experts.tables[layer]          # chip-major [n_dev, E, ...]
        E = tables[0].shape[1]
        uniq, inv = np.unique(idx_np, return_inverse=True)
        Eh = len(uniq)
        if Eh >= 0.75 * E:                                 # prefill: ship the whole table, no gather at all
            return tuple(tables), idx_np.astype(np.int32), E
        Ep = next(b for b in self.BUCKETS if b >= Eh) if Eh <= self.BUCKETS[-1] else Eh
        outs = [np.empty((self.n, Ep) + t.shape[2:], t.dtype) for t in tables]
        for o in outs:
            o[:, Eh:] = 0                                  # padding experts must not hold NaN-decodable garbage

        def task(j, c, i):                                 # one contiguous memcpy per (table, chip, expert)
            outs[j][c, i] = tables[j][c, uniq[i]]

        list(self._pool.map(lambda a: task(*a), [(j, c, i) for j in range(len(tables)) for c in range(self.n)
                                                  for i in range(Eh)]))
        return tuple(outs), inv.reshape(idx_np.shape).astype(np.int32), Ep

    def _gather_and_put(self, layer, idx_np):
        """Per-chip pipeline: each chip's slices are memcpy'd then immediately device_put (8 chips in parallel),
        so transfer of chip c overlaps the gather of the others. Returns device arrays sharded P('x')."""
        tables = self.host_experts.tables[layer]          # chip-major [n_dev, E, ...]
        N, k = idx_np.shape
        flat = idx_np.reshape(-1)
        devs = list(self.mesh.devices.flat)
        sh = NamedSharding(self.mesh, P(AXIS))

        def chip_task(c):
            pieces = []
            for t in tables:
                buf = np.empty((1, N, k) + t.shape[2:], t.dtype)
                dst = buf.reshape((N * k,) + t.shape[2:])
                src = t[c]
                for i in range(N * k):
                    dst[i] = src[flat[i]]
                pieces.append(jax.device_put(buf, devs[c]))
            return pieces

        per_chip = list(self._pool.map(chip_task, range(self.n)))
        out = []
        for j, t in enumerate(tables):
            shape = (self.n, N, k) + t.shape[2:]
            out.append(jax.make_array_from_single_device_arrays(shape, sh, [per_chip[c][j] for c in range(self.n)]))
        return tuple(out)

    def _run(self, tokens, caches, pos0, use_recurrent, length=None):
        B, T = tokens.shape
        length = T if length is None else length
        tokens = jnp.asarray(tokens)
        streams = self._prog_embed(B, T)(self.params, tokens)
        new_caches = []
        self.stats = {"gather_s": 0.0, "transfer_s": 0.0}
        for i in range(self.cfg.n_layers):
            has_cache = caches is not None
            pre = self._prog_pre(i, B, T, has_cache, use_recurrent)
            streams, h, post, comb, nc, idx, w = pre(self.params, streams, jnp.int32(pos0), jnp.int32(length),
                                                     caches[i] if has_cache else None)
            new_caches.append(nc)
            if self.cfg.mlp_types[i] == "sparse":
                t0 = time.perf_counter()
                idx_np = np.asarray(idx)
                arrays, idx_local, Ep = self._gather_unique(i, idx_np)
                t1 = time.perf_counter()
                dev = self.transport(arrays)
                post_prog = self._prog_post(i, B, T, (Ep, tuple(len(c) for c in dev)))
                streams = post_prog(self.params, streams, h, post, comb, w, jnp.asarray(idx_local), *dev)
                streams.block_until_ready()
                t2 = time.perf_counter()
                self.stats["gather_s"] += t1 - t0
                self.stats["transfer_s"] += t2 - t1
            else:
                post_prog = self._prog_post(i, B, T)
                streams = post_prog(self.params, streams, h, post, comb, w, idx)
        logits = self._prog_head(B, T)(self.params, streams, jnp.int32(length))
        return logits, new_caches

    def prefill(self, tokens: np.ndarray):
        tokens = np.asarray(tokens)
        B, T = tokens.shape
        Tp = self._bucket(T)
        padded = np.zeros((B, Tp), np.int32)
        padded[:, :T] = tokens
        logits, caches = self._run(padded, None, 0, False, length=T)
        return logits, caches, T

    def decode(self, token: np.ndarray, caches, pos: int):
        B = token.shape[0]
        logits, caches = self._run(np.asarray(token).reshape(B, 1), caches, pos, True)
        return logits, caches, pos + 1
'''
FILES["glm53/gguf_reader.py"] = r'''"""Minimal GGUF reader for the Unsloth GLM-5.3-Flash mirrors: header/tensor-map parsing plus positional reads that
work either on whole .gguf files or on the 4 GB byte-range pieces produced by kaggle/glm_mirror_gguf.py
(manifest.json next to the pieces). Reads use pread (no mmap → no page-cache pressure on the TPU host cgroup).

Tensor naming (llama.cpp glm5next): blk.{L}.ffn_{gate,up,down}_exps.weight with GGUF dims [ne0=in, ne1=out, ne2=E]
i.e. row-major numpy shape (E, out, in); quant blocks run along `in`.
"""
import json
import os
import struct
from concurrent.futures import ThreadPoolExecutor

GGML_SIZES = {0: (1, 4), 1: (1, 2), 8: (32, 34), 10: (256, 84), 11: (256, 110), 12: (256, 144), 13: (256, 176),
              14: (256, 210), 16: (256, 66), 17: (256, 74), 18: (256, 98), 19: (256, 50), 20: (32, 18), 21: (256, 110),
              22: (256, 82), 23: (256, 136), 29: (256, 56), 30: (1, 2)}
GGML_NAMES = {0: "F32", 1: "F16", 8: "Q8_0", 10: "Q2_K", 11: "Q3_K", 12: "Q4_K", 13: "Q5_K", 14: "Q6_K", 16: "IQ2_XXS",
              17: "IQ2_XS", 18: "IQ3_XXS", 19: "IQ1_S", 20: "IQ4_NL", 21: "IQ3_S", 22: "IQ2_S", 23: "IQ4_XS",
              29: "IQ1_M", 30: "BF16"}


class PiecewiseFile:
    """A logical file backed by one real file or by ordered byte-range pieces."""

    def __init__(self, pieces):          # pieces: list of (path, start, length) covering [0, size)
        self.pieces = sorted(pieces, key=lambda p: p[1])
        self.size = self.pieces[-1][1] + self.pieces[-1][2]
        self._fds = {}

    @classmethod
    def single(cls, path):
        return cls([(path, 0, os.path.getsize(path))])

    def _fd(self, path):
        if path not in self._fds:
            self._fds[path] = os.open(path, os.O_RDONLY)
        return self._fds[path]

    def pread(self, offset, length):
        out = bytearray(length)
        mv = memoryview(out)
        done = 0
        for path, start, plen in self.pieces:
            if start + plen <= offset + done or start > offset + length:
                continue
            lo = max(offset + done, start)
            hi = min(offset + length, start + plen)
            if hi <= lo:
                continue
            got = os.pread(self._fd(path), hi - lo, lo - start)
            mv[lo - offset:hi - offset] = got
            done = hi - offset
            if done >= length:
                break
        assert done == length, f"short read {done}/{length} at {offset}"
        return out

    def close(self):
        for fd in self._fds.values():
            os.close(fd)
        self._fds.clear()


def open_mirror(dirs):
    """dirs: dataset mount dirs containing manifest.json + pieces (or raw .gguf files). Returns {shard_name: PiecewiseFile}."""
    files = {}
    for d in dirs:
        man = os.path.join(d, "manifest.json")
        if os.path.exists(man):
            m = json.load(open(man))
            for name, info in m["files"].items():
                files[name] = PiecewiseFile([(os.path.join(d, p["name"]), p["start"], p["length"]) for p in info["pieces"]])
        for fn in os.listdir(d):
            if fn.endswith(".gguf"):
                files[fn] = PiecewiseFile.single(os.path.join(d, fn))
    return files


class GGUFShard:
    def __init__(self, pf: PiecewiseFile):
        self.pf = pf
        self.kv, self.tensors, self.data0 = self._parse()

    def _parse(self):
        buf = self.pf.pread(0, min(self.pf.size, 64 << 20))
        pos = [0]

        def rd(fmt):
            v = struct.unpack_from(fmt, buf, pos[0])[0]
            pos[0] += struct.calcsize(fmt)
            return v

        def rstr():
            n = rd("<Q")
            s = bytes(buf[pos[0]:pos[0] + n]).decode(errors="replace")
            pos[0] += n
            return s

        def rval(t):
            if t in (0, 1): return rd("<B" if t == 0 else "<b")
            if t in (2, 3): return rd("<H" if t == 2 else "<h")
            if t in (4, 5): return rd("<I" if t == 4 else "<i")
            if t == 6: return rd("<f")
            if t == 7: return rd("<B")
            if t == 8: return rstr()
            if t == 9:
                et = rd("<I"); n = rd("<Q"); return [rval(et) for _ in range(n)]
            if t in (10, 11): return rd("<Q" if t == 10 else "<q")
            if t == 12: return rd("<d")
            raise ValueError(t)

        assert bytes(buf[:4]) == b"GGUF", "not a GGUF file"
        pos[0] = 4
        rd("<I"); nt = rd("<Q"); nkv = rd("<Q")
        kv = {}
        for _ in range(nkv):
            k = rstr(); t = rd("<I"); v = rval(t)
            if not (isinstance(v, list) and len(v) > 64):
                kv[k] = v
        align = kv.get("general.alignment", 32)
        tensors = {}
        for _ in range(nt):
            name = rstr(); nd = rd("<I"); dims = [rd("<Q") for _ in range(nd)]; ty = rd("<I"); off = rd("<Q")
            n = 1
            for d in dims:
                n *= d
            bs, tb = GGML_SIZES[ty]
            tensors[name] = {"dims": dims, "type": GGML_NAMES.get(ty, ty), "offset": off, "nbytes": n // bs * tb}
        data0 = (pos[0] + align - 1) // align * align
        return kv, tensors, data0

    def read_tensor(self, name, offset=0, length=None):
        t = self.tensors[name]
        length = t["nbytes"] - offset if length is None else length
        return self.pf.pread(self.data0 + t["offset"] + offset, length)


class GGUFModel:
    """All shards of a split GGUF merged into one tensor map."""

    def __init__(self, files):
        self.shards = {n: GGUFShard(pf) for n, pf in sorted(files.items())}
        self.where = {}
        self.kv = {}
        for n, s in self.shards.items():
            self.kv.update(s.kv)
            for t in s.tensors:
                self.where[t] = n

    def info(self, name):
        return self.shards[self.where[name]].tensors[name]

    def read(self, name, offset=0, length=None):
        return self.shards[self.where[name]].read_tensor(name, offset, length)

    def expert_bytes(self, name):
        """bytes of one expert (dims [in, out, E]) = out * in/bs * tb."""
        t = self.info(name)
        return t["nbytes"] // t["dims"][2]

    def read_expert(self, name, e):
        return self.read(name, e * self.expert_bytes(name), self.expert_bytes(name))

    def read_experts(self, name, es, threads=8):
        with ThreadPoolExecutor(threads) as pool:
            return list(pool.map(lambda e: self.read_expert(name, e), es))
'''
FILES["glm53/hf_convert.py"] = r'''"""Convert Glm5Next weights to the JAX param layout of `glm53.model`.

Two sources:
  * `from_hf_module(model)`: a transformers `Glm5NextTextModel` (used for tiny random-init tests).
  * `dequant_fp8(w, scale_inv, block=128)`: helper for real checkpoints (fp8 e4m3 blockwise).
"""
from __future__ import annotations

import numpy as np


def _np(t):
    import torch
    if t.dtype == torch.bfloat16:
        return t.detach().float().cpu().numpy()
    return t.detach().cpu().numpy()


def _lin(t):
    """torch Linear weight [out, in] -> JAX [in, out]."""
    return np.ascontiguousarray(_np(t).T)


def convert_hc(mod):
    return {"fn": _lin(mod.fn), "base": _np(mod.base), "scale": _np(mod.scale)}


def convert_kda(a):
    C = a.qkv_dim
    conv_w = _np(a.conv1d.weight)[:, 0, :]                     # [3C, K]
    return {
        "q": _lin(a.q_proj.weight), "k": _lin(a.k_proj.weight), "v": _lin(a.v_proj.weight),
        "conv_q": conv_w[:C], "conv_k": conv_w[C:2 * C], "conv_v": conv_w[2 * C:],
        "f_a": _lin(a.forget_gate.f_a_proj.weight), "f_b": _lin(a.forget_gate.f_b_proj.weight),
        "dt_bias": _np(a.forget_gate.dt_bias), "A_log": _np(a.forget_gate.A_log),
        "b": _lin(a.b_proj.weight),
        "g_a": _lin(a.g_a_proj.weight), "g_b": _lin(a.g_b_proj.weight),
        "o_norm": _np(a.o_norm.weight), "o": _lin(a.o_proj.weight),
    }


def convert_mla(a):
    p = {
        "q_a": _lin(a.q_a_proj.weight), "q_a_norm": _np(a.q_a_layernorm.weight), "q_b": _lin(a.q_b_proj.weight),
        "kv_a": _lin(a.kv_a_proj_with_mqa.weight), "kv_a_norm": _np(a.kv_a_layernorm.weight),
        "kv_b": _lin(a.kv_b_proj.weight), "o": _lin(a.o_proj.weight),
    }
    if a.indexer is not None:
        ix = a.indexer
        p["indexer"] = {
            "wq_b": _lin(ix.wq_b.weight), "wk": _lin(ix.wk.weight),
            "k_norm_w": _np(ix.k_norm.weight), "k_norm_b": _np(ix.k_norm.bias),
            "weights_proj": _lin(ix.weights_proj.weight),
            "ape": _np(ix.index_kpool_compress_ape), "gate": _lin(ix.index_kpool_compress_gate),
        }
    return p


def convert_mlp(m):
    return {"gate": _lin(m.gate_proj.weight), "up": _lin(m.up_proj.weight), "down": _lin(m.down_proj.weight)}


def convert_moe(m):
    gu = _np(m.experts.gate_up_proj)                              # [E, 2mi, D]
    dn = _np(m.experts.down_proj)                                 # [E, D, mi]
    return {
        "router_w": _lin(m.gate.weight), "router_bias": _np(m.gate.e_score_correction_bias),
        "gate_up": np.ascontiguousarray(np.transpose(gu, (0, 2, 1))),   # [E, D, 2mi]
        "down": np.ascontiguousarray(np.transpose(dn, (0, 2, 1))),      # [E, mi, D]
        "shared": convert_mlp(m.shared_experts),
    }


def from_hf_module(model, lm_head=None):
    """`model`: transformers Glm5NextTextModel. Returns nested dict of numpy arrays."""
    layers = []
    for L in model.layers:
        p = {
            "ln1": _np(L.input_layernorm.weight), "ln2": _np(L.post_attention_layernorm.weight),
            "hc_attn": convert_hc(L.attn_hc), "hc_ffn": convert_hc(L.ffn_hc),
        }
        p["attn"] = convert_kda(L.self_attn) if L.block_type == "linear_attention" else convert_mla(L.self_attn)
        p["mlp"] = convert_moe(L.mlp) if hasattr(L.mlp, "experts") else convert_mlp(L.mlp)
        layers.append(p)
    params = {"embed": _np(model.embed_tokens.weight), "norm": _np(model.norm.weight), "layers": layers}
    if lm_head is not None:
        params["lm_head"] = _lin(lm_head.weight)
    return params


def dequant_fp8(w_fp8: np.ndarray, scale_inv: np.ndarray, block: int = 128) -> np.ndarray:
    """fp8 e4m3 weight [O, I] (as float32/np) with per-block scales [ceil(O/b), ceil(I/b)] -> float32 [O, I]."""
    O, I = w_fp8.shape
    so, si = scale_inv.shape
    s = np.repeat(np.repeat(scale_inv.astype(np.float32), block, axis=0), block, axis=1)[:O, :I]
    assert s.shape == (O, I), (s.shape, O, I, so, si)
    return w_fp8.astype(np.float32) * s
'''
FILES["glm53/iq_grids_data.py"] = r'''"""Auto-generated: llama.cpp IQ codebook grids (from gguf-py), npz base64 — so the package needs no data files."""
import base64, io
import numpy as np

_B64 = (
    "UEsDBC0AAAAIAAAAIQB/XAco//////////8JABQASVEyX1MubnB5AQAQAIAgAAAAAAAAdQcAAAAAAACdmb2OXkUMhkNncQfutluQ"
    "UxBEgbgAOhANBRWKyCKQIoI2iAa4Cm6YPWM/79iTLynY1ZH32z0/Y/v98Zz999vvv/nuh4+e/fnsr/tXD29/erz/6u7+719f3D+/"
    "u//5zeMfjy9/+/HN46uH6/dfv3z99uHp929/efn7w9PnT1589vkXz+++/PT53T93/+/rY6uvqOg+fxEV3XycEBU95gVR0X3ewK1u"
    "UDeKik+njRtHRff5ILcYD4yKHnMBUfHpqrEgN7+5sKePeV3U5zpYcFS8ruoJRMVrmT0ht7iZWFS8Tu+JuvlIOCp6zAK4VyGiPlsV"
    "5ChMVHSvaHGzUBzXzXrh3PxmAa/yRJWlFzQqXun2Al9p9EJft8v7zMKHZeGpufXkeFhrSFRc1WkNWqttjYqK6ymtcVFxta01Miq6"
    "+WhsVFzlb42OiqvMbWkAICp6TECssuS6K5Wo62IAht5wAKCo6OYDUFFxLacBbD3Os/DeSgfwouI6rQExKrpPYLrFTYBGRfcCbNRn"
    "85sApnX0msNjAtt9Atx9At3NJ+BLGVapPkAAME9tOQCJ9aSiHt6IstAdyYuCnncCRcXV5UaoqLiqGVW9qGyjsmzEW3+NolWUTjWo"
    "n4SMigu2jaALjo2gCy5RMImCwyBueKfSWlgjtJsPYofFIHgcRL+u7lS01mOJSRMAyARYAQtFJvk4BIJW+GAlbNnCEd7RuYUkrKMK"
    "FG3psNGtLTjuU3i8qnQKEBDxcVdXTflGoKJiyswWrKiYcrIFjCspQVRMGdjCFhWT3lvoeOIpeEA66bgFkJUhhKyQ0oNJepqw3xmc"
    "Apmw3ZkhmGRo7xFMKIYmwmEqwoFIISaIAKRAYKmg9aS1uC28UTHLuIWYyp+CjARwaZZhd+gUbDoGdNFEOG69pr4PhJ1OnwKPBLF0"
    "BB9EnMIPQkSdwwBIHc0ESachgCxR7zCGS9868qBm+NaybhzkygFSERfISktWoRqSt8HEQLakoEmyt5biCTDgNCYYISkpg7pcKO8X"
    "HzSqq7jdsLAEoIQXwThrNZY5NmND7KOmLhgqEepglPu/a3zpPtsAowwQhksih0rFxqJNQ8TiRAnr6NwGmd19v1HKg+o7Fxo1t0ZZ"
    "TDfSOkMp5ZkoUUlMQS005OI9cIqhBbPXsZ+4xbcZNGJyGnUWvxv2nKyRCFqAUp6GjnJmQbexx6LtNvhEfjd6PDUN3xdct/FLmmxr"
    "ZVfmbPceDHaR8wPKbVjjKkckO/rsAFYZHPYtVkT5bVOUElk1SZ6C5qEJcKw7h7WcrbrfkYAU205hmwAivEURUXMTgua2DNKJLDam"
    "Lw02FFPF6ic3a4B6OB4ehuYe2xRc14UV6263pQ7I8SiQj0vgCqj9OVDBENQWdUU1pZI1aKFmUicxa6jC1nY4ZoNNsEfOLjYU2kGv"
    "UDtQsK3VZvU1GdjMVlpvc1XiIE/jar6xEKhMC1EMTBlTxbzOgRBlQUSRXsRGTyjyAl7rzWhSYv1i37MCXopmn4MlPeBA+rmFMqqB"
    "c7m9a3zYmR4DKApZdq+BlCWpMgkwYViVEgUW0Ja79smMSiKhUAVFZkZihmCSOwdcsHfY3Wlv2Nnu1DEIo/yQDhJQcjWnFglX1WlR"
    "27qMn/KNbHdN6zKqkRHLptUgiBnukDV5Ld6DJ6AlNnryjgwpJYZRhkmf8oFcyBEZHhgtgCTIRqxBOCLkk54qJaAFZDTx3AjwELBO"
    "zzn0p5qWfcK3Se6AmzYMUEm3rlkNTRYTq3aXwfWNhM/2aST12RZBB5VHvX2WkzJKW32WRznrKPb6TKdZyliOTIau+rytSOnzz23x"
    "uXHhBymO1Wy9HDnz6BsZJiMuZATlBmphCpBuKOUqrOZmpvDZHiRlS2HqJFoboDXV5vOjb4hYIIrIuy7bFB8LZ09DAufGiYTQIhJT"
    "r97ZUGHlmTBDCgXAlCmEoF2iSWFEJoErF+srpZsbsDa1VkHaxozCsiWgwKJUCqcKLkeQhlj0RkiVpA7JFkZUmoSkqhvJODVSTpNC"
    "q2tX6dpGb82ykTNJb3jqsP79wr9btBEECEgGS2GvJiQbzmbSSjShAwas0hOOE0BlDQISTslwxRCExQA0WgvgZF5lGsxkABGRA5CQ"
    "nkPgqiannVfyjaH1elZARlIBtBy8PASA4+hszQG8dctsBACyECH1cc+YEAPPhyDSwloiuemQ+3cX3YTC7SCWLKJShWhSZ6lqI1Mj"
    "oppUyXiyLRrLGlo2Yeka1VC2tUp2ghA6abqHhSWCbbuYGpoaOF8pxxAAKyaxfbMxymyB0JOZrLKR2okiHKwMAZFHFWTAnHqjYyt6"
    "FxqsGcHR8FBmWuOvhEjkMMbJXDRKkWm7Xq0jXFQSAdMoUNBG0DTeaGxJTufLgC14Ggdk82XPJcUIIWITmfySxS6MsqdSichSYh+Z"
    "VnvhUKvQG4GSRWRQAst/bZEz/Zey/gmB8EqOSmYEpVQGDadZSt6O8VZrv4kAmUG0W2839o8IvOAouC04RCIt9lFtlZVUGSqdvex6"
    "RnYjylH2KyydNlfbllan1dd/UEsDBC0AAAAIAAAAIQCfBHCG//////////8JABQASVEzX1MubnB5AQAQAIAIAAAAAAAA6QMAAAAA"
    "AACdlc2KHUcMhacRhZCF0Cvc3Y1hNmOcjR/Au4RsvMjKDPGYBIwnzJhs4jxFXjjfUbXJ3rc5VHd1tf7Oke6/P7/76Zdfj5u/bv6+"
    "fnh4/u3p+uZy/frH3fX2cv34+PTl6f7z+8enDw/af3v/6fmB/eff7/984PmHH+9e3V5ev7y9/HP5zt+Lg5+BBRL0uWFggQB1HjCQ"
    "wJ3nYAskaGB87DJQ7APDUIA+DepaIE7jBvJ04qCBOXsgcGo4TbB6O9fXCfoMxEEBW+wvBbWOACt4Bg2MYF0BN6vAZeb4JYHlxwJG"
    "Mg4KRLCf7IMu3jVneyfrFkcDI+kAy0k62Ad2FkERhqq4iBQUcIpikcSUHGGv2GtWLrPCLqsXn7B2DQPKMsTE4h4UcOc52c9dUNt0"
    "WJzFtV0+S9BnsSmL0jXSMqWDe8O93BomhwjbR7Q1VgLY4t7Z970hopbIas71Jk1XggYEaQGWiIxlDjK5L/Z6G1MFyWKi1MVha0Bi"
    "xMdzwGjx3LxFdVQSyzEXBomFZ4RREkeymzswZey2A4RszHKf3K8ipsIHaxTf1SQAs1MhRU5wls4aPJMYYh2x2ahDLIqBLT5lGWNV"
    "yfMeICmJSjIi1JHEyi0D0b+gcwQLdStP4YqWY7KXd56xliom6asEK87CYiKGfZ3ABUDjq30XG12vBiS4oncD6ECqm8znixAh7gt9"
    "c4iI01fVJgklTbPoFIEjeUgL7mmcbCmcqGggEsEnTRSJT9ZWQxW+apRJ4VYma0ntIpuKr91wTmBodoKzYVoM8cZ2sLZbz2k3wphW"
    "8pZQ1Db1v2DUpcjWpQSbjEYiNC2JQmn1bmgpUGzoIl2adwvsGPXgfirhimw8S3gcxCddmty3wokRI6qZopksJR1U4QE0odAcvpKq"
    "sBJ9RHrJQLPSaRZbyGiNeHpYQ2czSDQ1RABqoNqHKgQJZBMzOtT2mlyMzmnFCUaMq4VSEashknsgSbqmMNJBMiH1uO8BBU3In/te"
    "M6XUREpGKhXhKgkRzxBTKHwSLQa4tY6ZWK1GI/lVFECd5xUMMPwxyayHLTgmHk1bop/OnHE8gxBrvOEZY1XboBpWBUxNZxuli6VU"
    "K6rl1Mi+RsaJVKfImuJ9jARQk2MXGfGGAyorNpkGgFJgK5JIUxNIymTRSapRI96QgOkc9CiSio/L9vivuRiRdHOhZI2METgKKFUT"
    "K6WLmnKOYiA9NFVIpuSA8vE2eEdkgDCxmKWpf0QxgPhXOYd87L/GGfbUS383M/T1NzEeNKT4ALFAixG4xvrSv0xrepfGh8aDOiC/"
    "NZwU6CM2asK/DJRKSmuoxA/l4w+E3p2uUQj/AVBLAwQtAAAACAAAACEAuFJVlf//////////CgAUAElRMl9YUy5ucHkBABAAgBAA"
    "AAAAAADaAwAAAAAAAJ1XMW4UQRA02YgfVHbZgcqJkZAQDyADkRAQIQsfAglhdEYkwCv4MLfdXT09s7sJZ63bt7sz011dVTP+++bd"
    "67fvH139vPp1vDs9fDwfXx6Ov7/cHK8Px0/35x/n228f7s93p+X+q9uvD6fL/YfPt99Pl+9Pnt88uz68eHp9+HP4z8/jFh9GBMYb"
    "jIiG4QVGBMcBjAiME6DFBDERI15eGyZmRGBcCI2bCzLi5a0hATQMiVyGDQkhLiXGiMtbNVFGXNKpiTMiGjcLYcRlWC0MDdsFkvE8"
    "Ct0pWNfycgUADZtAoAUQEyDL8hWQyy+yYNJqMpqkAMaIVkUB0FbfAJIRDcYCLBoGgDkBraX2gBYWugQ8GjYboJTnRsBXzoaopLkh"
    "C41qY6gGwSiRjVJvlJMuRhQocyOtK+xQrhrLmemkz+tR0O81PkpoVsagCA5EuCzDuG9RrWwFiz53J4yahYHNyGJmIqlUDN1W99YE"
    "I4lKqT3CCToMb/efmYi6syJkEhEWGVEjlTJ3icqBsGqluCkOKIOZyMpIxdhkldiWTie4Mt8lehLcarH5WCqdic9ZALCB/qgIQakQ"
    "iRDMIxf6FGGIukq5Ve6l266FI6R3heOszQ7sCogegVFI+pkFpU7OwgK5KTBHyEZahAOSTKBjtFwcBFhNWCZBnyWe0zOTQE2hjn6M"
    "Z4xjqcC/xih2JrbeOqU+CFyVizvigLCchS+EREG1IpZMxs/GgHBPNXXPKHRT1FaLpZzZQHTtGQmCzd6RrrjZWBgRo72m0QgNVacq"
    "lJWywGhbmdu+IY1KXxkUBrkvWqsydkrE3NXA5BQrA4ueAdE7uEyYJtBySdQMne4m9rjfCs1SGbIgSVGpr4yQcToZ27syRowwJ9g+"
    "czdI9kVaSd9tqThjIhklYVzG+epldFFWY2VWyfhOlD+aNwi+gxUj9qnzCCJjhptEeGjUw6hnY6LsfFK8Zc791EKz6L67F2M3zzIr"
    "iaOBMT5dgjL4JLNr1yqAMw7oZ1kzfUuQQc7mezcQzhfvdAcUIt4qjoXNG4YOJyo4GT1tHHJWmWMMTICkAEoJKdKGEUBMGw6HjYe5"
    "VnXdDnAqLKwlFs7Od/eIY4ZvPn3jCkW60HzvNYSpnjZtZM0ppyz8bT3VThKbBNMEtb00PfEYTGEgl4RIR0hLdJuOgmSbuVl4OXna"
    "SGb42n0DFdPTYSRj14oT1O/bFIzo7AyLozHOKOZiR0qwxUPRjqLfSKtsrWijK9qQM9ps5QQtsrUkgWDvG3f+q4UiEEvSczZlqBPl"
    "LJJDGfmwXI6+H0X9yEWUjT9eyxOZPv8AUEsDBC0AAAAIAAAAIQBJNKzI//////////8LABQASVEzX1hYUy5ucHkBABAAgAQAAAAA"
    "AAAaAgAAAAAAAJ2TMavUMRDEn5BKttxyiysWTmEbZbWwSGmn2FhYycN3oiA+uRMb9VP4hf1NcmBvwpD/7SXZ2dnJn9dvX7159+Dm"
    "x83P493p8uF8fHE4/vr85FiH48f78/fz7df39+e7k+Ivb79cTsQvn26/nfj96Omz53Xox3X4ffjP8XAwHCQwGyPAtB10YMHKnzHH"
    "KGDMANrswNxG+T7owNJG1r9LNBOY7QsdWPgIMIEVK6jp6+YJlNRBOd8zuDpZcmSyVsIl2Z/sL/YXeUHVqOiR3ZyfI0GDyDms52Ic"
    "m74oi45orGJIYwmuV67ixFhDRbrvD7HSaU0FR/A1d/F+FcAirBM037AoMAp4WYJGHJtFvLkCkXILJXVsS7hEk1TtOyn6r8SiCr0l"
    "phhoKnVw1QRovkTWcVu3b8E10JpGODJt4qo+bTchQUUiI0D16XQ0i7g6O4lNR0uv3MWOVfkuWq1Es6gmdm2eXxuomSRpoNIILrWh"
    "QEIwt2AaNIq2JjE6sTpZMK6ogpl1aAaiRmCIlhuWclnBCrAAdLHIxCy+SsIwoB1L8JsEldBhixVO6qRCRWqZCielHOByQWgdNOxq"
    "MFzaght0OG2L/RaK0jVxEIQTG3PtTDm6LGTKRtW5xEwfZB3dIFsGsNakT/x2FFwVttjpIM3aRY+5OqzXAJnJM5Shd2NgCK9leLlR"
    "ycRMLxYCU4KNDlJImVyvWK8NLyJJz79QSwMELQAAAAgAAAAhACXOngv//////////woAFABrc2lnbnMubnB5AQAQAAABAAAAAAAA"
    "zQAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBeU2qorqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGRhY6mjoKtQpk"
    "Ay6GxibmFla29g5Orm6e3j7+CYJCk0WmThOXmDlLeo6s3PwFikqLVZYuU9dYuUp7ja7eeoONm4y3mJpt32Fptdtm7z77A45Oh12O"
    "HnP3OHnK+4yv3/mAi5eCr4SGXb8RGXU75u69+ISHj5KfpKY9f5GZ9Trn7bv8D4VFn0u+fiuv+Pmr+k9t3X8AUEsDBC0AAAAIAAAA"
    "IQC18arc//////////8RABQAa3ZhbHVlc19pcTRubC5ucHkBABAAkAAAAAAAAABWAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6l"
    "oF6Taaiuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GoZmOpo6CrUK5AOuxhlr95+/++obI6+kmqlrZCEAUEsBAi0DLQAA"
    "AAgAAAAhAH9cByh1BwAAgCAAAAkAAAAAAAAAAAAAAIABAAAAAElRMl9TLm5weVBLAQItAy0AAAAIAAAAIQCfBHCG6QMAAIAIAAAJ"
    "AAAAAAAAAAAAAACAAbAHAABJUTNfUy5ucHlQSwECLQMtAAAACAAAACEAuFJVldoDAACAEAAACgAAAAAAAAAAAAAAgAHUCwAASVEy"
    "X1hTLm5weVBLAQItAy0AAAAIAAAAIQBJNKzIGgIAAIAEAAALAAAAAAAAAAAAAACAAeoPAABJUTNfWFhTLm5weVBLAQItAy0AAAAI"
    "AAAAIQAlzp4LzQAAAAABAAAKAAAAAAAAAAAAAACAAUESAABrc2lnbnMubnB5UEsBAi0DLQAAAAgAAAAhALXxqtxWAAAAkAAAABEA"
    "AAAAAAAAAAAAAIABShMAAGt2YWx1ZXNfaXE0bmwubnB5UEsFBgAAAAAGAAYAVgEAAOMTAAAAAA=="
)


def load():
    return np.load(io.BytesIO(base64.b64decode("".join(_B64))))
'''
FILES["glm53/iqquant.py"] = r'''"""Dequantization of llama.cpp "IQ" codebook formats (as used by Unsloth's UD-IQ3_XXS / UD-Q2_K_XL GLM-5.3-Flash GGUFs)
written in jax.numpy so the same code runs on CPU (tests vs gguf-py) and on TPU (inside the resident engine).

Block = 256 weights (QK_K). Inputs are the raw GGUF block bytes, uint8 [..., block_bytes]; output float [..., 256].
Layouts (bytes): IQ2_S 82 = d2|qs32|signs32|qh8|scales8 · IQ3_S 110 = d2|qs64|qh8|signs32|scales4 ·
IQ4_XS 136 = d2|sh2|sl4|qs128 · IQ2_XS 74 = d2|qs64(u16)|scales8 · IQ3_XXS 98 = d2|qs64|scales32(u32).
The grid lookup is `jnp.take(grid, idx)`; `LOOKUP` can be swapped for a faster TPU implementation.
"""
import os
import numpy as np
import jax
import jax.numpy as jnp

QK_K = 256
BLOCK_BYTES = {"IQ2_S": 82, "IQ3_S": 110, "IQ4_XS": 136, "IQ2_XS": 74, "IQ3_XXS": 98, "Q2_K": 84, "Q3_K": 110}
from glm53 import iq_grids_data as _gd
_g = _gd.load()
GRIDS = {k: np.asarray(_g[k]) for k in ("IQ2_S", "IQ3_S", "IQ2_XS", "IQ3_XXS")}   # int8 [entries, 8 or 4]
KSIGNS = np.asarray(_g["ksigns"])                                                  # uint8 [128]
KVALUES_IQ4NL = np.asarray(_g["kvalues_iq4nl"])                                    # int8 [16]


def _u16(b):   # uint8 [..., 2k] -> uint16 [..., k] (little endian)
    return jax.lax.bitcast_convert_type(b.reshape(b.shape[:-1] + (-1, 2)), jnp.uint16)


def _u32(b):
    return jax.lax.bitcast_convert_type(b.reshape(b.shape[:-1] + (-1, 4)), jnp.uint32)


def _f16(b):   # uint8 [..., 2] -> float32 [...]
    return jax.lax.bitcast_convert_type(b, jnp.float16).astype(jnp.float32)


def _bits8(b):   # uint8 [..., n] -> [..., n, 8] bit k of each byte (LSB first)
    return (b[..., None] >> jnp.arange(8, dtype=jnp.uint8)) & 1


def _nibbles(b):  # uint8 [..., n] -> [..., 2n] low nibble first
    return jnp.stack([b & 0x0F, b >> 4], -1).reshape(b.shape[:-1] + (-1,))


# ---------------------------------------------------------------------------------------------------- grid lookup
# XLA's gather is ~0.1 G lookups/s on v5e (measured) — useless. The "tree" implementation is gather-free: every grid
# entry is encoded as level indices (2 or 3 bits per value), bit-sliced into 32-entry uint32 words, and each output bit
# is fetched with a binary tree of `where`s over the word index (n_words-1 selects) + one variable shift.
LOOKUP_IMPL = "tree"          # "take" (jnp.take) or "tree"
LEVELS = {"IQ2_S": np.array([8, 25, 43], np.int8), "IQ2_XS": np.array([8, 25, 43], np.int8),
          "IQ3_S": np.array([1, 3, 5, 7, 9, 11, 13, 15], np.int8),
          "IQ3_XXS": np.array([4, 12, 20, 28, 36, 44, 52, 62], np.int8)}


class _TreeLUT:
    def __init__(self, name):
        grid, levels = GRIDS[name], LEVELS[name]
        self.levels = levels
        self.w = grid.shape[1]
        self.lb = int(np.ceil(np.log2(len(levels))))                       # bits per level index (2 or 3)
        n = grid.shape[0]
        lv = np.searchsorted(levels, grid)                                 # [n, w] level indices
        assert np.array_equal(levels[lv], grid)
        code = np.zeros(n, np.int64)
        for k in range(self.w):
            code |= lv[:, k].astype(np.int64) << (self.lb * k)
        self.nbits = self.lb * self.w
        self.n_words = n // 32
        # words[b][j] = 32 bits (entries 32j..32j+31) of output bit b
        self.words = np.zeros((self.nbits, self.n_words), np.uint32)
        for b in range(self.nbits):
            bits = ((code >> b) & 1).astype(np.uint32)
            self.words[b] = (bits.reshape(self.n_words, 32) << np.arange(32, dtype=np.uint32)).sum(1, dtype=np.uint32)

    def __call__(self, idx):
        wi = idx >> 5
        sh = (idx & 31).astype(jnp.uint32)
        bits = [((wi >> k) & 1).astype(bool) for k in range(int(np.log2(self.n_words)))]
        code = jnp.zeros(idx.shape, jnp.uint32)
        for b in range(self.nbits):
            nodes = [jnp.uint32(int(x)) for x in self.words[b]]
            for k, bit in enumerate(bits):                                 # select tree over word index bits
                nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
            word = nodes[0] if len(nodes) == 1 else jnp.broadcast_to(jnp.uint32(int(self.words[b][0])), idx.shape)
            code = code | (((word >> sh) & 1) << b)
        mask = (1 << self.lb) - 1
        lv = jnp.stack([((code >> (self.lb * k)) & mask).astype(jnp.int32) for k in range(self.w)], -1)   # [..., w]
        levels = self.levels
        out = jnp.full(lv.shape, int(levels[0]), jnp.int8)
        for i in range(1, len(levels)):
            out = jnp.where(lv == i, jnp.int8(int(levels[i])), out)
        return out


_TREES = {}


def lookup(name, idx):
    """name: grid name; idx int32 [...] -> int8 [..., w] grid values."""
    if LOOKUP_IMPL == "take":
        return jnp.take(jnp.asarray(GRIDS[name]), idx, axis=0)
    if name not in _TREES:
        _TREES[name] = _TreeLUT(name)
    return _TREES[name](idx)


def _signs_from_bits(signs_u8):   # uint8 [..., 32] -> float32 [..., 256] of ±1
    s = _bits8(signs_u8).reshape(signs_u8.shape[:-1] + (QK_K,))
    return 1.0 - 2.0 * s.astype(jnp.float32)


def dequant_iq2_s(blk):
    d = _f16(blk[..., 0:2])                                   # [...]
    qs = blk[..., 2:34].astype(jnp.int32)                     # [..., 32] low 8 bits of grid index (32 groups of 8)
    signs = _signs_from_bits(blk[..., 34:66])                 # [..., 256]
    qh = blk[..., 66:74]                                      # [..., 8] 2 bits per group, 4 groups per byte
    sc = _nibbles(blk[..., 74:82]).astype(jnp.float32)        # [..., 16] one 4-bit scale per 16 weights
    hi = jnp.stack([(qh >> (2 * k)) & 3 for k in range(4)], -1).reshape(qh.shape[:-1] + (32,)).astype(jnp.int32)
    idx = qs | (hi << 8)                                      # [..., 32]
    g = lookup("IQ2_S", idx).astype(jnp.float32).reshape(idx.shape[:-1] + (16, 16))  # 2 groups per scale
    db = d[..., None, None] * (0.5 + sc)[..., None] * 0.25
    return (db * g).reshape(idx.shape[:-1] + (QK_K,)) * signs


def dequant_iq3_s(blk):
    d = _f16(blk[..., 0:2])
    qs = blk[..., 2:66].astype(jnp.int32)                     # [..., 64] low 8 bits (64 groups of 4)
    qh = _bits8(blk[..., 66:74]).reshape(blk.shape[:-1] + (64,)).astype(jnp.int32)   # high bit per group
    signs = _signs_from_bits(blk[..., 74:106])
    sc = _nibbles(blk[..., 106:110]).astype(jnp.float32)      # [..., 8] one 4-bit scale per 32 weights
    idx = qs | (qh << 8)
    g = lookup("IQ3_S", idx).astype(jnp.float32).reshape(idx.shape[:-1] + (8, 32))   # 8 groups of 4 per scale
    db = d[..., None, None] * (1.0 + 2.0 * sc)[..., None]
    return (db * g).reshape(idx.shape[:-1] + (QK_K,)) * signs


def dequant_iq4_xs(blk):
    d = _f16(blk[..., 0:2])
    sh = _u16(blk[..., 2:4])[..., 0]                          # [...] 2 bits per 32-block (8 blocks)
    sl = _nibbles(blk[..., 4:8]).astype(jnp.int32)            # [..., 8]
    qs = blk[..., 8:136]                                      # [..., 128] two 4-bit values per byte, low nibble first
    shb = jnp.stack([(sh >> (2 * k)) & 3 for k in range(8)], -1).astype(jnp.int32)   # [..., 8]
    scales = (sl | (shb << 4)) - 32                           # [..., 8]
    q = qs.reshape(qs.shape[:-1] + (8, 16))                   # 8 sub-blocks of 16 bytes = 32 weights each
    q = jnp.concatenate([q & 0x0F, q >> 4], -1).astype(jnp.int32)   # [..., 8, 32]: first 16 low nibbles then 16 high
    v = _kv16(q)
    out = d[..., None, None] * scales.astype(jnp.float32)[..., None] * v
    return out.reshape(qs.shape[:-1] + (QK_K,))


def dequant_iq2_xs(blk):
    d = _f16(blk[..., 0:2])
    qs = _u16(blk[..., 2:66]).astype(jnp.int32)               # [..., 32]: 9-bit grid index | 7-bit sign index << 9
    sc = _nibbles(blk[..., 66:74]).astype(jnp.float32)        # [..., 16]
    sidx = qs >> 9
    sbits = jnp.take(jnp.asarray(KSIGNS), sidx, axis=0)       # uint8 [..., 32]
    signs = 1.0 - 2.0 * _bits8(sbits).astype(jnp.float32)     # [..., 32, 8]
    g = lookup("IQ2_XS", qs & 511).astype(jnp.float32) # [..., 32, 8]
    db = (d[..., None] * (0.5 + sc) * 0.25)                    # [..., 16]
    out = (g * signs).reshape(qs.shape[:-1] + (16, 16)) * db[..., None]
    return out.reshape(qs.shape[:-1] + (QK_K,))


def dequant_iq3_xxs(blk):
    d = _f16(blk[..., 0:2])
    qs = blk[..., 2:66].astype(jnp.int32)                     # [..., 64] 8-bit grid index (64 groups of 4)
    sc = _u32(blk[..., 66:98])                                 # [..., 8]: 4 x 7-bit sign idx | 4-bit scale << 28
    db = d[..., None] * (0.5 + (sc >> 28).astype(jnp.float32)) * 0.5     # [..., 8] one scale per 32 weights
    sidx = jnp.stack([(sc >> s) & 0x7F for s in (0, 7, 14, 21)], -1).astype(jnp.int32)   # [..., 8, 4]
    sbits = jnp.take(jnp.asarray(KSIGNS), sidx, axis=0)                                 # [..., 8, 4] uint8
    signs = 1.0 - 2.0 * _bits8(sbits).astype(jnp.float32)                               # [..., 8, 4, 8]
    g = lookup("IQ3_XXS", qs).astype(jnp.float32).reshape(qs.shape[:-1] + (8, 4, 8))   # 8 x (4 groups x 4... see below)
    # each 32-weight sub-block = 8 groups of 4 = the [8 groups] x [4 values]; regroup to match sign layout (4 sign-bytes x 8)
    g = g.reshape(qs.shape[:-1] + (8, 32)); signs = signs.reshape(qs.shape[:-1] + (8, 32))
    return (db[..., None] * g * signs).reshape(qs.shape[:-1] + (QK_K,))


# ------------------------------------------------------------------------------------ K-quants (the MTP layer's experts)
def _f16_at(blk, off):
    """f16 at byte offset `off` of every block -> f32 [..., nblk]."""
    u = blk[..., off].astype(jnp.uint32) | (blk[..., off + 1].astype(jnp.uint32) << 8)
    return jax.lax.bitcast_convert_type(u.astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _kq_layout(q):
    """q u8 [..., nblk, 64] -> list of 8 (n, s) groups of 32 2-bit values [..., nblk, 32] in ggml order
    (weight index 128 n + 32 s + l; byte q[32 n + l] shifted by 2 s)."""
    q = q.astype(jnp.int32)
    return [((q[..., 32 * n:32 * (n + 1)] >> (2 * s)) & 3) for n in range(2) for s in range(4)]


def q3k_scales(sc12):
    """Q3_K 12 packed bytes -> 16 six-bit scales (uint8 [..., 16]) in `is` order (ggml `dequantize_row_q3_K`)."""
    sc = np.asarray(sc12).astype(np.uint32)
    aux = [sc[..., 4 * i] | (sc[..., 4 * i + 1] << 8) | (sc[..., 4 * i + 2] << 16) | (sc[..., 4 * i + 3] << 24) for i in range(3)]
    k1, k2 = np.uint32(0x03030303), np.uint32(0x0F0F0F0F)
    tmp = aux[2]
    a2 = ((aux[0] >> 4) & k2) | (((tmp >> 4) & k1) << 4)
    a3 = ((aux[1] >> 4) & k2) | (((tmp >> 6) & k1) << 4)
    a0 = (aux[0] & k2) | (((tmp >> 0) & k1) << 4)
    a1 = (aux[1] & k2) | (((tmp >> 2) & k1) << 4)
    words = np.stack([a0, a1, a2, a3], axis=-1)                                   # [..., 4] u32
    return np.stack([(words >> (8 * b)) & 255 for b in range(4)], axis=-1).reshape(words.shape[:-1] + (16,)).astype(np.uint8)


def dequant_q2_k(blk):
    """blk uint8 [..., nblk, 84] -> float32 [..., nblk, 256] (natural order)."""
    sc = blk[..., 0:16].astype(jnp.int32)
    d, dmin = _f16_at(blk, 80), _f16_at(blk, 82)
    outs = []
    for g, v in enumerate(_kq_layout(blk[..., 16:80])):                          # g = 4 n + s
        for h in range(2):                                                        # l < 16 / l >= 16
            b = sc[..., 2 * g + h]
            dl, ml = d * (b & 15).astype(jnp.float32), dmin * (b >> 4).astype(jnp.float32)
            outs.append(dl[..., None] * v[..., 16 * h:16 * (h + 1)].astype(jnp.float32) - ml[..., None])
    return jnp.concatenate(outs, axis=-1)


def dequant_q3_k(blk):
    """blk uint8 [..., nblk, 110] -> float32 [..., nblk, 256] (natural order)."""
    hm = blk[..., 0:32].astype(jnp.int32)
    sc = jnp.asarray(q3k_scales(np.asarray(blk[..., 96:108]))).astype(jnp.int32) - 32
    d = _f16_at(blk, 108)
    outs = []
    for g, v in enumerate(_kq_layout(blk[..., 32:96])):
        hbit = (hm >> g) & 1                                                      # bit 4 n + s of hmask byte l
        val = v - 4 * (1 - hbit)
        for h in range(2):
            dl = d * sc[..., 2 * g + h].astype(jnp.float32)
            outs.append(dl[..., None] * val[..., 16 * h:16 * (h + 1)].astype(jnp.float32))
    return jnp.concatenate(outs, axis=-1)


DEQUANT = {"IQ2_S": dequant_iq2_s, "IQ3_S": dequant_iq3_s, "IQ4_XS": dequant_iq4_xs,
           "IQ2_XS": dequant_iq2_xs, "IQ3_XXS": dequant_iq3_xxs, "Q2_K": dequant_q2_k, "Q3_K": dequant_q3_k}


def dequant_rows(raw, qtype, ne0):
    """raw uint8 [nbytes] of a row-major GGUF tensor slice with ne0 weights per row -> float32 [rows, ne0]."""
    bb = BLOCK_BYTES[qtype]
    blk = jnp.asarray(raw).reshape(-1, ne0 // QK_K, bb)
    return DEQUANT[qtype](blk).reshape(blk.shape[0], ne0)


# ------------------------------------------------------------------------------------ position-major ("pm") dequant
# TPU pads the last two dims of every materialised array to (8,128) tiles, so [..., groups, 8] int8 intermediates cost
# 16x their size. Instead each codebook *position* is produced as its own lane-dense [..., n_groups] array and the row is
# emitted in position-major order: flat index = pos * G + group, G = groups per row (all blocks). The contraction dim of a
# matmul does not care about order as long as the activation is permuted the same way: x_pm = x[..., perm_pm(...)].
GROUP_W = {"IQ2_S": 8, "IQ2_XS": 8, "IQ3_S": 4, "IQ3_XXS": 4, "IQ4_XS": 2}   # positions per group (IQ4_XS: 2 nibbles)


def perm_pm(qtype, n_blocks):
    """Natural index of each position-major slot: x_pm = x[..., perm]; out_natural = out_pm[..., inv]."""
    w = GROUP_W[qtype]
    G = n_blocks * QK_K // w
    slots = np.arange(n_blocks * QK_K)
    if qtype == "IQ4_XS":   # weight d = sub*32 + pos*16 + j ; group g = sub*16 + j ; pos in {0 (low nibble), 1 (high)}
        pos, g = slots // G, slots % G
        return (32 * (g // 16) + 16 * pos + (g % 16)).astype(np.int32)
    pos, g = slots // G, slots % G                         # weight d = g*w + pos
    return (g * w + pos).astype(np.int32)


def _tree_pos(name, idx):
    """Codebook lookup returning one lane-dense int8 array per position: list of w arrays shaped like idx."""
    lut = _TREES.setdefault(name, _TreeLUT(name))
    wi = idx >> 5
    sh = (idx & 31).astype(jnp.uint32)
    bits = [((wi >> k) & 1).astype(bool) for k in range(int(np.log2(lut.n_words)))]
    code = jnp.zeros(idx.shape, jnp.uint32)
    for b in range(lut.nbits):
        nodes = [jnp.uint32(int(x)) for x in lut.words[b]]
        for bit in bits:
            nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
        code = code | (((nodes[0] >> sh) & 1) << b)
    mask = (1 << lut.lb) - 1
    outs = []
    for k in range(lut.w):
        lv = ((code >> (lut.lb * k)) & mask).astype(jnp.int32)
        v = jnp.full(lv.shape, int(lut.levels[0]), jnp.int8)
        for i in range(1, len(lut.levels)):
            v = jnp.where(lv == i, jnp.int8(int(lut.levels[i])), v)
        outs.append(v)
    return outs


def _rowflat(a):   # [..., nblk, m] -> [..., nblk*m]
    return a.reshape(a.shape[:-2] + (a.shape[-2] * a.shape[-1],))


def dequant_pm_iq2_s(blk):
    """blk uint8 [..., nblk, 82] -> float32 [..., 8 * nblk*32] position-major."""
    d = _f16(blk[..., 0:2])                                          # [..., nblk]
    qs = _rowflat(blk[..., 2:34].astype(jnp.int32))                  # [..., G]  G = nblk*32
    qh = blk[..., 66:74]
    hi = _rowflat(jnp.stack([(qh >> (2 * k)) & 3 for k in range(4)], -1).reshape(qh.shape[:-1] + (32,)).astype(jnp.int32))
    idx = qs | (hi << 8)                                             # [..., G]
    sc = _rowflat(_nibbles(blk[..., 74:82]).astype(jnp.float32))     # [..., nblk*16]  one per 2 groups
    scale = _rowflat(jnp.repeat(d, 16, axis=-1).reshape(d.shape + (16,))) * (0.5 + sc) * 0.25   # [..., nblk*16]
    scale = jnp.repeat(scale, 2, axis=-1)                            # [..., G]
    signs = _rowflat(blk[..., 34:66])                                # [..., G] one byte per group
    vals = _tree_pos("IQ2_S", idx)
    outs = [scale * v.astype(jnp.float32) * (1.0 - 2.0 * ((signs >> k) & 1).astype(jnp.float32)) for k, v in enumerate(vals)]
    return jnp.concatenate(outs, -1)


def dequant_pm_iq3_s(blk):
    """blk uint8 [..., nblk, 110] -> float32 [..., 4 * nblk*64] position-major."""
    d = _f16(blk[..., 0:2])
    qs = _rowflat(blk[..., 2:66].astype(jnp.int32))                  # [..., G]  G = nblk*64
    qh = _rowflat(_bits8(blk[..., 66:74]).reshape(blk.shape[:-1] + (64,)).astype(jnp.int32))
    idx = qs | (qh << 8)
    sc = _rowflat(_nibbles(blk[..., 106:110]).astype(jnp.float32))   # [..., nblk*8] one per 8 groups
    scale = _rowflat(jnp.repeat(d, 8, axis=-1).reshape(d.shape + (8,))) * (1.0 + 2.0 * sc)
    scale = jnp.repeat(scale, 8, axis=-1)                            # [..., G]
    sb = _rowflat(blk[..., 74:106])                                  # [..., nblk*32] sign bytes: byte j = groups 2j, 2j+1
    s_even = jnp.repeat(sb, 2, axis=-1)                              # [..., G] byte for each group
    shift = jnp.tile(jnp.array([0, 4], jnp.uint8), sb.shape[-1])     # group parity picks the nibble
    vals = _tree_pos("IQ3_S", idx)
    outs = [scale * v.astype(jnp.float32) * (1.0 - 2.0 * ((s_even >> (shift + k)) & 1).astype(jnp.float32))
            for k, v in enumerate(vals)]
    return jnp.concatenate(outs, -1)


def dequant_pm_iq4_xs(blk):
    """blk uint8 [..., nblk, 136] -> float32 [..., 2 * nblk*128] position-major (pos = nibble)."""
    d = _f16(blk[..., 0:2])
    sh = _u16(blk[..., 2:4])[..., 0]
    sl = _nibbles(blk[..., 4:8]).astype(jnp.int32)                   # [..., nblk, 8]
    shb = jnp.stack([(sh >> (2 * k)) & 3 for k in range(8)], -1).astype(jnp.int32)
    scales = ((sl | (shb << 4)) - 32).astype(jnp.float32) * d[..., None]          # [..., nblk, 8] per 32 weights
    scale = _rowflat(jnp.repeat(scales, 16, axis=-1))                # [..., nblk*128] per group (16 groups per sub-block)
    qs = _rowflat(blk[..., 8:136])                                   # [..., G] G = nblk*128
    lo = _kv16(qs & 0x0F)
    hi = _kv16(qs >> 4)
    return jnp.concatenate([scale * lo, scale * hi], -1)


def _kv16(nib):
    """16-entry IQ4_NL value table without a gather (XLA's generic gather ran at 0.11 G lookups/s = 74 ms per
    IQ4_XS layer): 4-bit nibble -> float32 via a 4-level select tree (15 selects)."""
    kv = [float(v) for v in KVALUES_IQ4NL]
    bits = [((nib >> k) & 1).astype(bool) for k in range(4)]
    nodes = [jnp.float32(v) for v in kv]
    for bit in bits:
        nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
    return nodes[0]


DEQUANT_PM = {"IQ2_S": dequant_pm_iq2_s, "IQ3_S": dequant_pm_iq3_s, "IQ4_XS": dequant_pm_iq4_xs}


def pm_permute(qtype, x):
    """x[..., n] natural order -> position-major order along the last axis, via reshapes/transposes only
    (XLA's generic gather costs ~2.5 ms per call on v5e, a transpose is free). Equals jnp.take(x, perm_pm(...), -1)."""
    n = x.shape[-1]
    lead = x.shape[:-1]
    if qtype == "IQ4_XS":                      # d = sub*32 + pos*16 + j  ->  [pos, sub, j]
        y = x.reshape(lead + (n // 32, 2, 16))
        return jnp.swapaxes(y, -3, -2).reshape(lead + (n,))
    w = GROUP_W[qtype]                          # d = g*w + pos  ->  [pos, g]
    y = x.reshape(lead + (n // w, w))
    return jnp.swapaxes(y, -2, -1).reshape(lead + (n,))
'''
FILES["glm53/model.py"] = r'''"""Glm5Next text model in pure JAX (functional, params = nested dict of arrays).

Reference: transformers `modeling_glm5_next.py` (v5.16). Every block is written so that
prefill (full sequence) and single-token decode share the same math; caches are explicit
pytrees threaded through the functions.

Parameter layout (all Linear weights stored as [in, out], i.e. y = x @ W):
  embed [V, D], norm [D], lm_head [D, V]
  layers[i]:
    ln1 [D], ln2 [D]
    hc_attn / hc_ffn: fn [hc*D, (2+hc)*hc], base [(2+hc)*hc], scale [3]
    attn (KDA):  q,k,v [D, H*hd]; conv_q, conv_k, conv_v [H*hd, K]; f_a [D, hd]; f_b [hd, H*hd];
                 dt_bias [H*hd]; A_log [H]; b [D, H]; g_a [D, hd]; g_b [hd, H*hd]; o_norm [hd]; o [H*hd, D]
    attn (MLA):  q_a [D, qr]; q_a_norm [qr]; q_b [qr, H*qk]; kv_a [D, kvr]; kv_a_norm [kvr];
                 kv_b [kvr, H*(qk+v)]; o [H*v, D]; indexer {...} (unused in v0)
    mlp (dense): gate [D, I], up [D, I], down [I, D]
    mlp (moe):   router_w [D, E], router_bias [E], gate_up [E, D, 2*mi], down [E, mi, D], shared {gate, up, down}
"""
from __future__ import annotations

import dataclasses
import functools
from typing import Any

import jax
import numpy as np
import jax.numpy as jnp
from jax import lax

Params = dict[str, Any]


@dataclasses.dataclass(frozen=True)
class Cfg:
    hidden: int
    vocab: int
    n_layers: int
    layer_types: tuple[str, ...]      # "linear_attention" | "deepseek_sparse_attention"
    mlp_types: tuple[str, ...]        # "dense" | "sparse"
    eps: float
    # KDA
    kda_heads: int
    kda_hd: int
    conv_k: int
    gate_lb: float | None
    # MLA (NoPE)
    n_heads: int
    q_lora: int
    kv_lora: int
    qk_nope: int
    v_hd: int
    # indexer
    idx_heads: int
    idx_hd: int
    idx_topk: int
    idx_kpool: int
    # mHC
    hc: int
    hc_iters: int
    hc_eps: float
    # MLP / MoE
    inter: int
    moe_inter: int
    n_exp: int
    topk: int
    n_shared: int
    scaling: float
    norm_topk: bool
    swiglu_limit: float
    # compute dtype for matmuls (fp32 for CPU tests, bf16 on TPU)
    dtype: Any = jnp.float32
    # tensor-parallel reduction applied after every row-parallel matmul (None = single device)
    tp_reduce: Any = None
    tp_axis: Any = None               # shard_map axis name (collectives inside the model)
    seq_shard: int = 1                # MLA/indexer caches sharded over this many chips (interleaved by k-pool)
    q_block: int = 128                # queries per attention block (bounds the per-query temporaries)
    union_pools: int = 512            # per-block key-dedup budget (local pools); over budget -> per-query gathers; 0 = always per-query
    per_query_max: int = 4            # query blocks of up to this many queries skip the dedup (decode, short verifies)
    topk_local: int = 128             # first-stage local top-k per chip (exact: falls back to the full K when a chip's list was cut short)
    cache_cap: int = 0                # set inside mla_attention: global cache capacity
    cache_q8: bool = False            # MLA latent cache stored as int8 rows + a per-token f32 scale ("cs"): half the HBM
    pool_gather: int = 0              # per-query key gathers by whole k-pools ((kp, kvr) windows, one per selected pool,
                                      # instead of a row per token; the scales as 128-lane rows + a lane select) from this
                                      # many queries per block (1 = decode too; 0 = never). OFF: in a micro-benchmark a
                                      # T = 4 verify's layer gather fell from 16.7 + 50.8 us (token rows + element-
                                      # gathered scales) to 11.4 + 9.7 (j148), but inside the programs XLA re-lays the
                                      # whole per-layer cache out for the window gather ({2,0,1:T(1,128)}, a copy per
                                      # call: j150 2026-10-05, plain B = 3 +0.6 ms per step, the verify's gain eaten)
    select_buckets: tuple = ()        # indexer selection (sharded cache): the local top-k over the smallest of these
                                      # pool counts holding every pool visible from the block's positions, chosen by a
                                      # lax.switch. OFF: the conditional per MLA layer cost more than the sorts it saved
                                      # (v6e 2026-10-05, j143: one-stream DFlash step +0.5 ms with (512, 2048))

    def reduce(self, y):
        return y if self.tp_reduce is None else self.tp_reduce(y)

    @classmethod
    def from_hf(cls, c, dtype=jnp.float32) -> "Cfg":
        return cls(
            hidden=c.hidden_size, vocab=c.vocab_size, n_layers=c.num_hidden_layers,
            layer_types=tuple(c.layer_types), mlp_types=tuple(c.mlp_layer_types), eps=c.rms_norm_eps,
            kda_heads=c.linear_num_heads, kda_hd=c.linear_head_dim, conv_k=c.linear_conv_kernel_dim,
            gate_lb=c.linear_lower_bound,
            n_heads=c.num_attention_heads, q_lora=c.q_lora_rank, kv_lora=c.kv_lora_rank,
            qk_nope=c.qk_nope_head_dim, v_hd=c.v_head_dim,
            idx_heads=c.index_n_heads, idx_hd=c.index_head_dim, idx_topk=c.index_topk, idx_kpool=c.index_kpool,
            hc=c.hc_mult, hc_iters=c.hc_sinkhorn_iters, hc_eps=c.hc_eps,
            inter=c.intermediate_size, moe_inter=c.moe_intermediate_size, n_exp=c.n_routed_experts,
            topk=c.num_experts_per_tok, n_shared=c.n_shared_experts, scaling=c.routed_scaling_factor,
            norm_topk=c.norm_topk_prob, swiglu_limit=c.swiglu_limit, dtype=dtype,
        )


# ----------------------------------------------------------------------------- basics
def rmsnorm(x, w, eps):
    xf = x.astype(jnp.float32)
    xf = xf * lax.rsqrt(jnp.mean(xf * xf, -1, keepdims=True) + eps)
    return (w.astype(jnp.float32) * xf).astype(x.dtype)


def unweighted_rmsnorm_f32(x, eps):
    return x * lax.rsqrt(jnp.mean(x * x, -1, keepdims=True) + eps)


def l2norm(x, eps=1e-6):
    return x / jnp.sqrt(jnp.sum(x * x, -1, keepdims=True) + eps)


def silu(x):
    return x * jax.nn.sigmoid(x)


def swiglu_clamped(gate, up, limit):
    gate = jnp.minimum(gate, limit)
    up = jnp.clip(up, -limit, limit)
    return silu(gate) * up


def mlp_partial(p, x, cfg: Cfg):
    """Column/row-parallel MLP without the final TP reduction."""
    return swiglu_clamped(x @ p["gate"], x @ p["up"], cfg.swiglu_limit) @ p["down"]


def mlp(p, x, cfg: Cfg):
    return cfg.reduce(mlp_partial(p, x, cfg))


# ----------------------------------------------------------------------------- mHC
def hc_pre(p, streams, cfg: Cfg):
    """streams [B,T,hc,D] -> (post [B,T,hc], comb [B,T,hc,hc], collapsed [B,T,D])."""
    hc = cfg.hc
    B, T = streams.shape[:2]
    flat = unweighted_rmsnorm_f32(streams.reshape(B, T, -1).astype(jnp.float32), cfg.eps)
    if "fn" in p:
        mix = flat @ p["fn"].astype(jnp.float32)                   # [B,T,(2+hc)*hc]
    else:                                                          # stored transposed for the kernels (mhc_params)
        mix = jnp.einsum("btk,mk->btm", flat, p["fnT"].astype(jnp.float32))
    pre_w, post_w, comb_w = mix[..., :hc], mix[..., hc:2 * hc], mix[..., 2 * hc:]
    base, scale = p["base"].astype(jnp.float32), p["scale"].astype(jnp.float32)
    pre = jax.nn.sigmoid(pre_w * scale[0] + base[:hc]) + cfg.hc_eps
    post = 2.0 * jax.nn.sigmoid(post_w * scale[1] + base[hc:2 * hc])
    comb_logits = comb_w.reshape(B, T, hc, hc) * scale[2] + base[2 * hc:].reshape(hc, hc)
    comb = jax.nn.softmax(comb_logits, -1) + cfg.hc_eps
    # Sinkhorn on an hc x hc matrix. Row/column sums are written as explicit slice adds instead of reductions so the
    # whole (2*iters-1)-step chain stays one elementwise XLA fusion: with reductions every step is its own tiny kernel
    # (~40 launches per layer per token ≈ 10 ms/token on v5e at decode).
    rsum = lambda c: sum(c[..., :, j:j + 1] for j in range(hc))          # [B,T,hc,1]
    csum = lambda c: sum(c[..., j:j + 1, :] for j in range(hc))          # [B,T,1,hc]
    comb = comb / (csum(comb) + cfg.hc_eps)
    for _ in range(cfg.hc_iters - 1):
        comb = comb / (rsum(comb) + cfg.hc_eps)
        comb = comb / (csum(comb) + cfg.hc_eps)
    collapsed = jnp.einsum("bth,bthd->btd", pre, streams.astype(jnp.float32)).astype(streams.dtype)
    return post, comb, collapsed


def mhc_params(p):
    """An mHC site's params as the kernels read them: fn [hc*D, M] stored transposed, fnT [M, hc*D] (a [., 24] minor
    dim would be lane-padded in VMEM; the HBM bytes are the same). hc_pre's XLA path reads either."""
    if "fn" not in p:
        return p
    fn = p["fn"]
    fnT = np.ascontiguousarray(np.asarray(fn).T) if isinstance(fn, np.ndarray) else jnp.transpose(fn)
    return {"fnT": fnT, **{k: v for k, v in p.items() if k != "fn"}}


def hc_post(post, comb, y, residual):
    """new_streams[h] = post[h] * y + sum_h' comb[h', h] * residual[h']  (matches torch code)."""
    dt = residual.dtype
    return post.astype(dt)[..., None] * y[..., None, :] + jnp.einsum("bthk,bthd->btkd", comb.astype(dt), residual)


# ----------------------------------------------------------------------------- KDA
def causal_conv1d(x, w, state=None, length=None, out_dtype=None):
    """Depthwise causal conv over time. x [B,T,C], w [C,K], state [B,K-1,C] or None (zeros).
    `length` (traced int, <= T): only the first `length` tokens are real; the new state is taken from them.
    Returns (y [B,T,C] in x's dtype (or out_dtype), new_state [B,K-1,C])."""
    B, T, C = x.shape
    K = w.shape[-1]
    if state is None:
        state = jnp.zeros((B, K - 1, C), x.dtype)
    xp = jnp.concatenate([state.astype(x.dtype), x], axis=1)        # [B, T+K-1, C]
    y = jnp.zeros((B, T, C), jnp.float32)
    for j in range(K):
        y = y + xp[:, j:j + T].astype(jnp.float32) * w[:, j].astype(jnp.float32)
    if length is None:
        new_state = xp[:, -(K - 1):]
    else:
        new_state = lax.dynamic_slice_in_dim(xp, length, K - 1, axis=1)   # inputs at t in [length-K+1, length-1]
    return y.astype(out_dtype or x.dtype), new_state


def kda_gates(p, x, cfg: Cfg):
    """Forget gate g [B,T,H,hd] (log-decay, <= 0) and beta [B,T,H] in fp32."""
    B, T, _ = x.shape
    f = (x @ p["f_a"]) @ p["f_b"]
    g = f.astype(jnp.float32) + p["dt_bias"].astype(jnp.float32)
    g = g.reshape(B, T, cfg.kda_heads, cfg.kda_hd)
    decay_rate = jnp.exp(p["A_log"].astype(jnp.float32))[None, None, :, None]
    if cfg.gate_lb is not None:
        g = cfg.gate_lb * jax.nn.sigmoid(decay_rate * g)
    else:
        g = -decay_rate * jax.nn.softplus(g)
    beta = jax.nn.sigmoid((x @ p["b"]).astype(jnp.float32))
    return g, beta


def kda_chunked(q, k, v, g, beta, state0=None, chunk=64):
    """Chunked KDA (HF `chunk_kimi_delta_attention`). q,k,v [B,T,H,d]; g [B,T,H,d]; beta [B,T,H].
    Returns out [B,T,H,d] (fp32) and final state [B,H,dk,dv] (fp32).
    Memory is O(one chunk) in the per-channel decay `exp(g_i - g_j)` [c,c,d]: it is computed per chunk inside a
    `lax.map` (intra-chunk term) and again inside the state scan, never for all chunks at once (that tensor was
    262 KB/token/chip and capped a single prefill call at ~2048 tokens)."""
    B, T, H, dk = k.shape
    dv = v.shape[-1]
    f32 = jnp.float32
    q, k, v, g = [jnp.swapaxes(t.astype(f32), 1, 2) for t in (q, k, v, g)]   # [B,H,T,*]
    beta = jnp.swapaxes(beta.astype(f32), 1, 2)                              # [B,H,T]
    q = l2norm(q) * (dk ** -0.5)
    k = l2norm(k)
    pad = (-T) % chunk
    Tp = T + pad
    if pad:
        q, k, v, g = [jnp.pad(t, ((0, 0), (0, 0), (0, pad), (0, 0))) for t in (q, k, v, g)]
        beta = jnp.pad(beta, ((0, 0), (0, 0), (0, pad)))
    n = Tp // chunk
    q, k, v, g = [t.reshape(B, H, n, chunk, -1) for t in (q, k, v, g)]
    beta = beta.reshape(B, H, n, chunk)
    k_beta = k * beta[..., None]
    v_beta = v * beta[..., None]
    g = jnp.cumsum(g, axis=-2)                                             # per-channel cumsum in chunk
    ii = jnp.arange(chunk)
    strict_lower = ii[:, None] > ii[None, :]
    lower_incl = ii[:, None] >= ii[None, :]
    eye = jnp.eye(chunk, dtype=f32)

    def chunk_decay(g_i):
        # decay[i,j,d] = exp(g_i[d] - g_j[d]) for i >= j (<= 1); the upper triangle is never used, clamp it so no
        # inf/NaN is ever produced (g spans down to -320 per chunk)
        return jnp.exp(jnp.minimum(g_i[..., :, None, :] - g_i[..., None, :, :], 0.0))   # [B,H,c,c,d]

    def intra(xs):
        k_i, g_i, kb_i = xs                                                  # [B,H,c,d]
        a = -jnp.einsum("bhid,bhjd,bhijd->bhij", kb_i, k_i, chunk_decay(g_i))
        return jnp.where(strict_lower, a, 0.0)

    per_chunk = lambda *ts: tuple(jnp.moveaxis(t, 2, 0) for t in ts)      # [n,B,H,c,*]
    attn = jnp.moveaxis(lax.map(intra, per_chunk(k, g, k_beta)), 0, 2)     # [B,H,n,c,c]
    # HF: forward substitution computing (I - attn)^{-1} with unit diagonal, i.e. inverse of (I + M)
    L = eye - attn                                                          # lower unit-triangular
    attn = jnp.vectorize(lambda m: jax.scipy.linalg.solve_triangular(m, eye, lower=True, unit_diagonal=True),
                         signature="(c,c)->(c,c)")(L)
    u = attn @ v_beta                                                       # [B,H,n,c,dv]
    w = attn @ (k_beta * jnp.exp(g))                                        # [B,H,n,c,dk]
    S0 = jnp.zeros((B, H, dk, dv), f32) if state0 is None else state0.astype(f32)

    def step(S, xs):
        q_i, k_i, g_i, u_i, w_i = xs                                          # leading dims [B,H,...]
        attn_inter = (q_i * jnp.exp(g_i)) @ S
        attn_intra = jnp.einsum("bhid,bhjd,bhijd->bhij", q_i, k_i, chunk_decay(g_i))
        attn_intra = jnp.where(lower_incl, attn_intra, 0.0)
        v_new = u_i - w_i @ S
        out = attn_inter + attn_intra @ v_new
        g_last = g_i[..., -1:, :]
        S = S * jnp.exp(g_last).swapaxes(-1, -2) + jnp.swapaxes(k_i * jnp.exp(g_last - g_i), -1, -2) @ v_new
        return S, out

    S, out = lax.scan(step, S0, per_chunk(q, k, g, u, w))
    out = jnp.moveaxis(out, 0, 2).reshape(B, H, Tp, dv)[:, :, :T]
    return jnp.swapaxes(out, 1, 2), S


KDA_UNROLL = 0             # kda_recurrent unrolls its token loop for 2..KDA_UNROLL tokens instead of a lax.scan (a
                           # while loop: 0.19 ms per one-stream DFlash step on the v6e). OFF: unrolled, the step's
                           # device time ROSE 13.16 -> 13.63 ms (j150 2026-10-05: loop fusions +0.25, all-reduces
                           # +0.15, the while gone) — the scan body is the better-fused form. Results agree to an ulp.


def kda_state_step(S, k_i, v_i, g_i, b_i):
    """The KDA state update of one token (f32; k_i l2-normed): S [B,H,dk,dv], k/v/g [B,H,d], b [B,H]."""
    S = S * jnp.exp(g_i)[..., None]
    kv_mem = jnp.einsum("bhkv,bhk->bhv", S, k_i)
    delta = (v_i - kv_mem) * b_i[..., None]
    return S + k_i[..., None] * delta[..., None, :]


def kda_recurrent(q, k, v, g, beta, state, hist=False):
    """One (or a few) tokens, recurrent form. Shapes as kda_chunked; state [B,H,dk,dv] fp32.
    hist=True also returns the state after every token ([T,B,H,dk,dv]; speculative-decoding rollback); hist="kin"
    instead returns every token's state-update inputs [T,B,H,4,d] f32 (k, v, g, beta broadcast over d: `kda_replay`
    recomputes the state after any prefix from them, 4/dv of the states' bytes)."""
    f32 = jnp.float32
    B, T, H, dk = k.shape
    q = l2norm(q.astype(f32)) * (dk ** -0.5)
    k = l2norm(k.astype(f32))
    v, g, beta = v.astype(f32), g.astype(f32), beta.astype(f32)
    keep = hist is True

    def step(S, xs):
        q_i, k_i, v_i, g_i, b_i = xs                                          # [B,H,d] / [B,H]
        S = kda_state_step(S, k_i, v_i, g_i, b_i)
        o = jnp.einsum("bhkv,bhk->bhv", S, q_i)
        return S, ((o, S) if keep else o)

    xs = tuple(jnp.moveaxis(t, 1, 0) for t in (q, k, v, g, beta))
    if 2 <= T <= KDA_UNROLL:              # a verify of a few tokens: straight-line code, no while loop (T = 1: XLA
        # inlines the one-iteration scan already; its bits are the plain decode fingerprint's)
        S, outs = state.astype(f32), []
        for t in range(T):
            S, o = step(S, tuple(x[t] for x in xs))
            outs.append(o)
        out = jax.tree.map(lambda *a: jnp.stack(a), *outs) if T > 1 else jax.tree.map(lambda a: a[None], outs[0])
    else:
        S, out = lax.scan(step, state.astype(f32), xs)
    if keep:
        out, states = out
        return jnp.moveaxis(out, 0, 1), S, states
    if hist == "kin":
        kin = jnp.stack([k, v, g, jnp.broadcast_to(beta[..., None], k.shape)], axis=3)     # [B,T,H,4,d]
        return jnp.moveaxis(out, 0, 1), S, jnp.moveaxis(kin, 1, 0)
    return jnp.moveaxis(out, 0, 1), S


def kda_replay(state, kin, n_keep):
    """The KDA state after the first n_keep (traced, 0..T) tokens of a verify, from the state before it [B,H,dk,dv]
    and the tokens' update inputs kin [T,B,H,4,d] (`kda_recurrent(hist="kin")`): the same step, unrolled and masked."""
    S = state.astype(jnp.float32)
    for t in range(kin.shape[0]):
        S = jnp.where(t < n_keep, kda_state_step(S, kin[t, :, :, 0], kin[t, :, :, 1], kin[t, :, :, 2],
                                                 kin[t, :, :, 3, 0]), S)
    return S


def kda_project(p, x, cfg: Cfg):
    """Position-free, batched part of a KDA layer: x [B,T,D] (input-normed) -> dict(qkv [B,T,3C] before the conv,
    conv_w [3C,K], g [B,T,H,hd], beta [B,T,H], gate [B,T,H,hd])."""
    B, T, _ = x.shape
    H, hd = cfg.kda_heads, cfg.kda_hd
    qkv = jnp.concatenate([x @ p["q"], x @ p["k"], x @ p["v"]], -1)            # [B,T,3C]
    conv_w = jnp.concatenate([p["conv_q"], p["conv_k"], p["conv_v"]], 0)        # [3C,K]
    g, beta = kda_gates(p, x, cfg)
    gate = ((x @ p["g_a"]) @ p["g_b"]).reshape(B, T, H, hd)
    return {"qkv": qkv, "conv_w": conv_w, "g": g, "beta": beta, "gate": gate}


def kda_core(proj, cfg: Cfg, cache=None, use_recurrent=False, length=None, hist=False):
    """Conv + KDA recurrence of `proj` (kda_project; any B in lockstep, ONE cache) -> (core [B,T,H,hd] f32,
    new_cache). cache: None or {"conv": [B,K-1,3C], "state": [B,H,dk,dv]}. `length`: number of real tokens (rest
    is padding: beta=0, g=0 so the state ignores them). hist=True (recurrent, T > 1: speculative verify) adds
    "state_hist" [T+1,B,H,dk,dv] and "conv_hist" [T+1,B,K-1,3C] = the cache after 0..T of the tokens (entry 0 =
    before them), for `rollback_states`; hist="kin": "kin_hist" [T,B,H,4,d] (the tokens' state-update inputs) instead
    of "state_hist", and the returned "state" is the one BEFORE the tokens (the rollback replays the accepted ones:
    `kda_replay`)."""
    qkv_raw, conv_w, g, beta = proj["qkv"], proj["conv_w"], proj["g"], proj["beta"]
    B, T, _ = qkv_raw.shape
    H, hd = cfg.kda_heads, cfg.kda_hd
    conv_state = None if cache is None else cache["conv"]
    qkv, new_conv = causal_conv1d(qkv_raw, conv_w, conv_state, length)
    qkv = silu(qkv)
    C = H * hd
    q, k, v = qkv[..., :C], qkv[..., C:2 * C], qkv[..., 2 * C:]
    q, k, v = [t.reshape(B, T, H, hd) for t in (q, k, v)]
    if length is not None:
        valid = jnp.arange(T)[None, :] < length                                    # [1,T]
        beta = jnp.where(valid[..., None], beta, 0.0)
        g = jnp.where(valid[..., None, None], g, 0.0)
    state = None if cache is None else cache["state"]
    extra = {}
    if use_recurrent:
        assert state is not None
        if hist:
            core, new_state, states = kda_recurrent(q, k, v, g, beta, state, hist=hist)
            K = conv_w.shape[-1]
            full = jnp.concatenate([conv_state.astype(qkv_raw.dtype), qkv_raw], axis=1)       # [B,K-1+T,3C]
            # entry t = the conv state after t tokens; entry 0 (before the block) lets a rollback to n_keep = 0 undo
            # a verify that was dispatched ahead of its tokens (`ResidentLayerEngine.dflash_step`)
            extra = {"conv_hist": jnp.stack([full[:, t:t + K - 1] for t in range(T + 1)])}
            if hist == "kin":
                extra["kin_hist"] = states                     # [T,B,H,4,d]: `kda_replay` masks by n_keep
                new_state = state                              # (replayed to the accepted prefix by the rollback)
            else:
                extra["state_hist"] = jnp.concatenate([state.astype(states.dtype)[None], states])   # [T+1,B,H,dk,dv]
        else:
            core, new_state = kda_recurrent(q, k, v, g, beta, state)
    else:
        core, new_state = kda_chunked(q, k, v, g, beta, state)
    return core, {"conv": new_conv, "state": new_state, **extra}


def kda_out(p, core, gate, cfg: Cfg, dtype):
    """Gated RMSNorm over hd (fp32, sigmoid gate) + output projection (TP-reduced): -> y [B,T,D]."""
    B, T, H, hd = gate.shape
    o = core.astype(jnp.float32)
    o = o * lax.rsqrt(jnp.mean(o * o, -1, keepdims=True) + cfg.eps) * p["o_norm"].astype(jnp.float32)
    o = o * jax.nn.sigmoid(gate.astype(jnp.float32))
    return cfg.reduce(o.astype(dtype).reshape(B, T, H * hd) @ p["o"])


KDA_KERNEL = False         # the KDA layers' recurrent core of decode / short-verify calls (at most KDA_MAX_TOKENS tokens
KDA_MAX_TOKENS = 8         # per row; silu, the norms, the gates, the delta-rule recurrence, the gated output norm) as
KDA_INTERPRET = False      # glm53.pallas_kda kernels on the TPU (elsewhere only with KDA_INTERPRET); the conv taps and
                           # the projections stay XLA. Read at trace time. hist=True (the state form) stays XLA.
                           # OFF: v6e 2026-10-05 (j143 / j150) the kernel costs ~9 us per layer call (96 lane-to-
                           # sublane transposes of [128, 128] per call: three per token and head) against ~10 us for
                           # XLA's chain (the while loop + one fusion), and the conv / gate glue left outside grew:
                           # one-stream DFlash step 13.42 -> 13.75 ms, plain B = 3 14.06 -> 14.98. Its exact-f32
                           # contractions also moved the logits by ~2 % of the max (XLA's einsums round to bf16 there).


def kda_kernel_active(n_tokens):
    """Whether the KDA core kernel serves a recurrent call of n_tokens per row on this backend."""
    return bool(KDA_KERNEL) and n_tokens <= KDA_MAX_TOKENS and (KDA_INTERPRET or jax.default_backend() == "tpu")


def _kda_attention_kernel(p, x, cfg: Cfg, cache, hist):
    """kda_attention's recurrent step as the glm53.pallas_kda kernel: x [B,T,D], cache {"conv": [B,K-1,3C], "state":
    [B,H,dk,dv]} -> (y [B,T,D], new cache (+ "kin_hist" [T,B,H,4,d] and "conv_hist" [T+1,B,K-1,3C] with hist)). The
    conv runs in f32 into the kernel (XLA's fused chain keeps f32 there too)."""
    from glm53 import pallas_kda as PK
    B, T, _ = x.shape
    H, hd = cfg.kda_heads, cfg.kda_hd
    qkv = jnp.concatenate([x @ p["q"], x @ p["k"], x @ p["v"]], -1)                                 # [B,T,3C]
    conv_w = jnp.concatenate([p["conv_q"], p["conv_k"], p["conv_v"]], 0)
    xc, new_conv = causal_conv1d(qkv, conv_w, cache["conv"], None, out_dtype=jnp.float32)
    f = (x @ p["f_a"]) @ p["f_b"]
    gate = (x @ p["g_a"]) @ p["g_b"]
    y, new_state, kin = PK.kda_core_out(xc, f, x @ p["b"], gate, cache["state"], p["dt_bias"], p["A_log"], p["o_norm"],
                                        hd=hd, eps=cfg.eps, gate_lb=cfg.gate_lb, hist=bool(hist), interpret=KDA_INTERPRET)
    new_cache = {"conv": new_conv, "state": new_state}
    if hist:
        K = conv_w.shape[-1]
        full = jnp.concatenate([cache["conv"].astype(qkv.dtype), qkv], axis=1)
        new_cache["kin_hist"] = jnp.moveaxis(kin, 1, 0)
        new_cache["conv_hist"] = jnp.stack([full[:, t:t + K - 1] for t in range(T + 1)])
    return cfg.reduce(y.astype(x.dtype).reshape(B, T, H * hd) @ p["o"]), new_cache


def kda_attention(p, x, cfg: Cfg, cache=None, use_recurrent=False, length=None, hist=False):
    """x [B,T,D] (already input-normed). cache: None or {"conv": [B,K-1,3C], "state": [B,H,dk,dv]} (one cache, all
    rows in lockstep). Returns (y [B,T,D], new_cache); see kda_core for `length` / `hist`."""
    if use_recurrent and cache is not None and length is None and hist is not True and kda_kernel_active(x.shape[1]):
        return _kda_attention_kernel(p, x, cfg, cache, hist)
    proj = kda_project(p, x, cfg)
    core, new_cache = kda_core(proj, cfg, cache, use_recurrent, length, hist)
    return kda_out(p, core, proj["gate"], cfg, x.dtype), new_cache


def kda_attention_rows(p, x, cfg: Cfg, caches, hist=False):
    """Recurrent (decode) step of B INDEPENDENT streams: x [B,T,D], caches[b] = row b's own cache (batch dim 1).
    The projections and the output run batched, the conv/recurrence per row -> (y [B,T,D], [new cache per row]);
    hist=True (T > 1: a speculative verify of every row) adds each row's per-token histories (kda_core)."""
    if hist is not True and kda_kernel_active(x.shape[1]):              # the kernel per row (its own cache / state)
        ys, new = [], []
        for b, c in enumerate(caches):
            y, nc = _kda_attention_kernel(p, x[b:b + 1], cfg, c, hist)
            ys.append(y); new.append(nc)
        return jnp.concatenate(ys, 0), new
    proj = kda_project(p, x, cfg)
    cores, new = [], []
    for b, c in enumerate(caches):
        pr = {k: (v if k == "conv_w" else v[b:b + 1]) for k, v in proj.items()}
        core, nc = kda_core(pr, cfg, c, True, None, hist)
        cores.append(core); new.append(nc)
    return kda_out(p, jnp.concatenate(cores, 0), proj["gate"], cfg, x.dtype), new


def layernorm(x, w, b, eps):
    xf = x.astype(jnp.float32)
    mu = xf.mean(-1, keepdims=True)
    var = jnp.mean((xf - mu) ** 2, -1, keepdims=True)
    return ((xf - mu) * lax.rsqrt(var + eps) * w.astype(jnp.float32) + b.astype(jnp.float32)).astype(x.dtype)


# ----------------------------------------------------------------------------- DSA indexer (k-pool)
# Cache layouts. Replicated (cfg.seq_shard == 1): slot t of the MLA/indexer caches holds position t.
# Sequence-sharded over n = cfg.seq_shard chips, interleaved by k-pool: position t belongs to pool p = t // kp, lives
# on chip p % n at local slot (p // n) * kp + t % kp. Every chip therefore holds an equal share of complete pools, a
# prefill piece scatters T/n rows per chip, and the indexer scores its own pools with all heads.
def _pos_to_slot(t, kp, n):
    """Global position(s) -> (owner chip, local slot)."""
    pool = t // kp
    return pool % n, (pool // n) * kp + t % kp


def _slot_to_pos(s, kp, n, chip):
    return ((s // kp) * n + chip) * kp + s % kp


def cache_write(cache, new, pos0, cfg: Cfg, chip=None, valid=None):
    """Write new [B,T,R] at global positions pos0.. into cache [B,S_local,R] (see layouts above). `valid` [T] bool
    (optional) keeps the cache untouched for rows that are False."""
    new = new.astype(cache.dtype)
    n, kp = cfg.seq_shard, cfg.idx_kpool
    if n == 1 and valid is None:
        return lax.dynamic_update_slice_in_dim(cache, new, pos0, axis=1)
    B, T = new.shape[:2]
    S_local = cache.shape[1]
    if n == 1:
        mine, slot = jnp.ones((T,), bool), pos0 + jnp.arange(T)
    else:
        owner, slot = _pos_to_slot(pos0 + jnp.arange(T), kp, n)
        mine = owner == chip
    if valid is not None:
        mine = mine & valid
    if T <= 2:                                     # decode: read-select-write the row(s), no scatter
        for t in range(T):
            cur = lax.dynamic_slice_in_dim(cache, slot[t], 1, axis=1)
            cache = lax.dynamic_update_slice_in_dim(cache, jnp.where(mine[t], new[:, t:t + 1], cur), slot[t], axis=1)
        return cache
    # one scatter: rows of other chips (or invalid rows) go to distinct out-of-bounds slots, which XLA drops
    slot = jnp.where(mine, slot, S_local + jnp.arange(T))
    return cache.at[:, slot].set(new, mode="drop", unique_indices=True)


def mla_cache_shapes(cfg: Cfg, B, S, has_indexer, n=1):
    """Global cache shapes of one MLA layer for capacity S (per chip: the sequence/pool axis divided by n):
    "c" latent [B,S,kvr] (+ "cs" [B,S] per-token scales when cfg.cache_q8: "c" then holds the int8 rows packed four
    to an int32 word, [B,S,kvr/4] — `pack_latent`); with an indexer "pk"
    pooled indexer keys [B,S/kp,ihd] (one row per complete k-pool) and the raw keys/gates of the incomplete tail
    pool "tk"/"tg" [B,kp,ihd] (replicated, tiny). Dtypes: `cache_dtype`."""
    kp = cfg.idx_kpool
    out = {"c": (B, S // n, cfg.kv_lora // 4 if cfg.cache_q8 else cfg.kv_lora)}
    if cfg.cache_q8:
        out["cs"] = (B, S // n)
    if has_indexer:
        out.update(pk=(B, S // kp // n, cfg.idx_hd), tk=(B, kp, cfg.idx_hd), tg=(B, kp, cfg.idx_hd))
    return out


ROW_KEYS = ("c", "cs", "pk")         # positional cache entries (rows per token / per pool); the rest are tail buffers


def cache_dtype(cfg: Cfg, key, dtype):
    """dtype of one MLA cache entry: int8 latents (packed in int32 words) + f32 scales under cache_q8, else the model
    dtype."""
    if cfg.cache_q8 and key == "c":
        return jnp.int32
    if key == "cs":
        return jnp.float32
    return dtype


def pack_latent(q):
    """int8 rows [..., kvr] -> int32 words [..., kvr/4] (the q8 latent cache's storage): word j holds the bytes of
    elements j, j + kvr/4, j + kvr/2, j + 3 kvr/4 (byte k = element j + k kvr/4). An int8 cache has (32,128) tiles
    while XLA writes and reads single rows in (8,128) tiles, so every program that touched it copied the whole
    per-layer cache (16.8 MB per chip at 262k) in and out (v6e xprof 2026-10-04: ~1.7 ms per DFlash step); 32-bit
    words have one natural tiling, and the strided byte order lets `unpack_latent` rebuild a row from four 128-lane
    blocks with shifts (a bitcast to int8 needed a relayout of every gathered row). Values are unchanged."""
    w = q.shape[-1] // 4
    b = q.reshape(q.shape[:-1] + (4, w)).astype(jnp.int32) & 0xFF
    return b[..., 0, :] | (b[..., 1, :] << 8) | (b[..., 2, :] << 16) | (b[..., 3, :] << 24)


def unpack_latent(w):
    """int32 words [..., kvr/4] (`pack_latent`) -> the int8 values [..., kvr], as int32 (sign-extended bytes)."""
    return jnp.concatenate([(w << (24 - 8 * k)) >> 24 for k in range(4)], -1)


def latent_dequant(q, s, dtype):
    """int8 latent rows [..., kvr] (or their int32 packing, `pack_latent`) x per-row scales [...] -> dtype."""
    if q.dtype == jnp.int32:
        q = unpack_latent(q)
    return (q.astype(jnp.float32) * s.astype(jnp.float32)[..., None]).astype(dtype)


def gather_scalars(a, idx):
    """a [B, S] (per-row latent scales), idx [B, Tq, W] in-range int32 -> a[b, idx] [B, Tq, W]: the 128-value rows
    holding them (a row gather, like the latent rows' own) and a lane select — exact. An element gather of scalars is
    slower on the TPU (v6e 2026-10-04, jobs/j127, gather + dequant per MLA layer: Tq = 1 34 -> 27 us, Tq = 4 67 -> 35
    us in isolation; but in the T = 4 verify the 4 MB of rows per layer made the step's other fusions slower: used
    for single-query steps only)."""
    B, S = a.shape
    if S % 128:
        return jnp.take_along_axis(a[:, None], idx, axis=2, mode="promise_in_bounds")
    rows = jnp.take_along_axis(a.reshape(B, 1, S // 128, 128), (idx // 128)[..., None], axis=2,
                               mode="promise_in_bounds")                                     # [B, Tq, W, 128]
    lane = jnp.arange(128, dtype=idx.dtype) == (idx % 128)[..., None]
    return jnp.sum(jnp.where(lane, rows, jnp.zeros((), rows.dtype)), -1)


def write_latent(cache, new_c, pos0, cfg: Cfg, chip=None):
    """New latents [B,T,kvr] into the cache at pos0 -> the updated {"c"} (or {"c", "cs"}: per-token absmax int8)."""
    if not cfg.cache_q8:
        return {"c": cache_write(cache["c"], new_c, pos0, cfg, chip)}
    x = new_c.astype(jnp.float32)
    s = jnp.maximum(jnp.max(jnp.abs(x), axis=-1), 1e-12) / 127.0                  # [B,T]
    q = jnp.clip(jnp.round(x / s[..., None]), -127, 127).astype(jnp.int8)
    return {"c": cache_write(cache["c"], pack_latent(q), pos0, cfg, chip), "cs": cache_write(cache["cs"], s, pos0, cfg, chip)}


def pool_new_keys(ix, tail_k, tail_g, new_k, new_g, pos0, cfg: Cfg, length, hist=False):
    """k-pool compression of the NEW tokens (the first `length` of new_k/new_g, at positions pos0..) together with
    the raw keys of the incomplete pool they may extend (tail_k/tail_g [B,kp,ihd]: row i = position (pos0//kp)*kp + i
    for i < pos0 % kp). -> (pooled [B,NP,ihd] f32, complete [NP] bool, pool0 = global id of pooled[0], new tails)."""
    B, T, ihd = new_k.shape
    kp = cfg.idx_kpool
    f32 = jnp.float32
    a = pos0 % kp                                                             # tokens of the tail pool already seen
    NP = (T + 2 * kp - 2) // kp + 1                                           # pools covering a + T rows (+1 spare)
    L = NP * kp

    def fill(tail, new):
        buf = jnp.zeros((B, L, ihd), f32).at[:, :kp].set(tail.astype(f32))
        return lax.dynamic_update_slice_in_dim(buf, new.astype(f32), a, axis=1)

    kb, gb = fill(tail_k, new_k), fill(tail_g, new_g)
    present = jnp.arange(L) < a + length                                      # [L] rows that hold real tokens
    logits = jnp.where(present[None, :, None], gb + jnp.tile(ix["ape"].astype(f32), (L // kp, 1))[None], -jnp.inf)
    probs = jnp.nan_to_num(jax.nn.softmax(logits.reshape(B, NP, kp, ihd), axis=2))
    pooled = (probs * kb.reshape(B, NP, kp, ihd)).sum(2)                      # [B,NP,ihd]
    complete = present.reshape(NP, kp).all(-1)
    start = ((a + length) // kp) * kp                                         # first row of the (incomplete) last pool
    tail_k2 = lax.dynamic_slice_in_dim(kb, start, kp, axis=1)
    tail_g2 = lax.dynamic_slice_in_dim(gb, start, kp, axis=1)
    if hist:                                                                  # tails after 0..T of the tokens
        st = [((a + t) // kp) * kp for t in range(T + 1)]                     # (entry 0 = before them)
        th_k = jnp.stack([lax.dynamic_slice_in_dim(kb, s_, kp, axis=1) for s_ in st])
        th_g = jnp.stack([lax.dynamic_slice_in_dim(gb, s_, kp, axis=1) for s_ in st])
        return pooled, complete, pos0 // kp, tail_k2, tail_g2, (th_k, th_g)
    return pooled, complete, pos0 // kp, tail_k2, tail_g2


def indexer_pool(ix, keys, gates, cfg: Cfg):
    """k-pool compression of ALL cached keys (query independent). keys/gates [B,S_local,ihd] ->
    (pool_keys [B,P_local,ihd] f32, pool_valid [P_local] bool). Pools straddling the end of a replicated cache
    whose size is not a multiple of kp are invalid (model-level calls with cap=None)."""
    B, S, ihd = keys.shape
    kp = cfg.idx_kpool
    P = -(-S // kp)
    pad = P * kp - S
    kpad = jnp.pad(keys, ((0, 0), (0, pad), (0, 0))).reshape(B, P, kp, ihd).astype(jnp.float32)
    gpad = jnp.pad(gates, ((0, 0), (0, pad), (0, 0))).reshape(B, P, kp, ihd).astype(jnp.float32)
    tok_valid = jnp.arange(P * kp).reshape(P, kp) < S                             # [P,kp]
    logits = jnp.where(tok_valid[None, :, :, None], gpad + ix["ape"].astype(jnp.float32)[None, None], -jnp.inf)
    probs = jnp.nan_to_num(jax.nn.softmax(logits, axis=2))
    return (probs * kpad).sum(2), tok_valid.all(-1)


def top_k(x, k, chunk=None):
    """`lax.top_k` over the last axis with a chunked first stage: the top k of every chunk of `chunk` values (default
    max(4k, 256)), then the top k of those candidates. Same values and ids as lax.top_k (the k largest are among their
    chunks' k largest; ties keep lower indices first), but much cheaper on the TPU, where a top_k over a long axis is
    one full sort (v6e 2026-10-04, jobs/j122: 8192 values k = 512: 37 -> 18 us; 4 x 8192 k = 128: 37 -> 30 us;
    3 x 19360 k = 16: 462 -> 26 us). Float inputs (the last chunk is padded with -inf)."""
    n = x.shape[-1]
    c = chunk or max(4 * k, 256)
    if n <= 2 * c or 2 * k > c:
        return lax.top_k(x, k)
    pad = (-n) % c
    if pad:
        x = jnp.pad(x, [(0, 0)] * (x.ndim - 1) + [(0, pad)], constant_values=-jnp.inf)
    xc = x.reshape(x.shape[:-1] + (-1, c))
    v, i = lax.top_k(xc, k)                                                           # [..., n/c, k]
    i = i + (jnp.arange(xc.shape[-2], dtype=i.dtype) * c)[:, None]
    v, i = v.reshape(x.shape[:-1] + (-1,)), i.reshape(x.shape[:-1] + (-1,))
    v, j = lax.top_k(v, k)
    return v, jnp.take_along_axis(i, j, -1)


def indexer_select(scores, pool_valid, qpos, cfg: Cfg, chip=None):
    """scores [B,Tq,P_local] f32 (this chip's pools); qpos [Tq] (or [B,Tq]) global query positions -> (pools [B,Tq,K] global pool
    ids, valid [B,Tq,K]): the top-K pools per query (two-stage top-k across chips when the cache is sharded: local
    top-K, all-gather the candidates, global top-K)."""
    B, Tq, P_local = scores.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    lo = jnp.finfo(jnp.float32).min
    j = jnp.arange(P_local)
    gpool = j * n + (chip if n > 1 else 0)                                            # global pool ids [P_local]
    pool_end = gpool * kp + kp - 1
    if qpos.ndim == 1:
        visible = (pool_end[None, :] <= qpos[:, None]) & pool_valid[None, :]          # [Tq,P_local]
        scores = jnp.where(visible[None], scores, lo)
    else:                                                                             # qpos [B,Tq]: positions per row
        scores = jnp.where((pool_end <= qpos[..., None]) & pool_valid, scores, lo)
    K = min(cfg.idx_topk // kp, P_local)
    if n == 1:
        vals, sel = top_k(scores, K)
        return sel, vals > lo
    off = chip

    def two_stage(k_local, L=P_local):
        """local top-k_local -> all-gather -> global top-K; also returns whether any chip's list was cut short of
        the global threshold (then the result may be inexact). L = the local pools that can be visible (static):
        only the first L are scored (the rest sit beyond every query's position)."""
        if k_local >= L:                           # every candidate pool goes to the global stage: no local sort
            v, j = scores[..., :L], jnp.broadcast_to(jnp.arange(L, dtype=jnp.int32), (B, Tq, L))
        else:
            v, j = top_k(scores[..., :L], k_local)                                    # [B,Tq,k_local]
        j = j * n + off
        both = lax.all_gather(jnp.stack([lax.bitcast_convert_type(v, jnp.int32), j]), cfg.tp_axis, axis=-1,
                              tiled=True)                                             # one collective, not two
        va, ca = lax.bitcast_convert_type(both[0], jnp.float32), both[1]              # [B,Tq,n*k_local]
        kk = min(cfg.idx_topk // kp, n * k_local)
        neg, ca = lax.sort((-va, ca), dimension=-1, num_keys=1)                       # descending by score, ids carried
        vs, ss = -neg[..., :kk], ca[..., :kk]
        thr = vs[..., -1:]                                                            # weakest globally selected score
        cut = (k_local < K) & jnp.any(v[..., -1:] > thr)                              # a chip had more above the bar
        cut = lax.pmax(cut.astype(jnp.int32), cfg.tp_axis) > 0
        return vs, ss, cut

    k_fast = min(cfg.topk_local, K) if Tq > 1 else K       # a single query (decode) gains nothing from the fallback dance

    def select(L):
        vs, ss, cut = two_stage(k_fast, L)
        if k_fast < K:
            vs, ss = lax.cond(cut, lambda _: two_stage(K, L)[:2], lambda _: (vs, ss), None)
        return vs, ss

    # the local pools a query at position q can see are the first (q + 1) // (n * kp) + 1 (interleaved layout): the
    # sorts run over that bucket of the P_local scores, not all of them (a 262k cache scores 8,192 pools per chip at
    # any context; the TPU-v5e paper's "blocks sized for the max context" cost). Every chip takes the same branch.
    Ls = [L for L in cfg.select_buckets if L < P_local] + [P_local]
    if len(Ls) == 1:
        vs, ss = select(P_local)
    else:
        nv = (jnp.max(qpos) + 1) // (n * kp) + 1
        idx = sum((nv > L).astype(jnp.int32) for L in Ls[:-1])
        vs, ss = lax.switch(idx, [functools.partial(select, L) for L in Ls])
    return ss, vs > lo


def select_tokens(pools, valid, qpos, cfg: Cfg):
    """Selected pools -> global token indices [B,Tq,W] int32 (-1 = invalid), W = idx_topk + kp - 1 padded to a
    multiple of 16 (a tile-aligned slot count keeps the gathered keys free of relayout copies): the pools' tokens plus
    the incomplete tail pool of the visible prefix (positions 0..qpos)."""
    B, Tq, K = pools.shape
    kp, S = cfg.idx_kpool, cfg.cache_cap
    tok = (pools[..., None] * kp + jnp.arange(kp)).reshape(B, Tq, K * kp)
    tok = jnp.where(jnp.repeat(valid, kp, axis=-1), tok, -1)
    tok = jnp.pad(tok, ((0, 0), (0, 0), (0, cfg.idx_topk - K * kp)), constant_values=-1)
    vis_count = qpos + 1
    tail_count = vis_count % kp
    tail_start = vis_count - tail_count
    off = jnp.arange(kp - 1)
    tail = tail_start[:, None] + off[None, :]                                         # [Tq,kp-1]
    tail_ok = (off[None, :] < tail_count[:, None]) & (tail < S)
    tail = jnp.where(tail_ok, tail, -1)
    tok = jnp.concatenate([tok, jnp.broadcast_to(tail[None], (B, Tq, kp - 1))], -1)
    W = tok.shape[-1]
    align = 32 if cfg.cache_q8 else 16                                              # int8 rows tile by 32 sublanes
    return jnp.pad(tok, ((0, 0), (0, 0), (0, (-W) % align)), constant_values=-1).astype(jnp.int32)


def union_pools(pools, valid, qpos, cfg: Cfg, chip=None):
    """Per-block key deduplication. pools/valid [B,Tq,K] global pool ids -> (member [B,Tq,U] bool, upool [U] local
    pool ids of the block's union (sentinel P_local past the end), count): the union over the block's queries of the
    selected pools that live on this chip (plus each query's incomplete tail pool), compacted into U slots."""
    B, Tq, K = pools.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    P_local = -(-(cfg.cache_cap // n) // kp)                                           # pools per chip (last may be partial)
    U = min(cfg.union_pools, P_local)
    tail_pool = jnp.broadcast_to((qpos // kp)[None, :, None], (B, Tq, 1))
    tail_on = jnp.broadcast_to(((qpos + 1) % kp != 0)[None, :, None], (B, Tq, 1))
    cand = jnp.concatenate([pools, tail_pool], -1)                                    # [B,Tq,K+1] global pools
    ok = jnp.concatenate([valid, tail_on], -1)
    if n > 1:
        ok = ok & (cand % n == chip)
        cand = cand // n
    local = jnp.where(ok, cand, P_local)                                              # sentinel = P_local
    srt = jnp.sort(local.reshape(-1))
    first = jnp.concatenate([jnp.ones((1,), jnp.bool_), srt[1:] != srt[:-1]]) & (srt < P_local)
    count = first.sum()
    # the U smallest unique pool ids (top-k of the negated keys; a scatter compaction costs more on TPU)
    keys = jnp.where(first, -srt, -P_local)
    if keys.shape[0] < U:
        keys = jnp.pad(keys, (0, U - keys.shape[0]), constant_values=-P_local)
    upool = -lax.top_k(keys, U)[0]                                                    # ascending union, sentinel P_local
    member = (local[..., :, None] == upool[None, None, None, :]).any(-2) & (upool < P_local)[None, None]   # [B,Tq,U]
    return member, upool, count, U


def _attend(q, keys, mask, cfg: Cfg, per_query_keys):
    """Softmax attention of q [B,Tq,Hq,r] over keys ([B,Tq,W,r] if per_query_keys else [B,S,r]) restricted to
    `mask` ([B,Tq,W] or [Tq,S]). With a sharded cache Hq = all heads and every chip holds a slice of the keys:
    partial softmax with the global max, then reduce-scatter over heads -> [B,Tq,H_local,r]."""
    lo = jnp.finfo(jnp.float32).min
    scale = cfg.qk_nope ** -0.5
    if per_query_keys:
        s = jnp.einsum("bthr,btwr->bhtw", q, keys).astype(jnp.float32) * scale
        s = jnp.where(mask[:, None], s, lo)
    else:
        s = jnp.einsum("bthr,bsr->bhts", q, keys).astype(jnp.float32) * scale
        mask = mask if mask.ndim == 3 else mask[None]                                 # [B or 1, Tq, S]
        s = jnp.where(mask[:, None], s, lo)
    pv = "bhtw,btwr->bthr" if per_query_keys else "bhts,bsr->bthr"
    if cfg.seq_shard == 1:
        probs = jax.nn.softmax(s, -1).astype(keys.dtype)
        return jnp.einsum(pv, probs, keys)
    m = lax.pmax(s.max(-1), cfg.tp_axis)                                              # [B,Hq,Tq] global max
    e = jnp.exp(s - m[..., None])                                                     # 0 where masked / no local keys
    l = jnp.sum(e, -1)                                                                # [B,Hq,Tq]
    o = jnp.einsum(pv, e.astype(keys.dtype), keys, preferred_element_type=jnp.float32)   # [B,Tq,Hq,r]
    o = lax.psum_scatter(o, cfg.tp_axis, scatter_dimension=2, tiled=True)             # [B,Tq,H_local,r]
    l = lax.psum_scatter(l, cfg.tp_axis, scatter_dimension=1, tiled=True)             # [B,H_local,Tq]
    return (o / jnp.swapaxes(l, 1, 2)[..., None]).astype(keys.dtype)


# ----------------------------------------------------------------------------- MLA (NoPE)
def mla_project(p, x, cfg: Cfg):
    """Position-free, batched part of an MLA layer: x [B,T,D] -> dict(q_lat [B,T,H,kvr] absorbed queries, new_c
    [B,T,kvr] new latents; with an indexer also nk / ng [B,T,ihd] (its keys / gates), qi [B,T,IH,ihd] f32 and
    wts [B,T,IH] f32 (its queries / head weights))."""
    B, T, _ = x.shape
    H, qk, dv, kvr = cfg.n_heads, cfg.qk_nope, cfg.v_hd, cfg.kv_lora
    # T > 1 (a verify): XLA folds the reshape into a dot batched over heads and copies the dequantized q_b / wq_b into
    # that layout on every call; a barrier keeps the plain 2-D product (v6e 2026-10-04: DFlash step -0.17 ms)
    bar = (lambda a: a) if T == 1 else lax.optimization_barrier
    q_resid = rmsnorm(x @ p["q_a"], p["q_a_norm"], cfg.eps)
    q = bar(q_resid @ p["q_b"]).reshape(B, T, H, qk)
    new_c = rmsnorm(x @ p["kv_a"], p["kv_a_norm"], cfg.eps)                         # [B,T,kvr]
    kv_b = p["kv_b"].reshape(kvr, H, qk + dv)
    Wk = kv_b[:, :, :qk]
    out = {"q_lat": jnp.einsum("bthq,rhq->bthr", q, Wk), "new_c": new_c}            # absorbed query [B,T,H,kvr]
    if "indexer" in p:
        ix = p["indexer"]
        IH, ihd = cfg.idx_heads, cfg.idx_hd
        out["nk"] = layernorm(x @ ix["wk"], ix["k_norm_w"], ix["k_norm_b"], 1e-6)
        out["ng"] = x @ ix["gate"]
        out["qi"] = bar(q_resid @ ix["wq_b"]).reshape(B, T, IH, ihd).astype(jnp.float32)
        out["wts"] = (x @ ix["weights_proj"]).astype(jnp.float32) * (IH ** -0.5)     # [B,T,IH]
    return out


def _mla_write(p, proj, cfg: Cfg, cache, pos0, chip, length, hist):
    """`mla_core`'s cache update (cfg.cache_cap set) -> new cache: the new latents at pos0; with an indexer its pooled
    keys and tails (+ the tails after each token with hist)."""
    new_cache = write_latent(cache, proj["new_c"], pos0, cfg, chip)
    if "indexer" in p:
        ix = p["indexer"]
        res = pool_new_keys(ix, cache["tk"], cache["tg"], proj["nk"], proj["ng"], pos0, cfg, length, hist)
        pooled, complete, pool0, tk2, tg2 = res[:5]
        pcfg = dataclasses.replace(cfg, idx_kpool=1)                             # pool-level slots: pool % n, pool // n
        new_cache["pk"] = cache_write(cache["pk"], pooled, pool0, pcfg, chip, valid=complete)
        new_cache["tk"], new_cache["tg"] = tk2.astype(cache["tk"].dtype), tg2.astype(cache["tg"].dtype)
        if hist:
            new_cache["tk_hist"] = res[5][0].astype(cache["tk"].dtype)
            new_cache["tg_hist"] = res[5][1].astype(cache["tg"].dtype)
    return new_cache


def _indexer_scores(qi, wts, pool_keys, cfg: Cfg):
    """The indexer's score of every (query, local pool): qi [B,Tq,IH,ihd], wts [B,Tq,IH], pool_keys [B,P,ihd] f32
    -> [B,Tq,P]."""
    sc = jax.nn.relu(jnp.einsum("bthd,bpd->bthp", qi, pool_keys) * (cfg.idx_hd ** -0.5))
    return jnp.einsum("bth,bthp->btp", wts, sc)


def _per_query_keys(c_all, cs_all, pools, pvalid, qpos, cfg: Cfg, chip, dtype):
    """The latents of every query's selected tokens that live on this chip (cfg.cache_cap set): (keys [B,Tq,W,kvr],
    valid [B,Tq,W])."""
    n, kp = cfg.seq_shard, cfg.idx_kpool
    tok = select_tokens(pools, pvalid, qpos, cfg)                                  # [B,Tq,W] global positions
    valid = (tok >= 0) & (tok <= qpos[None, :, None])
    if n > 1:
        owner, slot = _pos_to_slot(jnp.maximum(tok, 0), kp, n)
        valid = valid & (owner == chip)
    else:
        slot = tok
    slot = jnp.where(valid, slot, 0)
    c_sel = jnp.take_along_axis(c_all[:, None], slot[..., None], axis=2, mode="promise_in_bounds")
    if cfg.cache_q8:
        # the scales [B,Tq,W]: 128-value rows + a lane select for one query (decode, `gather_scalars`); by pools for
        # a verify (`_token_scales`: the 4 x 2080 element gathers were 50.8 us per MLA layer, 4 x 520 pool rows 9.7)
        if slot.shape[1] == 1:
            s_sel = gather_scalars(cs_all, slot)
        else:
            s_sel = _token_scales(cs_all, pools, pvalid, qpos, cfg, chip, slot.shape[2])
        c_sel = latent_dequant(c_sel, s_sel, dtype)
    return c_sel, valid


def _token_scales(cs_all, pools, pvalid, qpos, cfg: Cfg, chip, W):
    """The q8 scales of `select_tokens`' slots, [B,Tq,W] in its order (the K pools' tokens pool-major, zeros up to
    idx_topk, the tail pool's first kp-1 tokens, zeros up to W), gathered by POOLS: `_pool_scales` over the K + 1
    pools reads the 128-value row holding each pool's kp consecutive scales and selects them by lane — 4x fewer
    gather rows than the tokens' scales one by one (v6e 2026-10-05, j148: 9.7 vs 50.8 us per MLA layer of a T = 4
    verify). Invalid or foreign pools read local pool 0 (their slots are masked by `valid`)."""
    B, Tq, K = pools.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    tail = jnp.broadcast_to((qpos // kp)[None, :, None], (B, Tq, 1))                   # the query's incomplete pool
    gpool = jnp.concatenate([pools, tail], -1)                                           # [B,Tq,K+1] global pool ids
    ok = jnp.concatenate([pvalid, jnp.ones((B, Tq, 1), bool)], -1)
    if n > 1:
        ok = ok & (gpool % n == chip)
        gpool = gpool // n
    sc = _pool_scales(cs_all, jnp.where(ok, gpool, 0), kp)                               # [B,Tq,K+1,kp]
    parts = [sc[:, :, :K].reshape(B, Tq, K * kp)]
    if K * kp < cfg.idx_topk:
        parts.append(jnp.zeros((B, Tq, cfg.idx_topk - K * kp), sc.dtype))
    parts.append(sc[:, :, K, :kp - 1])
    s = jnp.concatenate(parts, -1)
    return jnp.pad(s, ((0, 0), (0, 0), (0, W - s.shape[-1])))


POOL_GATHER_WINDOWS = True    # `_per_query_pool_keys` gathers (kp, kvr) windows (lax.gather) instead of rows of a
                              # pool-major reshape of the cache (read at trace time)


def _per_query_pool_keys(c_all, cs_all, pools, pvalid, qpos, cfg: Cfg, chip, dtype):
    """`_per_query_keys` with the latents gathered by whole k-pools: one gathered row of kp tokens per selected pool
    (plus the query's incomplete tail pool) instead of one row per token — the same keys and per-token validity, in
    pool-major order, from 4x fewer gather rows (a TPU gather pays per row, not per byte). The q8 scales come as the
    128-value rows holding each pool's kp consecutive scales plus a lane select (`gather_scalars`). -> (keys
    [B,Tq,W,kvr], valid [B,Tq,W]) with W = (K + 1) * kp padded to a multiple of 32."""
    B, Tq, K = pools.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    S_local, R = c_all.shape[1], c_all.shape[2]
    P_local = -(-S_local // kp)
    tail = (qpos // kp)[None, :, None]                                                # [1,Tq,1] the query's own pool
    tail_on = ((qpos + 1) % kp != 0)[None, :, None]                                   # incomplete: not in pk, its tokens <= qpos
    gpool = jnp.concatenate([pools, jnp.broadcast_to(tail, (B, Tq, 1))], -1)          # [B,Tq,K+1] global pool ids
    ok = jnp.concatenate([pvalid, jnp.broadcast_to(tail_on, (B, Tq, 1))], -1)
    if n > 1:
        ok = ok & (gpool % n == chip)
        local = gpool // n
    else:
        local = gpool
    local = jnp.where(ok, local, 0)
    tok = gpool[..., None] * kp + jnp.arange(kp, dtype=gpool.dtype)                   # [B,Tq,K+1,kp] global positions
    valid = (ok[..., None] & (tok <= qpos[None, :, None, None])).reshape(B, Tq, (K + 1) * kp)
    if POOL_GATHER_WINDOWS and S_local % kp == 0:
        # (kp, R) windows straight from the token-major cache: no pool-major reshape (a relayout copy of the whole
        # per-layer cache per call on the TPU)
        bidx = jnp.broadcast_to(jnp.arange(B, dtype=jnp.int32)[:, None, None], local.shape)
        start = jnp.stack([bidx, local * kp], -1)                                                    # [B,Tq,K+1,2]
        dn = lax.GatherDimensionNumbers(offset_dims=(3, 4), collapsed_slice_dims=(0,), start_index_map=(0, 1))
        rows = lax.gather(c_all, start, dn, slice_sizes=(1, kp, R), mode="promise_in_bounds")        # [B,Tq,K+1,kp,R]
        c_sel = rows.reshape(B, Tq, (K + 1) * kp, R)
        if cfg.cache_q8:                     # (a (1, kp) window gather of the scales is 6x slower than this, j148)
            c_sel = latent_dequant(c_sel, _pool_scales(cs_all, local, kp).reshape(B, Tq, (K + 1) * kp), dtype)
        W = c_sel.shape[2]
        pad = (-W) % 32
        if pad:
            c_sel = jnp.pad(c_sel, ((0, 0), (0, 0), (0, pad), (0, 0)))
            valid = jnp.pad(valid, ((0, 0), (0, 0), (0, pad)))
        return c_sel, valid
    c_pools = jnp.pad(c_all, ((0, 0), (0, P_local * kp - S_local), (0, 0))) if S_local % kp else c_all
    c_view = c_pools.reshape(B, P_local, kp * R)
    rows = jnp.take_along_axis(c_view[:, None], local[..., None], axis=2, mode="promise_in_bounds")   # [B,Tq,K+1,kp*R]
    c_sel = rows.reshape(B, Tq, (K + 1) * kp, R)
    if cfg.cache_q8:
        c_sel = latent_dequant(c_sel, _pool_scales(cs_all, local, kp).reshape(B, Tq, (K + 1) * kp), dtype)
    W = c_sel.shape[2]
    pad = (-W) % 32
    if pad:
        c_sel = jnp.pad(c_sel, ((0, 0), (0, 0), (0, pad), (0, 0)))
        valid = jnp.pad(valid, ((0, 0), (0, 0), (0, pad)))
    return c_sel, valid


def _pool_scales(cs_all, local, kp):
    """The kp consecutive per-token scales of every pool in `local` [B,Tq,P] from cs_all [B,S_local] -> [B,Tq,P,kp]:
    the 128-value rows holding them (a row gather) and a lane select per position (`gather_scalars`' way; v6e
    2026-10-05, j148: 9.7 us per MLA layer at 4 x 520 pools vs 50.8 for the element gather of the 4 x 2080 scalars)."""
    B, S_local = cs_all.shape
    Tq, P = local.shape[1:]
    if S_local % 128 or 128 % kp:
        slot = (local[..., None] * kp + jnp.arange(kp)).reshape(B, Tq, P * kp)
        return jnp.take_along_axis(cs_all[:, None], jnp.minimum(slot, S_local - 1), axis=2,
                                   mode="promise_in_bounds").reshape(B, Tq, P, kp)
    srows = jnp.take_along_axis(cs_all.reshape(B, 1, S_local // 128, 128), (local * kp // 128)[..., None], axis=2,
                                mode="promise_in_bounds")                                              # [B,Tq,P,128]
    lane = jnp.arange(128, dtype=jnp.int32)
    base = (local * kp % 128)[..., None]
    return jnp.stack([jnp.sum(jnp.where(lane == base + t, srows, jnp.zeros((), srows.dtype)), -1) for t in range(kp)], -1)


def _query_keys(c_all, cs_all, pools, pvalid, qpos, cfg: Cfg, chip, dtype):
    """The per-query key gather of a block: by pools (`_per_query_pool_keys`) from cfg.pool_gather queries, else by
    tokens (`_per_query_keys`)."""
    f = _per_query_pool_keys if cfg.pool_gather and pools.shape[1] >= cfg.pool_gather else _per_query_keys
    return f(c_all, cs_all, pools, pvalid, qpos, cfg, chip, dtype)


def _attend_rows(q, keys, masks, cfg: Cfg):
    """`_attend` with per-query keys for B rows whose keys stay separate (keys[b] [1,Tq,W,r], masks[b] [1,Tq,W]; no
    concatenation of the gathered latents): the scores and products per row, the cross-chip max and the two
    reduce-scatters once for all rows. Per row the same operations as `_attend`."""
    lo = jnp.finfo(jnp.float32).min
    scale = cfg.qk_nope ** -0.5
    pv = "bhtw,btwr->bthr"
    ss = [jnp.where(m[:, None], jnp.einsum("bthr,btwr->bhtw", q[b:b + 1], k).astype(jnp.float32) * scale, lo)
          for b, (k, m) in enumerate(zip(keys, masks))]
    if cfg.seq_shard == 1:
        return jnp.concatenate([jnp.einsum(pv, jax.nn.softmax(s, -1).astype(k.dtype), k) for s, k in zip(ss, keys)], 0)
    m = lax.pmax(jnp.concatenate([s.max(-1) for s in ss], 0), cfg.tp_axis)          # [B,Hq,Tq] global max
    es = [jnp.exp(s - m[b:b + 1, ..., None]) for b, s in enumerate(ss)]
    l = jnp.concatenate([jnp.sum(e, -1) for e in es], 0)                             # [B,Hq,Tq]
    o = jnp.concatenate([jnp.einsum(pv, e.astype(k.dtype), k, preferred_element_type=jnp.float32)
                         for e, k in zip(es, keys)], 0)                              # [B,Tq,Hq,r]
    o = lax.psum_scatter(o, cfg.tp_axis, scatter_dimension=2, tiled=True)
    l = lax.psum_scatter(l, cfg.tp_axis, scatter_dimension=1, tiled=True)
    return (o / jnp.swapaxes(l, 1, 2)[..., None]).astype(keys[0].dtype)


ROWS_SHARED_MIN = 3        # mla_core_rows shares the collectives from this many queries over all rows (B*T; 0 = never;
                           # read at trace). v6e 2026-10-04 (j135, same engine, shared vs per row): plain decode B=2
                           # 12.53 vs 12.19 ms, B=3 14.90 vs 15.67; DFlash step of 2 / 3 streams 22.86 / 32.5 vs
                           # 23.47 / 32.8


def mla_core_rows(p, proj, cfg: Cfg, caches, pos, sparse="auto", cap=None, hist=False):
    """`mla_core` for B INDEPENDENT rows (caches[b] = row b's own cache, batch 1, at position pos[b]: traced scalars)
    -> (o_lat [B,T,H_local,kvr], [new cache per row]). On the DSA path of decode and short verifies (sparse, at most
    per_query_max queries) every row updates and gathers from its own cache, but the query all-gather, the indexer's
    selection (top-k, the candidates' all-gather) and the softmax combine across the chips run once for all rows:
    latency-bound collectives and sorts (~10-20 us each, about six per row and MLA layer; v6e 2026-10-04). Otherwise
    (one row, fewer than `ROWS_SHARED_MIN` queries in all: two decode rows ran faster apart) `mla_core` per row."""
    B = len(caches)
    q_lat = proj["q_lat"]
    T = q_lat.shape[1]
    n = cfg.seq_shard
    S = caches[0]["c"].shape[1] * n
    use_sparse = sparse == "always" or (sparse == "auto" and S > cfg.idx_topk)
    if (B == 1 or not ROWS_SHARED_MIN or B * T < ROWS_SHARED_MIN
            or not (use_sparse and "indexer" in p and T <= min(cfg.per_query_max, cfg.q_block))):
        outs, new = [], []
        for b, (c, pb) in enumerate(zip(caches, pos)):
            o_b, nc = mla_core(p, {k: v[b:b + 1] for k, v in proj.items()}, cfg, c, pb, sparse, cap, None, hist)
            outs.append(o_b); new.append(nc)
        return jnp.concatenate(outs, 0), new
    chip = lax.axis_index(cfg.tp_axis) if n > 1 else None
    cfg = dataclasses.replace(cfg, cache_cap=S)
    dtype = proj["new_c"].dtype
    new, scores, qpos = [], [], []
    for b, (c, pb) in enumerate(zip(caches, pos)):
        pr = {k: v[b:b + 1] for k, v in proj.items()}
        nc = _mla_write(p, pr, cfg, c, pb, chip, T, hist)
        new.append(nc)
        scores.append(_indexer_scores(pr["qi"], pr["wts"], nc["pk"].astype(jnp.float32), cfg))
        qpos.append(pb + jnp.arange(T))
    qpos = jnp.stack(qpos)                                                          # [B,T]
    pools, pvalid = indexer_select(jnp.concatenate(scores, 0), jnp.ones((new[0]["pk"].shape[1],), bool), qpos, cfg,
                                   chip)                                            # [B,T,K]
    kv = [_query_keys(nc["c"], nc.get("cs"), pools[b:b + 1], pvalid[b:b + 1], qpos[b], cfg, chip, dtype)
          for b, nc in enumerate(new)]
    q = lax.all_gather(q_lat, cfg.tp_axis, axis=2, tiled=True) if n > 1 else q_lat   # all heads [B,T,H*n,kvr]
    return _attend_rows(q, [k for k, _ in kv], [v for _, v in kv], cfg), new


def mla_core(p, proj, cfg: Cfg, cache=None, pos0=0, sparse=None, cap=None, length=None, hist=False):
    """Position-dependent part of an MLA layer for ONE cache (all rows in lockstep at pos0): cache update, DSA
    selection and attention -> (o_lat [B,T,H_local,kvr], new_cache). Arguments as in `mla_attention`."""
    q_lat, new_c = proj["q_lat"], proj["new_c"]
    B, T, H, kvr = q_lat.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    chip = lax.axis_index(cfg.tp_axis) if n > 1 else None
    has_ix = "indexer" in p
    q8 = cfg.cache_q8
    if cache is None:
        S = T if cap is None else cap
        assert S % (kp * n) == 0 or n == 1, (S, kp, n)
        S_alloc = -(-S // kp) * kp if n == 1 else S                                # a whole number of pools
        cache = {k: jnp.zeros(sh, cache_dtype(cfg, k, new_c.dtype)) for k, sh in mla_cache_shapes(cfg, B, S_alloc, has_ix, n).items()}
        if n == 1 and S_alloc != S:
            cache["c"] = cache["c"][:, :S]
            if q8:
                cache["cs"] = cache["cs"][:, :S]
    S = cache["c"].shape[1] * n                                                    # global capacity
    cfg = dataclasses.replace(cfg, cache_cap=S)
    new_cache = _mla_write(p, proj, cfg, cache, pos0, chip, T if length is None else length, hist)
    c_all = new_cache["c"]                                                         # [B,S_local,kvr] (int8 under q8)
    cs_all = new_cache.get("cs")                                                   # [B,S_local] scales under q8
    dtype = new_c.dtype
    qpos = pos0 + jnp.arange(T)
    use_sparse = sparse == "always" or (sparse == "auto" and S > cfg.idx_topk)
    if use_sparse:
        pool_keys = new_cache["pk"].astype(jnp.float32)                            # [B,P_local,ihd] cached pooled keys
        pool_valid = jnp.ones((pool_keys.shape[1],), bool)                         # only complete pools are written;
        qi, wts = proj["qi"], proj["wts"]                                          # later ones are masked by position
    if n > 1 and not use_sparse:
        kpos = _slot_to_pos(jnp.arange(c_all.shape[1]), kp, n, chip)               # global position per local slot
    elif not use_sparse:
        kpos = jnp.arange(S)
    if use_sparse:                                                                 # pool-major view for the key-dedup path (one relayout per call)
        S_local = c_all.shape[1]
        P_local = -(-S_local // kp)
        c_pools = jnp.pad(c_all, ((0, 0), (0, P_local * kp - S_local), (0, 0))) if S_local % kp else c_all
        c_view = c_pools.reshape(B, P_local, kp * c_all.shape[-1])                   # (kvr/4 int32 words under q8)
        if q8:
            cs_pools = jnp.pad(cs_all, ((0, 0), (0, P_local * kp - S_local))) if S_local % kp else cs_all
            cs_view = cs_pools.reshape(B, P_local, kp)

    def per_query_attend(q_b, pools, pvalid, qpos_b):
        """Per-query gather of the selected keys that live on this chip (the pre-dedup path; decode + fallback)."""
        c_sel, valid = _query_keys(c_all, cs_all, pools, pvalid, qpos_b, cfg, chip, dtype)
        return _attend(q_b, c_sel, valid, cfg, True)

    def block(q_lat_b, qi_b, wts_b, qpos_b):
        """One block of queries -> o_lat [B,Tq,H_local,kvr]."""
        if n > 1:
            q_lat_b = lax.all_gather(q_lat_b, cfg.tp_axis, axis=2, tiled=True)      # all heads [B,Tq,H*n,kvr]
        if use_sparse:
            sc = _indexer_scores(qi_b, wts_b, pool_keys, cfg)                       # [B,Tq,P_local]
            pools, pvalid = indexer_select(sc, pool_valid, qpos_b, cfg, chip)      # [B,Tq,K] global pools
            if qpos_b.shape[0] <= cfg.per_query_max or cfg.union_pools == 0:       # decode / spec verify / dedup off
                return per_query_attend(q_lat_b, pools, pvalid, qpos_b)             # (no union bookkeeping, no conds)
            member, upool, count, U = union_pools(pools, pvalid, qpos_b, cfg, chip)
            over = count > U
            if n > 1:
                over = lax.pmax(over.astype(jnp.int32), cfg.tp_axis) > 0          # every chip takes the same branch
            def dedup(args):
                q_b, member, upool = args
                idx = jnp.minimum(upool, P_local - 1)
                keys = c_view.at[:, idx].get(mode="promise_in_bounds")                # [B,U,kp*kvr]
                keys = keys.reshape(B, U * kp, c_all.shape[-1])                     # the union's tokens, pool-major
                if q8:
                    keys = latent_dequant(keys, cs_view.at[:, idx].get(mode="promise_in_bounds").reshape(B, U * kp), dtype)
                slots = (upool[:, None] * kp + jnp.arange(kp)[None, :]).reshape(-1)
                kpos_u = _slot_to_pos(slots, kp, n, chip) if n > 1 else slots
                mask = jnp.repeat(member, kp, axis=-1) & (kpos_u[None, None, :] <= qpos_b[None, :, None])
                return _attend(q_b, keys, mask, cfg, False)

            per_query = lambda args: per_query_attend(args[0], pools, pvalid, qpos_b)
            return lax.cond(over, per_query, dedup, (q_lat_b, member, upool))
        c_dense = latent_dequant(c_all, cs_all, dtype) if q8 else c_all
        return _attend(q_lat_b, c_dense, kpos[None, :] <= qpos_b[:, None], cfg, False)

    Tq = min(cfg.q_block, T)
    nb = -(-T // Tq)
    if nb == 1:
        o_lat = block(q_lat, qi if use_sparse else None, wts if use_sparse else None, qpos)
    else:
        Tpad = nb * Tq - T
        padq = lambda a: jnp.pad(a, ((0, 0), (0, Tpad)) + ((0, 0),) * (a.ndim - 2)) if Tpad else a
        blk = lambda a: jnp.moveaxis(padq(a).reshape((B, nb, Tq) + a.shape[2:]), 1, 0)   # [nb,B,Tq,...]
        qpos_blk = jnp.pad(qpos, (0, Tpad)).reshape(nb, Tq)
        xs = (blk(q_lat), blk(qi) if use_sparse else None, blk(wts) if use_sparse else None, qpos_blk)
        o_lat = lax.map(lambda t: block(*t), xs)                                    # [nb,B,Tq,H,kvr]
        o_lat = jnp.moveaxis(o_lat, 0, 1).reshape(B, nb * Tq, H, kvr)[:, :T]
    return o_lat, new_cache


def mla_out(p, o_lat, cfg: Cfg):
    """o_lat [B,T,H_local,kvr] -> value up-projection + output projection (TP-reduced) [B,T,D]."""
    B, T, H, kvr = o_lat.shape
    qk, dv = cfg.qk_nope, cfg.v_hd
    Wv = p["kv_b"].reshape(kvr, cfg.n_heads, qk + dv)[:, :, qk:]
    o = jnp.einsum("bthr,rhv->bthv", o_lat, Wv).reshape(B, T, H * dv)
    return cfg.reduce(o @ p["o"])


def mla_attention(p, x, cfg: Cfg, cache=None, pos0=0, sparse=None, cap=None, length=None, hist=False):
    """Causal NoPE MLA with absorbed kv_b and a FIXED-CAPACITY latent cache (no recompiles as the sequence grows).
    x [B,T,D]. cache: None (prefill; allocates `cap` slots, or exactly T when cap is None) or
    {"c": [B,S_local,kvr], "pk": [B,P_local,ihd] pooled indexer keys, "tk"/"tg": [B,kp,ihd] tail}; new tokens are written at pos0 (may be
    traced). Slots beyond the current query position hold stale/zero data and are always masked by causality.
    sparse: None -> dense causal; "auto" -> DSA indexer when the (global) capacity > idx_topk; "always".
    Queries are processed in blocks of cfg.q_block (lax.map) so the per-query temporaries (indexer scores over all
    pools, the gathered top-k keys) never scale with the prompt length. cfg.seq_shard > 1: cache sharded over the
    chips (see the layout notes above), queries all-gathered over heads, partial softmax combined across chips.
    = mla_project (batched, position-free) -> mla_core (one cache, all rows at pos0) -> mla_out."""
    proj = mla_project(p, x, cfg)
    o_lat, new_cache = mla_core(p, proj, cfg, cache, pos0, sparse, cap, length, hist)
    return mla_out(p, o_lat, cfg), new_cache


def mla_attention_rows(p, x, cfg: Cfg, caches, pos, sparse="auto", cap=None, hist=False):
    """Decode step of B INDEPENDENT streams: x [B,T,D]; caches[b] = row b's own cache (batch dim 1) at position
    pos[b] (traced scalars). Projections and the output run batched, the cache updates and key gathers per row, the
    DSA selection's and the attention's collectives batched (`mla_core_rows`) -> (y [B,T,D], [new cache per row]);
    hist=True (T > 1, a verify per row): the pool-tail histories."""
    proj = mla_project(p, x, cfg)
    o_lat, new = mla_core_rows(p, proj, cfg, caches, pos, sparse, cap, hist)
    return mla_out(p, o_lat, cfg), new


# ----------------------------------------------------------------------------- MoE
def moe_route(p, x, cfg: Cfg):
    """x [N,D] -> (topk_idx [N,k] int32, topk_w [N,k] fp32). n_group == 1 (no group masking)."""
    logits = x.astype(jnp.float32) @ p["router_w"].astype(jnp.float32)
    scores = jax.nn.sigmoid(logits)
    choice = scores + p["router_bias"].astype(jnp.float32)
    _, idx = lax.top_k(choice, cfg.topk)
    # one-hot pick instead of take_along_axis: XLA's generic gather costs ~2.5 ms per call on TPU v5e
    w = jnp.sum(jax.nn.one_hot(idx, scores.shape[-1], dtype=scores.dtype) * scores[:, None, :], axis=-1)
    if cfg.norm_topk:
        w = w / (w.sum(-1, keepdims=True) + 1e-20)
    return idx.astype(jnp.int32), w * cfg.scaling


def moe_apply(x, gate_up_w, down_w, w, limit):
    """x [N,D]; gate_up_w [N,k,D,2mi]; down_w [N,k,mi,D]; w [N,k] -> [N,D]."""
    gu = jnp.einsum("nd,nkdm->nkm", x, gate_up_w)
    mi = gu.shape[-1] // 2
    h = swiglu_clamped(gu[..., :mi], gu[..., mi:], limit)
    y = jnp.einsum("nkm,nkmd->nkd", h, down_w)
    return jnp.einsum("nkd,nk->nd", y.astype(jnp.float32), w).astype(x.dtype)


def moe_apply_grouped(x, gu_w, dn_w, idx_local, w, limit, chunk=256):
    """Grouped experts: x [N,D]; gu_w [Eh,D,2mi]; dn_w [Eh,mi,D]; idx_local [N,k] (ids into Eh, -1 = none);
    w [N,k]. Every token is run through every gathered expert (dense, masked) — Eh/k× extra FLOPs but no per-token
    weight copies, so prefill fits HBM. Token chunks of `chunk` bound the [N,Eh,mi] intermediate."""
    N, D = x.shape
    Eh = gu_w.shape[0]
    mi = dn_w.shape[1]
    # routing weights as a dense [N, Eh] matrix
    onehot = jax.nn.one_hot(jnp.maximum(idx_local, 0), Eh, dtype=jnp.float32) * (idx_local >= 0)[..., None]
    rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32), onehot)            # [N,Eh]
    outs = []
    for s0 in range(0, N, chunk):
        xs, rws = x[s0:s0 + chunk], rw[s0:s0 + chunk]
        gu = jnp.einsum("nd,edm->nem", xs, gu_w)                              # [n,Eh,2mi]
        h = swiglu_clamped(gu[..., :mi], gu[..., mi:], limit)
        h = jnp.where(rws[..., None] > 0, h.astype(jnp.float32) * rws[..., None], 0.0).astype(x.dtype)   # fold routing weights in
        outs.append(jnp.einsum("nem,emd->nd", h, dn_w))
    return jnp.concatenate(outs, 0) if len(outs) > 1 else outs[0]


def moe_dense(p, x, cfg: Cfg, fetch=None, layer=0):
    """MoE block for x [B,T,D]. `fetch(layer, idx)` returns (gate_up_w [N,k,D,2mi], down_w [N,k,mi,D]) for
    idx [N,k]; default = gather from the on-device tables p["gate_up"] [E,D,2mi], p["down"] [E,mi,D].
    Under TP the expert/shared outputs are partial sums; one reduction covers both."""
    B, T, D = x.shape
    xf = x.reshape(-1, D)
    idx, w = moe_route(p, xf, cfg)
    if fetch is not None and hasattr(fetch, "apply"):      # resident quantized experts (glm53.resident)
        y = fetch.apply(p, xf, idx, w, layer, cfg.swiglu_limit, seq_tokens=T).astype(xf.dtype)
    else:
        if fetch is None:
            gu_w, dn_w = p["gate_up"][idx], p["down"][idx]
        else:
            gu_w, dn_w = fetch(layer, idx)
        y = moe_apply(xf, gu_w, dn_w, w, cfg.swiglu_limit)
    y = cfg.reduce(y + mlp_partial(p["shared"], xf, cfg))
    return y.reshape(B, T, D)


# ----------------------------------------------------------------------------- layer / model
MHC_KERNEL = "bf16"        # "bf16" / "f32": the mHC sites of decode and short-verify layer calls (at most
MHC_MAX_TOKENS = 16        # MHC_MAX_TOKENS = B*T tokens) run as glm53.pallas_mhc kernels with that precision for the
MHC_INTERPRET = False      # two small contractions, on the TPU (elsewhere only with MHC_INTERPRET); None = XLA's
                           # chain. Read at trace time. v6e 2026-10-04 (j139, j136b): plain decode B=1 10.47 -> 9.65 ms,
                           # B=3 15.28 -> 14.14, DFlash 3 streams 30.99 -> 29.68, one-stream DFlash device 14.29 -> 13.56


def mhc_active():
    """Whether the mHC kernels serve decode / short-verify layer calls on this backend."""
    return bool(MHC_KERNEL) and (MHC_INTERPRET or jax.default_backend() == "tpu")


def _mhc(streams):
    B, T = streams.shape[:2]
    return MHC_KERNEL if mhc_active() and B * T <= MHC_MAX_TOKENS else None


def decoder_layer(p, streams, cfg: Cfg, ltype, mtype, cache=None, pos0=0, use_recurrent=False, fetch=None,
                  sparse="auto", layer=0, cap=None, length=None, hist=False):
    kern = _mhc(streams)
    post, comb, h = _site_pre(p["hc_attn"], p["ln1"], streams, cfg, kern)
    if ltype == "linear_attention":
        h, new_cache = kda_attention(p["attn"], h, cfg, cache, use_recurrent, length, hist)
    else:
        h, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length, hist)
    streams, post, comb, h = _site_post_pre(post, comb, h, streams, p["hc_ffn"], p["ln2"], cfg, kern)
    if mtype == "sparse":
        h = moe_dense(p["mlp"], h, cfg, fetch, layer)
    else:
        h = mlp(p["mlp"], h, cfg)
    return _site_post(post, comb, h, streams, kern), new_cache


def _site_pre(p_hc, ln, streams, cfg: Cfg, kern):
    """A layer's first mHC site: (post, comb, normed block input). With `kern` (the pallas_mhc path) comb is flat."""
    if kern:
        from glm53 import pallas_mhc as PMH
        return PMH.mhc_pre(streams, p_hc, ln, cfg, kern, MHC_INTERPRET)
    post, comb, h = hc_pre(p_hc, streams, cfg)
    return post, comb, rmsnorm(h, ln, cfg.eps)


def _site_post_pre(post, comb, y, streams, p_hc, ln, cfg: Cfg, kern):
    """The middle site: the attention block's residual update, then the MLP block's pre."""
    if kern:
        from glm53 import pallas_mhc as PMH
        return PMH.mhc_post_pre(post, comb, y, streams, p_hc, ln, cfg, kern, MHC_INTERPRET)
    streams = hc_post(post, comb, y, streams)
    return (streams,) + _site_pre(p_hc, ln, streams, cfg, None)


def _site_post(post, comb, y, streams, kern):
    if kern:
        from glm53 import pallas_mhc as PMH
        return PMH.mhc_post(post, comb, y, streams, MHC_INTERPRET)
    return hc_post(post, comb, y, streams)


def decoder_layer_rows(p, streams, cfg: Cfg, ltype, mtype, caches, pos, fetch=None, sparse="auto", layer=0, cap=None,
                       hist=False):
    """Recurrent step of B INDEPENDENT streams: streams [B,T,hc,D] (T = 1: decode; T > 1 with hist=True: every row
    verifies its own T-token block); caches[b] = row b's own cache (batch dim 1), pos[b] its position (traced
    scalar). Attention runs per row on its own cache, everything else (mHC, projections, MoE: T consecutive tokens
    per row) batched over the rows. Returns (streams, [new cache per row], with the *_hist entries when hist)."""
    kern = _mhc(streams)
    post, comb, h = _site_pre(p["hc_attn"], p["ln1"], streams, cfg, kern)
    if ltype == "linear_attention":
        h, new = kda_attention_rows(p["attn"], h, cfg, caches, hist)
    else:
        h, new = mla_attention_rows(p["attn"], h, cfg, caches, pos, sparse, cap, hist)
    streams, post, comb, h = _site_post_pre(post, comb, h, streams, p["hc_ffn"], p["ln2"], cfg, kern)
    h = moe_dense(p["mlp"], h, cfg, fetch, layer) if mtype == "sparse" else mlp(p["mlp"], h, cfg)
    return _site_post(post, comb, h, streams, kern), new


def layer_pre(p, streams, cfg: Cfg, ltype, mtype, cache=None, pos0=0, use_recurrent=False, sparse="auto", cap=None,
              length=None):
    """First half of a decoder layer up to (and including) MoE routing. Returns
    (streams_after_attn, h [B,T,D] normed MLP input, post, comb, new_cache, idx [N,k] or None, w [N,k] or None)."""
    post, comb, h = hc_pre(p["hc_attn"], streams, cfg)
    h = rmsnorm(h, p["ln1"], cfg.eps)
    if ltype == "linear_attention":
        h, new_cache = kda_attention(p["attn"], h, cfg, cache, use_recurrent, length)
    else:
        h, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length)
    streams = hc_post(post, comb, h, streams)
    post, comb, h = hc_pre(p["hc_ffn"], streams, cfg)
    h = rmsnorm(h, p["ln2"], cfg.eps)
    idx = w = None
    if mtype == "sparse":
        idx, w = moe_route(p["mlp"], h.reshape(-1, h.shape[-1]), cfg)
    return streams, h, post, comb, new_cache, idx, w


def layer_post(p, streams, h, post, comb, cfg: Cfg, mtype, gu_w=None, dn_w=None, w=None, idx_local=None):
    """Second half: MLP / experts (weights already gathered for this layer's routing) + residual update.
    If idx_local is given the experts are grouped ([Eh,...] tables + local ids), else per-token ([N,k,...])."""
    B, T, D = h.shape
    if mtype == "sparse":
        xf = h.reshape(-1, D)
        if idx_local is not None:
            y = moe_apply_grouped(xf, gu_w, dn_w, idx_local, w, cfg.swiglu_limit)
        else:
            y = moe_apply(xf, gu_w, dn_w, w, cfg.swiglu_limit)
        y = cfg.reduce(y + mlp_partial(p["mlp"]["shared"], xf, cfg)).reshape(B, T, D)
    else:
        y = mlp(p["mlp"], h, cfg)
    return hc_post(post, comb, y, streams)


HIST_KEYS = {"state_hist": "state", "conv_hist": "conv", "tk_hist": "tk", "tg_hist": "tg", "kin_hist": "state"}


def split_hist(cache):
    """cache dict with *_hist entries -> (cache without them, hist dict or None)."""
    hist = {k: cache[k] for k in cache if k in HIST_KEYS}
    return {k: v for k, v in cache.items() if k not in HIST_KEYS}, (hist or None)


def rollback_states(hists, n_keep, dtypes, cur=None):
    """Speculative verify of T tokens produced `hists` (per layer: the recurrent/tail state after 0..T of the tokens
    (entry 0 = before them), or None); return per layer the small state entries as after the first `n_keep` (traced,
    0..T; 0 undoes the whole verify) tokens (the positional cache rows beyond are overwritten later). `dtypes`: per
    layer {key: dtype} of the cache entries. A "kin_hist" (the KDA tokens' update inputs, T entries) replays the
    accepted tokens from cur[layer]["state"] (the state the verify left unchanged)."""
    out = []
    for i, (h, dt) in enumerate(zip(hists, dtypes)):
        if not h:
            out.append(None); continue
        o = {}
        for hk, k in HIST_KEYS.items():
            if hk not in h:
                continue
            if hk == "kin_hist":
                o[k] = kda_replay(cur[i][k], h[hk], n_keep).astype(dt[k])
            else:
                o[k] = lax.dynamic_index_in_dim(h[hk], n_keep, axis=0, keepdims=False).astype(dt[k])
        out.append(o)
    return out


def mtp_layer(p, embeds, h_prev, cfg: Cfg, cache=None, pos0=0, fetch=None, sparse="auto", cap=None, length=None,
              layer=None):
    """GLM-5 MTP (NextN) block (vLLM `glm5next/nvidia/mtp.py`): x = eh_proj(cat(rmsnorm(embeds, enorm),
    rmsnorm(h_prev, hnorm))) — embeds [B,T,D] of the token AFTER each position (zero at position 0), h_prev [B,T,D]
    the main model's post-norm hidden at that position — then one plain pre-norm MLA + MoE block (no
    hyper-connections) and the shared-head norm. Returns (h [B,T,D] for the shared lm_head, new attention cache)."""
    x = jnp.concatenate([rmsnorm(embeds.astype(cfg.dtype), p["enorm"], cfg.eps),
                         rmsnorm(h_prev.astype(cfg.dtype), p["hnorm"], cfg.eps)], -1) @ p["eh_proj"]
    h = rmsnorm(x, p["ln1"], cfg.eps)
    a, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length)
    x = x + a
    h = rmsnorm(x, p["ln2"], cfg.eps)
    x = x + moe_dense(p["mlp"], h, cfg, fetch, layer)
    return rmsnorm(x, p["head_norm"], cfg.eps), new_cache


def forward(params, tokens, cfg: Cfg, caches=None, pos0=0, use_recurrent=False, fetch=None, sparse="auto",
            embeds=None, cap=None, length=None):
    """tokens [B,T] -> (hidden [B,T,D] after final norm, new_caches list). Prefill: caches=None.
    `embeds` [B,T,D] overrides the embedding lookup (used by the TP engine with a vocab-sharded table)."""
    x = params["embed"][tokens].astype(cfg.dtype) if embeds is None else embeds.astype(cfg.dtype)
    streams = jnp.broadcast_to(x[:, :, None, :], x.shape[:2] + (cfg.hc, x.shape[-1]))
    new_caches = []
    for i in range(cfg.n_layers):
        c = None if caches is None else caches[i]
        streams, nc = decoder_layer(params["layers"][i], streams, cfg, cfg.layer_types[i], cfg.mlp_types[i],
                                    c, pos0, use_recurrent, fetch, sparse, layer=i, cap=cap, length=length)
        new_caches.append(nc)
    h = rmsnorm(streams.mean(2).astype(cfg.dtype), params["norm"], cfg.eps)
    return h, new_caches


def logits(params, h):
    return h @ params["lm_head"]
'''
FILES["glm53/pallas_kda.py"] = r'''"""Pallas TPU kernel for a KDA layer's recurrent core at decode / short-verify shapes (B rows x T tokens, T <= 8).

After the conv taps (XLA, f32) it does what model.kda_core + model.kda_out do up to the output projection, in ONE
kernel per layer call instead of ~10 small fusions and a while loop: silu, the per-head l2 norms of q and k, the forget
gate g and beta from their projections, the T-token delta-rule recurrence on the f32 state [H, dk, dv] and the gated
RMS norm of the outputs. hist=True: also the tokens' update inputs ("kin", [B, T, H, 4, d]: k, v, g, beta) for the
rollback replay, and the state comes back as BEFORE the block (model.kda_core's hist="kin" contract).

Layout: everything is one grid step per row with whole arrays in VMEM (the state is 4 x 64 KB per chip); token t's
vectors are taken out of the [T, W] inputs by a masked sublane reduction (exact), a per-head vector becomes a matrix
with constant rows by broadcast + transpose ([d, d] f32: what the recurrence multiplies the state by), so no sublane
slicing of packed dtypes is needed. The arithmetic follows model.py op by op in f32; the sums of the two state
contractions run in a different order than XLA's reductions, so results agree to rounding, not bit for bit. On the TPU
the head dim must be a multiple of 128 (lane-aligned head slices); in interpret mode any shape works."""
import functools

import jax
import jax.numpy as jnp
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu


def _row(x, t):
    """Row t of a small [T, W] f32 value as [1, W] (exact: one nonzero term per column)."""
    sub = lax.broadcasted_iota(jnp.int32, x.shape, 0)
    return jnp.sum(jnp.where(sub == t, x, 0.0), axis=0, keepdims=True)


def _lane(x, j):
    """Lane j of a small [N, W] f32 value as [N, 1]."""
    lane = lax.broadcasted_iota(jnp.int32, x.shape, 1)
    return jnp.sum(jnp.where(lane == j, x, 0.0), axis=1, keepdims=True)


def _colmat(r, n):
    """[1, n] row -> [n, n] with M[i, j] = r[i] (row i constant): the matrix the state is scaled / contracted with."""
    return jnp.transpose(jnp.broadcast_to(r, (n, n)))


def _rows(*rows):
    """[1, n] f32 rows -> [len(rows), n]."""
    n = rows[0].shape[1]
    sub = lax.broadcasted_iota(jnp.int32, (len(rows), n), 0)
    out = jnp.zeros((len(rows), n), jnp.float32)
    for i, r in enumerate(rows):
        out = jnp.where(sub == i, jnp.broadcast_to(r, (len(rows), n)), out)
    return out


def _softplus(x):
    return jnp.maximum(x, 0.0) + jnp.log1p(jnp.exp(-jnp.abs(x)))      # (= jax.nn.softplus's logaddexp(x, 0))


def _kernel(x_ref, f_ref, b_ref, gate_ref, s_ref, dtb_ref, on_ref, alog_ref, y_ref, ns_ref, *kin_refs, H, hd, T, eps,
            gate_lb, hist):
    f32 = jnp.float32
    C = H * hd
    x = x_ref[...].astype(f32)                       # [T, 3C]: the conv output (q | k | v), before silu
    x = x * jax.nn.sigmoid(x)                        # silu
    f = f_ref[...].astype(f32) + dtb_ref[...]        # [T, C] forget-gate pre-activations + dt_bias
    bp = b_ref[...].astype(f32)                      # [T, H] beta pre-activations
    gt = gate_ref[...].astype(f32)                   # [T, C] output-gate pre-activations
    onorm = on_ref[...]                              # [1, hd]
    states = [s_ref[h] for h in range(H)]            # [hd, hd] f32 each (rows = dk)
    sub = lax.broadcasted_iota(jnp.int32, (T, C), 0)
    y = jnp.zeros((T, C), f32)
    for t in range(T):
        xt, ft, bt, gtt = _row(x, t), _row(f, t), _row(bp, t), _row(gt, t)
        outs = []
        for h in range(H):
            lo, hi = h * hd, (h + 1) * hd
            q, k, v = xt[:, lo:hi], xt[:, C + lo:C + hi], xt[:, 2 * C + lo:2 * C + hi]
            q = q / jnp.sqrt(jnp.sum(q * q, axis=1, keepdims=True) + 1e-6) * (hd ** -0.5)      # model.l2norm
            k = k / jnp.sqrt(jnp.sum(k * k, axis=1, keepdims=True) + 1e-6)
            dr = jnp.exp(alog_ref[h])                                                           # decay rate
            gh = ft[:, lo:hi]
            g = gate_lb * jax.nn.sigmoid(dr * gh) if gate_lb is not None else -dr * _softplus(gh)   # model.kda_gates
            beta = jax.nn.sigmoid(_lane(bt, h))                                                 # [1, 1]
            kmat = _colmat(k, hd)
            S = states[h] * _colmat(jnp.exp(g), hd)                                             # S[k, v] *= exp(g[k])
            kv = jnp.sum(S * kmat, axis=0, keepdims=True)                                       # k^T S  [1, dv]
            delta = (v - kv) * beta
            S = S + kmat * delta                                                                # + k (x) delta
            states[h] = S
            o = jnp.sum(S * _colmat(q, hd), axis=0, keepdims=True)                              # q^T S
            o = o * lax.rsqrt(jnp.mean(o * o, axis=1, keepdims=True) + eps) * onorm             # model.kda_out
            o = o * jax.nn.sigmoid(gtt[:, lo:hi])
            outs.append(o)
            if hist:
                kin_refs[0][t, h] = _rows(k, v, g, jnp.broadcast_to(beta, (1, hd)))
        row = outs[0] if H == 1 else jnp.concatenate(outs, axis=1)                             # [1, C]
        y = jnp.where(sub == t, jnp.broadcast_to(row, (T, C)), y)
    y_ref[...] = y.astype(y_ref.dtype)
    for h in range(H):
        ns_ref[h] = s_ref[h] if hist else states[h]  # hist: the state before the block (the rollback replays)


@functools.partial(jax.jit, static_argnames=("hd", "eps", "gate_lb", "hist", "interpret"))
def kda_core_out(x, f, bpre, gate, state, dt_bias, A_log, o_norm, *, hd, eps, gate_lb, hist=False, interpret=False):
    """x [B, T, 3C] (the conv output, any float dtype), f [B, T, C] = (x @ f_a) @ f_b, bpre [B, T, H] = x @ b, gate
    [B, T, C] = (x @ g_a) @ g_b, state [B, H, hd, hd] f32, dt_bias [C], A_log [H], o_norm [hd] -> (y [B, T, C] in
    gate's dtype = the gated, normed outputs before the o projection, new state [B, H, hd, hd] f32 (hist: the input
    state), kin [B, T, H, 4, hd] f32 or None)."""
    B, T, C3 = x.shape
    C = C3 // 3
    H = C // hd
    f32 = jnp.float32
    kern = functools.partial(_kernel, H=H, hd=hd, T=T, eps=eps, gate_lb=gate_lb, hist=hist)

    def row(*dims):
        return pl.BlockSpec((None,) + dims, lambda b: (b,) + (0,) * len(dims))
    in_specs = [row(T, C3), row(T, C), row(T, H), row(T, C), row(H, hd, hd),
                pl.BlockSpec((1, C), lambda b: (0, 0)), pl.BlockSpec((1, hd), lambda b: (0, 0)),
                pl.BlockSpec(memory_space=pltpu.SMEM)]
    out_shape = [jax.ShapeDtypeStruct((B, T, C), gate.dtype), jax.ShapeDtypeStruct((B, H, hd, hd), f32)]
    out_specs = [row(T, C), row(H, hd, hd)]
    if hist:
        out_shape.append(jax.ShapeDtypeStruct((B, T, H, 4, hd), f32))
        out_specs.append(row(T, H, 4, hd))
    outs = pl.pallas_call(kern, grid=(B,), out_shape=tuple(out_shape), in_specs=in_specs, out_specs=tuple(out_specs),
                          interpret=interpret)(
        x, f, bpre.astype(f32), gate, state.astype(f32), dt_bias.astype(f32).reshape(1, C),
        o_norm.astype(f32).reshape(1, hd), A_log.astype(f32))
    return outs[0], outs[1], (outs[2] if hist else None)
'''
FILES["glm53/pallas_mhc.py"] = r'''"""Pallas TPU kernels for the mHC sites of a decoder layer at decode / short-verify shapes (N = B*T tokens, a few to a
few dozen). One kernel per site instead of XLA's chain of small fusions (norm, the [N, hc*D] x [hc*D, (2+hc)*hc]
projection, sigmoids, softmax, the Sinkhorn iterations, the collapse, the block norm; plus the residual update of
the previous block at the middle site):

  mhc_pre(streams, p, ln_w, cfg)                      == model.hc_pre, then model.rmsnorm(h, ln_w)
  mhc_post_pre(post, comb, y, streams, p, ln_w, cfg)  == model.hc_post(post, comb, y, streams), then the above
  mhc_post(post, comb, y, streams)                    == model.hc_post (the end of a layer)

Shapes as in model.py: streams [B, T, hc, D] (model dtype), y and h [B, T, D], post [B, T, hc] f32; comb is FLAT,
[B, T, hc*hc] f32 with comb[h, k] (the weight of stream h in new stream k) at h*hc + k, so it moves between the
kernels of a layer without relayouts (mhc_pre / mhc_post_pre return it so, mhc_post_pre / mhc_post take it so). The arithmetic follows model.py op by op
(hc_post in the model dtype with f32 accumulation, hc_pre in f32); sums run in a different order than XLA's
reductions, so results agree to rounding, not bit for bit. `prec`: "bf16" rounds the operands of the two small
contractions (the mix projection, the collapse) to bf16 as XLA's default matmul precision does for f32 dots on the
TPU; "f32" keeps them exact (the reference model's f32 math).

The projection weight is read transposed, fnT [(2+hc)*hc, hc*D]: [hc*D, 24] would be padded to 128 lanes in VMEM
(and in HBM). `p` may hold "fnT" (made once at load) or "fn" (transposed here, one XLA transpose per call).
Everything is one grid step with whole arrays in VMEM (a decode site moves ~0.1 MB per token plus the 0.8 MB
weight). The 16 Sinkhorn entries are separate [N, 1] columns, so every step is the reference's own elementwise op."""
import functools

import jax
import jax.numpy as jnp
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu


def _col(x, j):
    """Column j of a small [N, W] f32 value as [N, 1] (exact: one nonzero term per row)."""
    lane = lax.broadcasted_iota(jnp.int32, x.shape, 1)
    return jnp.sum(jnp.where(lane == j, x, 0.0), axis=1, keepdims=True)


def _assemble(cols, n_rows):
    """[N, 1] columns -> [N, len(cols)]."""
    W = len(cols)
    lane = lax.broadcasted_iota(jnp.int32, (n_rows, W), 1)
    out = jnp.zeros((n_rows, W), jnp.float32)
    for j, c in enumerate(cols):
        out = jnp.where(lane == j, jnp.broadcast_to(c, (n_rows, W)), out)
    return out


def _sigmoid(x):
    return jax.nn.sigmoid(x)


def _kernel(*refs, hc, D, iters, eps, hc_eps, with_post, prec, with_pre=True, opts=()):
    """opts (static): "once" = the residual update rounded once (f32 coefficients, products and sums: what XLA's
    fusions compute on the TPU, which keep f32 intermediates), "nocr" = the collapse not rounded to the model dtype
    before the norm (likewise), "recip" = Sinkhorn divisions as one reciprocal per row/column sum, "fnany" = the
    projection weight DMA'd by hand, overlapped with the residual update; timing stubs: "empty", "it1", "nodot"."""
    f32 = jnp.float32
    if "fnany" in opts:
        *refs, fn_vmem, sem = refs
    if not with_pre:
        pin_ref, cin_ref, y_ref, s_ref, ns_ref = refs
    elif with_post:
        pin_ref, cin_ref, y_ref, s_ref, fnT_ref, sc_ref, ln_ref, ns_ref, po_ref, co_ref, h_ref = refs
    else:
        s_ref, fnT_ref, sc_ref, ln_ref, po_ref, co_ref, h_ref = refs
    dt = s_ref.dtype
    N = s_ref.shape[0]
    M = (2 + hc) * hc
    if "fnany" in opts:
        cp = pltpu.make_async_copy(fnT_ref, fn_vmem, sem)
        cp.start()
        fnT_ref = fn_vmem
    if "empty" in opts:
        if with_post:
            ns_ref[...] = s_ref[...]
        if with_pre:
            if "fnany" in opts:
                cp.wait()
            po_ref[...] = jnp.zeros(po_ref.shape, f32)
            co_ref[...] = jnp.zeros(co_ref.shape, f32)
            h_ref[...] = s_ref[:, :D]
        return
    if "it1" in opts:
        iters = 1
    if with_post:
        # hc_post: new[k] = post[k] * y + sum_h comb[h, k] * streams[h], in the model dtype (f32 products and sums,
        # rounded where the reference's ops round: the product, the einsum, the add)
        y = y_ref[...].astype(f32)
        pin = pin_ref[...]
        cin = cin_ref[...]
        once = "once" in opts
        rnd = (lambda v: v) if once else (lambda v: v.astype(dt).astype(f32))   # noqa: E731
        pc = [rnd(_col(pin, k)) for k in range(hc)]
        cc = [rnd(_col(cin, j)) for j in range(hc * hc)]
        olds = [s_ref[:, h * D:(h + 1) * D].astype(f32) for h in range(hc)]
        xs = []
        for k in range(hc):
            a = rnd(pc[k] * y)
            e = cc[k] * olds[0]
            for h in range(1, hc):
                e = e + cc[h * hc + k] * olds[h]
            nk = (a + rnd(e)).astype(dt)
            ns_ref[:, k * D:(k + 1) * D] = nk
            xs.append(nk.astype(f32))
    else:
        xs = [s_ref[:, h * D:(h + 1) * D].astype(f32) for h in range(hc)]
    if not with_pre:
        return
    # hc_pre: unweighted RMS norm over all hc*D values of a token, the mix projection
    ss = jnp.sum(xs[0] * xs[0], axis=1, keepdims=True)
    for h in range(1, hc):
        ss = ss + jnp.sum(xs[h] * xs[h], axis=1, keepdims=True)
    r = lax.rsqrt(ss / (hc * D) + eps)
    if "fnany" in opts:
        cp.wait()
    mix = None
    for h in range(hc):
        fl = xs[h] * r
        w = fnT_ref[:, h * D:(h + 1) * D]
        if prec == "bf16":
            part = lax.dot_general(fl.astype(jnp.bfloat16), w.astype(jnp.bfloat16), (((1,), (1,)), ((), ())),
                                   preferred_element_type=f32)
        else:
            part = lax.dot_general(fl, w.astype(f32), (((1,), (1,)), ((), ())), preferred_element_type=f32,
                                   precision=lax.Precision.HIGHEST)
        mix = part if mix is None else mix + part                                     # [N, M]
    if "nodot" in opts:
        mix = jnp.zeros_like(mix)
    s0, s1, s2 = sc_ref[M], sc_ref[M + 1], sc_ref[M + 2]
    base = [sc_ref[j] for j in range(M)]
    m = [_col(mix, j) for j in range(M)]
    pre = [_sigmoid(m[j] * s0 + base[j]) + hc_eps for j in range(hc)]
    post = [2.0 * _sigmoid(m[hc + j] * s1 + base[hc + j]) for j in range(hc)]
    lg = [m[2 * hc + j] * s2 + base[2 * hc + j] for j in range(hc * hc)]              # [i * hc + j]
    c = []
    for i in range(hc):                                                               # softmax over j, + hc_eps
        row = lg[i * hc:(i + 1) * hc]
        mx = row[0]
        for v in row[1:]:
            mx = jnp.maximum(mx, v)
        ex = [jnp.exp(v - mx) for v in row]
        den = ex[0]
        for v in ex[1:]:
            den = den + v
        c += [v / den + hc_eps for v in ex]

    def csum(c, j):
        out = c[j]
        for i in range(1, hc):
            out = out + c[i * hc + j]
        return out

    def rsum(c, i):
        out = c[i * hc]
        for j in range(1, hc):
            out = out + c[i * hc + j]
        return out

    if "recip" in opts:
        div = lambda a, b: a * b                                                      # noqa: E731
        inv = lambda v: 1.0 / v                                                       # noqa: E731
    else:
        div = lambda a, b: a / b                                                      # noqa: E731
        inv = lambda v: v                                                             # noqa: E731
    cs = [inv(csum(c, j) + hc_eps) for j in range(hc)]
    c = [div(c[i * hc + j], cs[j]) for i in range(hc) for j in range(hc)]
    for _ in range(iters - 1):
        rs = [inv(rsum(c, i) + hc_eps) for i in range(hc)]
        c = [div(c[i * hc + j], rs[i]) for i in range(hc) for j in range(hc)]
        cs = [inv(csum(c, j) + hc_eps) for j in range(hc)]
        c = [div(c[i * hc + j], cs[j]) for i in range(hc) for j in range(hc)]
    po_ref[...] = _assemble(post, N)
    co_ref[...] = _assemble(c, N)
    # collapse with pre, then the block's RMS norm (weighted)
    if prec == "bf16":
        pr = [v.astype(jnp.bfloat16).astype(f32) for v in pre]
        col = pr[0] * xs[0].astype(jnp.bfloat16).astype(f32)
        for h in range(1, hc):
            col = col + pr[h] * xs[h].astype(jnp.bfloat16).astype(f32)
    else:
        col = pre[0] * xs[0]
        for h in range(1, hc):
            col = col + pre[h] * xs[h]
    cf = col if "nocr" in opts else col.astype(dt).astype(f32)
    r2 = lax.rsqrt(jnp.sum(cf * cf, axis=1, keepdims=True) / D + eps)
    h_ref[...] = (ln_ref[...].astype(f32) * (cf * r2)).astype(dt)


@functools.partial(jax.jit, static_argnames=("hc", "interpret"))
def _call_post(post, comb, y, streams, *, hc, interpret):
    N, HD = streams.shape
    D = HD // hc
    V = pltpu.VMEM
    kern = functools.partial(_kernel, hc=hc, D=D, iters=0, eps=0.0, hc_eps=0.0, with_post=True, prec="f32",
                             with_pre=False)
    return pl.pallas_call(
        kern, out_shape=jax.ShapeDtypeStruct((N, HD), streams.dtype),
        in_specs=[pl.BlockSpec(memory_space=V)] * 4, out_specs=pl.BlockSpec(memory_space=V),
        interpret=interpret,
    )(post, comb, y, streams)


def _scalars(p):
    return jnp.concatenate([p["base"].astype(jnp.float32).reshape(-1), p["scale"].astype(jnp.float32).reshape(-1)])


def _fnT(p):
    return p["fnT"] if "fnT" in p else p["fn"].T


@functools.partial(jax.jit, static_argnames=("hc", "iters", "eps", "hc_eps", "prec", "interpret", "opts"))
def _call_pre(streams, fnT, sc, ln_w, *, hc, iters, eps, hc_eps, prec, interpret, opts=()):
    N, HD = streams.shape
    D = HD // hc
    V = pltpu.VMEM
    kern = functools.partial(_kernel, hc=hc, D=D, iters=iters, eps=eps, hc_eps=hc_eps, with_post=False, prec=prec,
                             opts=opts)
    fspec, scratch = _fn_specs(opts, fnT)
    return pl.pallas_call(
        kern,
        out_shape=(jax.ShapeDtypeStruct((N, hc), jnp.float32), jax.ShapeDtypeStruct((N, hc * hc), jnp.float32),
                   jax.ShapeDtypeStruct((N, D), streams.dtype)),
        in_specs=[pl.BlockSpec(memory_space=V), fspec, pl.BlockSpec(memory_space=pltpu.SMEM),
                  pl.BlockSpec(memory_space=V)],
        out_specs=(pl.BlockSpec(memory_space=V),) * 3,
        scratch_shapes=scratch,
        interpret=interpret,
    )(streams, fnT, sc, ln_w.reshape(1, D))


def _fn_specs(opts, fnT):
    """(in_spec of the projection weight, scratch shapes)."""
    if "fnany" in opts:
        return pl.BlockSpec(memory_space=pl.ANY), [pltpu.VMEM(fnT.shape, fnT.dtype), pltpu.SemaphoreType.DMA(())]
    return pl.BlockSpec(memory_space=pltpu.VMEM), []


@functools.partial(jax.jit, static_argnames=("hc", "iters", "eps", "hc_eps", "prec", "interpret", "opts"))
def _call_post_pre(post, comb, y, streams, fnT, sc, ln_w, *, hc, iters, eps, hc_eps, prec, interpret, opts=()):
    N, HD = streams.shape
    D = HD // hc
    V = pltpu.VMEM
    kern = functools.partial(_kernel, hc=hc, D=D, iters=iters, eps=eps, hc_eps=hc_eps, with_post=True, prec=prec,
                             opts=opts)
    fspec, scratch = _fn_specs(opts, fnT)
    return pl.pallas_call(
        kern,
        out_shape=(jax.ShapeDtypeStruct((N, HD), streams.dtype), jax.ShapeDtypeStruct((N, hc), jnp.float32),
                   jax.ShapeDtypeStruct((N, hc * hc), jnp.float32), jax.ShapeDtypeStruct((N, D), streams.dtype)),
        in_specs=[pl.BlockSpec(memory_space=V)] * 4 + [fspec, pl.BlockSpec(memory_space=pltpu.SMEM),
                                                       pl.BlockSpec(memory_space=V)],
        out_specs=(pl.BlockSpec(memory_space=V),) * 4,
        scratch_shapes=scratch,
        interpret=interpret,
    )(post, comb, y, streams, fnT, sc, ln_w.reshape(1, D))


OPTS = ("once", "nocr")     # rounding as XLA's TPU fusions do (j140, v6e 2026-10-04: closest to XLA's chain)


def mhc_pre(streams, p, ln_w, cfg, prec="bf16", interpret=False, opts=OPTS):
    """streams [B, T, hc, D] -> (post [B, T, hc] f32, comb [B, T, hc*hc] f32 (flat: comb[h, k] at h*hc + k),
    h [B, T, D]): model.hc_pre + model.rmsnorm(h, ln_w)."""
    B, T, hc, D = streams.shape
    po, co, h = _call_pre(streams.reshape(B * T, hc * D), _fnT(p), _scalars(p), ln_w, hc=hc, iters=cfg.hc_iters,
                          eps=cfg.eps, hc_eps=cfg.hc_eps, prec=prec, interpret=interpret, opts=tuple(opts))
    return po.reshape(B, T, hc), co.reshape(B, T, hc * hc), h.reshape(B, T, D)


def mhc_post_pre(post, comb, y, streams, p, ln_w, cfg, prec="bf16", interpret=False, opts=OPTS):
    """model.hc_post(post, comb, y, streams), then mhc_pre of the result -> (streams, post, comb, h); comb flat
    [B, T, hc*hc] in and out (as mhc_pre returns it)."""
    B, T, hc, D = streams.shape
    N = B * T
    ns, po, co, h = _call_post_pre(post.reshape(N, hc), comb.reshape(N, hc * hc), y.reshape(N, D),
                                   streams.reshape(N, hc * D), _fnT(p), _scalars(p), ln_w, hc=hc, iters=cfg.hc_iters,
                                   eps=cfg.eps, hc_eps=cfg.hc_eps, prec=prec, interpret=interpret, opts=tuple(opts))
    return ns.reshape(B, T, hc, D), po.reshape(B, T, hc), co.reshape(B, T, hc * hc), h.reshape(B, T, D)


def mhc_post(post, comb, y, streams, interpret=False):
    """model.hc_post with the flat comb [B, T, hc*hc] -> streams [B, T, hc, D]."""
    B, T, hc, D = streams.shape
    N = B * T
    ns = _call_post(post.reshape(N, hc), comb.reshape(N, hc * hc), y.reshape(N, D), streams.reshape(N, hc * D), hc=hc,
                    interpret=interpret)
    return ns.reshape(B, T, hc, D)
'''
FILES["glm53/pallas_moe.py"] = r'''"""Pallas TPU kernel: fused codebook-dequant + matvec of routed experts straight from the planar HBM tables.

`moe_matvec(planes, qtype, nblk, idx, X)` computes, for every expert slot i (one (token, expert) pair):
    out[i, r] = sum_n  W_{idx[i]}[r, n] * x_i[n]              (W dequantized on the fly, never written anywhere)
where X[i, w, c] = x_i[input(w, c)] is the per-word activation matrix of glm53.planes (see `planes.pm_x`).

Grid = expert slots; the expert id comes from a scalar-prefetch operand so each grid step DMAs exactly the planes
of the chosen expert (double-buffered by Pallas). Inside, one (8, 128) vreg of the qs plane = 8 packed words x 128
matrix rows; the codebook index of every group is looked up with in-vreg lane gathers (two codes per int32 table
word: one gather per 128-word table row + a select tree over the high index bits + a shift for the half), the values
come out of the code with shifts (IQ2_S: one int multiply builds the f32 bits), the sign bits are XORed into the f32
sign, and the values are multiplied-accumulated on the VPU against the activation broadcast along lanes:
acc[s, r] += v[s, r] * X[8b + s, c]. The lane broadcasts of X are computed once per expert slot into VMEM. The
sublane sum of the accumulator over all words is the output row block. `moe_matvec_gu` does gate and up of the same
slots in one kernel (one grid step per slot, the broadcasts shared). Everything is 32-bit (no packed dtypes), loads
are (n, 128) windows at 8-aligned sublane offsets, so the kernels also run under `interpret=True` on CPU for the tests.

Why this shape (speed plan §4/§12, measured on the v6e 2026-10-03 with jobs/j65): the decode kernel is bound by
cross-lane ops (each lane gather or lane broadcast ~1.5 ns per vreg) and then by VALU ops per weight; one gather per
128 table entries (8 for IQ2_S's 1024-entry grid) + a lane broadcast per weight vreg cost 15.7 us per expert, the
pair table + hoisted broadcasts + fused gate/up + pre-encoded values + XOR sign + one grid step per down slot 8.4 us,
with identical outputs. Mosaic in libtpu 0.0.42 has no multi-dimension in-vreg gather ("Zero or multiple gather
dimensions"), so a one-op 1024-entry lookup is not available.

Codebook tables: `code_table(qtype)` (see there).
"""
import functools
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

from glm53 import iqquant as Q
from glm53 import planes as PL
from glm53 import model as M

LANES = 128
SUB = 8


SIGN = 0x80000000
IQ2S_F32 = (0x41000000, 0x640000)          # f32 bits of the IQ2_S levels 8, 25, 43 = base + u * step, u = 0, 2, 3


@functools.lru_cache(None)
def code_table(qtype):
    """int32 [n // 256, 128]: the grid, TWO entries per int32 (entry 2w in the low 16 bits of word w, 2w + 1 in the
    high 16; word w at [w // 128, w % 128]). One field per weight k of the entry: IQ2_S stores its level lv in
    {0, 1, 2} as u = 0, 2, 3 in bits 2k..2k+1 (so the value's f32 bits are IQ2S_F32[0] + u * IQ2S_F32[1]), IQ3_S
    stores the value 2 lv + 1 in bits 4k..4k+3."""
    grid, levels = Q.GRIDS[qtype], Q.LEVELS[qtype]
    lv = np.searchsorted(levels, grid).astype(np.int64)                 # [n, w]
    assert np.array_equal(levels[lv], grid)
    if qtype == "IQ2_S":
        assert np.array_equal(levels, [8, 25, 43])
        f, fw = np.array([0, 2, 3])[lv], 2
    elif qtype == "IQ3_S":
        assert np.array_equal(levels, 2 * np.arange(8) + 1)
        f, fw = 2 * lv + 1, 4
    else:
        raise ValueError(qtype)
    code = np.zeros(grid.shape[0], np.int64)
    for k in range(grid.shape[1]):
        code |= f[:, k] << (fw * k)
    return (code[0::2] | (code[1::2] << 16)).astype(np.uint32).view(np.int32).reshape(-1, LANES)


@functools.lru_cache(None)
def kv16_table():
    t = np.zeros((1, LANES), np.float32)
    t[0, :16] = Q.KVALUES_IQ4NL
    return t


def _f16_bits_to_f32(h):
    """u32 holding f16 bits (low 16) -> f32 (normal + subnormal + zero; no inf/nan expected)."""
    h = h & 0xFFFF
    s = (h >> 15) & 1
    e = (h >> 10) & 31
    m = h & 1023
    bits = (s << 31) | ((e + 112) << 23) | (m << 13)
    normal = lax.bitcast_convert_type(bits.astype(jnp.uint32), jnp.float32)
    sub = m.astype(jnp.int32).astype(jnp.float32) * (2.0 ** -24)
    sub = jnp.where(s == 1, -sub, sub)
    return jnp.where(e == 0, sub, normal)


def _rows8(src, reps):
    """src (n, 128) with n * reps == 8 -> (8, 128) whose sublane s is src[s // reps]."""
    n = src.shape[0]
    if n == 1:
        return jnp.broadcast_to(src, (SUB, LANES))
    row = lax.broadcasted_iota(jnp.int32, (SUB, LANES), 0)
    out = jnp.broadcast_to(src[n - 1:n], (SUB, LANES))
    for r in range(n - 2, -1, -1):
        out = jnp.where(row < (r + 1) * reps, jnp.broadcast_to(src[r:r + 1], (SUB, LANES)), out)
    return out


def _sub_iota():
    return lax.broadcasted_iota(jnp.uint32, (SUB, LANES), 0)


_GATHER_DN = lax.GatherDimensionNumbers(offset_dims=(), collapsed_slice_dims=(1,), start_index_map=(1,),
                                        operand_batching_dims=(0,), start_indices_batching_dims=(0,))


def _take(t, i):
    """In-vreg lane gather t[s, i[s, l]] (Mosaic's dynamic_gather; indices are in range, so no negative-index fix-up
    as jnp.take_along_axis adds)."""
    return lax.gather(t, i[..., None], _GATHER_DN, (1, 1), mode=lax.GatherScatterMode.PROMISE_IN_BOUNDS)


def _lookup(tb, q, j, hw, hb):
    """Code of grid entry idx = (byte j of q) | (high index bits << 8), from `code_table`'s rows `tb` (each (8, 128),
    one table row broadcast to all sublanes): table word = idx >> 1 (low 7 bits = bits 1..7 of the byte, row = the high
    index bits, bit hb.. of hw), half = bit 0 of the byte. Returns int32 (8, 128) with the code in its low 16 bits (the
    high bits are the other entry or a sign extension; the decoders never read them)."""
    vals = [_take(t, ((q >> (8 * j + 1)) & 127).astype(jnp.int32)) for t in tb]
    bit = 0
    while len(vals) > 1:
        sel = (hw & (1 << (hb + bit))) != 0
        vals = [jnp.where(sel, vals[2 * m + 1], vals[2 * m]) for m in range(len(vals) // 2)]
        bit += 1
    half = ((q << 4) if j == 0 else (q >> (8 * j - 4))) & 16
    return vals[0] >> half.astype(jnp.int32)


def _signed(v, word, bit):
    """f32 v negated where bit `bit` of the u32 `word` is set (XOR into the f32 sign bit: shl, and, xor)."""
    m = (word << (31 - bit)) & jnp.uint32(SIGN)
    return lax.bitcast_convert_type(lax.bitcast_convert_type(v, jnp.uint32) ^ m, jnp.float32)


# ------------------------------------------------------------------------------------------ per-format word decoders
# `rows(name, r0, n)` returns rows r0..r0+n-1 of that plane for the current expert as an (n, 128) value (static lanes).
# Each decoder returns a list of (scale (8,128) f32, [(c, value (8,128) f32), ...]) for the 8 words 8b..8b+7.
def _decode_iq2_s(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sgw = rows("sg", 8 * b, 8)
    sub = _sub_iota()
    shift8 = (sub & 3) * 8
    qhw = (_rows8(rows("qh", 2 * b, 2), 4) >> shift8) & 255                 # qh byte of word 8b+s
    scw = (_rows8(rows("sc", 2 * b, 2), 4) >> shift8) & 255                 # sc byte of word 8b+s
    dw = rows("d", b // 2, 1) >> (16 * (b % 2))                             # block b for all 8 words
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = [dv * (0.5 + ((scw >> (4 * h)) & 15).astype(jnp.int32).astype(jnp.float32)) * 0.25 for h in range(2)]
    groups = [(scale[0], []), (scale[1], [])]
    base, step = IQ2S_F32
    for j in range(4):
        code = _lookup(tb, q, j, qhw, 2 * j)
        for k in range(8):
            bits = (code & (3 << (2 * k))) * (step >> (2 * k)) + base                    # u << 2k times step >> 2k
            v = lax.bitcast_convert_type(bits, jnp.float32)
            groups[j // 2][1].append((8 * j + k, _signed(v, sgw, 8 * j + k)))
    return groups


def _decode_iq3_s(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    sgh = _rows8(rows("sg", 4 * b, 4), 2) >> ((sub & 1) * 16)              # the 16 sign bits of word 8b+s (low half)
    qhn = (_rows8(rows("qh", b, 1), 8) >> (sub * 4)) & 15                    # 4 high bits of word 8b+s
    scn = (_rows8(rows("sc", b // 2, 1), 8) >> ((4 * (b % 2) + (sub >> 1)) * 4)) & 15
    dw = rows("d", b // 4, 1) >> (16 * ((b // 2) % 2))
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = dv * (1.0 + 2.0 * scn.astype(jnp.int32).astype(jnp.float32))
    items = []
    for j in range(4):
        code = _lookup(tb, q, j, qhn, j)
        for k in range(4):
            v = ((code >> (4 * k)) & 15 if k else code & 15).astype(jnp.float32)     # 2 lv + 1
            items.append((4 * j + k, _signed(v, sgh, 4 * j + k)))
    return [(scale, items)]


def _decode_iq4_xs(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    i = (2 * b + (sub >> 2)) % 8                                             # sub-block of word 8b+s within its block
    slw = (_rows8(rows("sl", b // 4, 1), 8) >> (4 * i)) & 15
    shw = (_rows8(rows("sh", b // 4, 1), 8) >> (2 * i)) & 3
    dw = rows("d", b // 8, 1) >> (16 * ((b // 4) % 2))
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = dv * ((slw | (shw << 4)).astype(jnp.int32) - 32).astype(jnp.float32)
    kv = tb[0]                                                               # (8,128) f32, entries at lanes 0..15
    items = []
    for pos in range(2):
        for j in range(4):
            nib = ((q >> (8 * j + 4 * pos)) & 15).astype(jnp.int32)
            items.append((4 * pos + j, jnp.take_along_axis(kv, nib, axis=1)))
    return [(scale, items)]


def _kq_scales(rows, b, sub):
    """K-quant scale bytes for the 8 words of vreg b (half n = b % 2 of block b // 2): per shift s an (8,128) u32 with
    the scale byte `is` = 8 n + 2 s + (sublane >= 4) of the block's 16, plus the block's d word (1,128)."""
    blk, n = b // 2, b % 2
    scw = rows("sc", 4 * blk, 4)                                                 # (4,128): the 16 scale bytes
    hi = (sub >= 4).astype(jnp.uint32)
    out = []
    for s_ in range(4):
        r = 2 * n + s_ // 2
        row = jnp.broadcast_to(scw[r:r + 1], (SUB, LANES))
        out.append((row >> (8 * (2 * (s_ % 2) + hi))) & 255)
    return out, rows("d", blk, 1)


def _decode_q2_k(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    sc, dw = _kq_scales(rows, b, sub)
    d = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    dmin = jnp.broadcast_to(_f16_bits_to_f32(dw >> 16), (SUB, LANES))
    items = []
    for s_ in range(4):
        dl = d * (sc[s_] & 15).astype(jnp.int32).astype(jnp.float32)
        ml = dmin * (sc[s_] >> 4).astype(jnp.int32).astype(jnp.float32)
        for j in range(4):
            v = ((q >> (8 * j + 2 * s_)) & 3).astype(jnp.int32).astype(jnp.float32)
            items.append((4 * s_ + j, dl * v - ml))
    return [(jnp.ones((SUB, LANES), jnp.float32), items)]


def _decode_q3_k(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    blk, n = b // 2, b % 2
    hm = rows("hm", 8 * blk, 8)                                                  # (8,128): u32 q = hmask bytes 4q..4q+3
    sc, dw = _kq_scales(rows, b, sub)
    d = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    items = []
    for s_ in range(4):
        dl = d * ((sc[s_] & 255).astype(jnp.int32) - 32).astype(jnp.float32)
        for j in range(4):
            v = ((q >> (8 * j + 2 * s_)) & 3).astype(jnp.int32)
            hb = ((hm >> (8 * j + 4 * n + s_)) & 1).astype(jnp.int32)
            items.append((4 * s_ + j, dl * (v - 4 + 4 * hb).astype(jnp.float32)))
    return [(jnp.ones((SUB, LANES), jnp.float32), items)]


DECODE = {"IQ2_S": _decode_iq2_s, "IQ3_S": _decode_iq3_s, "IQ4_XS": _decode_iq4_xs, "Q2_K": _decode_q2_k,
          "Q3_K": _decode_q3_k}


def _shared(*static):
    """jax.jit with these static arguments around a kernel entry point: every call site with the same configuration and
    shapes shares ONE trace and lowering (XLA inlines the call; same HLO). Without it each call site re-traced and
    re-lowered its kernel in Python — v6e 2026-10-04, jobs/j125: ~3.6 s per MoE layer per program for the per-slot
    kernels and ~25 s for the grouped ones (845 s for the five T = 4 verify programs), while the XLA/Mosaic compile
    of 1 or 4 identical kernels took the same time."""
    return lambda f: jax.jit(f, static_argnames=static)


def _table_arrays(qtype):
    if qtype == "IQ4_XS":
        return jnp.asarray(kv16_table())
    if qtype in ("Q2_K", "Q3_K"):
        return jnp.zeros((1, LANES), jnp.int32)                              # no codebook
    return jnp.asarray(code_table(qtype))


def _pick_rows(block, off, r0, n):
    """block (8, 128) value; rows off+r0 .. off+r0+n-1 where `off` is a traced scalar in [0, 8): select tree
    (Mosaic has no unaligned dynamic sublane loads)."""
    outs = []
    for r in range(n):
        want = off + r0 + r
        v = block[0:1]
        for q in range(1, SUB):
            v = jnp.where(want == q, block[q:q + 1], v)
        outs.append(v)
    return outs[0] if n == 1 else jnp.concatenate(outs, axis=0)


# ------------------------------------------------------------------------------------------------------- the kernel
def _table_rows(tbl_ref, qtype):
    """The decoders' `tb`: each table row broadcast to all sublanes (IQ4_XS: its one 16-entry row)."""
    n = 1 if qtype == "IQ4_XS" else tbl_ref.shape[0]
    return [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n)]


def _fill_bx(x_ref, bx_ref, nb, C):
    """bx[b*C + c] = X[8b:8b+8, c] broadcast along lanes, once per expert slot (instead of once per weight vreg)."""
    for b in range(nb):
        xb = x_ref[SUB * b:SUB * (b + 1), :]
        for c in range(C):
            bx_ref[b * C + c] = jnp.broadcast_to(xb[:, c:c + 1], (SUB, LANES))


def _matvec_lanes(decode, rows, tb, bx_ref, nb, C):
    """sum_n W[r, n] x[n] for the 128 rows of one lane vreg -> (1, 128) f32."""
    out = jnp.zeros((SUB, LANES), jnp.float32)
    for b in range(nb):
        for scale, items in decode(b, rows, tb):
            acc = jnp.zeros((SUB, LANES), jnp.float32)
            for c, val in items:
                acc = acc + val * bx_ref[b * C + c]
            out = out + acc * scale
    return jnp.sum(out, axis=0, keepdims=True)


def _rows_fn(prefs, offs, lanes):
    """rows(plane, r0, n) of the current expert at `lanes` (planes with < 8 rows per expert: picked from their 8-row
    block with selects on the scalar-prefetched offset)."""
    def rows(k, r0, n):
        if offs[k] is None:
            return prefs[k][r0:r0 + n, lanes]
        return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)
    return rows


def _offsets(keys, rows_p, e):
    return {k: (lax.rem(e * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}


@_shared("qtype", "nblk", "interpret", "lane_block")
def moe_matvec(planes, qtype, nblk, idx, X, *, interpret=False, lane_block=None):
    """planes: {k: u32 [E*rows_p, R] or [1, E*rows_p, R]}; idx int32 [Nk]; X f32 [Nk, W, C] -> f32 [Nk, R].

    Grid = (expert slot, lane block of `lane_block` matrix rows, default all rows up to 4096: one grid step per slot,
    every lane vreg unrolled; 2026-10-03 on the v6e the down matrix took 2.99 us per expert this way vs 3.35 with two
    steps and 6.0 with a fori_loop over the lane vregs); every ref access uses static indices (Mosaic rejects dynamic
    sublane/lane offsets that are not provably tile aligned): planes with fewer than 8 rows per expert are fetched as
    their 8-row block and the expert's rows are picked with selects on the scalar-prefetched offset."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    first = planes[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    Nk = idx.shape[0]
    assert X.shape == (Nk, W, C), (X.shape, (Nk, W, C))
    assert R % LANES == 0 and W % SUB == 0, (R, W)
    LB = lane_block or min(R, 4096)
    assert R % LB == 0 and LB % LANES == 0, (R, LB)
    n_lb = R // LB
    n_vreg = LB // LANES
    nb = W // SUB
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]

    def kernel(idx_ref, *refs):
        plane_refs = dict(zip(keys, refs[:len(keys)]))
        tbl_ref, x_ref, o_ref, bx_ref = refs[len(keys):]
        offs = _offsets(keys, rows_p, idx_ref[pl.program_id(0)])
        tb = _table_rows(tbl_ref, qtype)
        pl.when(pl.program_id(1) == 0)(lambda: _fill_bx(x_ref, bx_ref, nb, C))
        for v in range(n_vreg):
            lanes = slice(v * LANES, (v + 1) * LANES)
            o_ref[:, lanes] = _matvec_lanes(decode, _rows_fn(plane_refs, offs, lanes), tb, bx_ref, nb, C)

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda i, cb, idx_ref: (0, (idx_ref[i] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda i, cb, idx_ref: ((idx_ref[i] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys]
    in_specs += [pl.BlockSpec(tbl.shape, lambda i, cb, idx_ref: (0, 0)),
                 pl.BlockSpec((None, W, C), lambda i, cb, idx_ref: (i, 0, 0))]
    out_spec = pl.BlockSpec((None, 1, LB), lambda i, cb, idx_ref: (i, 0, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=1, grid=(Nk, n_lb), in_specs=in_specs,
                                             out_specs=out_spec,
                                             scratch_shapes=[pltpu.VMEM((nb * C, SUB, LANES), jnp.float32)])
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((Nk, 1, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary", "arbitrary")))
    args = [planes[k] for k in keys] + [tbl, X]
    return fn(idx, *args).reshape(Nk, R)


@_shared("qtype", "nblk", "interpret", "k")
def moe_matvec_gu(planes_g, planes_u, qtype, nblk, idx, X, *, interpret=False, k=1):
    """gate and up of the same expert slots in ONE kernel: planes_{g,u} as in `moe_matvec` (same qtype and shape,
    R <= 1024 rows: all lane vregs unrolled), X f32 [Nk / k, W, C] = one row per TOKEN, whose k consecutive slots
    share it -> (g, u) f32 [Nk, R] each. One grid step per slot; the lane broadcasts of X are made once per token
    (the first of its k slots; the scratch persists across the sequential grid) and serve both matrices (2026-10-03 on
    the v6e: 5.47 us per expert for gate + up vs 2 x 3.43 as two hoisted kernels and 2 x 4.75 shipped)."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    first = planes_g[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    Nk = idx.shape[0]
    assert Nk % k == 0 and X.shape == (Nk // k, W, C), (X.shape, (Nk // k, W, C))
    assert R % LANES == 0 and R <= 1024 and W % SUB == 0, (R, W)
    assert all(planes_u[kk].shape == planes_g[kk].shape for kk in keys)
    n_vreg = R // LANES
    nb = W // SUB
    nk = len(keys)
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]

    def kernel(idx_ref, *refs):
        g_refs, u_refs = dict(zip(keys, refs[:nk])), dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, bx_ref = refs[2 * nk:]
        offs = _offsets(keys, rows_p, idx_ref[pl.program_id(0)])
        tb = _table_rows(tbl_ref, qtype)
        if k == 1:
            _fill_bx(x_ref, bx_ref, nb, C)
        else:                                      # the token's first slot fills, its other k-1 slots reuse
            pl.when(pl.program_id(0) % k == 0)(lambda: _fill_bx(x_ref, bx_ref, nb, C))
        for m, prefs in enumerate((g_refs, u_refs)):
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)
                o_ref[m:m + 1, lanes] = _matvec_lanes(decode, _rows_fn(prefs, offs, lanes), tb, bx_ref, nb, C)

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, R), lambda i, idx_ref: (0, (idx_ref[i] * rows_p[k]) // B, 0))
        return pl.BlockSpec((B, R), lambda i, idx_ref: ((idx_ref[i] * rows_p[k]) // B, 0))

    in_specs = [plane_spec(kk) for kk in keys] * 2
    in_specs += [pl.BlockSpec(tbl.shape, lambda i, idx_ref: (0, 0)),
                 pl.BlockSpec((None, W, C), lambda i, idx_ref: (i // k, 0, 0))]
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=1, grid=(Nk,), in_specs=in_specs,
                                             out_specs=pl.BlockSpec((None, 2, R), lambda i, idx_ref: (i, 0, 0)),
                                             scratch_shapes=[pltpu.VMEM((nb * C, SUB, LANES), jnp.float32)])
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((Nk, 2, R), jnp.float32),
                        interpret=interpret, compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",)))
    out = fn(idx, *[planes_g[k] for k in keys], *[planes_u[k] for k in keys], tbl, X)
    return out[:, 0], out[:, 1]


# ---------------------------------------------------------------- grouped slots (a multi-token step of one sequence)
# A verify of T tokens of ONE sequence routes S = T*k slots, and neighbouring tokens share experts (v6e j64: ~24
# distinct of 32 slots at T = 4). These variants take one grid step per DISTINCT expert: its weights are decoded once
# and multiplied into the accumulators of every slot routed to it, each in exactly the per-slot kernel's order
# (bit-identical results). The step runs the body of the smallest size class >= its slot count (the classes are
# unrolled bodies of 1..tm accumulators; a group below its class repeats its last slot, which recomputes and rewrites
# the same values), so a single-token group pays one MAC per weight, not tm. Activations and outputs stay whole in
# VMEM (dynamic slot indices, no reordering outside the kernel); gate+up broadcasts every token's activation once for
# the whole call. Groups past n_groups have count 0: they repeat the last expert id (no new plane DMA) and skip compute.
# (The first version, 2026-10-04, ran every group at tm = T with per-group broadcasts: 21.9 vs 17.3 ms per DFlash step.)
def group_plan(idx):
    """idx int32 [T, k] (a token's k experts distinct) -> eids int32 [S] (the distinct experts in first-slot order,
    then the last one repeated), cnt int32 [S] (slots per group, 0 past the distinct count), slots int32 [S * T]
    (row g = the slots of group g in slot order; entries past cnt repeat the group's last slot). Exact integer ops on
    S x S compares (no sort, no gather)."""
    T, k = idx.shape
    S = T * k
    flat = idx.reshape(-1)
    eq = flat[:, None] == flat[None, :]
    earlier = jnp.tril(eq, -1)
    first = ~jnp.any(earlier, axis=1)                                   # the first slot of each expert
    gid = jnp.cumsum(first.astype(jnp.int32)) - 1
    g_of = jnp.sum(jnp.where(eq & first[None, :], gid[None, :], 0), axis=1)     # [S] the group of every slot
    rank = jnp.sum(earlier, axis=1, dtype=jnp.int32)                    # [S] its place in the group
    n_groups = jnp.sum(first, dtype=jnp.int32)
    ga = jnp.arange(S, dtype=jnp.int32)
    member = g_of[None, :] == ga[:, None]                               # [G, S]
    cnt = jnp.sum(member, axis=1, dtype=jnp.int32)
    eids = jnp.sum(jnp.where(member & first[None, :], flat[None, :], 0), axis=1)
    last = jnp.sum(jnp.where(ga == n_groups - 1, eids, 0))
    eids = jnp.where(ga < n_groups, eids, last)
    want = jnp.minimum(jnp.arange(T, dtype=jnp.int32)[None, :], jnp.maximum(cnt - 1, 0)[:, None])     # [G, T]
    hit = member[:, None, :] & (rank[None, None, :] == want[:, :, None])                              # [G, T, S]
    slots = jnp.sum(jnp.where(hit, ga[None, None, :], 0), axis=2)
    return eids.astype(jnp.int32), cnt, slots.reshape(-1).astype(jnp.int32)


def _class_bodies(classes, cnt, body):
    """Run body(c) for the size class c that holds `cnt` slots (classes ascending; cnt 0 runs nothing)."""
    lo = 1
    for c in classes:
        pl.when((cnt >= lo) & (cnt <= c))(functools.partial(body, c))
        lo = c + 1


def _fill_bx_at(x_ref, xi, bx_ref, base, nb, C):
    """`_fill_bx` for row xi of a whole-array X ref into bx_ref[base:base + nb*C]."""
    for b in range(nb):
        xb = x_ref[xi, SUB * b:SUB * (b + 1), :]
        for c in range(C):
            bx_ref[base + (b * C + c)] = jnp.broadcast_to(xb[:, c:c + 1], (SUB, LANES))


def _matvec_lanes_grouped(decode, rows, tb, bx_ref, bases, nb, C):
    """`_matvec_lanes` for the slots sharing the expert: each weight vreg decoded once, one accumulator per slot whose
    activation broadcasts start at bx_ref[bases[j]] -> len(bases) x (1, 128), each in `_matvec_lanes`' order."""
    n = len(bases)
    outs = [jnp.zeros((SUB, LANES), jnp.float32) for _ in range(n)]
    for b in range(nb):
        for scale, items in decode(b, rows, tb):
            accs = [jnp.zeros((SUB, LANES), jnp.float32) for _ in range(n)]
            for c, val in items:
                for j in range(n):
                    accs[j] = accs[j] + val * bx_ref[bases[j] + (b * C + c)]
            for j in range(n):
                outs[j] = outs[j] + accs[j] * scale
    return [jnp.sum(o, axis=0, keepdims=True) for o in outs]


def _decode_store(decode, rows, tb, w_ref, s_ref, wi, si, nb):
    """Decode one lane vreg's weights once into VMEM: values to w_ref[wi:], scales to s_ref[si:], in `_matvec_lanes`'
    order -> (wi, si) after them and the static plan [(b, [c, ...]) per scale] for `_mac_stored`."""
    plan = []
    for b in range(nb):
        for scale, items in decode(b, rows, tb):
            s_ref[si] = scale
            si += 1
            for c, val in items:
                w_ref[wi] = val
                wi += 1
            plan.append((b, [c for c, _ in items]))
    return wi, si, plan


def _mac_stored(w_ref, s_ref, wi, si, plan, bx_ref, base, C):
    """`_matvec_lanes` of one slot from the values `_decode_store` wrote (same products and sums, same order)."""
    out = jnp.zeros((SUB, LANES), jnp.float32)
    for b, cs in plan:
        acc = jnp.zeros((SUB, LANES), jnp.float32)
        for c in cs:
            acc = acc + w_ref[wi] * bx_ref[base + (b * C + c)]
            wi += 1
        out = out + acc * s_ref[si]
        si += 1
    return jnp.sum(out, axis=0, keepdims=True)


def _n_values(qtype, nb):
    """(values, scales) that DECODE[qtype] produces for one lane vreg of nb word blocks (an abstract trace)."""
    counts = []

    def f():
        z = lambda k, r0, n: jnp.zeros((n, LANES), jnp.uint32)                             # noqa: E731
        tbl = _table_arrays(qtype)
        tb = [jnp.zeros((SUB, LANES), tbl.dtype)] * (1 if qtype == "IQ4_XS" else tbl.shape[0])
        out = []
        for b in range(nb):
            for scale, items in DECODE[qtype](b, z, tb):
                counts.append(len(items))
                out.append(scale)
        return out
    jax.eval_shape(f)
    return sum(counts), len(counts)


def _n_steps(cnt, G, dynamic):
    return jnp.sum(cnt > 0, dtype=jnp.int32) if dynamic else G


def _grouped_geometry(planes, qtype, nblk, eids, cnt, slots, classes, loop):
    """Shapes of a grouped call; classes = the fused (unrolled) body sizes: without `loop` up to tm (default 1..tm),
    with `loop` the listed ones below tm (default (1,)) and larger counts take the decode-once loop."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    G = eids.shape[0]
    assert cnt.shape == (G,) and slots.shape[0] % G == 0, (eids.shape, cnt.shape, slots.shape)
    tm = slots.shape[0] // G
    classes = tuple(classes) or ((1,) if loop else tuple(range(1, tm)))
    classes = tuple(sorted(set(c for c in classes if c < tm))) + (() if loop else (tm,))
    first = planes[keys[0]]
    return (keys, rows_p, PL.WORDS_PER_BLOCK[qtype] * nblk, PL.PLANES_PER_WORD[qtype], first.ndim == 3,
            first.shape[-1], G, tm, classes, {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys})


@_shared("qtype", "nblk", "k", "classes", "interpret", "dynamic_grid", "loop", "vmem_mb")
def moe_matvec_gu_grouped(planes_g, planes_u, qtype, nblk, eids, cnt, slots, X, *, k, classes=(), interpret=False,
                          dynamic_grid=False, loop=False, vmem_mb=32):
    """`moe_matvec_gu` over grouped slots (`group_plan`): X f32 [T, W, C] one row per TOKEN (slot s is token s // k)
    -> (g, u) f32 [S, R] each, in slot order. classes: the unrolled body sizes below tm (= T, always one); () = all.
    dynamic_grid: one grid step per distinct expert (a traced grid bound) instead of G with empty steps at the end.
    loop: groups above the fused classes decode the expert once into VMEM and run one MAC pass per slot from it (a
    fori_loop over the group's slots: the kernel's code no longer grows with the class sizes; compile time).
    vmem_mb: the scoped VMEM limit (every token's activation broadcasts stay in VMEM: 2 MB per token at D = 4096)."""
    keys, rows_p, W, C, lead, R, G, tm, classes, blocks = _grouped_geometry(planes_g, qtype, nblk, eids, cnt, slots,
                                                                            classes, loop)
    T = X.shape[0]
    S = T * k
    assert X.shape == (T, W, C) and tm <= T, (X.shape, (T, W, C), tm)
    assert R % LANES == 0 and R <= 1024 and W % SUB == 0, (R, W)
    assert all(planes_u[kk].shape == planes_g[kk].shape for kk in keys)
    n_vreg = R // LANES
    nb = W // SUB
    nk = len(keys)
    tbl = _table_arrays(qtype)
    decode = DECODE[qtype]

    nv, ns = _n_values(qtype, nb)

    def kernel(eid_ref, cnt_ref, slot_ref, *refs):
        g_refs, u_refs = dict(zip(keys, refs[:nk])), dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, bx_ref = refs[2 * nk:2 * nk + 4]
        g = pl.program_id(0)                       # (scalars read outside pl.when: interpret mode cannot inside)
        offs = _offsets(keys, rows_p, eid_ref[g])
        n = cnt_ref[g]
        sl = [slot_ref[g * tm + j] for j in range(tm)]
        bases = [(s // k) * (nb * C) for s in sl]

        @pl.when(g == 0)
        def _():                                   # every token's broadcasts, once for the call
            for t in range(T):
                _fill_bx_at(x_ref, t, bx_ref, t * nb * C, nb, C)

        def body(c):
            tb = _table_rows(tbl_ref, qtype)
            for m, prefs in enumerate((g_refs, u_refs)):
                for v in range(n_vreg):
                    lanes = slice(v * LANES, (v + 1) * LANES)
                    res = _matvec_lanes_grouped(decode, _rows_fn(prefs, offs, lanes), tb, bx_ref, bases[:c], nb, C)
                    for j in range(c):
                        o_ref[sl[j], m:m + 1, lanes] = res[j]
        _class_bodies(classes, n, body)

        def decode_once_loop():
            w_ref, s_ref = refs[2 * nk + 4:]
            tb = _table_rows(tbl_ref, qtype)
            plans, wi, si = [], 0, 0
            for m, prefs in enumerate((g_refs, u_refs)):
                for v in range(n_vreg):
                    wi0, si0 = wi, si
                    wi, si, plan = _decode_store(decode, _rows_fn(prefs, offs, slice(v * LANES, (v + 1) * LANES)),
                                                 tb, w_ref, s_ref, wi, si, nb)
                    plans.append((m, v, wi0, si0, plan))

            def one_slot(j, carry):
                s_ = slot_ref[g * tm + j]
                base = (s_ // k) * (nb * C)
                for m, v, wi0, si0, plan in plans:
                    o_ref[s_, m:m + 1, v * LANES:(v + 1) * LANES] = _mac_stored(w_ref, s_ref, wi0, si0, plan, bx_ref,
                                                                                 base, C)
                return carry
            lax.fori_loop(0, n, one_slot, 0)
        if loop:
            pl.when(n > (classes[-1] if classes else 0))(decode_once_loop)

    def plane_spec(kk):
        B = blocks[kk]
        if lead:
            return pl.BlockSpec((None, B, R), lambda i, e, c, s: (0, (e[i] * rows_p[kk]) // B, 0))
        return pl.BlockSpec((B, R), lambda i, e, c, s: ((e[i] * rows_p[kk]) // B, 0))

    in_specs = [plane_spec(kk) for kk in keys] * 2
    in_specs += [pl.BlockSpec(tbl.shape, lambda i, e, c, s: (0, 0)),
                 pl.BlockSpec((T, W, C), lambda i, e, c, s: (0, 0, 0))]
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=3, grid=(_n_steps(cnt, G, dynamic_grid),),
                                             in_specs=in_specs,
                                             out_specs=pl.BlockSpec((S, 2, R), lambda i, e, c, s: (0, 0, 0)),
                                             scratch_shapes=[pltpu.VMEM((T * nb * C, SUB, LANES), jnp.float32)] + (
                                                 [pltpu.VMEM((2 * n_vreg * nv, SUB, LANES), jnp.float32),
                                                  pltpu.VMEM((2 * n_vreg * ns, SUB, LANES), jnp.float32)] if loop else []))
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((S, 2, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",),
                                                             vmem_limit_bytes=vmem_mb * 2 ** 20))
    out = fn(eids, cnt, slots, *[planes_g[kk] for kk in keys], *[planes_u[kk] for kk in keys], tbl, X)
    return out[:, 0], out[:, 1]


@_shared("qtype", "nblk", "classes", "interpret", "dynamic_grid", "loop", "vmem_mb")
def moe_matvec_grouped(planes, qtype, nblk, eids, cnt, slots, X, *, classes=(), interpret=False, dynamic_grid=False,
                       loop=False, vmem_mb=32):
    """`moe_matvec` over grouped slots (`group_plan`): X f32 [S, W, C] one row per SLOT -> f32 [S, R] in slot order
    (all R rows in one grid step per group; the group's slots are broadcast at the start of its step; `loop`: see
    `moe_matvec_gu_grouped`, each slot's broadcasts made in its pass)."""
    keys, rows_p, W, C, lead, R, G, tm, classes, blocks = _grouped_geometry(planes, qtype, nblk, eids, cnt, slots,
                                                                            classes, loop)
    S = X.shape[0]
    assert X.shape == (S, W, C), (X.shape, (S, W, C))
    assert R % LANES == 0 and R <= 4096 and W % SUB == 0, (R, W)
    n_vreg = R // LANES
    nb = W // SUB
    tbl = _table_arrays(qtype)
    decode = DECODE[qtype]

    nv, ns = _n_values(qtype, nb)

    def kernel(eid_ref, cnt_ref, slot_ref, *refs):
        plane_refs = dict(zip(keys, refs[:len(keys)]))
        tbl_ref, x_ref, o_ref, bx_ref = refs[len(keys):len(keys) + 4]
        g = pl.program_id(0)
        offs = _offsets(keys, rows_p, eid_ref[g])
        n = cnt_ref[g]
        sl = [slot_ref[g * tm + j] for j in range(tm)]

        def body(c):
            for j in range(c):
                _fill_bx_at(x_ref, sl[j], bx_ref, j * nb * C, nb, C)
            tb = _table_rows(tbl_ref, qtype)
            bases = [j * nb * C for j in range(c)]
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)
                res = _matvec_lanes_grouped(decode, _rows_fn(plane_refs, offs, lanes), tb, bx_ref, bases, nb, C)
                for j in range(c):
                    o_ref[sl[j], :, lanes] = res[j]
        _class_bodies(classes, n, body)

        def decode_once_loop():
            w_ref, s_ref = refs[len(keys) + 4:]
            tb = _table_rows(tbl_ref, qtype)
            plans, wi, si = [], 0, 0
            for v in range(n_vreg):
                wi0, si0 = wi, si
                wi, si, plan = _decode_store(decode, _rows_fn(plane_refs, offs, slice(v * LANES, (v + 1) * LANES)),
                                             tb, w_ref, s_ref, wi, si, nb)
                plans.append((v, wi0, si0, plan))

            def one_slot(j, carry):
                s_ = slot_ref[g * tm + j]
                _fill_bx_at(x_ref, s_, bx_ref, 0, nb, C)
                for v, wi0, si0, plan in plans:
                    o_ref[s_, :, v * LANES:(v + 1) * LANES] = _mac_stored(w_ref, s_ref, wi0, si0, plan, bx_ref, 0, C)
                return carry
            lax.fori_loop(0, n, one_slot, 0)
        if loop:
            pl.when(n > (classes[-1] if classes else 0))(decode_once_loop)

    def plane_spec(kk):
        B = blocks[kk]
        if lead:
            return pl.BlockSpec((None, B, R), lambda i, e, c, s: (0, (e[i] * rows_p[kk]) // B, 0))
        return pl.BlockSpec((B, R), lambda i, e, c, s: ((e[i] * rows_p[kk]) // B, 0))

    in_specs = [plane_spec(kk) for kk in keys]
    in_specs += [pl.BlockSpec(tbl.shape, lambda i, e, c, s: (0, 0)),
                 pl.BlockSpec((S, W, C), lambda i, e, c, s: (0, 0, 0))]
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=3, grid=(_n_steps(cnt, G, dynamic_grid),),
                                             in_specs=in_specs,
                                             out_specs=pl.BlockSpec((S, 1, R), lambda i, e, c, s: (0, 0, 0)),
                                             scratch_shapes=[pltpu.VMEM((tm * nb * C, SUB, LANES), jnp.float32)] + (
                                                 [pltpu.VMEM((n_vreg * nv, SUB, LANES), jnp.float32),
                                                  pltpu.VMEM((n_vreg * ns, SUB, LANES), jnp.float32)] if loop else []))
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((S, 1, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",),
                                                             vmem_limit_bytes=vmem_mb * 2 ** 20))
    return fn(eids, cnt, slots, *[planes[kk] for kk in keys], tbl, X).reshape(S, R)


def moe_matvec_ref(planes, qtype, nblk, idx, X):
    """Pure-JAX reference (gathers the experts, dequantizes with glm53.planes, einsum)."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    sel = {}
    for k in keys:
        a = planes[k]
        a = a[0] if a.ndim == 3 else a
        R = a.shape[-1]
        a = a.reshape(-1, rows_p[k], R)
        sel[k] = jnp.take(a, idx, axis=0)                                    # [Nk, rows_p, R]
    V = PL.dequant_planes(sel, qtype, nblk)                                 # [Nk, C*W, R]
    x_pm = jnp.swapaxes(X, 1, 2).reshape(X.shape[0], -1)                   # [Nk, C*W]
    return jnp.einsum("ni,nir->nr", x_pm, V, precision="highest")


# ------------------------------------------------------------------------------------------ prefill (sweep) kernels
# Dequantize each expert ONCE into (K, 128) bf16 tiles of W^T in VMEM and MXU-matmul all T tokens against them.
# Tile rows are ordered (word block b, value plane c, word s): input(8b+s, c) -> row (b*C + c)*8 + s; `pm_mxu`
# permutes activations to that order. Expert slots come from scalar-prefetched `ids` (unique active experts first,
# inactive slots repeat the last active id so their planes are never re-DMA'd) and `n_active` gates the compute.
def pm_mxu(qtype, x):
    """x [..., n] natural -> [..., n] in the kernel tile order (b, c, s)."""
    X = PL.pm_x(qtype, x)                                                    # [..., W, C]
    lead = X.shape[:-2]
    W, C = X.shape[-2:]
    y = X.reshape(lead + (W // SUB, SUB, C))
    y = jnp.swapaxes(y, -1, -2)                                              # [..., W/8, C, 8]
    return y.reshape(lead + (W * C,))


def _tiles(decode, rows, tb, b0, nb, dtype=jnp.bfloat16):
    """(nb*C*8, 128) tile of scaled W^T rows for word blocks b0..b0+nb-1 at the current lanes, in `dtype`."""
    parts = []
    for b in range(b0, b0 + nb):
        items = []
        for scale, its in decode(b, rows, tb):
            items += [(c, v * scale) for c, v in its]
        items.sort(key=lambda t: t[0])
        parts += [v for _, v in items]
    return jnp.concatenate(parts, axis=0).astype(dtype)


def _dot(a, b):
    prec = lax.Precision.HIGHEST if a.dtype == jnp.float32 else None
    return jnp.dot(a, b, preferred_element_type=jnp.float32, precision=prec)


def active_slots(idx, E):
    """idx int32 [N, k] -> (ids int32 [E], n_active int32): unique routed experts first (ascending), the rest of the
    slots repeat the last active id."""
    hit = (jax.nn.one_hot(idx.reshape(-1), E, dtype=jnp.int32).sum(0) > 0)          # [E] bool
    order = jnp.argsort(jnp.where(hit, 0, 1) * E + jnp.arange(E))                     # active ids first, ascending
    n_active = hit.sum().astype(jnp.int32)
    last = order[jnp.maximum(n_active - 1, 0)]
    ids = jnp.where(jnp.arange(E) < n_active, order, last).astype(jnp.int32)
    return ids, n_active


def _k_group(qtype):
    """word blocks per MXU tile so that K >= 128."""
    return max(1, 128 // (PL.PLANES_PER_WORD[qtype] * SUB))


def _probe_tiles(probe, tiles_fn, K, dtype):
    """Cost-split probes: 'mxu' replaces the dequantized tile by a constant (matmul cost only), 'deq' keeps the
    dequant but returns None so the caller skips the dot (dequant cost only)."""
    if probe == "mxu":
        return jnp.full((K, LANES), 0.01, dtype)
    return tiles_fn()


def moe_sweep_gateup(planes_g, planes_u, qtype, nblk, ids, n_active, xk, limit, *, interpret=False, lane_block=None,
                     t_block=512, vmem_mb=48, probe=None):
    """planes_{g,u}: {k: u32 [1, E*rows_p, R]} (same qtype); ids [E] int32; n_active int32 scalar; xk [T, n_in] in
    pm_mxu order (bf16 on TPU; f32 -> f32 tiles + HIGHEST-precision dots for the CPU tests) ->
    h [E, T, R] (xk.dtype) = swiglu_clamped(x @ gate_e^T, x @ up_e^T) for active slots, 0 elsewhere.
    `probe` (timing only, wrong numbers): 'mxu' = constant tiles, 'deq' = dequant without the matmul."""
    dtype = xk.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    E = ids.shape[0]
    T, n_in = xk.shape
    assert n_in == nblk * PL.QK, (n_in, nblk)
    first = planes_g[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    LB = min(lane_block or 1024, R)
    TB = min(T, t_block)
    assert T % TB == 0 and R % LB == 0
    n_lb, n_vreg, n_tb = R // LB, LB // LANES, T // TB
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(ids_ref, nact_ref, *refs):
        g_refs = dict(zip(keys, refs[:nk]))
        u_refs = dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, acc_g, acc_u = refs[2 * nk:]
        e = pl.program_id(1)
        eid = ids_ref[e]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        tb = _table_rows(tbl_ref, qtype)

        @pl.when(e >= nact_ref[0])
        def _inactive():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

        @pl.when(e < nact_ref[0])
        def _active():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows_of(prefs, lanes=lanes):
                    def rows(k, r0, n):
                        if offs[k] is None:
                            return prefs[k][r0:r0 + n, lanes]
                        return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)
                    return rows

                rg, ru = rows_of(g_refs), rows_of(u_refs)
                ag = jnp.zeros((TB, LANES), jnp.float32)
                au = jnp.zeros((TB, LANES), jnp.float32)
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    xs = x_ref[:, k0:k0 + K]                                              # (TB, K) bf16
                    tg = _probe_tiles(probe, lambda: _tiles(decode, rg, tb, b0, nbg, dtype), K, dtype)
                    tu = _probe_tiles(probe, lambda: _tiles(decode, ru, tb, b0, nbg, dtype), K, dtype)
                    if probe == "deq":
                        ag = ag + jnp.sum(tg, axis=0, keepdims=True).astype(jnp.float32)
                        au = au + jnp.sum(tu, axis=0, keepdims=True).astype(jnp.float32)
                        continue
                    ag = ag + _dot(xs, tg)
                    au = au + _dot(xs, tu)
                acc_g[:, lanes] = ag
                acc_u[:, lanes] = au
            o_ref[...] = M.swiglu_clamped(acc_g[...], acc_u[...], limit).astype(o_ref.dtype)

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda t, e, cb, ids_ref, n_ref: (0, (ids_ref[e] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda t, e, cb, ids_ref, n_ref: ((ids_ref[e] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys] * 2
    in_specs += [pl.BlockSpec(tbl.shape, lambda t, e, cb, ids_ref, n_ref: (0, 0)),
                 pl.BlockSpec((TB, n_in), lambda t, e, cb, ids_ref, n_ref: (t, 0))]
    out_spec = pl.BlockSpec((None, TB, LB), lambda t, e, cb, ids_ref, n_ref: (e, t, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_tb, E, n_lb), in_specs=in_specs,
                                             out_specs=out_spec,
                                             scratch_shapes=[pltpu.VMEM((TB, LB), jnp.float32)] * 2)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((E, T, R), dtype),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",) * 3,
                                                             vmem_limit_bytes=vmem_mb << 20))
    args = [planes_g[k] for k in keys] + [planes_u[k] for k in keys] + [tbl, xk]
    return fn(ids, n_active.reshape(1), *args)


def moe_sweep_down(planes, qtype, nblk, ids, n_active, hk, *, interpret=False, lane_block=None, t_block=512,
                   vmem_mb=48, probe=None):
    """planes {k: u32 [1, E*rows_p, R]}; hk [E, T, n_in] (pm_mxu order, routing weights folded in; bf16 or f32) ->
    y f32 [T, R] = sum over active slots of hk[e] @ down_e^T."""
    dtype = hk.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    E, T, n_in = hk.shape
    assert n_in == nblk * PL.QK and ids.shape == (E,)
    first = planes[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    LB = min(lane_block or 1024, R)
    TB = min(T, t_block)
    assert T % TB == 0 and R % LB == 0
    n_lb, n_vreg, n_tb = R // LB, LB // LANES, T // TB
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(ids_ref, nact_ref, *refs):
        prefs = dict(zip(keys, refs[:nk]))
        tbl_ref, h_ref, o_ref = refs[nk:]
        e = pl.program_id(2)
        eid = ids_ref[e]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        tb = _table_rows(tbl_ref, qtype)

        @pl.when(e == 0)
        def _init():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

        @pl.when(e < nact_ref[0])
        def _active():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows(k, r0, n, lanes=lanes):
                    if offs[k] is None:
                        return prefs[k][r0:r0 + n, lanes]
                    return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)

                acc = o_ref[:, lanes]
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    td = _probe_tiles(probe, lambda: _tiles(decode, rows, tb, b0, nbg, dtype), K, dtype)
                    if probe == "deq":
                        acc = acc + jnp.sum(td, axis=0, keepdims=True).astype(jnp.float32)
                        continue
                    acc = acc + _dot(h_ref[:, k0:k0 + K], td)
                o_ref[:, lanes] = acc

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda t, cb, e, ids_ref, n_ref: (0, (ids_ref[e] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda t, cb, e, ids_ref, n_ref: ((ids_ref[e] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys]
    in_specs += [pl.BlockSpec(tbl.shape, lambda t, cb, e, ids_ref, n_ref: (0, 0)),
                 pl.BlockSpec((None, TB, n_in), lambda t, cb, e, ids_ref, n_ref: (e, t, 0))]
    out_spec = pl.BlockSpec((TB, LB), lambda t, cb, e, ids_ref, n_ref: (t, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_tb, n_lb, E), in_specs=in_specs,
                                             out_specs=out_spec)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((T, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",) * 3,
                                                             vmem_limit_bytes=vmem_mb << 20))
    return fn(ids, n_active.reshape(1), *[planes[k] for k in keys], tbl, hk)


# ------------------------------------------------------------------------------------- grouped (ragged) prefill GEMM
# The dense sweep multiplies every 512-token chunk against every active expert (masked): on real text ~all 288 experts
# are active, so ~36x the useful MXU work, each expert dequantized once per chunk, and an [E, T, ml] intermediate.
# Here the (token, expert) slots are SORTED BY EXPERT into blocks of `tm` rows (each expert's rows start at a block
# boundary, zero-padded); the kernels walk the blocks in order, DMA + dequantize an expert's planes ONCE (when the
# block's expert changes; the dequantized W^T tiles stay in VMEM scratch) and multiply only that expert's rows.
def ragged_plan(idx, w, E, tm):
    """Slot layout for the grouped kernels. idx int32 [T, k], w f32 [T, k] -> dict with
    blk_expert int32 [G] (expert of row block g; G = ceil(T*k / tm) + min(E, T*k) blocks of `tm` rows — every routed
    expert needs ceil(count / tm) <= count / tm + 1 blocks, and at most min(E, T*k) experts are routed — blocks >=
    n_blocks are padding and repeat the last expert), n_blocks int32 [1], tok_row int32 [Rp] (token of every sorted
    row, -1 = padding), w_row f32 [Rp] (its routing weight), row_slot int32 [T, k] (the row of every (token, slot)).
    (A verify of 8 tokens: 2 + 64 blocks instead of 2 + 288 — each padding block is a grid step.)"""
    T, k = idx.shape
    S = T * k
    G = -(-S // tm) + min(E, S)
    Rp = G * tm
    flat = idx.reshape(-1).astype(jnp.int32)
    order = jnp.argsort(flat)
    se = flat[order]
    counts = jnp.sum(jax.nn.one_hot(flat, E, dtype=jnp.int32), axis=0)                    # [E]
    padded = -(-counts // tm) * tm
    pstart = jnp.cumsum(padded) - padded
    ustart = jnp.cumsum(counts) - counts
    row_sorted = pstart[se] + (jnp.arange(S, dtype=jnp.int32) - ustart[se])              # row of sorted slot j
    tok_row = jnp.full((Rp,), -1, jnp.int32).at[row_sorted].set((order // k).astype(jnp.int32))
    w_row = jnp.zeros((Rp,), jnp.float32).at[row_sorted].set(w.reshape(-1)[order].astype(jnp.float32))
    row_slot = jnp.zeros((S,), jnp.int32).at[order].set(row_sorted).reshape(T, k)
    nb = padded // tm
    bstart = jnp.cumsum(nb) - nb
    n_blocks = jnp.sum(nb).astype(jnp.int32)
    g = jnp.arange(G, dtype=jnp.int32)
    be = jnp.sum((g[:, None] >= (bstart + nb)[None, :]).astype(jnp.int32), axis=1)        # experts ended at or before g
    be = jnp.minimum(be, E - 1)
    last = be[jnp.maximum(n_blocks - 1, 0)]
    be = jnp.where(g < n_blocks, be, last).astype(jnp.int32)
    return {"blk_expert": be, "n_blocks": n_blocks.reshape(1), "tok_row": tok_row, "w_row": w_row, "row_slot": row_slot}


def _ragged_call(kernel, planes_list, keys, rows_p, blocks, tbl, act, out_dtype, R, LB, tm, G, scratch, interpret, vmem_mb,
                 be, n_blocks):
    """Shared pallas_call plumbing of the two ragged kernels: grid (lane block, row block)."""
    lead = planes_list[0][keys[0]].ndim == 3
    n_lb = R // LB
    Rp, n_in = act.shape

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda cb, g, be_ref, nb_ref: (0, (be_ref[g] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda cb, g, be_ref, nb_ref: ((be_ref[g] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys] * len(planes_list)
    in_specs += [pl.BlockSpec(tbl.shape, lambda cb, g, be_ref, nb_ref: (0, 0)),
                 pl.BlockSpec((tm, n_in), lambda cb, g, be_ref, nb_ref: (jnp.minimum(g, nb_ref[0] - 1), 0))]
    out_spec = pl.BlockSpec((tm, LB), lambda cb, g, be_ref, nb_ref: (g, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_lb, G), in_specs=in_specs,
                                             out_specs=out_spec, scratch_shapes=scratch)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((Rp, R), out_dtype),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary", "arbitrary"),
                                                             vmem_limit_bytes=vmem_mb << 20))
    args = [p[k] for p in planes_list for k in keys] + [tbl, act]
    return fn(be, n_blocks, *args)


@_shared("qtype", "nblk", "limit", "tm", "interpret", "lane_block", "vmem_mb")
def moe_ragged_gateup(planes_g, planes_u, qtype, nblk, blk_expert, n_blocks, xs, limit, *, tm=32, interpret=False,
                      lane_block=None, vmem_mb=48):
    """Grouped prefill GEMM, gate + up. xs [Rp, n_in] in pm_mxu order, rows sorted by expert in blocks of `tm` rows
    (block g = rows g*tm.., expert blk_expert[g]; blocks >= n_blocks are padding) -> h [Rp, R] (xs.dtype) =
    swiglu_clamped(x @ gate_e^T, x @ up_e^T) per row with e = its block's expert; padding blocks are zero."""
    dtype = xs.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    Rp, n_in = xs.shape
    assert n_in == nblk * PL.QK and n_in == W * C and Rp % tm == 0, (n_in, nblk, W, C, Rp, tm)
    G = Rp // tm
    assert blk_expert.shape == (G,), (blk_expert.shape, G)
    R = planes_g[keys[0]].shape[-1]
    LB = min(lane_block or 1024, R)
    assert R % LB == 0
    n_vreg = LB // LANES
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(be_ref, nb_ref, *refs):
        g_refs = dict(zip(keys, refs[:nk]))
        u_refs = dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, wg, wu = refs[2 * nk:]
        g = pl.program_id(1)
        eid = be_ref[g]
        prev = be_ref[jnp.maximum(g - 1, 0)]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        tb = _table_rows(tbl_ref, qtype)

        @pl.when((g == 0) | (eid != prev))
        def _decode():                                     # this expert's W^T tiles for the lane block, once
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows_of(prefs, lanes=lanes):
                    def rows(k, r0, n):
                        if offs[k] is None:
                            return prefs[k][r0:r0 + n, lanes]
                        return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)
                    return rows

                rg, ru = rows_of(g_refs), rows_of(u_refs)
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    wg[k0:k0 + K, lanes] = _tiles(decode, rg, tb, b0, nbg, dtype)
                    wu[k0:k0 + K, lanes] = _tiles(decode, ru, tb, b0, nbg, dtype)

        @pl.when(g < nb_ref[0])
        def _rows():
            x = x_ref[...]                                                                  # (tm, n_in)
            o_ref[...] = M.swiglu_clamped(_dot(x, wg[...]), _dot(x, wu[...]), limit).astype(o_ref.dtype)

        @pl.when(g >= nb_ref[0])
        def _pad():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

    scratch = [pltpu.VMEM((n_in, LB), dtype)] * 2
    return _ragged_call(kernel, [planes_g, planes_u], keys, rows_p, blocks, tbl, xs, dtype, R, LB, tm, G, scratch,
                        interpret, vmem_mb, blk_expert, n_blocks)


@_shared("qtype", "nblk", "tm", "interpret", "lane_block", "vmem_mb", "out_dtype")
def moe_ragged_down(planes, qtype, nblk, blk_expert, n_blocks, hs, *, tm=32, interpret=False, lane_block=None,
                    vmem_mb=48, out_dtype=None):
    """Grouped prefill GEMM, down. hs [Rp, n_in] (pm_mxu order, sorted rows as in `moe_ragged_gateup`) ->
    y [Rp, R] (out_dtype, default hs.dtype) = h @ down_e^T per row; padding blocks are zero."""
    dtype = hs.dtype
    out_dtype = out_dtype or dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    Rp, n_in = hs.shape
    assert n_in == nblk * PL.QK and n_in == W * C and Rp % tm == 0, (n_in, nblk, W, C, Rp, tm)
    G = Rp // tm
    assert blk_expert.shape == (G,), (blk_expert.shape, G)
    R = planes[keys[0]].shape[-1]
    LB = min(lane_block or 1024, R)
    assert R % LB == 0
    n_vreg = LB // LANES
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(be_ref, nb_ref, *refs):
        prefs = dict(zip(keys, refs[:nk]))
        tbl_ref, h_ref, o_ref, wd = refs[nk:]
        g = pl.program_id(1)
        eid = be_ref[g]
        prev = be_ref[jnp.maximum(g - 1, 0)]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        tb = _table_rows(tbl_ref, qtype)

        @pl.when((g == 0) | (eid != prev))
        def _decode():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows(k, r0, n, lanes=lanes):
                    if offs[k] is None:
                        return prefs[k][r0:r0 + n, lanes]
                    return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)

                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    wd[k0:k0 + K, lanes] = _tiles(decode, rows, tb, b0, nbg, dtype)

        @pl.when(g < nb_ref[0])
        def _rows():
            o_ref[...] = _dot(h_ref[...], wd[...]).astype(o_ref.dtype)

        @pl.when(g >= nb_ref[0])
        def _pad():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

    scratch = [pltpu.VMEM((n_in, LB), dtype)]
    return _ragged_call(kernel, [planes], keys, rows_p, blocks, tbl, hs, out_dtype, R, LB, tm, G, scratch, interpret,
                        vmem_mb, blk_expert, n_blocks)
'''
FILES["glm53/planes.py"] = r'''"""Planar ("lane = matrix row") storage of codebook-quantized expert matrices for the resident engine.

The raw GGUF block layout (82/110/136 bytes per 256 weights) makes every field a byte slice at an odd offset, which
costs XLA one relayout copy + one "header parsing" fusion per field per step and is unusable from a Pallas kernel.
Here each block field becomes its own dense 32-bit plane, transposed so that the matrix ROW is the lane (minor) axis
and the 4-byte "word" of packed codes is the sublane axis:

    word w of a row = 4 consecutive qs bytes = 4 codebook groups (4w .. 4w+3); byte j of the word is group 4w+j and
    each group holds GROUP_W[qtype] weights (positions k). So one (8, 128) vreg of the qs plane covers 8 words x 128
    rows and the weight it describes at (byte j, position k) is input index INPUT(w, j, k) of those 128 rows.

Per expert the planes hold exactly the bytes of the GGUF blocks (no expansion) except `d`, which is stored as pairs
of f16 bit patterns in one u32 per two blocks. All planes of one table are E-merged: shape [E * rows_p, R] so that
every array is (8, 128)-tile dense whatever rows_p is (a [E, 1, 4096] array would pad 1 -> 8 sublanes = 8x waste).

Formats (W = words per row, C = value planes per word = bytes x positions, `input(w, c)`):
  IQ2_S : qs u32 [W, R]; sg u32 [W, R] (sign byte j of word w = group 4w+j, bit k = position k);
          qh u32 [W/4, R] (byte w%4 of u32 w//4: bits 2j..2j+1 = high index bits of group 4w+j);
          sc u32 [W/4, R] (byte w%4 of u32 w//4: nibble h = scale of groups 4w+2h, 4w+2h+1);
          d  u32 [ceil(nblk/2), R] (f16 bits of block b at half b%2 of u32 b//2; block b = w // 8).
          W = 8*nblk, C = 32, input = 32w + 8j + k.
  IQ3_S : qs u32 [W, R]; sg u32 [W/2, R] (half w%2 of u32 w//2: nibble j = group 4w+j, bit k = position k);
          qh u32 [W/8, R] (bits 4(w%8)+j of u32 w//8 = high bit of group 4w+j);
          sc u32 [W/16, R] (nibble (w//2)%8 of u32 w//16 = scale of words 2n, 2n+1); d as above, block b = w // 16.
          W = 16*nblk, C = 16, input = 16w + 4j + k.
  IQ4_XS: qs u32 [W, R] (byte j of word w: low nibble = position 0, high = position 1); sl u32 [nblk, R] (nibble i =
          low scale bits of sub-block i); sh u32 [nblk, R] (bits 2i..2i+1 = high scale bits); d as above.
          W = 32*nblk, C = 8 (c = 4*pos + j), input = 32*(w//4) + 16*pos + 4*(w%4) + j.

The XLA dequant returns each expert as [C*W, R] with input index c*W + w ("pm order"); `pm_x` permutes activations
to match: x_pm[c*W + w] = x[input(w, c)], and the per-word activation matrix used by the Pallas kernel is
X[w, c] = x[input(w, c)] (same data as x_pm reshaped [C, W] and transposed).
"""
import numpy as np
import jax
import jax.numpy as jnp

from glm53 import iqquant as Q

QK = Q.QK_K
U32 = np.dtype("<u4")

WORDS_PER_BLOCK = {"IQ2_S": 8, "IQ3_S": 16, "IQ4_XS": 32, "Q2_K": 16, "Q3_K": 16}
PLANES_PER_WORD = {"IQ2_S": 32, "IQ3_S": 16, "IQ4_XS": 8, "Q2_K": 16, "Q3_K": 16}   # C
PLANE_KEYS = {"IQ2_S": ("qs", "sg", "qh", "sc", "d"), "IQ3_S": ("qs", "sg", "qh", "sc", "d"),
              "IQ4_XS": ("qs", "sl", "sh", "d"), "Q2_K": ("qs", "sc", "d"), "Q3_K": ("qs", "hm", "sc", "d")}
# K-quants (the MTP layer's experts; no codebook): word w of a row = 4 qs bytes; block b = w // 16, half n = (w % 16)
# // 8, q = w % 8; byte j of the word is ggml byte l = 4 q + j of half n and holds 4 two-bit values (shift s):
#   input(w, c) = 256 b + 128 n + 32 s + 4 q + j with c = 4 s + j (C = 16).
#   Q2_K: sc u32 [4 nblk, R] (the block's 16 scale/min bytes: byte is = 8 n + 2 s + (q >= 4), low nibble scale,
#         high nibble min); d u32 [nblk, R] (f16 d in the low half, f16 dmin in the high half).
#   Q3_K: hm u32 [8 nblk, R] (u32 q of block b = hmask bytes 4 q .. 4 q + 3; bit 4 n + s of byte l = high bit);
#         sc u32 [4 nblk, R] (the 16 six-bit scales, pre-unpacked to bytes, same `is` indexing); d u32 [nblk, R].


def plane_rows(qtype, nblk):
    """rows_p per expert of every plane."""
    W = WORDS_PER_BLOCK[qtype] * nblk
    dd = (nblk + 1) // 2
    if qtype == "IQ2_S":
        return {"qs": W, "sg": W, "qh": W // 4, "sc": W // 4, "d": dd}
    if qtype == "IQ3_S":
        return {"qs": W, "sg": W // 2, "qh": W // 8, "sc": W // 16, "d": dd}
    if qtype == "IQ4_XS":
        return {"qs": W, "sl": nblk, "sh": nblk, "d": dd}
    if qtype == "Q2_K":
        return {"qs": W, "sc": 4 * nblk, "d": nblk}
    if qtype == "Q3_K":
        return {"qs": W, "hm": 8 * nblk, "sc": 4 * nblk, "d": nblk}
    raise ValueError(qtype)


def input_index(qtype, w, c):
    """Natural input index of value plane c of word w (numpy-vectorised over w, c)."""
    if qtype == "IQ2_S":
        return 32 * w + c                       # c = 8j + k
    if qtype == "IQ3_S":
        return 16 * w + c                       # c = 4j + k
    if qtype == "IQ4_XS":
        pos, j = c // 4, c % 4
        return 32 * (w // 4) + 16 * pos + 4 * (w % 4) + j
    if qtype in ("Q2_K", "Q3_K"):
        s_, j = c // 4, c % 4
        return 256 * (w // 16) + 128 * ((w % 16) // 8) + 32 * s_ + 4 * (w % 8) + j
    raise ValueError(qtype)


def pm_perm(qtype, n_inputs):
    """x_pm = x[perm]: perm[c*W + w] = input(w, c)."""
    W = n_inputs * WORDS_PER_BLOCK[qtype] // QK
    C = PLANES_PER_WORD[qtype]
    c, w = np.meshgrid(np.arange(C), np.arange(W), indexing="ij")
    return input_index(qtype, w, c).reshape(-1).astype(np.int32)


def pm_x(qtype, x):
    """x [..., n] natural -> X [..., W, C] with X[w, c] = x[input(w, c)] (reshape/transpose only, no gather)."""
    n = x.shape[-1]
    lead = x.shape[:-1]
    if qtype == "IQ2_S":
        return x.reshape(lead + (n // 32, 32))
    if qtype == "IQ3_S":
        return x.reshape(lead + (n // 16, 16))
    if qtype == "IQ4_XS":                                     # input = 32*sub + 16*pos + 4*wq + j -> [sub, pos, wq, j]
        y = x.reshape(lead + (n // 32, 2, 4, 4))
        y = jnp.swapaxes(y, -3, -2)                           # [sub, wq, pos, j]
        return y.reshape(lead + (n // 8, 8))                  # w = 4*sub + wq, c = 4*pos + j
    if qtype in ("Q2_K", "Q3_K"):                             # input = 256b + 128n + 32s + 4q + j -> [b, n, s, q, j]
        y = x.reshape(lead + (n // 256, 2, 4, 8, 4))
        y = jnp.swapaxes(y, -3, -2)                           # [b, n, q, s, j]
        return y.reshape(lead + (n // 16, 16))                # w = 16b + 8n + q, c = 4s + j
    raise ValueError(qtype)


def pm_flat(qtype, x):
    """x [..., n] natural -> x_pm [..., n] with x_pm[c*W + w] = x[input(w, c)]."""
    X = pm_x(qtype, x)
    return jnp.swapaxes(X, -1, -2).reshape(x.shape)


# ------------------------------------------------------------------------------------------------ numpy packing
def _u32(a):
    """[E, R, nblk, n] u8 -> [E, R, nblk*n/4] u32 (little-endian words, group order preserved)."""
    E, R = a.shape[:2]
    a = np.ascontiguousarray(a.reshape(E, R, -1))
    assert a.shape[-1] % 4 == 0, a.shape
    return a.view(U32)


def _merge(a):
    """[E, R, P] -> [E*P, R] (planes are stored expert-major, rows_p sublanes per expert, matrix rows on lanes)."""
    E, R, P = a.shape
    return np.ascontiguousarray(a.transpose(0, 2, 1)).reshape(E * P, R)


def pack_planes(t, qtype):
    """t uint8 [E, R, nblk, block_bytes] (one chip's slice of one expert table) -> {plane: u32 [E*rows_p, R]}."""
    E, R, nblk, bb = t.shape
    assert bb == Q.BLOCK_BYTES[qtype], (bb, qtype)
    f = lambda a, b: t[..., a:b]
    if qtype in ("Q2_K", "Q3_K"):
        if qtype == "Q2_K":
            out = {"qs": _u32(f(16, 80)), "sc": _u32(f(0, 16)), "d": _u32(f(80, 84))}
        else:
            sc16 = Q.q3k_scales(f(96, 108))                                              # [E, R, nblk, 16] u8
            d = np.concatenate([f(108, 110), np.zeros(t.shape[:3] + (2,), np.uint8)], -1)
            out = {"qs": _u32(f(32, 96)), "hm": _u32(f(0, 32)), "sc": _u32(sc16), "d": _u32(d)}
        rows = plane_rows(qtype, nblk)
        out = {k: _merge(out[k]) for k in PLANE_KEYS[qtype]}
        for k, a in out.items():
            assert a.shape == (E * rows[k], R), (k, a.shape, rows[k])
        return out
    d = t[..., 0:2].reshape(E, R, nblk * 2)
    if nblk % 2:
        d = np.concatenate([d, np.zeros((E, R, 2), np.uint8)], -1)
    out = {"d": d.copy().view(U32)}
    if qtype == "IQ2_S":
        out.update(qs=_u32(f(2, 34)), sg=_u32(f(34, 66)), qh=_u32(f(66, 74)), sc=_u32(f(74, 82)))
    elif qtype == "IQ3_S":
        out.update(qs=_u32(f(2, 66)), qh=_u32(f(66, 74)), sg=_u32(f(74, 106)), sc=_u32(f(106, 110)))
    elif qtype == "IQ4_XS":
        sh = np.concatenate([f(2, 4), np.zeros(t.shape[:3] + (2,), np.uint8)], -1)
        out.update(sh=_u32(sh), sl=_u32(f(4, 8)), qs=_u32(f(8, 136)))
    else:
        raise ValueError(qtype)
    rows = plane_rows(qtype, nblk)
    out = {k: _merge(out[k]) for k in PLANE_KEYS[qtype]}
    for k, a in out.items():
        assert a.shape == (E * rows[k], R), (k, a.shape, rows[k])
    return out


def planes_bytes(planes):
    return sum(int(a.nbytes) if hasattr(a, "nbytes") else int(np.prod(a.shape)) * 4 for a in planes.values())


# ------------------------------------------------------------------------------------------------ XLA dequant
def _f16_pairs(d, nblk):
    """d u32 [E, ceil(nblk/2), R] -> f32 [E, nblk, R] block scales."""
    E, _, R = d.shape
    halves = jnp.stack([d & 0xFFFF, d >> 16], axis=2).reshape(E, -1, R)[:, :nblk]        # [E, nblk, R] u32
    return jax.lax.bitcast_convert_type(halves.astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _bytes(a, n=4, bits=8):
    """u32 [E, P, R] -> [E, P*n, R] sub-fields of `bits` bits, field m of word p at index p*n + m."""
    E, P, R = a.shape
    mask = (1 << bits) - 1
    return jnp.stack([(a >> (bits * m)) & mask for m in range(n)], axis=2).reshape(E, P * n, R)


def _sgn(bits, k):
    return 1.0 - 2.0 * ((bits >> k) & 1).astype(jnp.float32)


def dequant_planes(planes, qtype, nblk):
    """planes {k: u32 [E, rows_p, R]} -> f32 [E, C*W, R] (pm order along axis 1)."""
    if qtype == "IQ2_S":
        return _deq_iq2_s(planes, nblk)
    if qtype == "IQ3_S":
        return _deq_iq3_s(planes, nblk)
    if qtype == "IQ4_XS":
        return _deq_iq4_xs(planes, nblk)
    if qtype in ("Q2_K", "Q3_K"):
        return _deq_kq(planes, qtype, nblk)
    raise ValueError(qtype)


def _f16_lo(u):
    return jax.lax.bitcast_convert_type((u & 0xFFFF).astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _kq_scale_index(W):
    """Scale byte of word w (into the block-major 16-byte scale lists): is = 8 n + 2 s + (q >= 4), block-offset 16 b."""
    w = np.arange(W)
    base = 16 * (w // 16) + 8 * ((w % 16) // 8) + ((w % 8) >= 4)
    return [base + 2 * s_ for s_ in range(4)]                                       # per shift s


def _deq_kq(p, qtype, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R], W = 16 nblk
    E, W, R = qs.shape
    scb = _bytes(p["sc"]).astype(jnp.int32)                                     # [E, 16 nblk, R] scale bytes (`is` order)
    idx = _kq_scale_index(W)
    d = jnp.repeat(_f16_lo(p["d"]), 16, axis=1)                                 # [E, W, R]
    if qtype == "Q2_K":
        dmin = jnp.repeat(_f16_lo(p["d"] >> 16), 16, axis=1)
    else:
        hmb = _bytes(p["hm"]).astype(jnp.int32)                                 # [E, 32 nblk, R] hmask byte l of block b
        w = np.arange(W)
        hidx = [32 * (w // 16) + 4 * (w % 8) + j for j in range(4)]              # byte l = 4 q + j
        nhalf = (w % 16) // 8
    outs = []
    for c in range(16):
        s_, j = c // 4, c % 4
        v = ((qs >> (8 * j + 2 * s_)) & 3).astype(jnp.float32)
        sc = jnp.take(scb, jnp.asarray(idx[s_]), axis=1)                        # [E, W, R]
        if qtype == "Q2_K":
            outs.append(d * (sc & 15).astype(jnp.float32) * v - dmin * (sc >> 4).astype(jnp.float32))
        else:
            hb = (jnp.take(hmb, jnp.asarray(hidx[j]), axis=1) >> jnp.asarray(4 * nhalf + s_)[None, :, None]) & 1
            outs.append(d * (sc - 32).astype(jnp.float32) * (v - 4.0 * (1 - hb).astype(jnp.float32)))
    return jnp.stack(outs, axis=1).reshape(E, 16 * W, R)


def _deq_iq2_s(p, nblk):
    qs, sg = p["qs"].astype(jnp.int32), p["sg"].astype(jnp.int32)               # [E, W, R]
    E, W, R = qs.shape
    qhb = _bytes(p["qh"]).astype(jnp.int32)                                     # [E, W, R] qh byte of word w
    scb = _bytes(p["sc"]).astype(jnp.int32)                                     # [E, W, R]
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 8, axis=1)                         # [E, W, R] block = w // 8
    scale = [d * (0.5 + ((scb >> (4 * h)) & 15).astype(jnp.float32)) * 0.25 for h in range(2)]
    outs = []
    for j in range(4):
        idx = ((qs >> (8 * j)) & 255) | (((qhb >> (2 * j)) & 3) << 8)
        sgn = (sg >> (8 * j)) & 255
        vals = Q._tree_pos("IQ2_S", idx)                                        # 8 int8 [E, W, R]
        outs += [scale[j // 2] * v.astype(jnp.float32) * _sgn(sgn, k) for k, v in enumerate(vals)]
    return jnp.stack(outs, axis=1).reshape(E, 32 * W, R)


def _deq_iq3_s(p, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R]
    E, W, R = qs.shape
    sgh = _bytes(p["sg"], 2, 16).astype(jnp.int32)                              # [E, W, R] 16 sign bits of word w
    qhn = _bytes(p["qh"], 8, 4).astype(jnp.int32)                               # [E, W, R] 4 high bits of word w
    scn = jnp.repeat(_bytes(p["sc"], 8, 4), 2, axis=1).astype(jnp.float32)      # [E, W, R] scale nibble of word w
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 16, axis=1)                        # [E, W, R] block = w // 16
    scale = d * (1.0 + 2.0 * scn)
    outs = []
    for j in range(4):
        idx = ((qs >> (8 * j)) & 255) | (((qhn >> j) & 1) << 8)
        sgn = (sgh >> (4 * j)) & 15
        vals = Q._tree_pos("IQ3_S", idx)                                        # 4 int8 [E, W, R]
        outs += [scale * v.astype(jnp.float32) * _sgn(sgn, k) for k, v in enumerate(vals)]
    return jnp.stack(outs, axis=1).reshape(E, 16 * W, R)


def _deq_iq4_xs(p, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R], W = 32*nblk
    E, W, R = qs.shape
    sl = _bytes(p["sl"], 8, 4).astype(jnp.int32)                                # [E, 8*nblk, R] sub-block i
    sh = _bytes(p["sh"], 8, 2).astype(jnp.int32)                                # [E, 8*nblk, R]
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 8, axis=1)                         # [E, 8*nblk, R]
    scale = jnp.repeat(d * ((sl | (sh << 4)) - 32).astype(jnp.float32), 4, axis=1)   # [E, W, R] sub-block = w // 4
    outs = []
    for pos in range(2):
        for j in range(4):
            nib = ((qs >> (8 * j + 4 * pos)) & 15)
            outs.append(scale * Q._kv16(nib))
    return jnp.stack(outs, axis=1).reshape(E, 8 * W, R)


def dequant_natural(planes, qtype, nblk):
    """Test helper: [E, C*W, R] pm order -> [E, R, n_inputs] natural order."""
    v = dequant_planes(planes, qtype, nblk)
    E, _, R = v.shape
    n = nblk * QK
    inv = np.argsort(pm_perm(qtype, n))
    return jnp.take(jnp.swapaxes(v, 1, 2), jnp.asarray(inv), axis=-1)
'''
FILES["glm53/quant8.py"] = r'''"""int8 storage for the big NON-expert matrices (attention projections, shared experts, dense MLPs): per-channel
absmax scales, dequantized to bf16 at the top of each layer program (XLA fuses the convert into the matmul operand).
Frees ~1.1 GB of HBM per chip on v5e-8 (18 GB bf16 -> 9 GB). Routers stay fp32; norms/biases/1-D/small arrays untouched.

A quantized leaf is the dict {"q": int8 [in, out], "s": scale} with the scale laid out so it shards like the weight:
weights sharded on axis 0 get per-row scales [in, 1]; everything else per-column scales [1, out]."""
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P

MIN_SIZE = 1 << 20
SKIP_KEYS = {"router_w", "router_bias", "fn", "base", "scale", "gate_up", "down_", "embed", "lm_head"}
Q8_KEYS = frozenset({"q", "s"})


def is_q8(x):
    return isinstance(x, dict) and set(x.keys()) == Q8_KEYS


def eligible(key, a, spec):
    return (key not in SKIP_KEYS and getattr(a, "ndim", 0) == 2 and a.size >= MIN_SIZE
            and isinstance(spec, P) and a.dtype in (jnp.bfloat16, jnp.float32, np.dtype("float32"), np.dtype("bfloat16")))


def quantize_array(w, axis):
    """w [in, out] -> {"q": int8, "s": fp32}; axis 0 = per-row scales [in,1], axis 1 = per-column [1,out]."""
    w32 = jnp.asarray(w).astype(jnp.float32)
    amax = jnp.max(jnp.abs(w32), axis=1 - axis, keepdims=True)
    s = jnp.maximum(amax, 1e-12) / 127.0
    q = jnp.clip(jnp.round(w32 / s), -127, 127).astype(jnp.int8)
    return {"q": q, "s": s}


def quantize_array_host(w, axis, rows=8192):
    """`quantize_array` in NumPy on the host, chunked over rows: for the 155k x 4096 embedding / lm_head (a jnp
    version materialises a 2.5 GB f32 copy on chip 0 and OOMs a full HBM)."""
    w = np.asarray(w)
    n = w.shape[0]
    if axis == 1:                                            # per-column scales need the whole column: two passes
        amax = np.zeros((1, w.shape[1]), np.float32)
        for a in range(0, n, rows):
            amax = np.maximum(amax, np.abs(w[a:a + rows].astype(np.float32)).max(axis=0, keepdims=True))
        s = np.maximum(amax, 1e-12) / 127.0
        q = np.empty(w.shape, np.int8)
        for a in range(0, n, rows):
            q[a:a + rows] = np.clip(np.round(w[a:a + rows].astype(np.float32) / s), -127, 127).astype(np.int8)
        return {"q": q, "s": s}
    q = np.empty(w.shape, np.int8)
    s = np.empty((n, 1), np.float32)
    for a in range(0, n, rows):
        w32 = w[a:a + rows].astype(np.float32)
        sa = np.maximum(np.abs(w32).max(axis=1, keepdims=True), 1e-12) / 127.0
        s[a:a + rows] = sa
        q[a:a + rows] = np.clip(np.round(w32 / sa), -127, 127).astype(np.int8)
    return {"q": q, "s": s}


def scale_axis(spec):
    return 0 if (len(spec) >= 1 and spec[0] is not None) else 1


def quantize_layer(p, spec, fn=quantize_array):
    """Recursively quantize eligible leaves of one layer's params; returns (params, specs) with matching structure."""
    if isinstance(p, dict):
        out_p, out_s = {}, {}
        for k, v in p.items():
            sp = spec[k] if isinstance(spec, dict) else spec
            if isinstance(v, dict):
                out_p[k], out_s[k] = quantize_layer(v, sp, fn)
            elif eligible(k, v, sp):
                out_p[k] = fn(v, scale_axis(sp))
                out_s[k] = {"q": sp, "s": sp}
            else:
                out_p[k], out_s[k] = v, sp
        return out_p, out_s
    return p, spec


def expand_specs(p, spec):
    """Specs for params that are already quantized (q8 dict nodes where the spec is a single PartitionSpec)."""
    if is_q8(p) and isinstance(spec, P):
        return {"q": spec, "s": spec}
    if isinstance(p, dict) and isinstance(spec, dict):
        return {k: expand_specs(p[k], spec[k]) for k in p}
    if isinstance(p, list) and isinstance(spec, list):
        return [expand_specs(a, b) for a, b in zip(p, spec)]
    return spec


def dequant_tree(p, dtype=jnp.bfloat16):
    """Replace q8 nodes with dequantized arrays (inside a jitted program)."""
    if is_q8(p):
        return (p["q"].astype(jnp.float32) * p["s"]).astype(dtype)
    if isinstance(p, dict):
        return {k: dequant_tree(v, dtype) for k, v in p.items()}
    if isinstance(p, list):
        return [dequant_tree(v, dtype) for v in p]
    return p


def count_bytes(p):
    if is_q8(p):
        return p["q"].size + p["s"].size * 4
    if isinstance(p, dict):
        return sum(count_bytes(v) for v in p.values())
    if isinstance(p, list):
        return sum(count_bytes(v) for v in p)
    return getattr(p, "nbytes", 0)
'''
FILES["glm53/resident.py"] = r'''"""HBM-resident codebook-quantized experts (Unsloth UD-IQ*/Q2_K_XL GGUF) for the fully-jitted `Engine`.

Storage per sparse layer (chip-major, `P(AXIS)` on axis 0 hands each chip its slice; TP over moe_inter): every table
(gate_q, up_q, down_q) is a dict of PLANAR arrays `{plane: u32 [n_dev, E * rows_p, R]}` (glm53.planes): the GGUF block
fields split into dense 32-bit planes with the matrix row on the lane axis, so a Pallas kernel can dequantize an
expert in VMEM without byte slicing and XLA needs no relayout copies. gate/up: R = ml rows (this chip's slice of
moe_inter), inputs D; down: R = D rows, inputs ml.

Tables live in `params["layers"][i]["mlp"]`; `M.moe_dense` calls `ResidentFetch.apply`:
  decode (N*k <= gather_max_rows and fewer than `ragged_min_seq` tokens of one sequence): `glm53.pallas_moe`
  `moe_matvec_gu` + `moe_matvec` per (token, expert) slot (fused dequant + VPU matvec, the expert planes DMA'd by
  index), or the XLA path (dynamic slices + glm53.planes.dequant_planes + einsum) when use_pallas=False;
  prefill and multi-token verifies of one sequence: the grouped (ragged) kernels — every routed expert dequantized
  once, so the cost follows the DISTINCT experts (consecutive tokens share many; independent streams almost none).
Activations are permuted to the planes' "pm" input order (planes.pm_x / pm_flat); outputs are in natural row order.
"""
import dataclasses
import queue
import threading

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax

from glm53 import aot
from glm53 import iqquant as Q
from glm53 import model as M
from glm53 import planes as PL
from glm53 import pallas_moe as K

QK = Q.QK_K
KEYS = ("gate_q", "up_q", "down_q")


def pack_layer_from_gguf(gm, layer: int, n_dev: int, threads: int = 8):
    """Read one layer's routed experts from a `gguf_reader.GGUFModel` -> (tables dict, qtypes dict)."""
    out, qtypes = {}, {}
    for key, tname in (("gate_q", "gate"), ("up_q", "up"), ("down_q", "down")):
        name = f"blk.{layer}.ffn_{tname}_exps.weight"
        info = gm.info(name)
        ne0, ne1, E = info["dims"]                       # ne0 = input dim (blocks along it), ne1 = rows, E experts
        bb = Q.BLOCK_BYTES[info["type"]]
        raws = gm.read_experts(name, range(E), threads=threads)
        t = np.stack([np.frombuffer(r, np.uint8).reshape(ne1, ne0 // QK, bb) for r in raws])   # [E, rows, nblk, bb]
        if key == "down_q":                              # split blocks (the mi input dim) across chips
            t = t.reshape(E, ne1, n_dev, ne0 // QK // n_dev, bb).transpose(2, 0, 1, 3, 4)
        else:                                            # split rows (the mi output dim) across chips
            t = t.reshape(E, n_dev, ne1 // n_dev, ne0 // QK, bb).transpose(1, 0, 2, 3, 4)
        out[key] = pack_chip_planes(t, info["type"])
        qtypes[key] = info["type"]
    return out, qtypes


def pack_chip_planes(t, qtype, threads=None):
    """t uint8 [n_dev, E, rows, nblk, bb] -> {plane: u32 [n_dev, E*rows_p, rows]} (glm53.planes layout per chip).
    The chips' slices are packed in parallel threads (NumPy releases the GIL on the big slices; the packing, not the
    GGUF read, dominated the 9-minute build: the dataset mount serves ~650 MB/s to 8-16 readers, 2026-09-14)."""
    n = t.shape[0]
    threads = n if threads is None else max(1, int(threads))
    if threads > 1 and n > 1:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(min(threads, n)) as ex:
            per_chip = list(ex.map(lambda c: PL.pack_planes(t[c], qtype), range(n)))
    else:
        per_chip = [PL.pack_planes(t[c], qtype) for c in range(n)]
    return {k: np.stack([pc[k] for pc in per_chip]) for k in per_chip[0]}


def hbm_bytes(tables: dict) -> int:
    """Bytes per chip of a layer's tables ({key: {plane: [n_dev, ...]}})."""
    return sum(int(a.nbytes) // int(a.shape[0]) for planes in tables.values() for a in planes.values())


class ResidentFetch:
    """Stored on the Engine; `M.moe_dense` calls `.apply(p, x, idx, w, layer, limit)` when present."""

    def __init__(self, qtypes: dict, D: int, ml: int, out_dtype=jnp.bfloat16, gather_max_rows: int = 64, chunk: int = 8,
                 use_pallas: bool = True, interpret: bool = False, sweep_mode: str = "ragged", tm: int = 32,
                 combine: str = "matmul"):
        self.qtypes = qtypes            # {layer: {"gate_q": "IQ2_S", "up_q": "IQ2_S", "down_q": "IQ3_S"}}
        self.rows = {"gate_q": ml, "up_q": ml, "down_q": D}     # logical rows per expert (this chip)
        self.nblk = {"gate_q": D // QK, "up_q": D // QK, "down_q": ml // QK}
        self.out_dtype = out_dtype
        self.gather_max_rows = gather_max_rows
        self.chunk = chunk
        self.use_pallas = use_pallas    # decode path: Pallas fused kernel (True) or XLA planes dequant + einsum
        self.interpret = interpret      # run the Pallas kernel in interpret mode (CPU tests)
        self.sweep_mode = sweep_mode    # prefill: "dense" = masked sweep over active experts, "ragged" = grouped GEMM
        self.tm = tm                    # ragged: rows per block (multiple of 16 for bf16)
        self.combine = combine          # ragged: per-token combine of slot outputs, "gather" (row gather) or "matmul"

    def _deq(self, layer, key, tbl, sel):
        """Dequantize the experts picked by `sel` from planes `tbl` -> [n, C*W, R] (pm order along inputs) in out_dtype."""
        qt = self.qtypes[layer][key]
        rows_p = PL.plane_rows(qt, self.nblk[key])
        picked = {k: sel(a, rows_p[k]).reshape(-1, rows_p[k], a.shape[-1]) for k, a in tbl.items()}
        return PL.dequant_planes(picked, qt, self.nblk[key]).astype(self.out_dtype)

    @staticmethod
    def _slice_experts(a, rows_p, flat):
        """a u32 [1, E*rows_p, R]; flat int32 [n] -> [n*rows_p, R] via n dynamic slices (DMA, never a generic gather
        and never a copy of the whole table)."""
        parts = [lax.dynamic_slice_in_dim(a, flat[j] * rows_p, rows_p, axis=1) for j in range(flat.shape[0])]
        return jnp.concatenate(parts, axis=1)[0]

    # Multi-token steps of ONE sequence (verifies): from `ragged_min_seq` tokens and up to `verify_max_rows` slots they
    # take the ragged kernels with these settings. v6e 2026-10-03 (j67-4c51, whole verify step): T=4 slot 18.2 / ragged
    # 17.9 ms, T=8 27.8 / 25.6, T=16 48.8 / 36.3 — below 8 tokens the per-slot decode kernels are as fast or faster.
    ragged_min_seq = 8
    verify_dtype = None        # activations/tiles (None = out_dtype; f32 = f32 tiles + HIGHEST dots: ~1.5x slower)
    verify_tm = 16             # rows per block (a verify has <= T rows per expert; 16 = the bf16 minimum)
    verify_lane_block = 4096   # lane block (the down kernel in one grid step per block: j66 -19 % MoE at T=8)
    verify_max_rows = 256
    group_min_seq = 2          # multi-token steps (2..ragged_min_seq-1 tokens of one sequence: the DFlash verify) decode
                               # each distinct expert once for all its slots (`_apply_kernel_grouped`, bit-identical).
                               # v6e 2026-10-04 (j126): DFlash step 15.95 -> 14.59 ms at T = 4 (with the chunked
                               # top-k). 0 = per slot.
    group_classes = ()         # unrolled body sizes below T (T itself is always one); () = 1..T
    group_dynamic_grid = True  # one kernel grid step per distinct expert (traced grid bound), no empty steps (j120b)
    group_loop = False         # groups above the fused classes: decode once into VMEM + a MAC loop over their slots
                               # (fixed code size; j120c: ~2 % slower than the classes at the same compile cost)
    group_rows = "all"         # a multi-token step of SEVERAL sequences (B rows x T tokens: the batched DFlash verify):
                               # "all" = one grouped call over every slot (an expert two rows share is decoded once;
                               # `group_rows_classes` + the decode-once loop), "row" = one grouped call per row (the
                               # one-row verify's kernels and shapes). v6e 2026-10-04 (j129, T = 4): DFlash step of
                               # 2 / 3 streams 24.1 / 34.7 ms per row, 23.4 / 32.7 all
    group_rows_classes = (1, 2, 3, 4)
    group_rows_vmem_mb = 64    # "all": every token's activation broadcasts in VMEM (2 MB each at D = 4096: 24 MB at 3 x 4)

    def apply(self, p, x, idx, w, layer, limit, seq_tokens=1):
        """x [N,D]; idx [N,k] int32 into E; w [N,k] fp32 -> partial MoE output [N,D] (TP-reduced by the caller).
        seq_tokens: consecutive tokens per sequence in x (a verify of T positions: T; B independent rows: 1)."""
        gate_q, up_q, down_q = p["gate_q"], p["up_q"], p["down_q"]         # {plane: [1, E*rows_p, R]} on this chip
        N, k = idx.shape
        if self.use_pallas and seq_tokens >= self.ragged_min_seq and N * k <= self.verify_max_rows:
            return self._apply_ragged_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit, self.verify_dtype,
                                             self.verify_tm, self.verify_lane_block)
        if (self.use_pallas and self.group_min_seq and self.group_min_seq <= seq_tokens < self.ragged_min_seq
                and N > seq_tokens and N % seq_tokens == 0):         # B rows of T tokens (a batched verify)
            return self._apply_rows_grouped(x, idx, w, layer, gate_q, up_q, down_q, limit, seq_tokens)
        if N * k <= self.gather_max_rows:
            if self.use_pallas and self.group_min_seq and seq_tokens >= self.group_min_seq and seq_tokens == N:
                return self._apply_kernel_grouped(x, idx, w, layer, gate_q, up_q, down_q, limit)
            if self.use_pallas:
                return self._apply_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
            return self._apply_gather(x, idx, w, layer, gate_q, up_q, down_q, limit)
        return self._apply_sweep(x, idx, w, layer, gate_q, up_q, down_q, limit)

    def _apply_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        N, k = idx.shape
        flat = idx.reshape(-1)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        assert self.qtypes[layer]["up_q"] == qt_gu, self.qtypes[layer]
        Xt = PL.pm_x(qt_gu, x.astype(jnp.float32))                                      # [N, W, C]: per token
        g, u = K.moe_matvec_gu(gate_q, up_q, qt_gu, self.nblk["gate_q"], flat, Xt, interpret=self.interpret, k=k)
        h = M.swiglu_clamped(g, u, limit)                                               # [Nk, ml]
        y = K.moe_matvec(down_q, qt_dn, self.nblk["down_q"], flat, PL.pm_x(qt_dn, h),
                         interpret=self.interpret).reshape(N, k, -1)                    # [N, k, D]
        return jnp.einsum("nkd,nk->nd", y, w)

    def _apply_kernel_grouped(self, x, idx, w, layer, gate_q, up_q, down_q, limit, classes=None, loop=None, vmem_mb=32):
        """`_apply_kernel` for the T tokens of ONE sequence (a verify): the T*k slots are grouped by expert (a token's k
        experts are distinct, so a group holds at most T slots) and each distinct expert is decoded once for all its
        slots (`pallas_moe.group_plan`, `moe_matvec_gu_grouped` / `moe_matvec_grouped`: bit-identical to the per-slot
        kernels, outputs already in slot order). classes / loop override `group_classes` / `group_loop`."""
        N, k = idx.shape
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        classes = self.group_classes if classes is None else classes
        loop = self.group_loop if loop is None else loop
        eids, cnt, slots = K.group_plan(idx)
        g, u = K.moe_matvec_gu_grouped(gate_q, up_q, qt_gu, self.nblk["gate_q"], eids, cnt, slots,
                                       PL.pm_x(qt_gu, x.astype(jnp.float32)), k=k, classes=classes,
                                       interpret=self.interpret, dynamic_grid=self.group_dynamic_grid, loop=loop,
                                       vmem_mb=vmem_mb)
        h = M.swiglu_clamped(g, u, limit)                                               # [Nk, ml]
        y = K.moe_matvec_grouped(down_q, qt_dn, self.nblk["down_q"], eids, cnt, slots, PL.pm_x(qt_dn, h),
                                 classes=classes, interpret=self.interpret,
                                 dynamic_grid=self.group_dynamic_grid, loop=loop, vmem_mb=vmem_mb).reshape(N, k, -1)
        return jnp.einsum("nkd,nk->nd", y, w)

    def _apply_rows_grouped(self, x, idx, w, layer, gate_q, up_q, down_q, limit, T):
        """B independent rows of T consecutive tokens each (x [B*T, D], row-major): `group_rows` "row" = the grouped
        kernels once per row (neighbouring tokens of a row share experts, independent rows hardly do), "all" = one
        grouped call over all B*T*k slots (groups of up to B*T slots: the unrolled classes `group_rows_classes`, larger
        groups through the decode-once loop)."""
        N = x.shape[0]
        if self.group_rows == "all":
            return self._apply_kernel_grouped(x, idx, w, layer, gate_q, up_q, down_q, limit,
                                              classes=self.group_rows_classes, loop=True,
                                              vmem_mb=self.group_rows_vmem_mb)
        return jnp.concatenate([self._apply_kernel_grouped(x[r:r + T], idx[r:r + T], w[r:r + T], layer, gate_q, up_q,
                                                           down_q, limit) for r in range(0, N, T)], 0)

    def _apply_gather(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        """XLA decode path on the planar tables (no Pallas): per-expert dynamic slices + dequant + einsum."""
        N, k = idx.shape
        flat = idx.reshape(-1)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        sel = lambda a, rows_p: self._slice_experts(a, rows_p, flat)
        g = self._deq(layer, "gate_q", gate_q, sel)                                     # [Nk, D_pm, ml]
        u = self._deq(layer, "up_q", up_q, sel)
        d = self._deq(layer, "down_q", down_q, sel)                                     # [Nk, ml_pm, D]
        xs = jnp.repeat(PL.pm_flat(qt_gu, x).astype(self.out_dtype), k, axis=0)         # [Nk, D_pm]
        h = M.swiglu_clamped(jnp.einsum("nd,ndm->nm", xs, g), jnp.einsum("nd,ndm->nm", xs, u), limit)   # [Nk, ml]
        h = PL.pm_flat(qt_dn, h).astype(d.dtype)
        y = jnp.einsum("nm,nmd->nd", h, d).astype(jnp.float32).reshape(N, k, -1)
        return jnp.einsum("nkd,nk->nd", y, w)

    def _apply_sweep(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        if self.use_pallas and self.sweep_mode == "ragged":
            return self._apply_ragged_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
        if self.use_pallas:
            return self._apply_sweep_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
        return self._apply_sweep_xla(x, idx, w, layer, gate_q, up_q, down_q, limit)

    def _apply_ragged_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit, dtype=None, tm=None, lane_block=None):
        """Prefill through the grouped GEMM kernels: (token, expert) slots sorted by expert into `tm`-row blocks,
        each expert dequantized once and multiplied against its own rows only; the per-token combine gathers each
        token's k slot rows (or, `combine="matmul"`, multiplies by a [T, Rp] weight matrix). dtype (default out_dtype)
        = activations, VMEM tiles and outputs; f32 = f32 tiles + HIGHEST-precision dots (a verify close to the decode
        kernels' f32 numerics)."""
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        E = gate_q["qs"].shape[1] // PL.plane_rows(qt_gu, self.nblk["gate_q"])["qs"]
        T, k = idx.shape
        tm = tm or self.tm
        plan = K.ragged_plan(idx, w, E, tm)
        tok = plan["tok_row"]
        dtype = dtype or self.out_dtype
        xk = K.pm_mxu(qt_gu, x.astype(jnp.float32)).astype(dtype)                             # [T, D]
        xs = jnp.take(xk, jnp.maximum(tok, 0), axis=0, mode="clip")
        xs = jnp.where((tok >= 0)[:, None], xs, jnp.zeros((), xs.dtype))                       # [Rp, D]
        h = K.moe_ragged_gateup(gate_q, up_q, qt_gu, self.nblk["gate_q"], plan["blk_expert"], plan["n_blocks"], xs,
                                limit, tm=tm, interpret=self.interpret, lane_block=lane_block)   # [Rp, ml]
        hk = K.pm_mxu(qt_dn, h.astype(jnp.float32)).astype(dtype)
        y = K.moe_ragged_down(down_q, qt_dn, self.nblk["down_q"], plan["blk_expert"], plan["n_blocks"], hk,
                              tm=tm, interpret=self.interpret, lane_block=lane_block)            # [Rp, D]
        if self.combine == "matmul" and dtype != jnp.float32:     # f32 (verify): the slot path's combine below
            wc = (tok[None, :] == jnp.arange(T, dtype=jnp.int32)[:, None]).astype(jnp.float32) * plan["w_row"][None, :]
            return jnp.dot(wc.astype(y.dtype), y, preferred_element_type=jnp.float32)
        ys = jnp.take(y, plan["row_slot"].reshape(-1), axis=0, mode="clip").reshape(T, k, -1).astype(jnp.float32)
        return jnp.einsum("tkd,tk->td", ys, w.astype(jnp.float32))

    SWEEP_TOKENS = 512     # token chunk per kernel launch (bounds the [E, T, ml] intermediate to ~75 MB per chip)

    def _apply_sweep_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        """Prefill through the Pallas sweep kernels: every routed expert is dequantized once into VMEM tiles and MXU-
        multiplied against all tokens; experts no token routes to are skipped (scalar-prefetched active slots)."""
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        E = gate_q["qs"].shape[1] // PL.plane_rows(qt_gu, self.nblk["gate_q"])["qs"]
        N = x.shape[0]
        ids, n_act = K.active_slots(idx, E)
        rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32),
                        (idx[:, :, None] == ids[None, None, :]).astype(jnp.float32))          # [N, E] per slot
        outs = []
        for t0 in range(0, N, self.SWEEP_TOKENS):
            xs = x[t0:t0 + self.SWEEP_TOKENS]
            xk = K.pm_mxu(qt_gu, xs.astype(jnp.float32)).astype(self.out_dtype)                # [T, D]
            h = K.moe_sweep_gateup(gate_q, up_q, qt_gu, self.nblk["gate_q"], ids, n_act, xk, limit,
                                   interpret=self.interpret)                                    # [E, T, ml] bf16
            hw = h.astype(jnp.float32) * jnp.swapaxes(rw[t0:t0 + self.SWEEP_TOKENS], 0, 1)[:, :, None]
            hk = K.pm_mxu(qt_dn, hw).astype(self.out_dtype)                                     # [E, T, ml]
            outs.append(K.moe_sweep_down(down_q, qt_dn, self.nblk["down_q"], ids, n_act, hk, interpret=self.interpret))
        return outs[0] if len(outs) == 1 else jnp.concatenate(outs, axis=0)

    def _apply_sweep_xla(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        E = gate_q["qs"].shape[1] // PL.plane_rows(self.qtypes[layer]["gate_q"], self.nblk["gate_q"])["qs"]
        CH = self.chunk
        assert E % CH == 0, (E, CH)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        onehot = jax.nn.one_hot(idx, E, dtype=jnp.float32)                  # [N,k,E]
        rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32), onehot)        # [N,E] routing weight (0 if unused)
        xg = PL.pm_flat(qt_gu, x).astype(self.out_dtype)                    # [N, D_pm]

        def body(acc, c):
            c0 = c * CH
            sel = lambda a, rows_p: lax.dynamic_slice_in_dim(a, c0 * rows_p, CH * rows_p, axis=1)[0]
            g = self._deq(layer, "gate_q", gate_q, sel)                                          # [CH, D_pm, ml]
            u = self._deq(layer, "up_q", up_q, sel)
            d = self._deq(layer, "down_q", down_q, sel)                                          # [CH, ml_pm, D]
            h = M.swiglu_clamped(jnp.einsum("nd,edm->nem", xg, g), jnp.einsum("nd,edm->nem", xg, u), limit)  # [N,CH,ml]
            rws = lax.dynamic_slice_in_dim(rw, c0, CH, 1)                                        # [N, CH]
            h = PL.pm_flat(qt_dn, h.astype(jnp.float32) * rws[..., None]).astype(d.dtype)
            return acc + jnp.einsum("nem,emd->nd", h, d).astype(jnp.float32), None

        acc, _ = lax.scan(body, jnp.zeros((xg.shape[0], x.shape[1]), jnp.float32), jnp.arange(E // CH))
        return acc


# ------------------------------------------------------------------------------------- per-layer-kind engine
from glm53.engine import Engine, AXIS, R, shard_map, cache_specs, device_sample, device_probs, spec_accept, DeviceSampler   # noqa: E402
from jax.sharding import NamedSharding, PartitionSpec as P           # noqa: E402


class _Reaper(threading.Thread):
    """Drops objects on a helper thread. A step leaves ~300 jax arrays behind (consumed caches, histories, the
    previous state); dropping one costs ~11-20 us of host time on the 8-chip mesh (j107), so the main thread hands
    them over instead of paying between its dispatches (`ResidentLayerEngine.reap`)."""

    def __init__(self):
        super().__init__(daemon=True, name="glm53-reaper")
        self.q = queue.SimpleQueue()
        self.start()

    def run(self):
        while True:
            item = self.q.get()
            del item

    def drop(self, *objs):
        self.q.put(objs)


class _HostShards:
    """A sharded device array parked in host memory as its per-device shards (a pytree leaf)."""

    def __init__(self, shards, devices, shape, dtype, sharding):
        self.shards, self.devices, self.shape, self.dtype, self.sharding = shards, devices, shape, dtype, sharding
        self.nbytes = sum(int(x.nbytes) for x in shards)

    @classmethod
    def of(cls, a):
        sh = sorted(a.addressable_shards, key=lambda s: s.device.id)
        return cls([np.asarray(s.data) for s in sh], [s.device for s in sh], a.shape, a.dtype, a.sharding)

    def to_device(self):
        parts = [jax.device_put(x, d) for x, d in zip(self.shards, self.devices)]
        return jax.make_array_from_single_device_arrays(self.shape, self.sharding, parts)


class ResidentLayerEngine(Engine):
    """Resident experts with ONE jitted program per layer *kind* (attention type, MLP type, expert quant types,
    indexer presence) instead of one 45-layer program: the layer's params are program arguments, so ~6 programs cover
    all layers and the compile stays small (a single unrolled program with 42 codebook dequants got the XLA compiler
    SIGKILLed on the Kaggle host). Per step: embed program, 45 layer programs, head program — all device-resident,
    no host gathers.

    Prefill is CHUNKED: the caches are allocated up front (`alloc_caches`) and the prompt is fed in pieces of
    `prefill_piece` tokens (the last one padded to a bucket) through cache-carrying programs, so no temporary scales
    with the prompt length; `prefill(tokens, caches, pos0)` continues an existing context (prefix caching). Cache
    buffers are DONATED to the layer programs (in-place update; never reuse a cache object after passing it in —
    `copy_caches` for snapshots)."""

    def __init__(self, cfg, params, expert_fetch, devices=None, max_len: int = 4096, layers_per_program: int = 1,
                 layers_per_program_prefill: int = 1, int8_nonexpert: bool = False, seq_shard: bool = False,
                 q_block: int = 128, prefill_piece: int = 2048, cache_q8: bool = False):
        super().__init__(cfg, params, None, devices, max_len, expert_fetch=expert_fetch, int8_nonexpert=int8_nonexpert,
                         seq_shard=seq_shard, q_block=q_block, cache_q8=cache_q8)
        self._progs = {}
        self._allocs = {}
        self.launches = 0
        assert prefill_piece in self.PREFILL_BUCKETS, prefill_piece
        self.prefill_piece = min(prefill_piece, max_len)
        # consecutive layer groups; a group's program is keyed by the tuple of its layer kinds (dispatch ≈ 2 ms per
        # launch on the Kaggle TPU VM, so 45 single-layer launches cost ~100 ms/token). Prefill keeps 1 layer per
        # program by default: a 4-layer prefill program (expert sweep scans inside) did not finish compiling in 50 min.
        self.lpp = {"decode": layers_per_program, "prefill": layers_per_program_prefill}
        self.groups = {m: [list(range(a, min(a + k, cfg.n_layers))) for a in range(0, cfg.n_layers, k)]
                       for m, k in self.lpp.items()}

    def _kind(self, i):
        qt = tuple(sorted(self.expert_fetch.qtypes.get(i, {}).items())) if self.cfg.mlp_types[i] == "sparse" else ()
        return (self.cfg.layer_types[i], self.cfg.mlp_types[i], qt, "indexer" in self.params["layers"][i]["attn"])

    def _prog_embed(self, B, T, override=False):
        """Token embeddings broadcast to the hc residual streams [B,T,hc,D]; with override=True the extra arguments
        (vectors [B,T,D], mask [B,T] bool) replace the table rows where mask is set (image tokens: glm53.vision)."""
        key = ("embed", B, T, override)
        if key not in self._progs:
            def prog(embed, tokens, *ov):
                x = self._embed(embed, tokens).astype(self.lcfg.dtype)
                if override:
                    vec, mask = ov
                    x = jnp.where(mask[..., None], vec.astype(x.dtype), x)
                return jnp.broadcast_to(x[:, :, None, :], (B, T, self.cfg.hc, x.shape[-1]))
            in_specs = (self.specs["embed"], R) + ((R, R) if override else ())
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=R, check_vma=False))
        return self._progs[key]

    def _prog_head(self, B, T, sample=False):
        """Logits [B,V] of the last valid position; with sample=True also the device-sampled token ids [B]
        (extra args: temperature, top_p, key — `DeviceSampler.bind(mesh)`), returned as (ids, logits, next key)."""
        key = ("head", B, T, sample)
        if key not in self._progs:
            def prog(norm, lm_head, streams, length, *samp):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                zl = self._logits_local(lm_head, last)
                logits = lax.all_gather(zl, AXIS, axis=-1, tiled=True)
                if not sample:
                    return logits
                temp, top_p, k = samp
                k1, k2 = jax.random.split(k)
                return device_sample(zl, temp, top_p, k1), logits, k2
            in_specs = (R, self.specs["lm_head"], R, R) + ((R, R, R) if sample else ())
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=in_specs,
                                                 out_specs=(R, R, R) if sample else R, check_vma=False))
        return self._progs[key]

    def _hist_spec(self, i, kin=False):
        """Sharding of the per-token histories a speculative verify step returns for layer i (None: no history);
        kin=True: the KDA layers' token update inputs instead of their states (`M.kda_recurrent(hist="kin")`)."""
        if self.cfg.layer_types[i] == "linear_attention":
            return {"kin_hist" if kin else "state_hist": P(None, None, AXIS), "conv_hist": P(None, None, None, AXIS)}
        if "indexer" in self.params["layers"][i]["attn"]:
            return {"tk_hist": R, "tg_hist": R}
        return None

    def _hist_layout(self, layers, kin=False):
        """How a group program returns the per-token histories of a verify: {hist key: positions j in the group that
        have it}, one array stacked over those layers per key (not one per layer: every array a program allocates for
        its outputs costs ~17.5 us of host time per dispatch on the 8-chip mesh, j101 2026-10-04)."""
        lay = {}
        for j, i in enumerate(layers):
            for k in (self._hist_spec(i, kin) or {}):
                lay.setdefault(k, []).append(j)
        return {k: tuple(v) for k, v in sorted(lay.items())}

    def _prog_group(self, layers, B, T, has_cache, use_recurrent, hist=False, capture=(), pend=None, embed=False):
        """capture = positions j in the group whose output's stream mean is returned as a last output, stacked
        [len(capture), B, T, D] (DFlash features); hist=True / "kin" returns the per-token histories stacked per key
        (`_hist_layout`; "kin": the KDA layers return their tokens' update inputs and leave their states as before the
        block, `M.kda_core`). pend = T of the previous verify whose rollback is still pending (`dflash_step`): the
        program then also takes that verify's histories of this group (DONATED: this verify's histories reuse their
        buffers) and its n_keep, and first rolls the small state entries of its layers back to the accepted prefix
        (the rollback program's own op, `M.rollback_states`), so no separate rollback program runs between the steps.
        embed=True: the streams argument is (embedding table, tokens [B,T]) and the program embeds them itself (the
        step's first group: no separate embed program and its output between the dispatches)."""
        kinds = tuple(self._kind(i) for i in layers)
        key = ("group", kinds, B, T, has_cache, use_recurrent, hist) + ((capture,) if capture else ()) + \
            ((("pend", pend),) if pend else ()) + (("embed",) if embed else ())
        if key not in self._progs:
            lts = [self.cfg.layer_types[i] for i in layers]
            mts = [self.cfg.mlp_types[i] for i in layers]
            css = [self._cache_spec(i) for i in layers]
            kin = hist == "kin"
            lay = self._hist_layout(layers, kin)
            hss = {k: P(None, *self._hist_spec(layers[js[0]], kin)[k]) for k, js in lay.items()}
            rep = list(layers)                     # representative layer ids (only used to look up expert qtypes)

            from glm53 import quant8 as Q8

            def prog(ps, streams, pos0, length, caches, hg=None, nk=None):
                new, hs, feats = [], [], []
                if embed:
                    streams = self._embed_streams(*streams)
                if pend:
                    caches = list(caches)
                    roll = M.rollback_states([{k: hg[k][js.index(j)] for k, js in lay.items() if j in js} or None
                                              for j in range(len(ps))], nk,
                                             [{k_: c[k_].dtype for k_ in c} for c in caches], caches)
                    caches = [c if r_ is None else {**c, **r_} for c, r_ in zip(caches, roll)]
                for j, p in enumerate(ps):
                    p = Q8.dequant_tree(p, self.lcfg.dtype)          # int8 non-expert matrices -> bf16 (fused into dots)
                    fetch = self.expert_fetch if mts[j] == "sparse" else None
                    streams, nc = M.decoder_layer(p, streams, self.lcfg, lts[j], mts[j],
                                                  caches[j] if has_cache else None, pos0, use_recurrent, fetch, "auto",
                                                  layer=rep[j], cap=self.max_len, length=None if use_recurrent else length,
                                                  hist=hist)
                    nc, h = M.split_hist(nc)
                    new.append(nc); hs.append(h)
                    if j in capture:
                        feats.append(self._stream_mean(streams))
                out = (streams, new)
                if hist:
                    out += ({k: jnp.stack([hs[j][k] for j in js]) for k, js in lay.items()},)
                return out + ((jnp.stack(feats),) if capture else ())
            in_specs = ([self.specs["layers"][i] for i in layers], (self.specs["embed"], R) if embed else R, R, R,
                        css if has_cache else None) + ((hss, R) if pend else ())
            out_specs = (R, css) + ((hss,) if hist else ()) + ((R,) if capture else ())
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=((4,) if has_cache else ()) + ((5,) if pend else ()))
        return self._progs[key]

    def _embed_streams(self, table, tokens):
        """Inside a program: tokens [B,T] -> the hc residual streams [B,T,hc,D] (what `_prog_embed` returns)."""
        x = self._embed(table, tokens).astype(self.lcfg.dtype)
        return jnp.broadcast_to(x[:, :, None, :], x.shape[:2] + (self.cfg.hc, x.shape[-1]))

    def _stream_mean(self, streams):
        """DFlash feature of a layer: the mean of its 4 output residual streams (SGLang's hc_contract) [B,T,D]; a
        verify shorter than the drafter's block is padded to the block (zero rows), so the next draft program takes
        the same shape after a verify of any T (one draft program per block size)."""
        m = streams.astype(jnp.float32).mean(2).astype(self.lcfg.dtype)
        T, blk = m.shape[1], self.dflash.cfg.block
        return jnp.pad(m, ((0, 0), (0, blk - T), (0, 0))) if T < blk else m

    def _prog_stream_mean(self, B, T):
        key = ("stream_mean", B, T)
        if key not in self._progs:
            self._progs[key] = aot.jit(key, shard_map(self._stream_mean, mesh=self.mesh, in_specs=(R,), out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_hidden(self, B, T):
        """Final-norm hidden states of every position [B,T,D] (MTP prefill input)."""
        key = ("hidden", B, T)
        if key not in self._progs:
            def prog(norm, streams):
                return M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=(R, R), out_specs=R, check_vma=False))
        return self._progs[key]

    def _prog_head_all(self, B, T, sample=False):
        """Logits [B,T,V] and hidden [B,T,D] of every position (speculative verify; T small); with sample=True
        (extra args as in `_prog_head`) also the device-sampled ids [B,T]: (ids, logits, hidden, next key)."""
        key = ("head_all", B, T, sample)
        if key not in self._progs:
            def prog(norm, lm_head, streams, *samp):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
                zl = self._logits_local(lm_head, h)
                logits = lax.all_gather(zl, AXIS, axis=-1, tiled=True)
                if not sample:
                    return logits, h
                temp, top_p, k = samp
                k1, k2 = jax.random.split(k)
                ids = device_sample(zl.reshape(B * T, -1), temp, top_p, k1)
                return ids.reshape(B, T), logits, h, k2
            in_specs = (R, self.specs["lm_head"], R) + ((R, R, R) if sample else ())
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=in_specs,
                                                 out_specs=(R, R, R, R) if sample else (R, R), check_vma=False))
        return self._progs[key]

    def _prog_rollback(self, B, T, kin=False):
        """the per-group stacked histories of a T-token verify (`_verify_layers`) + n_keep (0..T; 0 undoes the verify)
        + the current small state entries (`_hist_cur`, DONATED: the outputs reuse their buffers, so the dispatch
        allocates nothing) -> the small state entries (KDA state/conv, pool tails) as after the first n_keep tokens,
        per layer; spliced into the caches by the caller (the big positional arrays are not touched). kin=True: the
        histories hold the KDA tokens' update inputs and the KDA states are those before the block (replayed)."""
        key = ("rollback", B, T) + (("kin",) if kin else ())
        if key not in self._progs:
            groups = self.groups["decode"]
            lays = [self._hist_layout(g, kin) for g in groups]
            hgs = [{k: P(None, *self._hist_spec(g[js[0]], kin)[k]) for k, js in lay.items()} for g, lay in zip(groups, lays)]
            hss = [self._hist_spec(i, kin) for i in range(self.cfg.n_layers)]
            oss = [None if h is None else {M.HIST_KEYS[k]: P(*v[1:]) for k, v in h.items()} for h in hss]   # drop the T axis
            dts = [{M.HIST_KEYS[k]: self.cfg.dtype if M.HIST_KEYS[k] != "state" else jnp.float32 for k in h} if h else None
                   for h in hss]

            def prog(hg, n, cur):
                per = [None] * self.cfg.n_layers
                for g, lay, h in zip(groups, lays, hg):
                    for k, js in lay.items():
                        for s_, j in enumerate(js):
                            per[g[j]] = {**(per[g[j]] or {}), k: h[k][s_]}
                return M.rollback_states(per, n, dts, cur)
            sm = shard_map(prog, mesh=self.mesh, in_specs=(hgs, R, oss), out_specs=oss, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(2,))
        return self._progs[key]

    def _hist_cur(self, caches, kin=False):
        """The small state entries of a verify's output caches that `_prog_rollback` replaces (and takes donated)."""
        out = []
        for i, c in enumerate(caches[:self.cfg.n_layers]):
            h = self._hist_spec(i, kin)
            out.append(None if h is None else {M.HIST_KEYS[k]: c[M.HIST_KEYS[k]] for k in h})
        return out

    mtp = None                       # MTP (NextN) layer params on device, or None (see set_mtp)
    dflash = None                    # DFlash 2 drafter (glm53.dflash.Drafter), or None (see set_dflash)
    reap = True                      # DFlash steps drop the previous step's arrays on a helper thread (`_Reaper`):
                                     # v6e 2026-10-05 (j145) host work per step 12.0 -> 9.9 ms (one stream), 18.5 ->
                                     # 14.0 (3 streams); the wall time was device-bound either way
    run_ahead = False                # DFlash steps dispatch the NEXT block's verify before the host reads this block's
                                     # tokens (all its inputs are device arrays), so the device never waits for the
                                     # host's turnaround; a stream that stops undoes that verify (`dflash_settle`).
                                     # Off: measured on the v6e 2026-10-05 (j143 / j126 / j102 / j131) the step and the
                                     # served rates were unchanged (the queued draft already covers the turnaround and
                                     # the remaining wall - busy gap is ~20 us per program launch), while every stream
                                     # end wastes one verify and needs the undo program.

    def _drop(self, *objs):
        """Release `objs` (the previous step's arrays) now, on the reaper thread when `reap` is set."""
        if self.reap:
            if getattr(self, "_reaper", None) is None:
                self._reaper = _Reaper()
            self._reaper.drop(*objs)

    def _cache_spec(self, i):
        if i == self.cfg.n_layers and self.dflash is not None:                 # the drafter's context ring
            from glm53.dflash import RING_SPECS
            return dict(RING_SPECS)
        if i == self.cfg.n_layers:                                             # the MTP layer's cache entry
            mla = P(None, AXIS) if self.lcfg.seq_shard > 1 else R
            spec = {"c": mla, "pk": mla, "tk": R, "tg": R, "h": R}
            if self.cfg.cache_q8:
                spec["cs"] = mla
            return spec
        return super()._cache_spec(i)

    def _is_kda(self, i):
        return i < self.cfg.n_layers and self.cfg.layer_types[i] == "linear_attention"

    def alloc_caches(self, B):
        """Zero caches for a new context at the engine's capacity, allocated directly on the chips (+ the MTP
        layer's entry, index n_layers, when an MTP layer is installed: its attention cache and the carried final
        hidden state "h" [B,1,D] of the last processed position; or the DFlash drafter's context ring there)."""
        cfg = self.cfg
        out = []
        n_entries = cfg.n_layers + (1 if self.mtp is not None or self.dflash is not None else 0)
        for i in range(n_entries):
            spec = self._cache_spec(i)
            if i == cfg.n_layers and self.dflash is not None:
                dc = self.dflash.cfg
                shapes = {k: ((dc.n_layers, B, dc.window, dc.kv_heads, dc.hd), dc.dtype) for k in ("k", "v")}
                shapes["p"] = ((B, dc.window), jnp.int32)               # slot positions + 1 (0 = empty)
            elif i == cfg.n_layers:
                shapes = {k: (sh, M.cache_dtype(cfg, k, cfg.dtype)) for k, sh in M.mla_cache_shapes(cfg, B, self.max_len, True).items()}
                shapes["h"] = ((B, 1, cfg.hidden), cfg.dtype)
            elif cfg.layer_types[i] == "linear_attention":
                shapes = {"conv": ((B, cfg.conv_k - 1, 3 * cfg.kda_heads * cfg.kda_hd), cfg.dtype),
                          "state": ((B, cfg.kda_heads, cfg.kda_hd, cfg.kda_hd), jnp.float32)}
            else:
                has_ix = "indexer" in self.params["layers"][i]["attn"]
                shapes = {k: (sh, M.cache_dtype(cfg, k, cfg.dtype)) for k, sh in M.mla_cache_shapes(cfg, B, self.max_len, has_ix).items()}
            c = {}
            for k, (shape, dtype) in shapes.items():
                key = (shape, jnp.dtype(dtype), spec[k])
                if key not in self._allocs:
                    self._allocs[key] = jax.jit(lambda shape=shape, dtype=dtype: jnp.zeros(shape, dtype),
                                                out_shardings=NamedSharding(self.mesh, spec[k]))
                c[k] = self._allocs[key]()
            out.append(c)
        return out

    @staticmethod
    def copy_caches(caches):
        """Device-side copy (the programs update caches in place)."""
        return jax.tree.map(jnp.copy, caches)

    def _prefix_rows(self, n_tokens):
        """Local cache rows that positions < n_tokens can occupy (interleaved layout: 4-token pools round-robin)."""
        kp, n = self.cfg.idx_kpool, self.lcfg.seq_shard
        return -(-n_tokens // (kp * n)) * kp if n > 1 else n_tokens

    def snapshot_prefix(self, caches, n_tokens, rows_bucket=1):
        """Compact snapshot of a context of n_tokens (the KDA states and only the used MLA/indexer cache rows), e.g.
        a shared system prompt: ~2 KB/token/chip instead of a full cache set. Restore with `restore_prefix`.
        `rows_bucket` rounds the row count up (bounds the number of slice/restore programs compiled for arbitrary
        context lengths; the extra rows are never read back)."""
        rows = self._prefix_rows(n_tokens)
        cap = self.max_len // max(self.lcfg.seq_shard, 1)
        rows = min(-(-rows // rows_bucket) * rows_bucket, cap)
        key = ("snap", rows)
        if key not in self._progs:
            self._progs[key] = {}
        out = []
        for i, c in enumerate(caches):
            if self._is_kda(i):
                out.append(jax.tree.map(jnp.copy, c)); continue
            spec = self._cache_spec(i)
            snap = {}
            for k, a in c.items():
                if k not in M.ROW_KEYS:                                       # tail buffers: whole copies
                    snap[k] = jnp.copy(a); continue
                rk = rows // self.cfg.idx_kpool if k == "pk" else rows
                pk = (k, a.shape, str(a.dtype))
                if pk not in self._progs[key]:
                    sp = spec[k]
                    self._progs[key][pk] = aot.jit((key, pk), shard_map(lambda x, rk=rk: x[:, :rk], mesh=self.mesh, in_specs=(sp,),
                                                             out_specs=sp, check_vma=False))
                snap[k] = self._progs[key][pk](a)
            out.append(snap)
        return {"n_tokens": n_tokens, "rows": rows, "caches": out}

    def restore_prefix(self, snap, B=1):
        """Fresh caches at full capacity holding the snapshot's context (the snapshot stays valid)."""
        caches = self.alloc_caches(B)
        rows = snap.get("rows") or self._prefix_rows(snap["n_tokens"])
        key = ("restore", rows)
        if key not in self._progs:
            self._progs[key] = {}
        out = []
        for i, c in enumerate(caches):
            sc = snap["caches"][i]
            if self._is_kda(i):
                out.append(jax.tree.map(jnp.copy, sc)); continue
            spec = self._cache_spec(i)
            new = {}
            for k, a in c.items():
                if k not in M.ROW_KEYS:
                    new[k] = jnp.copy(sc[k]); continue
                pk = (k, a.shape, str(a.dtype))
                if pk not in self._progs[key]:
                    sp = spec[k]
                    self._progs[key][pk] = aot.jit((key, pk), shard_map(lambda x, p: lax.dynamic_update_slice_in_dim(x, p, 0, axis=1),
                                                             mesh=self.mesh, in_specs=(sp, sp), out_specs=sp, check_vma=False),
                                                   donate_argnums=(0,))
                new[k] = self._progs[key][pk](a, sc[k])
            out.append(new)
        return out

    @staticmethod
    def snapshot_to_host(snap):
        """Move a compact snapshot into host memory so it holds no HBM; a parked context costs ~17 KB/token on the
        host. Every array is kept as its per-device shards (no host-side assembly of the 8 shards, which ran at
        ~1.5 GB/s); `snapshot_from_host` brings it back for `restore_prefix`."""
        host = jax.tree.map(_HostShards.of, snap["caches"])
        n_bytes = sum(h.nbytes for h in jax.tree.leaves(host, is_leaf=lambda x: isinstance(x, _HostShards)))
        return {"n_tokens": snap["n_tokens"], "rows": snap["rows"], "caches": host, "bytes": n_bytes}

    @staticmethod
    def snapshot_from_host(hsnap):
        """Device snapshot (as `snapshot_prefix` returns it) from a host snapshot; the host copy stays valid."""
        caches = jax.tree.map(lambda h: h.to_device(), hsnap["caches"], is_leaf=lambda x: isinstance(x, _HostShards))
        return {"n_tokens": hsnap["n_tokens"], "rows": hsnap["rows"], "caches": caches}

    def _capture_plan(self, g, capture):
        """DFlash feature capture for layer group g: (positions in the middle of the group, whose program returns
        their stream mean; whether the group's last layer is captured: the mean of the group's output)."""
        if not capture:
            return (), False
        js = [j for j, i in enumerate(g) if i in self.dflash.cfg.target_layers]
        return tuple(j for j in js if j < len(g) - 1), bool(js) and js[-1] == len(g) - 1

    def _run_layers(self, tokens, caches, pos0, use_recurrent, length, override=None, capture=False):
        """embed + all layer groups -> (final residual streams, new caches[, DFlash features: the stream mean after
        each of the drafter's target layers, [B,T,D] each, with capture=True]). `override` = (vectors [B,T,D],
        mask [B,T]) replaces the token embeddings where mask is set."""
        B, T = tokens.shape
        if override is None:
            streams = self._prog_embed(B, T)(self.params["embed"], jnp.asarray(tokens))
        else:
            streams = self._prog_embed(B, T, True)(self.params["embed"], jnp.asarray(tokens), jnp.asarray(override[0]),
                                                   jnp.asarray(override[1]))
        new_caches, feats = [], []
        has_cache = caches is not None
        pos0, length = self.scalar(pos0), self.scalar(length)
        for g in self.groups["decode" if use_recurrent else "prefill"]:
            mid, last = self._capture_plan(g, capture)
            prog = self._prog_group(g, B, T, has_cache, use_recurrent, capture=mid)
            out = prog([self.params["layers"][i] for i in g], streams, pos0, length,
                       [caches[i] for i in g] if has_cache else None)
            streams, ncs = out[:2]
            new_caches.extend(ncs)
            if mid:
                feats.extend(out[2][j] for j in range(len(mid)))
            if last:
                feats.append(self._prog_stream_mean(B, T)(streams))
            self.launches += 1
        return (streams, new_caches, feats) if capture else (streams, new_caches)

    def _run(self, tokens, caches, pos0, use_recurrent, length=None, want_logits=True, want_streams=False, override=None,
             capture=False):
        B, T = tokens.shape
        length = T if length is None else length
        out = self._run_layers(tokens, caches, pos0, use_recurrent, length, override, capture)
        streams, new_caches = out[:2]
        logits = None
        if want_logits:
            logits = self._prog_head(B, T)(self.params["norm"], self.params["lm_head"], streams, self.scalar(length))
        return (logits, new_caches) + ((streams,) if want_streams else ()) + ((out[2],) if capture else ())

    def scalar(self, x):
        """A host int as a replicated int32 device scalar. A plain jnp.int32(x) is uncommitted, so every program it is
        passed to reshards it on the host (`cpp_pjit_shard_arg_fallback`, ~0.3 ms per argument and call: ~3.6 ms of a
        verify step's ~16 ms host time, j69 2026-10-03). A device array (e.g. a position computed on the chips)
        passes through."""
        if isinstance(x, jax.Array):
            return x
        x = int(x)
        if 0 <= x < 64:                                  # small constants (block sizes, lengths): placed once
            c = self.__dict__.setdefault("_small_scalars", {})
            if x not in c:
                c[x] = jax.device_put(np.int32(x), NamedSharding(self.mesh, P()))
            return c[x]
        return jax.device_put(np.int32(x), NamedSharding(self.mesh, P()))

    def decode_sample(self, token, caches, pos, sampler: DeviceSampler):
        """`decode` with the next token sampled on the device: returns (ids [B] int32 device array, logits [B,V],
        caches, pos + 1). Only the ids need to reach the host (one small transfer instead of the [B,V] logits)."""
        B = token.shape[0]
        n = self.cfg.n_layers
        extra = caches[n:] if len(caches) > n else []
        streams, new = self._run_layers(np.asarray(token).reshape(B, 1), caches[:n], pos, True, 1)
        if getattr(self, "_one", None) is None:          # a replicated constant (no per-step scalar transfer); lazy so
            self._one = jax.device_put(jnp.int32(1), NamedSharding(self.mesh, P()))   # live engines survive a reload
        ids, logits, nk = self._prog_head(B, 1, True)(self.params["norm"], self.params["lm_head"], streams, self._one,
                                                      *sampler.bind(self.mesh))
        sampler.advance(nk)
        return ids, logits, new + extra, pos + 1

    # ---- batched decode of independent streams (continuous batching): one cache set per stream, per-row positions
    def _prog_embed_rows(self, B):
        """(embed, tokens [B] int32, pos [B] int32) -> (streams [B,1,hc,D], pos + 1): the positions stay on the
        device (no per-step host transfer), the program advances them."""
        key = ("embed_rows", B)
        if key not in self._progs:
            def prog(embed, tokens, pos):
                x = self._embed(embed, tokens[:, None]).astype(self.lcfg.dtype)
                return jnp.broadcast_to(x[:, :, None, :], (B, 1, self.cfg.hc, x.shape[-1])), pos + 1
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=(self.specs["embed"], R, R),
                                                 out_specs=(R, R), check_vma=False))
        return self._progs[key]

    def _prog_group_rows(self, layers, B, capture=False):
        """Decode step of `layers` for B independent streams: (params, streams [B,1,hc,D], pos [B], caches) with
        caches[j][b] = layer j's cache of stream b (batch dim 1, donated) -> (streams, new caches[j][b][, the stream
        means after EACH layer stacked [len(layers), B, 1, D] with capture=True: DFlash features, one program per layer
        kinds as in the verify])."""
        kinds = tuple(self._kind(i) for i in layers)
        key = ("group_rows", kinds, B) + (("capture",) if capture else ())
        if key not in self._progs:
            lts = [self.cfg.layer_types[i] for i in layers]
            mts = [self.cfg.mlp_types[i] for i in layers]
            css = [[self._cache_spec(i)] * B for i in layers]
            rep = list(layers)
            from glm53 import quant8 as Q8

            def prog(ps, streams, pos, caches):
                pos_b = [pos[b] for b in range(B)]
                new, feats = [], []
                for j, p in enumerate(ps):
                    p = Q8.dequant_tree(p, self.lcfg.dtype)
                    fetch = self.expert_fetch if mts[j] == "sparse" else None
                    streams, ncs = M.decoder_layer_rows(p, streams, self.lcfg, lts[j], mts[j], caches[j], pos_b, fetch,
                                                        "auto", layer=rep[j], cap=self.max_len)
                    new.append(ncs)
                    if capture:
                        feats.append(streams.astype(jnp.float32).mean(2).astype(self.lcfg.dtype))
                return (streams, new) + ((jnp.stack(feats),) if capture else ())
            in_specs = ([self.specs["layers"][i] for i in layers], R, R, css)
            out_specs = (R, css) + ((R,) if capture else ())
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(3,))
        return self._progs[key]

    def _prog_head_ring(self, B, sample, fidx):
        """`decode_rows`' head with a DFlash drafter installed: the head (`_prog_head(B, 1, sample)`: logits [B,V][, ids,
        next key]) + every row's drafter ring takes the step's position (its features captured after the drafter's
        target layers -> the drafter's context K/V, `dflash.ingest`): positions decoded in batched or plain steps are
        never holes in a stream's ring. (norm, lm_head, streams, drafter params, feats (the stacked stream means of the
        groups holding target layers, [len(g), B, 1, D]; `fidx` = the target positions in each), rings [B] (each batch
        1, donated), pos [B][, temperature, top_p, key]) -> (logits, rings) or (ids, logits, key, rings)."""
        key = ("head_ring", B, sample, fidx)
        if key not in self._progs:
            from glm53 import dflash as DF
            from glm53 import quant8 as Q8
            dr = self.dflash

            def prog(norm, lm_head, streams, dparams, feats, rings, pos, *samp):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)[:, 0]
                zl = self._logits_local(lm_head, h)
                logits = lax.all_gather(zl, AXIS, axis=-1, tiled=True)
                dp = Q8.dequant_tree(dparams, dr.cfg.dtype)
                one = jnp.ones((1,), jnp.int32)
                fl = [f[j] for f, js in zip(feats, fidx) for j in js]
                new = [DF.ingest(dp, rings[b], [f[b:b + 1] for f in fl], pos[b:b + 1], one, dr.lcfg) for b in range(B)]
                if not sample:
                    return logits, new
                temp, top_p, k = samp
                k1, k2 = jax.random.split(k)
                return device_sample(zl, temp, top_p, k1), logits, k2, new
            rs = [dict(DF.RING_SPECS)] * B
            in_specs = (R, self.specs["lm_head"], R, dr.specs, [R] * len(fidx), rs, R) + ((R, R, R) if sample else ())
            out_specs = (R, R, R, rs) if sample else (R, rs)
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs,
                                                 check_vma=False), donate_argnums=(5,))
        return self._progs[key]

    def _prog_group_vrows(self, layers, B, T, embed=False, pend=None):
        """The batched DFlash verify of `layers`: B independent streams, each verifying its own T-token block (anchor +
        drafts) on its own cache set at its own position. (params, streams [B,T,hc,D], pos [B], caches[j][b] (batch 1,
        donated)) -> (streams, new caches[j][b], the per-token histories stacked per key over the group's layers with
        the rows on the batch axis ([n, T, B, ...], `_hist_layout`; the KDA layers return their tokens' update inputs
        and leave their states as before the block: the rollback replays the accepted tokens — 64 KB per layer and row
        instead of T states of 0.5 MB, which did not fit next to 3 streams at 262k), the stream means after EACH layer
        [len(layers), B, block, D] (DFlash features, padded to the drafter's block: one program per layer kinds)).
        pend = T of the previous batched verify whose rollback is still pending (`dflash_step_rows`): the program
        also takes that verify's histories of this group (rows on the batch axis; DONATED, this verify's reuse the
        buffers) and its n_keep [B], and first rolls every row's small state entries back to its own accepted prefix
        (`_prog_rollback_rows`'s op), so no rollback program runs between the steps."""
        kinds = tuple(self._kind(i) for i in layers)
        key = ("group_vrows", kinds, B, T) + (("embed",) if embed else ()) + ((("pend", pend),) if pend else ())
        if key not in self._progs:
            lts = [self.cfg.layer_types[i] for i in layers]
            mts = [self.cfg.mlp_types[i] for i in layers]
            css = [[self._cache_spec(i)] * B for i in layers]
            lay = self._hist_layout(layers, kin=True)
            hss = {k: P(None, *self._hist_spec(layers[js[0]], kin=True)[k]) for k, js in lay.items()}
            rep = list(layers)
            from glm53 import quant8 as Q8

            def prog(ps, streams, pos, caches, hg=None, nk=None):
                pos_b = [pos[b] for b in range(B)]
                new, hs, feats = [], [], []
                if embed:
                    streams = self._embed_streams(*streams)
                if pend:
                    caches = [list(c) for c in caches]
                    dts = [{k_: v.dtype for k_, v in caches[j][0].items()} for j in range(len(ps))]
                    for b in range(B):
                        per = [{k: hg[k][js.index(j)][:, b:b + 1] for k, js in lay.items() if j in js} or None
                               for j in range(len(ps))]
                        roll = M.rollback_states(per, nk[b], dts, [caches[j][b] for j in range(len(ps))])
                        for j, r_ in enumerate(roll):
                            if r_ is not None:
                                caches[j][b] = {**caches[j][b], **r_}
                for j, p in enumerate(ps):
                    p = Q8.dequant_tree(p, self.lcfg.dtype)
                    fetch = self.expert_fetch if mts[j] == "sparse" else None
                    streams, ncs = M.decoder_layer_rows(p, streams, self.lcfg, lts[j], mts[j], caches[j], pos_b, fetch,
                                                        "auto", layer=rep[j], cap=self.max_len, hist="kin")
                    split = [M.split_hist(c) for c in ncs]
                    new.append([c for c, _ in split])
                    h0 = split[0][1]
                    hs.append(None if h0 is None else {k: jnp.concatenate([h[k] for _, h in split], axis=1) for k in h0})
                    feats.append(self._stream_mean(streams))
                hist = {k: jnp.stack([hs[j][k] for j in js]) for k, js in lay.items()}
                return streams, new, hist, jnp.stack(feats)
            in_specs = ([self.specs["layers"][i] for i in layers], (self.specs["embed"], R) if embed else R, R, css) + \
                ((hss, R) if pend else ())
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=(R, css, hss, R), check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(3,) + ((4,) if pend else ()))
        return self._progs[key]

    def _pend_identity_rows(self, sets, T):
        """`_pend_identity` for B streams: histories that change nothing (zero KDA update inputs, the current conv
        states and pool tails repeated T times), n_keep = T for every row."""
        B = len(sets)
        key = ("pend_identity_rows", B, T)
        if key not in self._progs:
            groups = self.groups["decode"]
            lays = [self._hist_layout(g, kin=True) for g in groups]
            hgs = [{k: P(None, *self._hist_spec(g[js[0]], kin=True)[k]) for k, js in lay.items()}
                   for g, lay in zip(groups, lays)]
            cur_specs = [None if h is None else {M.HIST_KEYS[k]: P(*v[1:]) for k, v in h.items()}
                         for h in (self._hist_spec(i, kin=True) for i in range(self.cfg.n_layers))]
            hd = self.cfg.kda_hd

            def one(cur, k):
                if k == "kin_hist":
                    s = cur["state"]                                           # [1,H,dk,dv] -> kin [T,1,H,4,dk] f32
                    return jnp.zeros((T, 1, s.shape[1], 4, hd), jnp.float32)
                a = cur[M.HIST_KEYS[k]]
                return jnp.broadcast_to(a[None], (T + 1,) + a.shape)           # entries 0..T (`M.rollback_states`)

            def prog(curs):                                                     # curs[b][layer] -> per group {k: [n,T,B,...]}
                return [{k: jnp.stack([jnp.concatenate([one(curs[b][g[j]], k) for b in range(B)], axis=1) for j in js])
                         for k, js in lay.items()} for g, lay in zip(groups, lays)]
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=([cur_specs] * B,), out_specs=hgs,
                                                 check_vma=False))
        return self._progs[key]([self._hist_cur(c, kin=True) for c in sets]), self.device_positions([T] * B), T

    def dflash_settle_rows(self, st):
        """`dflash_settle` for a batched state: the verify dispatched ahead undone (every row back to before its block,
        n_keep = 0) or the pending rollback applied to every row's caches. Idempotent."""
        B = len(st["sets"])
        if st.get("ahead") is not None:
            hists, nk, T = st["ahead"][2], self.device_positions([0] * B), st["T"]
        elif st.get("pend"):
            hists, nk, T = st["pend"]
        else:
            return st
        rolled = self._prog_rollback_rows(B, T)(hists, nk, [self._hist_cur(c, kin=True) for c in st["sets"]])
        new = [[({**c, **x} if x else c) for c, x in zip(cs, rs)] for cs, rs in zip(st["sets"], rolled)]
        return {**{k: v for k, v in st.items() if k != "ahead"}, "sets": new, "pend": None}

    def _prog_rollback_rows(self, B, T):
        """`_prog_rollback` for B independent streams: (the per-group histories of `_verify_rows` (rows on the batch
        axis), n_keep [B], cur[b] = row b's current small state entries (`_hist_cur`, DONATED; the KDA states are
        those before the block)) -> per row the small state entries as after its own first n_keep[b] tokens (the KDA
        states by replaying them: `M.kda_replay`)."""
        key = ("rollback_rows", B, T)
        if key not in self._progs:
            groups = self.groups["decode"]
            lays = [self._hist_layout(g, kin=True) for g in groups]
            hgs = [{k: P(None, *self._hist_spec(g[js[0]], kin=True)[k]) for k, js in lay.items()}
                   for g, lay in zip(groups, lays)]
            hss = [self._hist_spec(i, kin=True) for i in range(self.cfg.n_layers)]
            oss = [None if h is None else {M.HIST_KEYS[k]: P(*v[1:]) for k, v in h.items()} for h in hss]
            dts = [{M.HIST_KEYS[k]: self.cfg.dtype if M.HIST_KEYS[k] != "state" else jnp.float32 for k in h} if h else None
                   for h in hss]

            def prog(hg, n, cur):
                out = []
                for b in range(B):
                    per = [None] * self.cfg.n_layers
                    for g, lay, h in zip(groups, lays, hg):
                        for k, js in lay.items():
                            for s_, j in enumerate(js):
                                per[g[j]] = {**(per[g[j]] or {}), k: h[k][s_][:, b:b + 1]}
                    out.append(M.rollback_states(per, n[b], dts, cur[b]))
                return out
            sm = shard_map(prog, mesh=self.mesh, in_specs=(hgs, R, [oss] * B), out_specs=[oss] * B, check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(2,))
        return self._progs[key]

    def _verify_rows(self, tokens, sets, pos, pend=None):
        """embed + the decode layer groups of a batched verify (`_prog_group_vrows`): tokens [B,T] (row b = stream b's
        block), sets[b] = its n layer caches (consumed), pos [B] device positions -> (final streams, new sets, histories
        per group, DFlash features (stacked per group holding target layers, the target positions in each)). pend =
        (histories per group, n_keep [B], T) of the previous verify, whose rollback the groups apply first."""
        B, T = tokens.shape
        streams = (self.params["embed"], jnp.asarray(tokens))       # the first group program embeds
        new_sets = [[] for _ in range(B)]
        hists, feats, fidx = [], [], []
        tl = self.dflash.cfg.target_layers
        for gi, g in enumerate(self.groups["decode"]):
            out = self._prog_group_vrows(g, B, T, embed=gi == 0, pend=pend[2] if pend else None)(
                [self.params["layers"][i] for i in g], streams, pos, [[sets[b][i] for b in range(B)] for i in g],
                *((pend[0][gi], pend[1]) if pend else ()))
            streams, ncs, hs, f = out
            for j in range(len(g)):
                for b in range(B):
                    new_sets[b].append(ncs[j][b])
            hists.append(hs)
            js = tuple(j for j, i in enumerate(g) if i in tl)
            if js:
                feats.append(f); fidx.append(js)
            self.launches += 1
        return streams, new_sets, hists, (feats, tuple(fidx))

    def device_positions(self, pos):
        """Host positions (ints) -> one replicated int32 [B] device array (the form `decode_rows` carries)."""
        return jax.device_put(np.asarray(pos, np.int32).reshape(-1), NamedSharding(self.mesh, P()))

    def decode_rows(self, tokens, cache_sets, pos, sampler: DeviceSampler | None = None):
        """One decode step of B independent streams in one batch: `tokens` [B] (their last tokens; a device array
        from the previous step or host ints), `cache_sets[b]` = stream b's own cache list (as `prefill` /
        `restore_prefix` return them, batch dim 1; consumed), `pos` [B] int32 replicated device array
        (`device_positions`) or host ints. Returns (ids [B] device int32 or None without a sampler, logits [B,V],
        new cache sets, pos + 1 on the device). Streams may sit at any positions; the MoE/mHC/projections run
        batched, the attention per row. Compiles one program set per B. With a DFlash drafter installed (cache sets
        with its ring) the step also writes every row's position into that stream's ring (`_prog_head_ring`)."""
        B = len(cache_sets)
        n = self.cfg.n_layers
        assert all(len(c) >= n for c in cache_sets), "every stream needs a full cache set"
        if not isinstance(pos, jax.Array):
            pos = self.device_positions(pos)
        if not (isinstance(tokens, jax.Array) and tokens.shape == (B,) and tokens.dtype == jnp.int32):
            tokens = jnp.asarray(np.asarray(tokens, np.int32).reshape(B))   # (a device array from the last step passes through)
        streams, pos_next = self._prog_embed_rows(B)(self.params["embed"], tokens, pos)
        cap = self.dflash is not None and all(len(c) > n for c in cache_sets)
        new_sets = [[] for _ in range(B)]
        feats, fidx = [], []
        for g in self.groups["decode"]:
            prog = self._prog_group_rows(g, B, cap)
            out = prog([self.params["layers"][i] for i in g], streams, pos, [[cache_sets[b][i] for b in range(B)] for i in g])
            streams, ncs = out[:2]
            js = tuple(j for j, i in enumerate(g) if i in self.dflash.cfg.target_layers) if cap else ()
            if js:
                feats.append(out[2]); fidx.append(js)
            for j in range(len(g)):
                for b in range(B):
                    new_sets[b].append(ncs[j][b])
            self.launches += 1
        if getattr(self, "_one", None) is None:
            self._one = jax.device_put(jnp.int32(1), NamedSharding(self.mesh, P()))
        if cap:                                              # the drafter's rings take this step's positions
            args = (self.params["norm"], self.params["lm_head"], streams, self.dflash.params, feats,
                    [cache_sets[b][n] for b in range(B)], pos)
            if sampler is None:
                logits, rings = self._prog_head_ring(B, False, tuple(fidx))(*args)
                ids = None
            else:
                ids, logits, nk, rings = self._prog_head_ring(B, True, tuple(fidx))(*args, *sampler.bind(self.mesh))
                sampler.advance(nk)
            for b in range(B):
                new_sets[b].append(rings[b])
                new_sets[b].extend(cache_sets[b][n + 1:])
            return ids, logits, new_sets, pos_next
        for b in range(B):                                   # extra entries (an MTP cache) pass through untouched
            new_sets[b].extend(cache_sets[b][n:])
        if sampler is None:
            logits = self._prog_head(B, 1)(self.params["norm"], self.params["lm_head"], streams, self._one)
            return None, logits, new_sets, pos_next
        ids, logits, nk = self._prog_head(B, 1, True)(self.params["norm"], self.params["lm_head"], streams, self._one,
                                                      *sampler.bind(self.mesh))
        sampler.advance(nk)
        return ids, logits, new_sets, pos_next

    def prefill(self, tokens, caches=None, pos0=0, embeds=None):
        """Prefill `tokens` [B,T] into a fresh context (caches=None) or continue one at position pos0 (the caches
        are consumed). Returns (logits of the last token, caches, pos0 + T). With an MTP layer installed the MTP
        cache (entry n_layers) is prefilled too: MTP position p takes (token p+1, final hidden p), so a piece
        covers positions [pos0-1, pos0+L-1) (or [0, L-1) for the first piece) and the last hidden is carried.
        `embeds` = (idx [M] int, vec [M, D]) replaces the embeddings of tokens[0, idx] (B = 1; image tokens,
        glm53.vision) — the MTP layer's own embedding input keeps the table rows (drafts only; verify is exact).
        With a DFlash drafter installed, the pieces that hold the last window-1 positions also capture the target
        features, and the drafter's ring (entry n_layers) takes their K/V."""
        tokens = np.asarray(tokens)
        B, T = tokens.shape
        assert T >= 1 and pos0 + T <= self.max_len, (pos0, T, self.max_len)
        if caches is None:
            caches = self.alloc_caches(B)
        if embeds is not None:
            assert B == 1
            e_idx, e_vec = np.asarray(embeds[0], np.int64), np.asarray(embeds[1])
        logits = None
        n = self.cfg.n_layers
        main, mc = (caches[:n], caches[n]) if self.mtp is not None else (caches[:n], None)
        ring = caches[n] if self.dflash is not None else None
        for a in range(0, T, self.prefill_piece):
            seg = tokens[:, a:a + self.prefill_piece]
            L = seg.shape[1]
            Tp = self._bucket(L)
            padded = np.zeros((B, Tp), np.int32)
            padded[:, :L] = seg
            ov = None
            if embeds is not None:
                sel = (e_idx >= a) & (e_idx < a + L)
                if sel.any():
                    vec = np.zeros((B, Tp, e_vec.shape[-1]), e_vec.dtype)
                    mask = np.zeros((B, Tp), bool)
                    vec[0, e_idx[sel] - a] = e_vec[sel]
                    mask[0, e_idx[sel] - a] = True
                    ov = (vec, mask)
            if ring is not None:
                cap = a + L > T - self.dflash.cfg.window          # positions a block can still see
                out = self._run(padded, main, pos0 + a, False, length=L, want_logits=a + L >= T, override=ov,
                                capture=cap)
                logits, main = out[:2]
                if cap:
                    ring = self.dflash.ingest(ring, out[2], self.device_positions([pos0 + a] * B),
                                              self.device_positions([L] * B))
                continue
            if mc is None:
                logits, main = self._run(padded, main, pos0 + a, False, length=L, want_logits=a + L >= T, override=ov)
                continue
            logits, main, streams = self._run(padded, main, pos0 + a, False, length=L, want_logits=a + L >= T,
                                              want_streams=True, override=ov)
            hidden = self._prog_hidden(B, Tp)(self.params["norm"], streams)                 # [B,Tp,D]
            p0 = pos0 + a
            if p0 == 0:                                        # rows 0..L-2: token j+1 with hidden j
                mt = np.zeros((B, Tp), np.int32); mt[:, :L - 1] = seg[:, 1:]
                hp = jnp.concatenate([hidden[:, :Tp - 1], jnp.zeros_like(hidden[:, :1])], axis=1)
                start, length = 0, L - 1
            else:                                              # rows p0-1..p0+L-2: token p0+j with hidden p0+j-1
                mt = padded
                hp = jnp.concatenate([mc["h"], hidden[:, :Tp - 1]], axis=1)
                start, length = p0 - 1, L
            mcache = {k: v for k, v in mc.items() if k != "h"}
            if length > 0:
                _, _, mcache = self._prog_mtp(B, Tp, True)(self.mtp, self.params["embed"], self.params["lm_head"],
                                                           jnp.asarray(mt), hp, self.scalar(start), self.scalar(length), mcache)
            mc = {**mcache, "h": lax.dynamic_index_in_dim(hidden, L - 1, axis=1, keepdims=True)}
        return logits, main + [e for e in (mc, ring) if e is not None], pos0 + T

    # ---- speculative decoding with the MTP (NextN) layer
    def set_mtp(self, params, qtypes, k=1):
        """Install the MTP layer: `params` from checkpoint.load_mtp_layer (2-D matrices bf16) plus its expert planes
        (pack_layer_from_gguf) under mlp["gate_q"/"up_q"/"down_q"]; `qtypes` their formats; `k` drafts per step.
        The layer's attention cache + carried hidden state become cache entry n_layers (alloc_caches)."""
        col, row = P(None, AXIS), P(AXIS, None)
        attn = {"q_a": R, "q_a_norm": R, "q_b": col, "kv_a": R, "kv_a_norm": R, "kv_b": col, "o": row,
                "indexer": {kk: R for kk in params["attn"]["indexer"]}}
        mlp = {"router_w": R, "router_bias": R, "shared": {"gate": col, "up": col, "down": row}}
        for kk in ("gate_q", "up_q", "down_q"):
            mlp[kk] = {pk: P(AXIS) for pk in params["mlp"][kk]}
        spec = {"ln1": R, "ln2": R, "attn": attn, "mlp": mlp, "enorm": R, "hnorm": R, "eh_proj": R, "head_norm": R}
        self.expert_fetch.qtypes[self.cfg.n_layers] = qtypes
        self.mtp = jax.tree.map(lambda a, sp: a if hasattr(a, "sharding") and a.sharding == NamedSharding(self.mesh, sp)
                                else jax.device_put(a, NamedSharding(self.mesh, sp)), params, spec)
        self.mtp_spec, self.mtp_k = spec, k

    def _prog_mtp(self, B, T, has_cache=True):
        """(mtp params, embed, lm_head, tokens [B,T] = the token AFTER each MTP position, h_prev [B,T,D], pos0,
        length, cache) -> (logits of the last real position [B,V], hidden [B,T,D], new cache)."""
        key = ("mtp", B, T, has_cache)
        if key not in self._progs:
            from glm53 import quant8 as Q8
            cs = {k: v for k, v in self._cache_spec(self.cfg.n_layers).items() if k != "h"}

            def prog(mp, embed, lm_head, tokens, h_prev, pos0, length, cache):
                mp = Q8.dequant_tree(mp, self.lcfg.dtype)
                e = self._embed(embed, tokens).astype(self.lcfg.dtype)
                e = jnp.where((pos0 + jnp.arange(T))[None, :, None] == 0, jnp.zeros((), e.dtype), e)   # no position -1
                h, nc = M.mtp_layer(mp, e, h_prev, self.lcfg, cache, pos0, self.expert_fetch, "auto", self.max_len,
                                    length, layer=self.cfg.n_layers)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                return self._logits(lm_head, last), h, nc
            in_specs = (self.mtp_spec, self.specs["embed"], self.specs["lm_head"], R, R, R, R, cs)
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=(R, R, cs), check_vma=False)
            self._progs[key] = aot.jit(key, sm, donate_argnums=(7,))
        return self._progs[key]

    def _verify_layers(self, tokens, caches, pos0, capture=False, pend=None):
        """embed + the decode layer groups of a recurrent T-token step with per-token histories -> (final streams,
        caches, histories (per group, stacked per key, the rows' "kin" form: `_prog_rollback(kin=True)` takes them),
        DFlash features when capture: (the stacked stream means [len(g), B, T, D] of the groups holding drafter target
        layers, the target positions in each) — the drafter's programs take that pair
        (`dflash.Drafter.draft(feat_index=...)`)). pend = (histories per group, n_keep, T) of the previous verify whose
        rollback the groups apply first (`dflash_step`)."""
        B, T = tokens.shape
        streams = (self.params["embed"], jnp.asarray(tokens))       # the first group program embeds (no embed program)
        new, hists, feats, fidx = [], [], [], []
        pos0, length = self.scalar(pos0), self.scalar(T)
        tl = self.dflash.cfg.target_layers if capture else ()
        for gi, g in enumerate(self.groups["decode"]):
            # with capture, every group program returns the stream mean after EACH of its layers: groups with the same
            # layer kinds then share one program whichever layer they capture (5 verify programs per T instead of 9)
            every = tuple(range(len(g))) if capture else ()
            prog = self._prog_group(g, B, T, True, True, hist="kin", capture=every, pend=pend[2] if pend else None,
                                    embed=gi == 0)
            out = prog([self.params["layers"][i] for i in g], streams, pos0, length, [caches[i] for i in g],
                       *((pend[0][gi], pend[1]) if pend else ()))
            streams, ncs, hs = out[:3]
            new.extend(ncs); hists.append(hs)
            js = tuple(j for j, i in enumerate(g) if i in tl)
            if js:
                feats.append(out[3]); fidx.append(js)
            self.launches += 1
        return streams, new, hists, (feats, tuple(fidx))

    def _run_verify(self, tokens, caches, pos0, sampler=None, capture=False):
        """Recurrent T-token step returning (logits [B,T,V], caches, per-layer histories, hidden [B,T,D], ids[,
        DFlash features: list of [B,T,D] per drafter target layer, with capture=True]): ids = the device-sampled token
        of every position [B,T] when a DeviceSampler is given, else None."""
        B, T = tokens.shape
        streams, new, hists, feats = self._verify_layers(tokens, caches, pos0, capture)
        extra = (feats,) if capture else ()
        if sampler is None:
            logits, hidden = self._prog_head_all(B, T)(self.params["norm"], self.params["lm_head"], streams)
            return (logits, new, hists, hidden, None) + extra
        ids, logits, hidden, nk = self._prog_head_all(B, T, True)(self.params["norm"], self.params["lm_head"], streams,
                                                                  *sampler.bind(self.mesh))
        sampler.advance(nk)
        return (logits, new, hists, hidden, ids) + extra

    def spec_decode(self, token, caches, pos, k=None, sampler=None, draft_override=None, stop_ids=()):
        """One speculative step from the committed token `token` [B] at position `pos` (B = 1): the MTP layer
        drafts k tokens, the main model verifies them in one (k+1)-token recurrent step, the longest accepted prefix
        is kept (recurrent states and pool tails rolled back to it). `sampler` decides the target's token per
        position: None = argmax, a host callable `sampler(logits_row) -> int`, or a `DeviceSampler` (sampled inside
        the head program; only the ids cross to the host). A draft is accepted iff it equals the target's own choice,
        which is exact for greedy and a valid, if conservative, scheme for sampling. A draft in `stop_ids` ends the
        accepted run (it is emitted as the last, unfed token). Returns (emitted tokens [1..k+1] — all but the last
        were fed to the model, logits of the last emitted position [B,V], caches, new position)."""
        k = self.mtp_k if k is None else k
        n = self.cfg.n_layers
        B = token.shape[0]
        assert B == 1 and self.mtp is not None
        mc = caches[n]
        mcache = {kk: v for kk, v in mc.items() if kk != "h"}
        hp = mc["h"]
        tok = jnp.asarray(np.asarray(token, np.int32).reshape(B, 1))
        drafts, tails = [], []                                  # the draft chain stays on the device (one host sync per step)
        for j in range(k):
            tails.append((mcache["tk"], mcache["tg"]))          # before draft j (donated: keep the objects, not copies)
            lg, hd, mcache = self._prog_mtp(B, 1, True)(self.mtp, self.params["embed"], self.params["lm_head"],
                                                        tok, hp, self.scalar(pos - 1 + j), self.scalar(1),
                                                        {**mcache, "tk": jnp.copy(mcache["tk"]), "tg": jnp.copy(mcache["tg"])})
            d = jnp.asarray([[int(draft_override[j])]], jnp.int32) if draft_override is not None else jnp.argmax(lg, axis=-1)[:, None].astype(jnp.int32)
            drafts.append(d); tok = d; hp = hd
        seq = jnp.concatenate([jnp.asarray(np.asarray(token, np.int32).reshape(B, 1))] + drafts, axis=1)   # [B,k+1]
        dev = isinstance(sampler, DeviceSampler)
        logits, new, hists, hidden, ids = self._run_verify(seq, caches[:n], pos, sampler if dev else None)
        seq_h = np.asarray(seq)[0]
        drafts_h = [int(x) for x in seq_h[1:]]
        if dev:                                                 # only k+1 ids cross to the host, not [k+1, V] logits
            preds = [int(x) for x in np.asarray(ids[0])]
        else:
            lg_h = np.asarray(logits[0])
            preds = [int(np.argmax(lg_h[j])) if sampler is None else int(sampler(lg_h[j])) for j in range(k + 1)]
        a = 0                                                   # a stop token is never fed: it ends the accepted run
        while a < k and preds[a] == drafts_h[a] and drafts_h[a] not in stop_ids:
            a += 1
        # the verify's KDA states are those before the block (the "kin" histories): replay the a + 1 accepted tokens
        # (every one of them when all drafts were accepted), the conv states and pool tails to after them
        rolled = self._prog_rollback(B, k + 1, kin=True)(hists, self.scalar(a + 1), self._hist_cur(new, kin=True))
        new = [({**c, **r} if r else c) for c, r in zip(new, rolled)]
        if a + 1 < k:                                           # drafts a+1.. were fed rejected tokens: tail as before draft a+1
            mcache = {**mcache, "tk": tails[a + 1][0], "tg": tails[a + 1][1]}
        emitted = drafts_h[:a] + [preds[a]]
        h_next = lax.dynamic_index_in_dim(hidden, a, axis=1, keepdims=True)
        return emitted, logits[:, a], new + [{**mcache, "h": h_next}], pos + a + 1

    # ---- speculative decoding with the DFlash 2 drafter (block diffusion, glm53.dflash)
    def set_dflash(self, params, dcfg, int8=False, temp_scale=0.7):
        """Install the DFlash 2 drafter (`glm53.dflash.load_params`) on the engine's mesh; it shares the target's
        embedding and lm_head. Its context ring becomes cache entry n_layers (`alloc_caches`), filled by `prefill`
        with the captured target features (the stream mean after each of dcfg.target_layers) and by `dflash_step`
        with the accepted positions'. int8: the drafter's big matrices as q8 (glm53.quant8). temp_scale: drafts are
        drawn at temp_scale x the request's temperature (rejection sampling accepts against the target's own; 0.5-0.7
        accepted ~1 % more than 1.0 and ~4 % more than greedy drafts at temperature 1.0, v6e 2026-10-04)."""
        from glm53 import dflash as DF
        assert self.mtp is None, "MTP and DFlash share cache entry n_layers"
        assert max(dcfg.target_layers) < self.cfg.n_layers, dcfg.target_layers
        self.dflash = DF.Drafter(dataclasses.replace(dcfg, dtype=self.cfg.dtype), params, self.mesh,
                                 self.specs["embed"], self.specs["lm_head"], int8=int8, temp_scale=temp_scale)

    def _dflash_zeros(self, B):
        """Zero DFlash features in the verify's layout (`_verify_layers`): for a draft with no new positions."""
        key = ("dflash_zeros", B)
        if key not in self._progs:
            tl, blk = self.dflash.cfg.target_layers, self.dflash.cfg.block
            shapes, fidx = [], []
            for g in self.groups["decode"]:
                js = tuple(j for j, i in enumerate(g) if i in tl)
                if js:
                    shapes.append((len(g), B, blk, self.cfg.hidden)); fidx.append(js)
            z = jax.jit(lambda: [jnp.zeros(sh, self.cfg.dtype) for sh in shapes],
                        out_shardings=[NamedSharding(self.mesh, R)] * len(shapes))()
            self._progs[key] = (z, tuple(fidx))
        return self._progs[key]

    def _prog_head_accept(self, B, T, stop_ids, per_row=False):
        """The DFlash verify's head with the acceptance on the chips: (norm, lm_head, streams, seq [B,T] = anchor +
        drafts, cand / q [B,T-1,k] = the drafter's candidates and its distribution over them, pos (), temperature,
        top_p, key) -> dict: emitted [B,T] (the accepted drafts, then the emitted token, -1 after), a [B] (accepted
        drafts), bonus [B] (the emitted token = the next anchor), n_keep () = a+1, pos_next () = pos + a + 1, pos_b [B] =
        pos, n_new_b [B] = a + 1, key. The acceptance is rejection sampling against the target's distribution after
        temperature and top-p (`engine.spec_accept`): every emitted token is distributed as plain sampling; at
        temperature 0 a draft is accepted iff it is the argmax (greedy). A draft in `stop_ids` ends the accepted run
        (B = 1: n_keep and pos_next are row 0's). per_row=True (B independent streams, `dflash_step_rows`): pos [B] and
        temperature / top_p per row ([B] or scalars); n_keep and pos_next [B]."""
        key = ("head_accept", B, T, tuple(stop_ids)) + (("rows",) if per_row else ())
        if key not in self._progs:
            def prog(norm, lm_head, streams, seq, cand, q, pos, temp, top_p, k):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
                k1, k2 = jax.random.split(k)
                rows = lambda v: jnp.broadcast_to(jnp.reshape(v, (-1, 1)), (B, T)).reshape(-1)   # noqa: E731
                gidx, p = device_probs(self._logits_local(lm_head, h).reshape(B * T, -1), rows(temp), rows(top_p))
                C = gidx.shape[-1]
                d = seq[:, 1:].astype(jnp.int32)
                a, bonus = spec_accept(gidx.reshape(B, T, C), p.reshape(B, T, C), d, cand, q, k1, stop_ids)
                j = jnp.arange(T)[None, :]
                dpad = jnp.concatenate([d, jnp.zeros((B, 1), jnp.int32)], 1)
                emitted = jnp.where(j < a[:, None], dpad, jnp.where(j == a[:, None], bonus[:, None], -1))
                if per_row:
                    return {"emitted": emitted, "a": a, "bonus": bonus, "n_keep": a + 1, "pos_next": pos + a + 1,
                            "pos_b": pos, "n_new_b": a + 1, "key": k2}
                return {"emitted": emitted, "a": a, "bonus": bonus, "n_keep": a[0] + 1, "pos_next": pos + a[0] + 1,
                        "pos_b": jnp.broadcast_to(pos, (B,)), "n_new_b": a + 1, "key": k2}
            outs = {k: R for k in ("emitted", "a", "bonus", "n_keep", "pos_next", "pos_b", "n_new_b", "key")}
            in_specs = (R, self.specs["lm_head"], R, R, R, R, R, R, R, R)
            self._progs[key] = aot.jit(key, shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=outs,
                                                 check_vma=False))
        return self._progs[key]

    def _dflash_draft(self, ring, feats, pos_b, n_new_b, anchor_b, T, temp, key):
        """One draft program (any temperature: the selector draws from its q, the argmax at temperature 0) ->
        (seq [B,T] = anchor + drafts, cand, q [B,T-1,k], the new ring)."""
        r = self.dflash.draft(ring, feats[0], pos_b, n_new_b, anchor_b, self.params["embed"], self.params["lm_head"], T=T,
                              temperature=temp, key=key, feat_index=feats[1])
        return r["seq"], r["cand"], r["q"], r["ring"]

    def dflash_start(self, token, caches, pos, T=None, pending=None, sampler: DeviceSampler | None = None):
        """Begin DFlash 2 decoding of one stream (B = 1) from the committed token `token` [B] at position `pos` (not yet
        fed; e.g. the token sampled from `prefill`'s logits): dispatches the first draft (drawn at the sampler's
        temperature; pass the same sampler to `dflash_step`). `pending` = the features of positions the ring has not
        seen yet (None right after `prefill`, which fills it). Returns the state that `dflash_step` advances:
        {"caches", "pos" (device scalar), "pos_h" (host int), "seq" (the drafted block, on the device), "cand", "q",
        "T"}."""
        n = self.cfg.n_layers
        T = self.dflash.cfg.block if T is None else T
        B = np.asarray(token).reshape(-1).shape[0]
        assert B == 1 and self.dflash is not None
        sampler = sampler if sampler is not None else self._dflash_greedy()
        temp, _, k = sampler.bind(self.mesh)
        feats, p0, n_new = pending if pending is not None else (self._dflash_zeros(B), pos, 0)
        seq, cand, q, ring = self._dflash_draft(caches[n], feats, self.device_positions([p0] * B),
                                                self.device_positions([n_new] * B),
                                                self.device_positions(np.asarray(token).reshape(B)), T, temp, k)
        return {"caches": caches[:n] + [ring], "pos": self.scalar(pos), "pos_h": int(pos), "seq": seq, "cand": cand,
                "q": q, "T": T}

    def dflash_step(self, st, sampler: DeviceSampler | None = None, stop_ids=(), draft_override=None, next_T=None):
        """One DFlash 2 step: the target verifies the drafted block [anchor, drafts] in one T-token step that also
        captures its features; the acceptance runs on the chips (the head program: rejection sampling of the drafts
        against the target's distribution, `sampler` None = greedy, where a draft is accepted iff it is the argmax; a
        draft in `stop_ids` ends the run and is emitted unfed); the KDA states and pool tails are rolled back to the
        accepted prefix; the NEXT draft (anchor = the emitted token, the accepted positions' features into the ring)
        and, with `run_ahead`, the verify of that next block are dispatched before the host reads this step's tokens,
        so the host's turnaround overlaps them (the draft only writes committed positions; the verify ahead is undone
        by `dflash_settle` when the stream stops: stopping after any step leaves a consistent state). `next_T` = the
        block size of the next draft (default: this one's). `draft_override` (T-1 ids, tests; exact only at
        temperature 0) replaces this step's drafts (a verify dispatched ahead is undone and run again). Returns
        (emitted tokens [1..T] — all but the last were fed to the model, the new state)."""
        n = self.cfg.n_layers
        T, B = st["T"], 1
        sampler = sampler if sampler is not None else self._dflash_greedy()
        grave = self.__dict__.pop("_grave", None)        # the previous step's dropped arrays (see the end)
        seq = st["seq"]
        ahead = st.get("ahead")                          # this block's verify, dispatched by the previous step
        if draft_override is not None:
            if ahead is not None:                        # (tests) the verify ahead saw the real drafts: undo it
                st, ahead = self.dflash_settle(st), None
            seq = jnp.concatenate([seq[:, :1], jnp.asarray(np.asarray(draft_override, np.int32).reshape(B, T - 1))], 1)
        caches = st["caches"]
        if ahead is None:
            pend = st.get("pend") or self._pend_identity(caches[:n], T)
            ahead = self._verify_layers(seq, caches[:n], st["pos"], capture=True, pend=pend)
        else:
            pend = None
        streams, new, hists, feats = ahead
        self._drop(grave, pend)
        del grave, pend, ahead                           # released while the device runs the verify
        temp, top_p, k = sampler.bind(self.mesh)
        r = self._prog_head_accept(B, T, tuple(stop_ids))(self.params["norm"], self.params["lm_head"], streams, seq,
                                                          st["cand"], st["q"], st["pos"], temp, top_p, k)
        sampler.advance(r["key"])
        # the rollback of this verify to its accepted prefix is deferred: the next verify's group programs apply it
        # first (or `dflash_settle`, when the stream leaves DFlash): no rollback program, its ~90 outputs and their
        # release between the steps (the one-stream step was host-bound, j141 2026-10-04). The draft stays a program
        # of its own, queued behind the head (merged into the head program, 2026-10-05 j143 / j150, the device idled
        # through the host's turnaround: +0.5 ms per step).
        next_T = T if next_T is None else next_T
        nseq, cand, q, ring = self._dflash_draft(caches[n], feats, r["pos_b"], r["n_new_b"], r["bonus"], next_T, temp,
                                                 r["key"])
        out = {"caches": new + [ring], "pend": (hists, r["n_keep"], T), "pos": r["pos_next"], "seq": nseq, "cand": cand,
               "q": q, "T": next_T}
        # run ahead: the next block's verify (its inputs — the drafted block, the position, the caches, this verify's
        # histories and n_keep — are all on the device) is queued behind the draft, so the device has ~a step of work
        # when the host wakes up below; only when this block and the next both fit the context
        if self.run_ahead and st["pos_h"] + T + next_T <= self.max_len:
            out["ahead"] = self._verify_layers(nseq, new, r["pos_next"], capture=True, pend=out.pop("pend"))
            out["caches"] = out["ahead"][1] + [ring]
        emitted = [int(x) for x in np.asarray(r["emitted"])[0] if x >= 0]     # the host waits here
        out["pos_h"] = st["pos_h"] + len(emitted)
        # Dropping a jax array costs ~11-20 us of host time on the 8-chip mesh (j107) and a step leaves ~300 behind (the
        # consumed caches, the pre-rollback states, histories, ...): keep them until the next step has dispatched its
        # head, so the release happens while the device works instead of between the steps (`dflash_end` frees them;
        # with `reap` the drop itself runs on the reaper thread).
        self._grave = (st, feats, streams, r)
        return emitted, out

    def _pend_identity(self, caches, T):
        """A pending rollback that changes nothing (the first step of a DFlash run): the rows' form for one stream
        (`_pend_identity_rows`: zero KDA update inputs, the current conv states and pool tails) with a scalar
        n_keep = T; the same group programs then serve every step."""
        hists, _, _ = self._pend_identity_rows([caches], T)
        return hists, self.scalar(T), T

    def dflash_settle(self, st):
        """The state with its deferred work resolved — a verify dispatched ahead of its tokens (`run_ahead`) undone,
        since its block is never accepted (the small state entries back to before it: n_keep = 0), or the last step's
        pending rollback applied. Call before using st["caches"] outside DFlash steps (a stream leaving DFlash mode,
        tests). Idempotent."""
        n = self.cfg.n_layers
        caches = st["caches"]
        if st.get("ahead") is not None:
            hists, nk, T = st["ahead"][2], self.scalar(0), st["T"]
        elif st.get("pend"):
            hists, nk, T = st["pend"]
        else:
            return st
        rolled = self._prog_rollback(1, T, kin=True)(hists, nk, self._hist_cur(caches[:n], kin=True))
        new = [({**c, **x} if x else c) for c, x in zip(caches[:n], rolled)]
        return {**{k: v for k, v in st.items() if k != "ahead"}, "caches": new + list(caches[n:]), "pend": None}

    def dflash_end(self):
        """Release what the last `dflash_step` kept for a deferred release (call when a stream leaves DFlash mode)."""
        self.__dict__.pop("_grave", None)

    # ---- DFlash 2 for B streams at once (continuous batching: each stream its own cache set, ring and position)
    def _dflash_draft_rows(self, rings, feats, pos_b, n_new_b, anchor_b, T, temp, key):
        r = self.dflash.draft_rows(rings, feats[0], pos_b, n_new_b, anchor_b, self.params["embed"],
                                   self.params["lm_head"], T=T, temperature=temp, key=key, feat_index=feats[1])
        return r["seq"], r["cand"], r["q"], r["rings"]

    def dflash_start_rows(self, tokens, cache_sets, pos, T=None, sampler: DeviceSampler | None = None):
        """Begin DFlash 2 decoding of B independent streams together (`decode_rows`' layout): cache_sets[b] = stream b's
        full cache set (its drafter ring included, holding every committed position: prefill, `decode_rows` and DFlash
        steps keep it so; consumed), tokens [B] = their committed, not yet fed tokens, pos [B] host ints. Dispatches
        the first draft of every row; `sampler` = one temperature / top_p per row (or scalars; None = greedy). Returns
        the state `dflash_step_rows` advances: {"sets" (each row's n layer caches), "rings", "pos" (device [B]),
        "pos_h" (host ints), "seq", "cand", "q", "T"}."""
        n = self.cfg.n_layers
        B = len(cache_sets)
        assert self.dflash is not None and all(len(c) > n for c in cache_sets)
        T = self.dflash.cfg.block if T is None else T
        sampler = sampler if sampler is not None else self._dflash_greedy()
        temp, _, k = sampler.bind(self.mesh)
        pos_d = self.device_positions(pos)
        seq, cand, q, rings = self._dflash_draft_rows([c[n] for c in cache_sets], self._dflash_zeros(B), pos_d,
                                                      self.device_positions([0] * B),
                                                      self.device_positions(np.asarray(tokens).reshape(B)), T, temp, k)
        return {"sets": [list(c[:n]) for c in cache_sets], "rings": rings, "pos": pos_d,
                "pos_h": [int(p) for p in pos], "seq": seq, "cand": cand, "q": q, "T": T}

    def dflash_step_rows(self, st, sampler: DeviceSampler | None = None, stop_ids=(), draft_override=None, next_T=None):
        """`dflash_step` for B streams at once: one batched verify (every row its own block on its own caches; the MoE
        groups each row's T tokens by expert), the acceptance per row on the chips, each row's states rolled back to
        its own accepted prefix, then the next draft of every row (its accepted positions into its ring) queued before
        the host reads the tokens. Rows are independent: at temperature 0 each row emits exactly what that stream's
        own greedy decoding would. draft_override [B, T-1] (tests). With `run_ahead` the next blocks' verify is
        dispatched too (undone by `dflash_settle_rows` when the state ends). Returns (emitted tokens per row, the new
        state)."""
        T, B = st["T"], len(st["sets"])
        sampler = sampler if sampler is not None else self._dflash_greedy()
        grave = self.__dict__.pop("_grave", None)        # the previous step's dropped arrays (as in dflash_step)
        seq = st["seq"]
        ahead = st.get("ahead")                          # this step's verify, dispatched by the previous step
        if draft_override is not None:
            if ahead is not None:                        # (tests) the verify ahead saw the real drafts: undo it
                st, ahead = self.dflash_settle_rows(st), None
            seq = jnp.concatenate([seq[:, :1], jnp.asarray(np.asarray(draft_override, np.int32).reshape(B, T - 1))], 1)
        if ahead is None:
            pend = st.get("pend") or self._pend_identity_rows(st["sets"], T)
            ahead = self._verify_rows(seq, st["sets"], st["pos"], pend)
        else:
            pend = None
        streams, new, hists, feats = ahead
        self._drop(grave, pend)
        del grave, pend, ahead
        temp, top_p, k = sampler.bind(self.mesh)
        r = self._prog_head_accept(B, T, tuple(stop_ids), per_row=True)(self.params["norm"], self.params["lm_head"],
                                                                      streams, seq, st["cand"], st["q"], st["pos"],
                                                                      temp, top_p, k)
        sampler.advance(r["key"])
        # the rollback of every row to its accepted prefix is deferred to the next verify's group programs (or
        # `dflash_settle_rows`), as in `dflash_step`; the draft is queued behind the head (see there)
        next_T = T if next_T is None else next_T
        nseq, cand, q, rings = self._dflash_draft_rows(st["rings"], feats, r["pos_b"], r["n_new_b"], r["bonus"], next_T,
                                                       temp, r["key"])
        out = {"sets": new, "rings": rings, "pos": r["pos_next"], "pend": (hists, r["n_keep"], T), "seq": nseq,
               "cand": cand, "q": q, "T": next_T}
        if self.run_ahead and max(st["pos_h"]) + T + next_T <= self.max_len:     # run ahead (see dflash_step)
            out["ahead"] = self._verify_rows(nseq, new, r["pos_next"], out.pop("pend"))
            out["sets"] = out["ahead"][1]
        emitted = [[int(x) for x in row if x >= 0] for row in np.asarray(r["emitted"])]   # the host waits here
        out["pos_h"] = [p + len(e) for p, e in zip(st["pos_h"], emitted)]
        self._grave = (st, feats, streams, r)
        return emitted, out

    def _dflash_greedy(self):
        if getattr(self, "_greedy_sampler", None) is None:
            self._greedy_sampler = DeviceSampler(0.0, 1.0, seed=0)
        return self._greedy_sampler

    def decode(self, token, caches, pos):
        B = token.shape[0]
        n = self.cfg.n_layers
        extra = caches[n:] if len(caches) > n else []
        logits, caches = self._run(np.asarray(token).reshape(B, 1), caches[:n], pos, True)
        return logits, caches + extra, pos + 1
'''
FILES["glm53/scheduler.py"] = r'''"""Continuous batching for `ResidentLayerEngine.decode_rows`: independent streams decoded together, admitted between
steps, prefilled piece by piece with decode steps of the running streams in between, each keeping its own cache set.

One engine thread (`Scheduler.run`) owns every JAX call. Client threads `submit` a `Request` and wait on its
`done` event (tokens arrive through `on_token`, called on the engine thread). Cache sets on the chips = the active
streams' + finished contexts kept LIVE for the next turn (LRU), at most `max_sets` in total; beyond that, finished
contexts are parked in host memory (`SnapStore`) and resumed when a later prompt extends them. Between steps only the
sampled ids cross to the host: token ids, positions and the sampler state stay on the device.

DFlash 2 (`spec`, with a drafter installed on the engine: `ResidentLayerEngine.set_dflash`): a stream that runs ALONE
decodes speculatively (`dflash_start` / `dflash_step`: a block of T-1 drafts verified per step); the block size is
picked per stream and step from an EMA of the tokens per step (`DFlashPolicy`). Several active streams decode together:
either speculatively in ONE batched step (`dflash_start_rows` / `dflash_step_rows`: every stream verifies its own block
of the smallest size on its own caches, accepted per row) when the policy's measured step costs (`rows`: B -> ms of a
B-row DFlash step and of a plain batched step) and the streams' EMAs say it yields more tokens per ms, or in the
batched plain steps (`decode_rows`, which also write every stream's positions into its own drafter ring, so a stream
that goes back to DFlash drafts with its full context). A DFlash state owns its streams' caches and hands them back
when the membership or the mode changes. With temperature > 0 the drafts are accepted by rejection sampling against
the target's distribution (every token distributed as plain sampling).
"""
import collections
import threading
import time

import numpy as np
import jax
import jax.numpy as jnp

from glm53.engine import DeviceSampler


def match_prefix(live_ids, pos, prompt, quiet=False):
    """Default matcher: (k, fed) when the context (live_ids[:pos]) is a prefix of `prompt`: prompt[k:] must still be
    prefilled and `fed` = the ids the engine will have seen after that; (0, None) otherwise."""
    if pos == 0 or len(prompt) < pos:
        return 0, None
    a = np.asarray(live_ids[:pos]); b = np.asarray(prompt[:pos])
    if a.shape != b.shape or not np.array_equal(a, b):
        return 0, None
    return pos, list(live_ids[:pos]) + list(prompt[pos:])


class Request:
    """A generation request. `prompt`: token ids (the server's signature ids: image tokens negative, `imgs` maps
    them; the scheduler's `feed` turns slices into real ids + embedding overrides). Results: `out` (emitted tokens,
    the stop token included when hit), `stop_reason` ("stop" | "length" | "cancelled" | "error"), timings."""

    def __init__(self, prompt, max_new, temperature=1.0, top_p=1.0, imgs=None, on_token=None, rid=None, on_done=None,
                 budget=None):
        self.prompt = [int(t) for t in prompt]
        self.max_new, self.temperature, self.top_p = int(max_new), float(temperature), float(top_p)
        self.imgs, self.on_token, self.rid, self.on_done = imgs, on_token, rid, on_done
        self.budget = budget            # (n, end_id, forced_ids): unless `end_id` was emitted within the first n output
                                        # tokens, the next tokens are forced to `forced_ids` (a thinking budget)
        self.out = []
        self.done = threading.Event()
        self.error = None
        self.cancelled = False
        self.stop_reason = None
        self.reused = 0
        self.prefill_s = 0.0
        self.t_submit, self.t_first, self.t_end = time.time(), None, None

    def cancel(self):
        self.cancelled = True

    @property
    def decode_s(self):
        return (self.t_end or time.time()) - (self.t_first or self.t_submit)


class _Stream:
    """An admitted request: its own cache set, its position, the ids fed so far and the last (unfed) token."""

    def __init__(self, req, caches, pos, fed, tok, logits):
        self.req, self.caches, self.pos, self.fed, self.tok = req, caches, pos, list(fed), tok
        self.seen = []                                   # tokens fed beyond `fed`
        self.logits = logits                             # logits [1,V] after the last fed token (predict `tok`)
        self.max_new = req.max_new
        self.open = req.budget is not None               # the budgeted block (thinking) is still open
        self.force = []                                  # tokens to feed instead of the sampled ones (budget reached)
        self.spec = None                                 # DFlash state (engine.dflash_start) while decoding alone
        self.sampler = None                              # its own device sampler in DFlash mode
        self.tau = None                                  # DFlashPolicy's per-stream EMAs


class SnapStore:
    """Host-memory LRU of compact context snapshots keyed by their token ids (see `ResidentLayerEngine.snapshot_prefix`).
    `park` moves a context off the chips, `lookup` finds the entry a prompt extends (the matcher handles dropped
    thinking blocks and boundary drift when the server passes its own), `restore` brings it back into fresh
    full-capacity caches. A parked conversation keeps its two latest turn boundaries; older strict prefixes are
    dropped unless pinned (pinned = a conversation's system section, the branch point new sessions start from)."""

    def __init__(self, eng, max_bytes, rows_bucket=1024, match_len=match_prefix, log=print, state=None):
        self.eng, self.max_bytes, self.rows_bucket = eng, max_bytes, rows_bucket
        self.match_len, self.log = match_len, log
        self.state = state if state is not None else {}
        self.entries = collections.OrderedDict()                     # tuple(ids) -> entry, oldest first
        self.bytes = 0

    def park(self, ids, caches, pos, logits, pinned=False):
        """Snapshot `caches` (a context of `pos` tokens = `ids`) into host memory; the caches stay valid."""
        key = tuple(int(t) for t in ids)
        if key in self.entries:
            e = self.entries[key]
            e["pinned"] = e["pinned"] or pinned
            if logits is not None and e["logits"] is None:
                e["logits"] = np.asarray(logits)
            self.entries.move_to_end(key)
            return e
        t = time.time()
        snap = self.eng.snapshot_prefix(caches, pos, rows_bucket=self.rows_bucket)
        host = self.eng.snapshot_to_host(snap)
        del snap
        e = {"ids": list(key), "n": pos, "snap": host, "logits": None if logits is None else np.asarray(logits),
             "bytes": host["bytes"], "pinned": pinned}
        if not pinned:                                                # supersede our own earlier turns but the latest
            older = sorted((k for k, o in self.entries.items() if not o["pinned"] and len(k) < len(key) and key[:len(k)] == k),
                           key=len)
            for k in older[:-1]:
                self._drop(k)
        self.entries[key] = e
        self.bytes += e["bytes"]
        while self.bytes > self.max_bytes and len(self.entries) > 1:
            victims = [k for k, o in self.entries.items() if not o["pinned"]] or list(self.entries)
            self._drop(victims[0])
        st = self.state
        st["snap_parks" if not pinned else "snap_pins"] = st.get("snap_parks" if not pinned else "snap_pins", 0) + 1
        st["snap_entries"], st["snap_bytes"] = len(self.entries), self.bytes
        st["snap_s"] = st.get("snap_s", 0.0) + time.time() - t
        self.log(f"parked {pos} tokens ({e['bytes'] / 1e6:.0f} MB, {'pinned' if pinned else 'lru'}) in {time.time() - t:.2f}s; "
                 f"store {len(self.entries)} entries, {self.bytes / 1e9:.2f} GB")
        return e

    def _drop(self, key):
        e = self.entries.pop(key)
        self.bytes -= e["bytes"]

    def lookup(self, prompt):
        """-> (k, fed, entry) for the entry that covers most of `prompt`, or (0, None, None)."""
        best = (0, None, None)
        for key, e in list(self.entries.items()):
            if e["n"] <= best[0]:
                continue
            k, fed = self.match_len(e["ids"], e["n"], prompt, quiet=True)
            if k > best[0] and not (k == len(prompt) and e["logits"] is None):
                best = (k, fed, e)
        if best[2] is not None:
            self.entries.move_to_end(tuple(best[2]["ids"]))
        return best

    def restore(self, e):
        t = time.time()
        caches = self.eng.restore_prefix(self.eng.snapshot_from_host(e["snap"]))
        self.state["snap_hits"] = self.state.get("snap_hits", 0) + 1
        self.state["snap_s"] = self.state.get("snap_s", 0.0) + time.time() - t
        return caches


def _argmax_first(logits_row, temperature, top_p):
    return int(np.argmax(logits_row))


class DFlashPolicy:
    """Block size per DFlash step: `cost` = {T: ms per step} (measured, e.g. v6e 2026-10-03: {4: 21.2, 8: 27.3});
    the stream's EMA of tokens per step for each T picks the T with the least ms per token. A step of T also tells
    what a smaller T' would have accepted (min(tokens, T')), and a run cut before T what a larger one would have
    (the same tokens); only a fully accepted block leaves the larger T unknown — a T without news for `probe` steps
    is tried once. `prior` = the EMAs' start values. `rows` = {B: (ms of a DFlash step of B streams at the smallest
    block, ms of a plain batched step of B)}: B streams decode speculatively together when the sum of their EMAs per
    DFlash ms beats B tokens per plain ms (`pick_rows`); no entry for B = always plain. Plain steps measure no
    acceptance, so the choice is sticky: a batched DFlash state, once started (by the costs or by a probe after
    `probe` plain steps), runs at least `hold` steps and ends only below `stay` x the plain rate (at temperature 1.0 the
    EMAs of prose sit near the break-even: without it one dip kept a pair of streams plain, v6e 2026-10-04 j131)."""

    def __init__(self, cost, prior=None, alpha=0.15, probe=32, rows=None, hold=8, stay=0.9):
        self.cost = {int(t): float(c) for t, c in cost.items()}
        self.prior = prior or {t: 1.0 + 0.5 * (t - 1) ** 0.5 for t in self.cost}
        self.alpha, self.probe = alpha, probe
        self.rows = {int(b): (float(c[0]), float(c[1])) for b, c in (rows or {}).items()}
        self.hold, self.stay = hold, stay

    @property
    def rows_T(self):
        """The block size of batched DFlash steps (one for all rows: the smallest compiled)."""
        return min(self.cost)

    def pick_rows(self, streams, margin=1.0):
        """True when B streams decoding speculatively together (each `start`ed) yield more tokens per ms than
        margin x plain."""
        c = self.rows.get(len(streams))
        if c is None:
            return False
        return sum(s.tau["ema"][self.rows_T] for s in streams) / c[0] > margin * len(streams) / c[1]

    def start(self, s):
        s.tau = {"ema": dict(self.prior), "last": {t: 0 for t in self.cost}, "n": 0}

    def pick(self, s):
        st = s.tau
        for t, last in st["last"].items():
            if st["n"] - last >= self.probe:
                return t
        return min(self.cost, key=lambda t: self.cost[t] / st["ema"][t])

    def update(self, s, T, n_tokens):
        st = s.tau
        st["n"] += 1
        for t in self.cost:
            if t <= T or n_tokens < T:                 # a run cut before T ends there for any larger block too
                st["ema"][t] += self.alpha * (min(n_tokens, t) - st["ema"][t])
                st["last"][t] = st["n"]


class Scheduler:
    """See the module docstring. `feed(req, a, b) -> (ids [n] int32, embeds or None)` renders req.prompt[a:b] for
    the engine (the server attaches its image data to the request); `match_len(live_ids, pos, prompt, quiet=False) -> (k, fed)`; `system_end(prompt)` = length of
    the system section worth pinning (0 = none); `first_sample(logits_row, temperature, top_p) -> int` samples the
    first token of a request on the host (the rest come from the device sampler); `spec` = a `DFlashPolicy` to
    decode a stream that runs alone with the engine's DFlash drafter (None = never)."""

    def __init__(self, eng, stop_ids, max_streams=4, max_sets=None, feed=None, match_len=match_prefix,
                 system_end=None, first_sample=_argmax_first, snaps=None, log=print, state=None,
                 base_min=512, snap_min=256, piece=None, seed=None, min_free_gb=0.65, max_wait_s=90.0, spec=None):
        self.eng, self.stop_ids = eng, set(int(t) for t in stop_ids)
        self.max_streams = max(1, int(max_streams))
        self.max_sets = max(self.max_streams, int(max_sets if max_sets is not None else max_streams + 1))
        self.feed = feed or (lambda req, a, b: (np.asarray(req.prompt[a:b], np.int32), None))
        self.match_len, self.system_end = match_len, (system_end or (lambda prompt: 0))
        self.first_sample = first_sample
        self.snaps = snaps if snaps is not None else SnapStore(eng, 0, 1, match_len, log, state)
        self.log, self.state = log, (state if state is not None else {})
        self.base_min, self.snap_min = base_min, snap_min
        self.piece = piece or eng.prefill_piece
        self.min_free_gb = min_free_gb                  # HBM an admission needs (free_gb: a set, prefill temporaries, programs); 0 = off
        self.max_wait_s = max_wait_s                    # a request queued longer than this fails ("queue_timeout")
        self.live = collections.OrderedDict()            # finished contexts on the chips (LRU): key -> ctx dict
        self._live_n = 0
        self.active = []
        self.pending = collections.deque()
        self.cond = threading.Condition()
        self.sampler = DeviceSampler(1.0, 1.0, seed=seed)
        self._members = None                             # ids of the streams the device state below belongs to
        self._toks = self._pos = None
        self._stop = False
        self._paused, self._idle = False, threading.Event()
        self._waits = 0
        self._t_step = None
        self.thread = None
        self.steps = 0
        self.spec = spec if spec is not None and getattr(eng, "dflash", None) is not None else None
        self.seed = seed
        self._buf_cache = None                           # (sets on the chips, live buffer bytes on chip 0): free_gb
        self._grave = None                               # the last batched step's consumed caches (deferred release)
        self._rows = None                                # DFlash of several streams: {"streams", "st", "sampler"}
        self._rows_skip = 0                              # plain batched steps since the policy last chose DFlash rows
        self._rows_hold = 0                              # steps the running batched DFlash state still runs regardless

    # ---- client side
    def submit(self, req: Request):
        if len(req.prompt) + 2 > self.eng.max_len:
            self._fail(req, ValueError(f"prompt of {len(req.prompt)} tokens exceeds the context capacity {self.eng.max_len}"))
            return req
        with self.cond:
            self.pending.append(req)
            self.cond.notify()
        return req

    def start(self):
        self.thread = threading.Thread(target=self.run, daemon=True, name="glm-scheduler")
        self.thread.start()
        return self.thread

    def stop(self):
        with self.cond:
            self._stop = True
            self.cond.notify_all()
        if self.thread is not None:
            self.thread.join(timeout=60)
        for ctx in self.live.values():
            self._free_set(ctx["caches"])
        for s in self.active:
            self._spec_end(s)
            self._free_set(s.caches)
        self.live.clear(); self.active.clear(); self._toks = self._pos = None

    @staticmethod
    def _free_set(caches):
        """Release a dropped cache set's HBM now (`Array.delete`), whatever else still references the arrays: a
        dropped set kept alive by a stray reference (seen once after a concurrent burst, 2026-09-13) cost a stream
        slot for the rest of the session — refcounts alone are not a guarantee."""
        for x in jax.tree.leaves(caches):
            try:
                x.delete()
            except Exception:  # noqa: BLE001
                pass

    @property
    def n_sets(self):
        return len(self.active) + len(self.live)

    def free_gb(self):
        """HBM an admission can count on, chip 0, in GB: bytes_limit minus the live device buffers (None when the backend
        reports no memory statistics: CPU tests). bytes_in_use is NOT the measure: it also holds the runtime's cache of
        loaded programs, which the runtime evicts when a buffer needs the room and reloads at a program's next run
        (2026-10-03, v6e under the v5e budget: five 262k sets allocated where bytes_limit - bytes_in_use promised two,
        and 0.3-0.6 GB of in-use were such programs). `min_free_gb` therefore covers a new set, one prefill piece's
        temporaries AND the programs a decode step and a prefill piece need at once. The buffer sum (global arrays,
        ~6 ms) is cached while the number of cache sets on the chips does not change."""
        st = self.eng.mesh.devices.flat[0].memory_stats() or {}
        if "bytes_limit" not in st:
            return None
        key = self.n_sets
        if self._buf_cache is None or self._buf_cache[0] != key:
            n = len(self.eng.mesh.devices.flat)
            tot = 0
            for a in jax.live_arrays():
                try:
                    if len(a.sharding.device_set) == n:
                        tot += a.nbytes if a.sharding.is_fully_replicated else a.nbytes // n
                except Exception:  # noqa: BLE001  (deleted arrays)
                    pass
            self._buf_cache = (key, tot)
        return (st["bytes_limit"] - self._buf_cache[1]) / 1e9

    def in_use_free_gb(self):
        """bytes_limit - bytes_in_use on chip 0 (GB; the loaded programs count as used): for logs."""
        st = self.eng.mesh.devices.flat[0].memory_stats() or {}
        return round((st["bytes_limit"] - st["bytes_in_use"]) / 1e9, 3) if "bytes_limit" in st else None

    def _headroom(self):
        """True when an admission can allocate a cache set and run its prefill: parks live contexts (LRU) until the
        set budget and the HBM margin allow it; False when only active streams hold the chips (wait for one)."""
        self._make_room()
        f = self.free_gb()
        return f is None or not self.min_free_gb or f >= self.min_free_gb

    # ---- engine thread
    def pause(self, timeout=60.0):
        """Stop the engine thread between steps (running streams stall, nothing is admitted) until `resume`; returns
        once the thread is idle. For probe jobs that need the chips."""
        with self.cond:
            self._paused = True
            self.cond.notify_all()
        return self._idle.wait(timeout)

    def resume(self):
        with self.cond:
            self._paused = False
            self.cond.notify_all()

    def run(self):
        while not self._stop:
            with self.cond:
                while not self._stop and (self._paused or (not self.pending and not self.active)):
                    if self._paused:
                        self._idle.set()
                    self._t_step = None
                    self.cond.wait(timeout=1.0)
                self._idle.clear()
                if self._stop:
                    break
                req = None
                while self.pending and self.max_wait_s and time.time() - self.pending[0].t_submit > self.max_wait_s:
                    old = self.pending.popleft()                        # queued too long: fail it (the client can retry)
                    self._fail(old, TimeoutError(f"queued for {time.time() - old.t_submit:.0f}s without a free cache set"), "queue_timeout")
                if self.pending and len(self.active) < self.max_streams and self._headroom():
                    req = self.pending.popleft()
                elif self.pending and not self.active:                  # nothing running and still no room
                    if self._waits % 10 == 0:
                        self.log(f"admission waits: {len(self.live)} live contexts, free HBM {self.free_gb():.3f} GB "
                                 f"(in-use figure {self.in_use_free_gb()} GB)" if self.free_gb() is not None else
                                 f"admission waits: {len(self.live)} live contexts")
                    self._waits += 1
                    self.cond.wait(timeout=1.0)
            if req is not None:
                if req.cancelled:
                    self._fail(req, None, "cancelled"); continue
                try:
                    self._admit(req)
                except Exception as e:  # noqa: BLE001
                    import gc, traceback
                    self.log("admission failed:", traceback.format_exc()[-1500:])
                    self._fail(req, RuntimeError(f"{type(e).__name__}: {str(e)[:800]}"))   # no traceback: its frames
                    del e                                                                    # would pin the half-built caches
                    gc.collect()
                continue
            if self.active:
                try:
                    self._step()
                except Exception as e:  # noqa: BLE001
                    import gc, traceback
                    self.log("decode step failed:", traceback.format_exc()[-1500:])
                    err = RuntimeError(f"{type(e).__name__}: {str(e)[:800]}")
                    del e
                    for s in list(self.active):
                        s.req.error = err
                        self._finish(s, "error", keep=False)
                    self.active = []; self._members = None; self._toks = self._pos = None
                    gc.collect()

    def _fail(self, req, error, reason="error"):
        """Finish a request that never became a stream (capacity error, admission failure, cancelled while queued)."""
        req.error, req.stop_reason, req.t_end = error, reason, time.time()
        if req.on_done is not None:
            try:
                req.on_done()
            except Exception:  # noqa: BLE001
                pass
        req.done.set()

    def _make_room(self):
        """Park LRU live contexts until a cache set can be allocated within the set budget AND the HBM margin
        (`min_free_gb`, prefill temporaries); a parked set is freed (donated caches)."""
        while self.live:                                                 # (no gc.collect() here: a full collection
            f = self.free_gb()                                           #  costs ~14 s in a process with hundreds of
            if self.n_sets < self.max_sets and (f is None or not self.min_free_gb or f >= self.min_free_gb):
                break                                                    #  compiled programs; refcounts free the set)
            key, ctx = self.live.popitem(last=False)
            if ctx["pos"] >= self.snap_min:
                self.snaps.park(ctx["ids"], ctx["caches"], ctx["pos"], ctx["logits"])
            self._free_set(ctx["caches"])
            del ctx

    def _prefill(self, req, a, b, caches, pos):
        """prompt[a:b] into `caches` at `pos`, one piece at a time with a decode step of the running streams between
        pieces (chunked admission). Returns (logits, caches, pos)."""
        logits = None
        for x in range(a, b, self.piece):
            y = min(b, x + self.piece)
            ids, emb = self.feed(req, x, y)
            ids = np.asarray(ids, np.int32).reshape(1, -1)
            logits, caches, pos = self.eng.prefill(ids, caches, pos, embeds=emb)
            if y < b and self.active:
                self._step()
        return logits, caches, pos

    def _admit(self, req):
        prompt = req.prompt
        t0 = time.time()
        st = self.state
        # 1. a finished context still on the chips that the prompt extends
        best = (0, None, None)
        for key, c in self.live.items():
            k, fed = self.match_len(c["ids"], c["pos"], prompt, quiet=True)
            if k > best[0] and not (k == len(prompt) and c["logits"] is None):
                best = (k, fed, key)
        c = None                                                         # (keep no reference to a context _make_room may drop)
        if best[2] is None and self.live:
            self.log(f"no live context matched ({len(self.live)} live, {len(self.snaps.entries)} parked)")
        if best[2] is not None:
            k, fed, key = best
            ctx = self.live.pop(key)
            st["prefix_hits"] = st.get("prefix_hits", 0) + 1
            st["prefix_tokens_reused"] = st.get("prefix_tokens_reused", 0) + k
            if k == len(prompt):
                logits, caches, pos = ctx["logits"], ctx["caches"], ctx["pos"]
            else:
                logits, caches, pos = self._prefill(req, k, len(prompt), ctx["caches"], ctx["pos"])
            req.reused = k
        else:
            self._make_room()
            k, fed, entry = self.snaps.lookup(prompt)
            if k > 0:                                                    # 2. a parked context the prompt extends
                st["prefix_tokens_reused"] = st.get("prefix_tokens_reused", 0) + k
                caches = self.snaps.restore(entry)
                if k == len(prompt):
                    logits, pos = entry["logits"], entry["n"]
                else:
                    logits, caches, pos = self._prefill(req, k, len(prompt), caches, entry["n"])
                req.reused = k
            else:                                                        # 3. from scratch (pin the system section)
                fed = list(prompt)
                n_sys = self.system_end(prompt)
                if n_sys >= self.base_min and n_sys < len(prompt):
                    _, caches, pos = self._prefill(req, 0, n_sys, None, 0)
                    self.snaps.park(prompt[:n_sys], caches, pos, None, pinned=True)
                    logits, caches, pos = self._prefill(req, n_sys, len(prompt), caches, pos)
                else:
                    logits, caches, pos = self._prefill(req, 0, len(prompt), None, 0)
        jax.block_until_ready(logits)
        self._t_step = None
        req.prefill_s = time.time() - t0
        st["prefill_s"] = st.get("prefill_s", 0.0) + req.prefill_s
        tok = self.first_sample(np.asarray(logits)[0], req.temperature, req.top_p)
        s = _Stream(req, caches, pos, fed, tok, logits)
        s.max_new = max(1, min(req.max_new, self.eng.max_len - pos - 1))
        req.t_first = time.time()
        self.active.append(s); self._members = None
        self._emit(s, tok)

    def _emit(self, s, tok):
        req = s.req
        req.out.append(tok)
        if s.open and tok == req.budget[1]:
            s.open = False
        if req.on_token is not None and not req.cancelled:
            try:
                req.on_token(tok)
            except Exception as e:  # noqa: BLE001
                self.log(f"on_token failed ({e!r}): cancelling {req.rid}")
                req.cancelled = True
        if tok in self.stop_ids:
            self._finish(s, "stop")
        elif len(req.out) >= s.max_new or s.pos + 1 >= self.eng.max_len:
            self._finish(s, "length")
        elif req.cancelled:
            self._finish(s, "cancelled")

    def _finish(self, s, reason, keep=True):
        """The stream's context becomes a live (on-chip) context: ids = everything the engine has seen (the last
        emitted token is never fed), logits = the prediction after them."""
        req = s.req
        self._spec_end(s)
        if s in self.active:
            self.active.remove(s); self._members = None
        if not keep:
            self._free_set(s.caches)
        if keep:
            self._live_n += 1
            lg = s.logits[0][s.logits[1]:s.logits[1] + 1] if isinstance(s.logits, tuple) else s.logits
            self.live[self._live_n] = {"ids": s.fed + s.seen, "caches": s.caches, "pos": s.pos, "logits": lg}
        req.stop_reason = req.stop_reason or reason
        req.t_end = time.time()
        self.state["tokens"] = self.state.get("tokens", 0) + len(req.out)
        if req.on_done is not None:
            try:
                req.on_done()
            except Exception as e:  # noqa: BLE001
                self.log(f"on_done failed ({e!r}) for {req.rid}")
        req.done.set()

    def _step(self):
        """One batched decode step of the active streams."""
        for s in list(self.active):
            if s.req.cancelled:
                self._finish(s, "cancelled")
        act = list(self.active)                                         # (streams finishing below leave self.active)
        if not act:
            return
        if len(act) == 1 and self._spec_ok(act[0]):
            self._rows_end()
            return self._spec_step(act[0])
        if len(act) > 1 and self._rows_ok(act):
            return self._rows_step(act)
        self._rows_end()
        for s in act:                                                   # batched again: DFlash states hand back
            self._spec_end(s)
        B = len(act)
        members = tuple(id(s) for s in act)
        if members != self._members:                                    # membership changed: re-place the small state
            self._pos = self.eng.device_positions([s.pos for s in act])
            self._toks = jnp.asarray([s.tok for s in act], jnp.int32)
            self.sampler.set([s.req.temperature for s in act], [s.req.top_p for s in act])
            self._members = members
        t0 = time.perf_counter()
        ids, logits, sets, pos_next = self.eng.decode_rows(self._toks, [s.caches for s in act], self._pos, self.sampler)
        self._grave = None                     # the last step's consumed caches: released while the device runs this one
        ids_h = np.asarray(ids)                                         # the one device -> host sync per step
        forced = False
        for r, s in enumerate(act):                                     # a budget reached: feed its forced tokens
            if s.open and not s.force and len(s.req.out) >= s.req.budget[0]:
                s.force = list(s.req.budget[2])
            if s.force:
                if not forced:
                    ids_h, forced = np.array(ids_h), True
                ids_h[r] = s.force.pop(0)
        if forced:
            ids = jnp.asarray(ids_h, jnp.int32)
        t1 = time.perf_counter()
        self._toks, self._pos = ids, pos_next
        self.steps += 1
        st = self.state
        st["steps"] = self.steps
        st["step_tokens"] = st.get("step_tokens", 0) + B
        st["decode_s"] = st.get("decode_s", 0.0) + (t1 - t0)          # inside the engine (dispatch + device + sync)
        if self._t_step is not None:                                    # back-to-back steps: the loop's own overhead
            st["step_s"] = st.get("step_s", 0.0) + (t1 - self._t_step)
            st["steps_busy"] = st.get("steps_busy", 0) + 1
        self._t_step = t1
        self._grave = [s.caches for s in act]           # (dropping ~126 arrays per set costs ~1.5 ms of host time)
        for r, s in enumerate(act):                                     # every row takes its new caches first
            s.caches, s.pos = sets[r], s.pos + 1
            s.seen.append(s.tok)
            s.tok = int(ids_h[r])
            s.logits = (logits, r)                                      # sliced only when the stream finishes
        for s in act:
            self._emit(s, s.tok)

    # ---- DFlash (a stream running alone)
    def _spec_ok(self, s):
        """DFlash for this stream now: a policy and a drafter, no forced tokens pending, and a whole block of room
        before the thinking budget, max_new and the context capacity (the tail decodes plainly)."""
        if self.spec is None or s.force:
            return False
        T, req = max(self.spec.cost), s.req
        if s.open and len(req.out) + T >= req.budget[0]:
            return False
        return len(req.out) + T < s.max_new and s.pos + T + 1 < self.eng.max_len

    def _spec_end(self, s):
        """Leave DFlash mode: the stream takes its caches (the drafter's ring included) and position back; the queued
        draft is dropped (it only wrote committed positions into the ring). A stream of a batched DFlash state ends
        that state for all its streams."""
        if self._rows is not None and any(x is s for x in self._rows["streams"]):
            self._rows_end()
        if s.spec is None:
            return
        st = self.eng.dflash_settle(s.spec)                            # the last step's rollback was deferred
        s.caches, s.pos, s.spec = st["caches"], st["pos_h"], None
        self.eng.dflash_end()
        self._members = None

    def _spec_step(self, s):
        req, st = s.req, self.state
        if s.spec is None:
            if s.tau is None:
                self.spec.start(s)
            if s.sampler is None:
                s.sampler = DeviceSampler(req.temperature, req.top_p, seed=self.seed)
            s.spec = self.eng.dflash_start(np.array([s.tok]), s.caches, s.pos, T=self.spec.pick(s), sampler=s.sampler)
            s.caches = None
            self._members = None
        T = s.spec["T"]
        t0 = time.perf_counter()                                        # the next block's T is chosen before this
        emitted, s.spec = self.eng.dflash_step(s.spec, s.sampler, tuple(sorted(self.stop_ids)),   # step's tokens are
                                               next_T=self.spec.pick(s))                         # known (it is queued)
        self.spec.update(s, T, len(emitted))
        t1 = time.perf_counter()
        self.steps += 1
        st["steps"] = self.steps
        st["step_tokens"] = st.get("step_tokens", 0) + len(emitted)
        st["spec_steps"] = st.get("spec_steps", 0) + 1
        st["spec_tokens"] = st.get("spec_tokens", 0) + len(emitted)
        st["decode_s"] = st.get("decode_s", 0.0) + (t1 - t0)
        if self._t_step is not None:
            st["step_s"] = st.get("step_s", 0.0) + (t1 - self._t_step)
            st["steps_busy"] = st.get("steps_busy", 0) + 1
        self._t_step = t1
        s.seen.extend([s.tok] + emitted[:-1])                           # the anchor and the accepted drafts were fed
        s.tok, s.pos, s.logits = emitted[-1], s.spec["pos_h"], None
        for tok in emitted:
            if s not in self.active:                                    # finished mid-block (length / cancelled)
                break
            self._emit(s, tok)

    # ---- DFlash for several streams at once (one batched verify per step)
    def _rows_ok(self, act):
        """Batched DFlash for these streams now: every one could run DFlash alone (`_spec_ok`), the policy has costs for
        this B, and their EMAs say it pays — or `probe` plain batched steps passed without (to measure again). A
        running state stays for its `hold` steps, then while it pays `stay` x plain (`DFlashPolicy`)."""
        pol = self.spec
        if pol is None or len(act) not in pol.rows or not all(self._spec_ok(s) for s in act):
            return False
        for s in act:
            if s.tau is None:
                pol.start(s)
        if self._rows is not None and [id(x) for x in self._rows["streams"]] == [id(x) for x in act]:
            self._rows_hold -= 1                       # (the running state)
            ok = self._rows_hold >= 0 or pol.pick_rows(act, pol.stay)
        else:
            ok = pol.pick_rows(act) or self._rows_skip >= pol.probe
            self._rows_hold = pol.hold
        self._rows_skip = 0 if ok else self._rows_skip + 1
        return ok

    def _rows_end(self):
        """Leave batched DFlash: every stream of the state takes its caches (ring included) and position back."""
        R = self._rows
        if R is None:
            return
        st = self.eng.dflash_settle_rows(R["st"])                         # the last step's rollbacks were deferred
        for b, s in enumerate(R["streams"]):
            s.caches, s.pos = st["sets"][b] + [st["rings"][b]], st["pos_h"][b]
        self._rows = None
        self.eng.dflash_end()
        self._members = None

    def _rows_step(self, act):
        st = self.state
        if self._rows is None or [id(x) for x in self._rows["streams"]] != [id(x) for x in act]:
            self._rows_end()                                            # membership changed: a new state
            for s in act:
                self._spec_end(s)
            samp = DeviceSampler([s.req.temperature for s in act], [s.req.top_p for s in act], seed=self.seed)
            rs = self.eng.dflash_start_rows([s.tok for s in act], [s.caches for s in act], [s.pos for s in act],
                                            T=self.spec.rows_T, sampler=samp)
            for s in act:
                s.caches = None
            self._rows = {"streams": list(act), "st": rs, "sampler": samp}
            self._members = None
            st["rows_starts"] = st.get("rows_starts", 0) + 1
        R = self._rows
        T = R["st"]["T"]
        t0 = time.perf_counter()
        emitted, R["st"] = self.eng.dflash_step_rows(R["st"], R["sampler"], tuple(sorted(self.stop_ids)))
        t1 = time.perf_counter()
        n_tok = sum(len(e) for e in emitted)
        self.steps += 1
        st["steps"] = self.steps
        st["step_tokens"] = st.get("step_tokens", 0) + n_tok
        st["rows_steps"] = st.get("rows_steps", 0) + 1
        st["rows_tokens"] = st.get("rows_tokens", 0) + n_tok
        st["decode_s"] = st.get("decode_s", 0.0) + (t1 - t0)
        if self._t_step is not None:
            st["step_s"] = st.get("step_s", 0.0) + (t1 - self._t_step)
            st["steps_busy"] = st.get("steps_busy", 0) + 1
        self._t_step = t1
        for b, s in enumerate(act):                     # every row's bookkeeping first (a finish ends the state)
            em = emitted[b]
            self.spec.update(s, T, len(em))
            s.seen.extend([s.tok] + em[:-1])
            s.tok, s.pos, s.logits = em[-1], R["st"]["pos_h"][b], None
        for b, s in enumerate(act):
            for tok in emitted[b]:
                if s not in self.active:                                # finished mid-block (length / cancelled)
                    break
                self._emit(s, tok)

    # ---- warm-up
    def warmup(self, max_B=None, token=0):
        """Compile the batched decode programs for every batch size up to max_streams (zero caches, position 0)."""
        max_B = max_B or self.max_streams
        for B in range(1, max_B + 1):
            t = time.time()
            sets = [self.eng.alloc_caches(1) for _ in range(B)]
            samp = DeviceSampler([1.0] * B, [0.95] * B, seed=0)
            ids, lg, sets, pos = self.eng.decode_rows(np.full((B,), token, np.int32), sets, [0] * B, samp)
            jax.block_until_ready(ids)
            del sets
            self.log(f"warm-up batched decode B={B}: {time.time() - t:.0f}s")
'''
FILES["glm53/vision.py"] = r'''"""GLM-5.3-Flash vision tower (HF `Glm5NextVisionModel`, weights `model.visual.*`, bf16, 1.13 GB) in pure JAX, plus the
image preprocessing of `Glm5NextImageProcessor` (PIL): dynamic-resolution resize to a 28-pixel grid, 14x14 patches
duplicated over the temporal axis (2), 2x2 spatial merge -> one language-model token per 28x28 pixels.

Forward: patch embed (Conv3d == linear over (c, t, ph, pw) = 1176 inputs) -> 24 pre-norm blocks (RMSNorm; attention
with q/k RMSNorm over head_dim 64, 2-D rotary over (h, w) patch positions, full attention within each image; SwiGLU
MLP with the ±10 clamps, all with biases) -> post RMSNorm -> the 2x2 downsample conv (== linear over (c, i, j)) to
4096 -> merger (proj, LayerNorm, GELU, clamped SwiGLU to 10240 and back). Patches are laid out block-major (the 4
patches of a merge block consecutive), so consecutive groups of 4 form one output token; output tokens are the
merge blocks in row-major order, i.e. the order of the `<|image|>` tokens in the prompt.
"""
import math
import numpy as np
import jax
import jax.numpy as jnp

PATCH, MERGE, TEMPORAL = 14, 2, 2
FACTOR = PATCH * MERGE                                                  # 28 pixels per token side
MEAN = np.array([0.48145466, 0.4578275, 0.40821073], np.float32)          # OPENAI_CLIP_MEAN / STD
STD = np.array([0.26862954, 0.26130258, 0.27577711], np.float32)
EPS = 1e-5


# ----------------------------------------------------------------------------- preprocessing (port of the HF processor)
def smart_resize(height, width, min_tokens=16, max_tokens=8000, factor=FACTOR):
    """The HF `smart_resize` for one image (num_frames = temporal_factor = 2): the padded canvas (H, W), both multiples
    of 28, holding between min_tokens and max_tokens 28x28 tokens."""
    ppt = TEMPORAL * factor ** 2
    min_px, max_px = min_tokens * ppt, max_tokens * ppt
    align = lambda v: math.ceil(v / factor) * factor  # noqa: E731
    frames = TEMPORAL
    ah, aw = align(height), align(width)
    if frames * ah * aw < min_px:
        s = math.sqrt(min_px / (TEMPORAL * height * width))
        ah, aw = align(max(1, math.ceil(height * s))), align(max(1, math.ceil(width * s)))
    if frames * ah * aw > max_px:
        lo, hi = 1, height
        bh, bw = factor, factor
        while lo <= hi:
            ch = (lo + hi) // 2
            cw = max(1, math.floor(width * ch / height))
            H, W = align(ch), align(cw)
            if frames * H * W <= max_px:
                bh, bw = H, W
                lo = ch + 1
            else:
                hi = ch - 1
        ah, aw = bh, bw
    return ah, aw


def preprocess(img, min_tokens=16, max_tokens=8000):
    """PIL image -> (patches float32 [gh*gw, 3*2*14*14] in the model's block-major order, (1, gh, gw)). Mirrors the
    HF processor: RGB, bicubic resize of the content to fit the smart_resize canvas (never upscaled unless the image
    is below min_tokens), zero (black) padding right/bottom, rescale 1/255, CLIP mean/std, patchify."""
    from PIL import Image
    img = img.convert("RGB")
    w, h = img.size
    H, W = smart_resize(h, w, min_tokens, max_tokens)
    ppt = TEMPORAL * FACTOR ** 2
    s = min(H / h, W / w)
    if TEMPORAL * h * w >= ppt * min_tokens:
        s = min(1.0, s)
    ch, cw = max(1, min(H, math.floor(h * s))), max(1, min(W, math.floor(w * s)))
    if (ch, cw) != (h, w):
        img = img.resize((cw, ch), Image.BICUBIC)
    x = np.zeros((H, W, 3), np.uint8)
    x[:ch, :cw] = np.asarray(img, np.uint8)
    x = ((x.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)             # [3, H, W]
    gh, gw = H // PATCH, W // PATCH
    p = x.reshape(3, gh // MERGE, MERGE, PATCH, gw // MERGE, MERGE, PATCH)
    p = p.transpose(1, 4, 2, 5, 0, 3, 6)                                              # [bh, bw, mi, mj, c, ph, pw]
    p = p.reshape(gh * gw, 3, 1, PATCH, PATCH)
    p = np.broadcast_to(p, (gh * gw, 3, TEMPORAL, PATCH, PATCH)).reshape(gh * gw, 3 * TEMPORAL * PATCH * PATCH)
    return np.ascontiguousarray(p), (1, gh, gw)


def n_tokens(grid):
    t, gh, gw = grid
    return t * gh * gw // (MERGE * MERGE)


def position_ids(grids):
    """[N, 2] (h, w) patch positions in the block-major order, for all images concatenated."""
    out = []
    for t, gh, gw in grids:
        hh, ww = np.meshgrid(np.arange(gh), np.arange(gw), indexing="ij")
        shp = (gh // MERGE, MERGE, gw // MERGE, MERGE)
        hh = hh.reshape(shp).transpose(0, 2, 1, 3).reshape(-1)
        ww = ww.reshape(shp).transpose(0, 2, 1, 3).reshape(-1)
        out.append(np.tile(np.stack([hh, ww], -1), (t, 1)))
    return np.concatenate(out, 0).astype(np.int32)


def segment_ids(grids):
    """[N] image index of every patch (attention stays within an image)."""
    return np.concatenate([np.full(t * gh * gw, i, np.int32) for i, (t, gh, gw) in enumerate(grids)])


# ----------------------------------------------------------------------------- parameters
def from_state_dict(get, depth, hidden, dtype=np.float32):
    """`get(name)` -> np array for the HF names without the `model.visual.` prefix -> our params (matrices
    transposed to [in, out]; the two convs flattened to matrices)."""
    t = lambda n: np.ascontiguousarray(np.asarray(get(n), dtype).T)  # noqa: E731
    v = lambda n: np.asarray(get(n), dtype)  # noqa: E731
    blocks = []
    for i in range(depth):
        b = f"blocks.{i}."
        blocks.append({"n1": v(b + "norm1.weight"), "n2": v(b + "norm2.weight"),
                       "qkv": t(b + "attn.qkv.weight"), "qkv_b": v(b + "attn.qkv.bias"),
                       "proj": t(b + "attn.proj.weight"), "proj_b": v(b + "attn.proj.bias"),
                       "qn": v(b + "attn.q_norm.weight"), "kn": v(b + "attn.k_norm.weight"),
                       "gate": t(b + "mlp.gate_proj.weight"), "gate_b": v(b + "mlp.gate_proj.bias"),
                       "up": t(b + "mlp.up_proj.weight"), "up_b": v(b + "mlp.up_proj.bias"),
                       "down": t(b + "mlp.down_proj.weight"), "down_b": v(b + "mlp.down_proj.bias")})
    pe = np.asarray(get("patch_embed.proj.weight"), dtype)                              # [C, 3, 2, 14, 14]
    ds = np.asarray(get("downsample.weight"), dtype)                                    # [O, C, 2, 2]
    return {"patch": np.ascontiguousarray(pe.reshape(pe.shape[0], -1).T), "patch_b": v("patch_embed.proj.bias"),
            "blocks": blocks, "post_ln": v("post_layernorm.weight"),
            "ds": np.ascontiguousarray(ds.reshape(ds.shape[0], -1).T), "ds_b": v("downsample.bias"),
            "m_proj": t("merger.proj.weight"), "m_ln": v("merger.post_projection_norm.weight"),
            "m_ln_b": v("merger.post_projection_norm.bias"), "m_gate": t("merger.gate_proj.weight"),
            "m_up": t("merger.up_proj.weight"), "m_down": t("merger.down_proj.weight"),
            "n_heads": None}


def from_hf_module(m, dtype=np.float32):
    sd = {k: v.detach().float().cpu().numpy() for k, v in m.state_dict().items()}
    p = from_state_dict(sd.__getitem__, m.config.depth, m.config.hidden_size, dtype)
    p["n_heads"] = m.config.num_heads
    return p


def load_vision(reader, dtype=np.float32, depth=24, hidden=1024, n_heads=16):
    """From a checkpoint `ShardReader` (names `model.visual.*`, all in the last shard)."""
    p = from_state_dict(lambda n: reader.get("model.visual." + n, dtype), depth, hidden, dtype)
    p["n_heads"] = n_heads
    return p


# ----------------------------------------------------------------------------- forward
def _rms(x, w, eps=EPS):
    xf = x.astype(jnp.float32)
    return (xf * jax.lax.rsqrt(jnp.mean(xf * xf, -1, keepdims=True) + eps) * w.astype(jnp.float32)).astype(x.dtype)


def _rope(pos, head_dim, theta=10000.0):
    d = head_dim // 2                                                    # rotary dim per axis pair = 32 -> 16 freqs
    inv = 1.0 / (theta ** (np.arange(0, d, 2, dtype=np.float32) / d))
    f = (pos.astype(np.float32)[:, :, None] * inv[None, None, :]).reshape(pos.shape[0], -1)   # [N, 32] (h freqs, w freqs)
    emb = np.concatenate([f, f], -1)                                     # [N, 64]
    return jnp.asarray(np.cos(emb)), jnp.asarray(np.sin(emb))


def _rot_half(x):
    a, b = jnp.split(x, 2, axis=-1)
    return jnp.concatenate([-b, a], -1)


def _attention(p, x, cos, sin, same, n_heads, dtype):
    N, C = x.shape
    hd = C // n_heads
    qkv = (jnp.dot(x, p["qkv"].astype(dtype), preferred_element_type=jnp.float32) + p["qkv_b"]).reshape(N, 3, n_heads, hd)
    q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
    q, k = _rms(q, p["qn"]), _rms(k, p["kn"])
    c, s = cos[:, None, :], sin[:, None, :]
    q = q * c + _rot_half(q) * s
    k = k * c + _rot_half(k) * s
    sc = jnp.einsum("nhd,mhd->hnm", q, k) * (hd ** -0.5)
    sc = jnp.where(same[None], sc, -jnp.inf)
    a = jax.nn.softmax(sc, axis=-1)
    o = jnp.einsum("hnm,mhd->nhd", a, v).reshape(N, C).astype(dtype)
    return jnp.dot(o, p["proj"].astype(dtype), preferred_element_type=jnp.float32) + p["proj_b"]


def _swiglu(g, u, limit=10.0):
    g = jnp.minimum(g, limit)
    u = jnp.clip(u, -limit, limit)
    return jax.nn.silu(g) * u


def forward(p, patches, grids, dtype=jnp.float32):
    """patches [N, 1176] (all images concatenated), grids [(t, gh, gw), ...] -> [sum n_tokens, out_hidden] f32.
    Shapes are static per call (one compile per (N, layout)); pad N to a bucket for the TPU."""
    hd = p["blocks"][0]["qn"].shape[0]                                   # static shapes (works under jit; the
    pos, seg = position_ids(grids), segment_ids(grids)                    # n_heads entry may be a traced leaf)
    same = jnp.asarray(seg[:, None] == seg[None, :])
    x = jnp.dot(jnp.asarray(patches, dtype), p["patch"].astype(dtype), preferred_element_type=jnp.float32) + p["patch_b"]
    x = x.astype(dtype)
    n_heads = x.shape[-1] // hd
    cos, sin = _rope(pos, hd)
    for b in p["blocks"]:
        x = x + _attention(b, _rms(x, b["n1"]), cos, sin, same, n_heads, dtype).astype(dtype)
        h = _rms(x, b["n2"])
        g = jnp.dot(h, b["gate"].astype(dtype), preferred_element_type=jnp.float32) + b["gate_b"]
        u = jnp.dot(h, b["up"].astype(dtype), preferred_element_type=jnp.float32) + b["up_b"]
        x = x + (jnp.dot(_swiglu(g, u).astype(dtype), b["down"].astype(dtype), preferred_element_type=jnp.float32) + b["down_b"]).astype(dtype)
    x = _rms(x, p["post_ln"])
    N, C = x.shape
    x4 = x.reshape(N // 4, 2, 2, C).transpose(0, 3, 1, 2).reshape(N // 4, C * 4)   # (c, i, j) like the conv weight
    y = jnp.dot(x4, p["ds"].astype(dtype), preferred_element_type=jnp.float32) + p["ds_b"]
    y = jnp.dot(y.astype(dtype), p["m_proj"].astype(dtype), preferred_element_type=jnp.float32)
    mu = y.mean(-1, keepdims=True)
    var = ((y - mu) ** 2).mean(-1, keepdims=True)
    y = (y - mu) * jax.lax.rsqrt(var + EPS) * p["m_ln"] + p["m_ln_b"]
    y = jax.nn.gelu(y, approximate=False).astype(dtype)
    g = jnp.dot(y, p["m_gate"].astype(dtype), preferred_element_type=jnp.float32)
    u = jnp.dot(y, p["m_up"].astype(dtype), preferred_element_type=jnp.float32)
    return jnp.dot(_swiglu(g, u).astype(dtype), p["m_down"].astype(dtype), preferred_element_type=jnp.float32)


# ----------------------------------------------------------------------------- TP-sharded forward (8 chips)
def shard_specs(p):
    """PartitionSpecs for `to_sharded(p)`: attention heads over AXIS (qkv columns / proj rows regrouped per head),
    MLP and merger intermediate columns over AXIS (down rows), the downsample columns and m_proj rows over AXIS;
    small vectors replicated. ~140 MB per chip in bf16 for the real tower."""
    from jax.sharding import PartitionSpec as P
    from glm53.engine import AXIS
    R = P()
    blk = {"n1": R, "n2": R, "qkv": P(None, None, AXIS, None), "qkv_b": P(None, AXIS, None),
           "proj": P(AXIS, None, None), "proj_b": R, "qn": R, "kn": R,
           "gate": P(None, AXIS), "gate_b": P(AXIS), "up": P(None, AXIS), "up_b": P(AXIS), "down": P(AXIS, None), "down_b": R}
    blk = {k: P(None, *v) for k, v in blk.items()}                  # blocks are STACKED on a leading axis (lax.scan)
    return {"patch": R, "patch_b": R, "blocks": blk, "post_ln": R,
            "ds": P(None, AXIS), "ds_b": P(AXIS), "m_proj": P(AXIS, None), "m_ln": R, "m_ln_b": R,
            "m_gate": P(None, AXIS), "m_up": P(None, AXIS), "m_down": P(AXIS, None), "n_heads": None}


def to_sharded(p, mesh, dtype=jnp.bfloat16):
    """Regroup the head-sharded matrices ([in, 3, H, hd] qkv / [H, hd, out] proj) and place every array on the mesh."""
    from jax.sharding import NamedSharding
    H = p["n_heads"]
    q = dict(p)
    blocks = []
    for b in p["blocks"]:
        C = b["qkv"].shape[0]
        hd = C // H
        blocks.append({**b, "qkv": np.asarray(b["qkv"]).reshape(C, 3, H, hd), "qkv_b": np.asarray(b["qkv_b"]).reshape(3, H, hd),
                       "proj": np.asarray(b["proj"]).reshape(H, hd, C)})
    q["blocks"] = {k: np.stack([np.asarray(b[k]) for b in blocks]) for k in blocks[0]}   # [depth, ...] for lax.scan
    specs = shard_specs(p)

    def place(a, sp):
        if sp is None or a is None:
            return a
        a = np.asarray(a)
        a = a.astype(np.float32) if a.dtype == np.float32 else a
        return jax.device_put(jnp.asarray(a, dtype if a.ndim >= 2 else jnp.float32), NamedSharding(mesh, sp))
    out = jax.tree.map(place, q, specs, is_leaf=lambda x: x is None or not isinstance(x, (dict, list)))
    out["n_heads"] = H
    return out


def make_forward_sharded(p_sharded, mesh, dtype=jnp.bfloat16, q_chunk=1024):
    """-> f(patches [N,1176] np, grids tuple) -> [n_tokens, out] f32, one jit per (N, grids). Per chip: its heads
    (attention scores chunked over q_chunk queries), its MLP/merger columns; two psums per block. The 24 blocks run
    as a `lax.scan` over the stacked parameters (one block body per executable: TPU executables live in HBM, and
    five unrolled buckets held ~0.4 GB)."""
    from glm53.engine import AXIS, shard_map
    from jax import lax
    from jax.sharding import PartitionSpec as P
    R = P()
    n_heads = p_sharded["n_heads"]
    specs = shard_specs(p_sharded)
    del specs["n_heads"]
    pp = {k: v for k, v in p_sharded.items() if k != "n_heads"}
    n_dev = mesh.devices.size
    Hl = n_heads // n_dev
    cache = {}

    def build(N):
        """One program per patch count N: positions / segments arrive as arrays, so images of any grid share it."""
        def attn(b, x, cos, sin, same):
            C = x.shape[-1]
            hd = C // n_heads
            qkv = jnp.einsum("nc,cthd->nthd", x, b["qkv"].astype(dtype), preferred_element_type=jnp.float32) + b["qkv_b"]
            q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]                                 # [N, Hl, hd]
            q, k = _rms(q, b["qn"]), _rms(k, b["kn"])
            c, s_ = cos[:, None, :], sin[:, None, :]
            q = q * c + _rot_half(q) * s_
            k = k * c + _rot_half(k) * s_
            outs = []
            for a in range(0, N, q_chunk):
                sc = jnp.einsum("nhd,mhd->hnm", q[a:a + q_chunk], k) * (hd ** -0.5)
                sc = jnp.where(same[a:a + q_chunk][None], sc, -jnp.inf)
                outs.append(jnp.einsum("hnm,mhd->nhd", jax.nn.softmax(sc, axis=-1), v))
            o = jnp.concatenate(outs, 0).astype(dtype)                                # [N, Hl, hd]
            y = jnp.einsum("nhd,hdc->nc", o, b["proj"].astype(dtype), preferred_element_type=jnp.float32)
            return lax.psum(y, AXIS) + b["proj_b"]

        def prog(p, patches, cos, sin, same):
            x = jnp.dot(patches.astype(dtype), p["patch"].astype(dtype), preferred_element_type=jnp.float32) + p["patch_b"]
            x = x.astype(dtype)

            def block(x, b):
                x = x + attn(b, _rms(x, b["n1"]), cos, sin, same).astype(dtype)
                h = _rms(x, b["n2"])
                g = jnp.dot(h, b["gate"].astype(dtype), preferred_element_type=jnp.float32) + b["gate_b"]
                u = jnp.dot(h, b["up"].astype(dtype), preferred_element_type=jnp.float32) + b["up_b"]
                y = jnp.dot(_swiglu(g, u).astype(dtype), b["down"].astype(dtype), preferred_element_type=jnp.float32)
                return x + (lax.psum(y, AXIS) + b["down_b"]).astype(dtype), None
            x, _ = lax.scan(block, x, p["blocks"])
            x = _rms(x, p["post_ln"])
            C = x.shape[-1]
            x4 = x.reshape(N // 4, 2, 2, C).transpose(0, 3, 1, 2).reshape(N // 4, C * 4)
            y = jnp.dot(x4, p["ds"].astype(dtype), preferred_element_type=jnp.float32) + p["ds_b"]   # local columns
            y = lax.psum(jnp.dot(y.astype(dtype), p["m_proj"].astype(dtype), preferred_element_type=jnp.float32), AXIS)
            mu = y.mean(-1, keepdims=True)
            var = ((y - mu) ** 2).mean(-1, keepdims=True)
            y = (y - mu) * lax.rsqrt(var + EPS) * p["m_ln"] + p["m_ln_b"]
            y = jax.nn.gelu(y, approximate=False).astype(dtype)
            g = jnp.dot(y, p["m_gate"].astype(dtype), preferred_element_type=jnp.float32)
            u = jnp.dot(y, p["m_up"].astype(dtype), preferred_element_type=jnp.float32)
            return lax.psum(jnp.dot(_swiglu(g, u).astype(dtype), p["m_down"].astype(dtype), preferred_element_type=jnp.float32), AXIS)
        return jax.jit(shard_map(prog, mesh=mesh, in_specs=(specs, R, R, R, R), out_specs=R, check_vma=False))

    hd = pp["blocks"]["qn"].shape[-1]

    def run(patches, grids):
        grids = tuple(tuple(int(v) for v in g) for g in grids)
        N = patches.shape[0]
        if N not in cache:
            cache[N] = build(N)
        pos, seg = position_ids(grids), segment_ids(grids)
        cos, sin = _rope(pos, hd)
        same = jnp.asarray(seg[:, None] == seg[None, :])
        return cache[N](pp, jnp.asarray(patches, dtype), cos, sin, same)
    return run
'''
for path, src in FILES.items():
    open(path, "w").write(src)
print(f"wrote {len(FILES)} files of the glm53 package")


### The serving script
The next cell writes the serving script (the same one `launch.py` pushes; the source and docs are in the [kaggle-tpu-lab repo](https://github.com/ARahim3/kaggle-tpu-lab)).


In [ ]:
%%writefile serve_glm53.py
"""
Serve GLM-5.3-Flash (320B MoE, 18B active) on a Kaggle TPU v5e-8 with our own JAX engine.

This script is pushed to Kaggle as a script kernel by ../../launch.py, which fills in the CFG line below and embeds
the engine package. It also runs standalone (pasted into a Kaggle notebook next to a `glm53/` folder written by the
previous cell) — then it prints instead of using ntfy.

Steps (each one is announced in the log):
  1/6  runtime  — pre-flight (datasets attached, Internet on, a real TPU: ~20 s, before anything slow), pinned libtpu
                  + a few pip packages (~1 min), cloudflared, the engine package
  2/6  weights  — the routed experts (3-bit codebook tables, Unsloth's UD-IQ3_XXS GGUF) and the non-expert weights
                  (int8) straight onto the eight chips (~6 min); the non-expert weights come from the serve dataset's
                  base store when it is attached, else from the FP8 checkpoint datasets
  3/6  vision   — the vision tower, sharded over the chips
  4/6  warm-up  — the prefill buckets, the batched decode and speculative-decoding programs, the snapshot programs:
                  loaded from the serve dataset's exported programs and compile cache; without them every program is
                  traced, exported and compiled (~28 min)
  5/6  tunnel   — a public cloudflared URL (three attempts; a URL that never resolves is replaced once)
  6/6  ready    — READY banner + self-test, then keep serving until keepalive_min elapses

A cold run leaves `jax_cache/` (everything it compiled), `exported/` (every program it traced) and, when it read the
FP8 checkpoint, `base/` (the non-expert weights, vision tower and tokenizer, ~11 GB) in /kaggle/working, so its output
can be turned into the serve dataset ("New dataset" from the kernel output) that later runs attach.
"""
import base64, collections, glob, hashlib, io, json, os, queue, re, secrets, shutil, subprocess, sys, tarfile, threading, time, urllib.request, uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

CFG = None  # __LAUNCHER_CONFIG__  (launch.py replaces this line)

DEFAULTS = {
    "expert_datasets": ["rahim3/glm53-flash-iq3xxs-1", "rahim3/glm53-flash-iq3xxs-2"],   # Unsloth UD-IQ3_XXS GGUF, mirrored
    "base_datasets": ["rahim3/glm53-flash-fp8-1", "rahim3/glm53-flash-fp8-2",            # HF FP8 checkpoint, mirrored:
                      "rahim3/glm53-flash-fp8-3", "rahim3/glm53-flash-fp8-4"],           # non-expert weights, vision, tokenizer
    "serve_dataset": "rahim3/glm53-flash-serve-v3",   # base/ (non-expert weights, vision, tokenizer) + exported/ (the
                                     # programs' exports) + jax_cache/ (their compiled executables); attached: the FP8
                                     # datasets are not needed and the warm-up neither traces nor compiles
    "serve_fallback": ["rahim3/glm53-flash-serve-v2", "rahim3/glm53-flash-serve"],   # used when `serve_dataset` is not
                                     # attached: an older serve dataset's base/ still fits; its compiled programs are
                                     # another build's (every program is traced and compiled, ~28 min)
    "aot": True,                     # AOT export of the programs (glm53.aot): a program that has to be traced is exported
                                     # to /kaggle/working/exported and a run that finds its file there, or in the serve
                                     # dataset's exported/, skips JAX's tracing and lowering (the compile cache only
                                     # skips XLA's compile); v6e 2026-10-06: warm-up 10.4 min with the cache -> 0.3
    "dump_base": True,               # a cold run writes base/ into its output (so the serve dataset can be made from it)
    "libtpu": "0.0.42.*",            # the image's runtime is 140x slower on gathers and cannot run Pallas kernels
    "max_len": 262144,               # context capacity (tokens); a multiple of 32
    "streams": 3,                    # requests decoded together (one program set per batch size, ~1-2.5 min compile
                                     # each); a 4th 262k set never passes the admission (`min_free_gb`) on the v5e budget
    "sets": 4,                       # cache sets on the chips: running streams + finished contexts kept for their next turn
    "piece": 1024,                   # prefill piece (tokens): 1024 = ~2,500-2,700 tok/s vs ~2,050-2,150 at 512 (v6e
    "sched_piece": 1024,             # 2026-10-04, 16k-128k prompts); three 262k streams either way since the admission
                                     # counts live buffers (`min_free_gb`; the old in-use figure fitted two or three)
    "q_block": 32,                   # queries per attention block during prefill
    "cache_q8": True,                # int8 latent cache (a 262k set costs ~0.2 GB/chip instead of 0.3)
    "mhc_kernel": "bf16",            # the mHC sites of decode / verify steps as Pallas kernels (glm53.pallas_mhc: plain
                                     # decode -8 %, DFlash 3 streams -4 % on the v6e); "" = XLA's chain (the numerics
                                     # of the builds before 2026-10-05)
    "bucket_min": 256,               # smallest prefill bucket compiled (short prompts pad to it)
    "min_free_gb": 0.75,             # HBM an admission needs beyond the live buffers: a 262k cache set (0.23 GB/chip), one
                                     # piece's prefill temporaries (~0.22) and the programs of a decode step + a prefill
                                     # piece (~0.25: loaded programs are a cache the runtime evicts when buffers need room)
    "max_queue": 8,                  # waiting requests beyond the streams before a 429
    "max_wait_s": 90.0,              # a request waiting longer than this gets a 503 (clients retry)
    "keepalive_s": 15,               # SSE ping when nothing was streamed for this long (a buffered tool call, a queued
                                     # request): Cloudflare drops a response that is silent for ~100 s
    "think_budget_default": 0,       # thinking tokens before </think> is forced, when the request sets no budget (0 = unlimited)
    "vision": True,                  # load the vision tower (images in both APIs); False saves ~1 min and 0.14 GB/chip
    "vision_max_tokens": 1024,       # 28x28-pixel tokens per image
    "reasoning_effort_default": "low",   # server-side default: low | high (anything else = the template's Max)
    "temperature": 1.0, "top_p": 0.95,   # generation_config defaults
    "max_new_default": 4096,
    "snap_host_gb": 48,              # host RAM for parked contexts (agent sessions that interleave)
    "snap_rows": 1024,               # snapshot row bucket (1024 rows = 8192 tokens sharded)
    "snap_min": 256,                 # contexts shorter than this are re-prefilled instead of parked
    "base_min": 512,                 # a system section at least this long is pinned for new sessions
    "snap_warm_tokens": 32768,       # warm the snapshot programs for contexts up to this many tokens
    "keepalive_min": 480,            # auto-shutdown guard (Kaggle TPU caps at 9h anyway)
    "api_key": "",                   # generated if empty
    "ntfy_topic": "",                # optional: publish progress to ntfy.sh/<topic> (launch.py watches it)
    "dflash": "hf:incoai/GLM-5.3-Flash-DFlash2@bf582e4eacc1810f76656d1811693ff6c6737d2a",
                                     # DFlash 2 speculative decoding (one stream alone, and batched for 2-3 streams): the
                                     # drafter as `hf:<repo>[@revision]` (fetched from Hugging Face at start: public, 2.3 GB,
                                     # ~1 min; incoai's drafter is CC BY-NC-ND 4.0, research use only), or its directory /
                                     # attached dataset (config.json + model.safetensors); "" = plain decoding (one stream
                                     # ~79 tok/s instead of ~166 on the v5e-8)
    "dflash_int8": True,             # the drafter's big matrices as int8 (159 instead of 295 MB per chip, same acceptance)
    "dflash_cost": {"4": 16.4},      # ms per DFlash step by block size (Kaggle v5e-8, 2026-10-05: 2.6 tokens per step) = the block sizes compiled:
                                     # {"4": ..., "8": ...} also drafts 8 (2026-10-03: math-like text x1.97 instead of
                                     # x1.64, no gain overall at temperature 1.0; its programs only add warm-up)
    "dflash_prior": {"4": 2.8, "8": 3.3},    # tokens per step a new stream assumes until it has measured its own
    "dflash_rows": {"2": [26.0, 14.9], "3": [35.5, 17.0]},   # DFlash for B streams at once: {"B": [ms of a B-stream
                                     # DFlash step (block = the smallest of dflash_cost), ms of a plain batched step of
                                     # B]} = the B compiled (Kaggle v5e-8, 2026-10-05; served 154 / 168 tok/s aggregate at
                                     # 2 / 3 streams vs plain 114 / 147; ~8 min of cold warm-up each); the scheduler picks
                                     # DFlash or plain per step from these and the streams' tokens per step; {} = plain
    "dflash_draft_temp": 0.7,        # drafts are drawn at this x the request's temperature; the verify accepts them by
                                     # rejection sampling (output distributed exactly as plain sampling)
    "served_model_name": "glm-5.3-flash",
    "tunnel": True,                  # False: no cloudflared (local testing)
    "skip_runtime": False,           # True: no pip installs (local testing)
    "port": 8000,
}
CFG = {**DEFAULTS, **(CFG or {}), **globals().get("CFG_PRESET", {})}
_cfg_file = Path("serve_config.json")            # notebook flow: overrides next to this script
if _cfg_file.exists():
    CFG.update(json.loads(_cfg_file.read_text()))
if not CFG["api_key"]:
    CFG["api_key"] = "glm-" + secrets.token_hex(12)

ENGINE_B64 = ""  # __ENGINE__  (launch.py embeds the glm53 package here; the notebook writes the files instead)

PORT = int(CFG["port"])
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/tmp")
CACHE_DIR = WORK / "jax_cache"
EXPORT_DIR = WORK / "exported"
SERVE_NAMES = [n for n in [CFG["serve_dataset"], *CFG["serve_fallback"]] if n]
CLOUDFLARED = Path("/tmp/cloudflared")
T0 = time.time()
LOG_LINES = []


def log(*parts):
    line = time.strftime("[%H:%M:%S] ") + " ".join(str(p) for p in parts)
    LOG_LINES.append(line)
    print(line, flush=True)


def elapsed():
    return f"{int(time.time() - T0) // 60} min {int(time.time() - T0) % 60:02d} s"


def banner(step, title, note=""):
    log("")
    log("=" * 70)
    log(f" STEP {step}/6  {title}" + (f"   ({note})" if note else "") + f"   [{elapsed()} so far]")
    log("=" * 70)


def publish(phase, **extra):
    """Progress event: always logged; also pushed to ntfy if a topic is set."""
    log(f"PHASE {phase}", json.dumps(extra) if extra else "")
    if not CFG["ntfy_topic"]:
        return
    try:
        body = {"topic": CFG["ntfy_topic"], "title": f"kaggle-tpu-lab {phase}", "message": json.dumps({"phase": phase, **extra})}
        req = urllib.request.Request("https://ntfy.sh", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:  # noqa: BLE001
        log(f"(ntfy publish failed: {e})")


def sh(cmd, tag):
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    log(f"   {tag}: rc {r.returncode} in {time.time() - t:.0f}s" + (f" | {(r.stderr or '')[-200:].strip()}" if r.returncode else ""))
    return r.returncode


def mounts_of(names):
    """Dataset names ('owner/slug') -> mounted paths, in the given order (missing ones dropped)."""
    out = []
    for n in names:
        slug = n.split("/")[-1]
        hits = glob.glob(f"/kaggle/input/datasets/{n}") + glob.glob(f"/kaggle/input/{slug}")
        if hits:
            out.append(hits[0])
    return out


def hbm():
    st = jax.devices()[0].memory_stats() or {}
    return st.get("bytes_in_use", 0) / 1e9, (st.get("bytes_limit") or 0) / 1e9


def fail(step, msg):
    """Stop with a plain message (the launcher prints `step` and `tail` of a "failed" phase)."""
    log("   " + msg)
    publish("failed", step=step, tail=msg)
    sys.exit(1)


def preflight():
    """Look before the slow steps (~20 s): the datasets attached, Internet on (pip, cloudflared) and a real TPU present.
    Kaggle sometimes starts a "TPU" session with no TPU (a CPU-only container, most often on new or not-yet-verified
    accounts): jax then sees one CPU device and the build dies minutes later with a sharding error."""
    need = list(CFG["expert_datasets"]) + ([] if mounts_of(SERVE_NAMES) else
                                           ([CFG["serve_dataset"]] if CFG["serve_dataset"] and not mounts_of(CFG["base_datasets"]) else list(CFG["base_datasets"])))
    missing = [n for n in need if not mounts_of([n])]
    if missing:
        fail("datasets", f"datasets not attached: {missing}. In the right sidebar, Add Input -> search each name -> attach, then run again.")
    try:
        urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=20).read(1)
    except Exception as e:  # noqa: BLE001
        fail("no-internet", f"no Internet from this session ({str(e)[:120]}): Session options -> Internet ON (a phone-verified "
                            "Kaggle account is needed for that), then run again. The pip packages and the tunnel need it.")
    code = ("import jax\n"
            "try:\n"
            "    d = jax.devices()\n"
            "    print('TPU_CHECK', len(d), d[0].platform, getattr(d[0], 'device_kind', ''))\n"
            "except Exception as e:\n"
            "    print('TPU_CHECK 0 none', str(e).replace(chr(10), ' ')[:200])\n")
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=180)
        m = re.search(r"TPU_CHECK (\d+) (\S+)(.*)", (r.stdout or "") + (r.stderr or ""))
    except Exception as e:  # noqa: BLE001
        log(f"   (TPU check skipped: {e})"); return
    if m is None:
        log("   (TPU check inconclusive: the image's jax did not answer; continuing)"); return
    n, platform, rest = int(m.group(1)), m.group(2), m.group(3).strip()
    if n == 8 and platform == "tpu":
        log(f"   pre-flight OK: datasets attached, Internet on, 8 TPU chips ({rest})"); return
    if n == 0 and not re.search(r"jellyfish|TPU initialization failed|initialize backend 'tpu'|No TPU|vfio", rest, re.I):
        log(f"   (TPU check inconclusive, continuing: {rest[:160]})"); return
    fail("no-tpu", f"this session has no working TPU: jax sees {n} {platform} device(s) {rest}. Kaggle sometimes starts a TPU "
                   "session without one (most often on new or not-yet-verified accounts); nothing in this notebook can fix "
                   "that. Stop the session and start it again; `import jax; print(jax.device_count())` in a fresh cell must "
                   "print 8 before this script is worth running.")


# ----------------------------------------------------------------------------- 1. runtime
if not CFG["skip_runtime"]:
    banner(1, "Runtime", "pre-flight, pinned libtpu + pip packages, cloudflared, the engine")
    preflight()
    sh([sys.executable, "-m", "pip", "install", "-q", "safetensors", "huggingface_hub", "transformers>=5.16", "pillow",
        "flatbuffers", "torch", "--index-url", "https://download.pytorch.org/whl/cpu", "--extra-index-url", "https://pypi.org/simple"], "pip packages")
    if CFG["libtpu"]:
        sh([sys.executable, "-m", "pip", "install", "-q", f"libtpu=={CFG['libtpu']}"], f"libtpu=={CFG['libtpu']}")
    if ENGINE_B64 and not ENGINE_B64.startswith("__"):
        with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ENGINE_B64)), mode="r:gz") as tf:
            tf.extractall(WORK)
        sys.path.insert(0, str(WORK))
        log(f"   engine package extracted to {WORK}/glm53")
    elif not Path("glm53").is_dir():
        sys.exit("no glm53/ package next to this script and nothing embedded — run the notebook's engine cell first")

if CFG["tunnel"] and not CLOUDFLARED.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
    log("   cloudflared downloaded")

import numpy as np                      # noqa: E402  (after the runtime pins: libtpu must be installed before jax loads)
import jax, jax.numpy as jnp            # noqa: E402
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P   # noqa: E402

SERVE_MOUNT = (mounts_of(SERVE_NAMES) or [None])[0]
if SERVE_MOUNT and CFG["serve_dataset"] and not mounts_of([CFG["serve_dataset"]]):
    log(f"   {CFG['serve_dataset']} is not attached: using {SERVE_MOUNT} (its weights fit; its compiled programs are "
        "another build's, so every program compiles)")
BASE_DIR = os.path.join(SERVE_MOUNT, "base") if SERVE_MOUNT and os.path.isdir(os.path.join(SERVE_MOUNT, "base")) else None
if "eng" not in globals():              # (a test harness may pre-set eng/tok/ids/eos/cfg and skip the build)
    cache_src = os.path.join(SERVE_MOUNT, "jax_cache") if SERVE_MOUNT and os.path.isdir(os.path.join(SERVE_MOUNT, "jax_cache")) else None
    if cache_src and not CACHE_DIR.exists():
        t = time.time()
        shutil.copytree(cache_src, CACHE_DIR)
        log(f"   compile cache restored from {cache_src} in {time.time() - t:.0f}s")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    jax.config.update("jax_compilation_cache_dir", str(CACHE_DIR))
    try:                                 # a process that already initialised the cache elsewhere keeps writing there
        from jax._src import compilation_cache as _cc
        _cc.reset_cache()
    except Exception as e:  # noqa: BLE001
        log(f"   (compile cache reset skipped: {e!r})")
    try:                                 # which cache entries this build loads or writes (the serve dataset's next
        if not hasattr(_cc, "_glm_use"):     # jax_cache/ is exactly those: jax_cache_used.txt in the output)
            _cc._glm_use = {"hit": [], "miss": [], "put": [], "read_s": 0.0}

            def _rec(fn, kind, use):         # (`use` is bound here: the warm-up below reuses the name _cc)
                def wrapped(cache_key, *a, **k):
                    t = time.time()
                    r = fn(cache_key, *a, **k)
                    try:
                        if kind == "put":
                            use["put"].append(cache_key)
                        else:
                            use["hit" if r[0] is not None else "miss"].append(cache_key)
                            use["read_s"] += time.time() - t
                    except Exception:  # noqa: BLE001  (the record must never cost a cache read)
                        pass
                    return r
                return wrapped
            _cc.get_executable_and_time = _rec(_cc.get_executable_and_time, "get", _cc._glm_use)
            _cc.put_executable_and_time = _rec(_cc.put_executable_and_time, "put", _cc._glm_use)
        CACHE_USE = _cc._glm_use
        for _k in ("hit", "miss", "put"):
            CACHE_USE[_k].clear()
        CACHE_USE["read_s"] = 0.0
    except Exception as e:  # noqa: BLE001
        log(f"   (compile cache use is not recorded: {e!r})")
    log(f"   jax {jax.__version__}, {jax.device_count()} devices, compile cache at {CACHE_DIR}")
if CFG["aot"]:
    from glm53 import aot as AOT
    _exp_src = os.path.join(SERVE_MOUNT, "exported") if SERVE_MOUNT and os.path.isdir(os.path.join(SERVE_MOUNT, "exported")) else None
    AOT.configure(EXPORT_DIR, [_exp_src], context=lambda: AOT.engine_context(eng), log=log)
    _n_exp = sum(len(glob.glob(os.path.join(d, "*" + AOT.EXT))) for d in AOT.STORE.read_dirs)
    log(f"   exported programs: {_n_exp} files in {', '.join(AOT.STORE.read_dirs)}"
        + ("" if _n_exp else " (none yet: every program is traced, then exported for the next run)"))

# ----------------------------------------------------------------------------- 2. weights -> the chips
if "eng" not in globals():
    banner(2, "Weights", "3-bit codebook experts + int8 non-expert weights onto the eight chips, ~9 min")
    from glm53 import basestore as BS
    from glm53 import checkpoint as C
    from glm53 import model as M
    from glm53.engine import AXIS
    from glm53 import gguf_reader as G
    from glm53.resident import ResidentFetch, ResidentLayerEngine, pack_layer_from_gguf, hbm_bytes
    from transformers import AutoConfig, AutoTokenizer
    gguf_mounts = mounts_of(CFG["expert_datasets"])
    fp8_mounts = [] if BASE_DIR else mounts_of(CFG["base_datasets"])
    if len(gguf_mounts) < len(CFG["expert_datasets"]) or (not BASE_DIR and len(fp8_mounts) < len(CFG["base_datasets"])):
        fail("datasets", f"datasets missing: found gguf {gguf_mounts}, fp8 {fp8_mounts}, serve {SERVE_MOUNT}; attach "
                         f"{CFG['expert_datasets']} + {CFG['serve_dataset'] or CFG['base_datasets']}")
    hf_dir = BASE_DIR or fp8_mounts[0]
    log(f"   experts from {gguf_mounts}; the rest from {'the base store ' + BASE_DIR if BASE_DIR else 'the FP8 checkpoint'}")
    publish("loading", note="weights")
    hf_cfg = AutoConfig.from_pretrained(hf_dir)
    tc = hf_cfg.text_config
    N_LAYERS = tc.num_hidden_layers
    cfg = M.Cfg.from_hf(tc, dtype=jnp.bfloat16)
    N_DEV = jax.device_count()
    mesh = Mesh(np.array(jax.devices()), (AXIS,))
    tbl_sh = NamedSharding(mesh, P(AXIS))
    gm = G.GGUFModel(G.open_mirror(gguf_mounts))
    t_load = time.time()
    if BASE_DIR:
        store = BS.load(BASE_DIR)
        params = {**store["top"], "layers": []}
        log(f"   base store read in {time.time() - t_load:.0f}s ({store['manifest'].get('n_layers')} layers)")
        r = None
    else:
        r = C.RawShardReader(":".join(fp8_mounts))
        top = C.load_top(r, np.float32)
        params = {"embed": top["embed"].astype(jnp.bfloat16), "norm": top["norm"], "lm_head": top["lm_head"].astype(jnp.bfloat16), "layers": []}
        del top
    qtypes, hbm_total = {}, 0
    for i in range(N_LAYERS):
        ti = time.time()
        if BASE_DIR:
            p = store["layers"][i]
        else:
            p = C.load_layer(r, i, tc.layer_types[i], tc.mlp_layer_types[i], np.float32)
            p = jax.tree.map(lambda a: a.astype(jnp.bfloat16) if a.ndim >= 2 else a, p)
        if tc.mlp_layer_types[i] == "sparse":
            tables, qt = pack_layer_from_gguf(gm, i, N_DEV, threads=16)
            qtypes[i] = qt
            hbm_total += hbm_bytes(tables) * N_DEV
            for k, t in tables.items():                    # each table straight onto the chips: host RAM holds one layer
                p["mlp"][k] = jax.tree.map(lambda a: jax.device_put(a, tbl_sh), t)
            jax.block_until_ready([p["mlp"][k] for k in tables])
            del tables
        params["layers"].append(p)
        if r is not None:
            for f in C.layer_shards(":".join(fp8_mounts), i):
                r.release(f)
        if i % 5 == 0 or i == N_LAYERS - 1:
            log(f"   layer {i + 1}/{N_LAYERS} ({time.time() - ti:.0f}s) | HBM chip0 {hbm()[0]:.2f} GB")
    log(f"   expert tables {hbm_total / 1e9:.1f} GB across {N_DEV} chips; weights read in {(time.time() - t_load) / 60:.1f} min")
    M.MHC_KERNEL = CFG["mhc_kernel"] or None
    fetch = ResidentFetch(qtypes, tc.hidden_size, tc.moe_intermediate_size // N_DEV, out_dtype=jnp.bfloat16, gather_max_rows=64, chunk=8,
                          use_pallas=True, sweep_mode="ragged", tm=32, combine="matmul")
    eng = ResidentLayerEngine(cfg, params, fetch, max_len=CFG["max_len"], layers_per_program=4, layers_per_program_prefill=1,
                              int8_nonexpert=False if BASE_DIR else "all", seq_shard=True, q_block=CFG["q_block"],
                              prefill_piece=CFG["piece"], cache_q8=CFG["cache_q8"])
    del params
    use, lim = hbm()
    log(f"   ENGINE READY in {(time.time() - t_load) / 60:.1f} min: HBM chip0 {use:.2f} / {lim:.2f} GB, context {CFG['max_len']}")
    publish("loaded", hbm_gb=round(use, 2), minutes=round((time.time() - t_load) / 60, 1))
    tok = AutoTokenizer.from_pretrained(hf_dir)
    ids = np.asarray(tok(tok.apply_chat_template([{"role": "user", "content": "Write a haiku about tensor processing units."}],
                                                 tokenize=False, add_generation_prompt=True, reasoning_effort="low"),
                         add_special_tokens=False).input_ids).reshape(1, -1)
    eos = set(json.load(open(os.path.join(hf_dir, "config.json")))["text_config"].get("eos_token_id", [tok.eos_token_id]))

# ----------------------------------------------------------------------------- 3. vision tower
VISION, VISION_FWD = globals().get("VISION"), globals().get("VISION_FWD")
_vp = None
if CFG["vision"] and VISION_FWD is None and "hf_dir" in globals():
    banner(3, "Vision", "the vision tower sharded over the chips")
    from glm53 import vision as VIS
    t = time.time()
    if BASE_DIR and store.get("vision") is not None:
        _vp = store["vision"]
    else:
        _vp = VIS.load_vision(C.RawShardReader(":".join(fp8_mounts)), dtype=np.float32)
        _vp = jax.tree.map(lambda a: a.astype(jnp.bfloat16) if getattr(a, "dtype", None) == np.float32 else a, _vp)
    VISION = VIS.to_sharded(_vp, eng.mesh, dtype=jnp.bfloat16)
    VISION_FWD = VIS.make_forward_sharded(VISION, eng.mesh, dtype=jnp.bfloat16, q_chunk=1024)
    from PIL import Image
    for target in [b for b in (256, 512, 1024, 2048, 4096) if b <= 4 * CFG["vision_max_tokens"]]:
        side = int(target ** 0.5) * 14 - 14
        img = Image.fromarray(np.random.default_rng(0).integers(0, 256, (side, side, 3), dtype=np.uint8))
        patches, grid = VIS.preprocess(img, max_tokens=CFG["vision_max_tokens"])
        n = patches.shape[0]
        bucket = max(256, 1 << (n - 1).bit_length())
        grids = (grid,) if bucket == n else (grid, (1, 2, (bucket - n) // 2))
        if bucket > n:
            patches = np.concatenate([patches, np.zeros((bucket - n, patches.shape[1]), patches.dtype)], 0)
        jax.block_until_ready(VISION_FWD(patches, grids))
    log(f"   vision tower ready in {time.time() - t:.0f}s; HBM chip0 {hbm()[0]:.2f} GB")
if CFG["dump_base"] and not BASE_DIR and "hf_dir" in globals():
    t = time.time()
    BS.dump(str(WORK / "base"), eng.params, vision=_vp, hf_dir=hf_dir, log=log,
            meta={"model": CFG["served_model_name"], "int8_nonexpert": "all", "experts": "resident (not stored)"})
    log(f"   base store dumped in {time.time() - t:.0f}s (this run's output can become the serve dataset)")
del _vp

# ----------------------------------------------------------------------------- 3b. DFlash 2 drafter (optional)
_d = CFG["dflash"]
if _d.startswith("hf:"):                 # fetched from Hugging Face at start (a public repo: no token)
    _repo, _, _rev = _d[3:].partition("@")
    DFLASH_DIR = f"/tmp/models/{_repo.split('/')[-1]}"
    try:
        from huggingface_hub import snapshot_download
        t = time.time()
        snapshot_download(_repo, revision=_rev or None, local_dir=DFLASH_DIR,
                          allow_patterns=["config.json", "model.safetensors"])
        log(f"   DFlash drafter {_repo}: fetched in {time.time() - t:.0f}s")
    except Exception as e:  # noqa: BLE001
        log(f"   DFlash: fetching {_repo} failed ({e!r}) -> plain decoding")
        DFLASH_DIR = None
else:
    DFLASH_DIR = (_d if os.path.isdir(_d) else (mounts_of([_d]) or [None])[0]) if _d else None
    if _d and not DFLASH_DIR:
        log(f"   DFlash: {_d} not found -> plain decoding")
if DFLASH_DIR and getattr(eng, "dflash", None) is None:
    from glm53 import dflash as DF
    t = time.time()
    _dcfg, _dp = DF.load_params(DFLASH_DIR, jnp.bfloat16)
    eng.set_dflash(_dp, _dcfg, int8=CFG["dflash_int8"], temp_scale=CFG["dflash_draft_temp"])
    del _dp
    log(f"   DFlash 2 drafter ({_dcfg.n_layers} layers, block {_dcfg.block}) from {DFLASH_DIR}: "
        f"{eng.dflash.nbytes_per_chip() / 1e6:.0f} MB per chip in {time.time() - t:.0f}s; HBM chip0 {hbm()[0]:.2f} GB")
DFLASH_COST = {int(k): float(v) for k, v in CFG["dflash_cost"].items()}
DFLASH_ROWS = {int(k): tuple(float(x) for x in v) for k, v in CFG["dflash_rows"].items() if 1 < int(k) <= CFG["streams"]}

# ----------------------------------------------------------------------------- 4. the server
from glm53.resident import ResidentLayerEngine as _RLE   # noqa: E402
from glm53.engine import DeviceSampler                    # noqa: E402
from glm53.scheduler import DFlashPolicy, Request, Scheduler, SnapStore   # noqa: E402
from glm53 import vision as VIS                           # noqa: E402

API_KEY = CFG["api_key"]
MAX_NEW_DEFAULT = CFG["max_new_default"]
KEEPALIVE_S = float(CFG["keepalive_s"] or 0)
THINK_BUDGET_DEFAULT = int(CFG["think_budget_default"] or 0)
MAX_STREAMS, MAX_SETS = CFG["streams"], CFG["sets"]
SCHED_PIECE = CFG["sched_piece"]
MIN_FREE_GB, MAX_WAIT_S, MAX_QUEUE = CFG["min_free_gb"], CFG["max_wait_s"], CFG["max_queue"]
VISION_MAX_TOKENS = CFG["vision_max_tokens"]
DEFAULT_EFFORT = CFG["reasoning_effort_default"]
DEFAULT_TEMP, DEFAULT_TOP_P = CFG["temperature"], CFG["top_p"]
MODEL = CFG["served_model_name"]
BUCKET_MIN = CFG["bucket_min"]
if BUCKET_MIN:
    eng.PREFILL_BUCKETS = tuple(b for b in type(eng).PREFILL_BUCKETS if b >= BUCKET_MIN)
STATE = {"url": None, "requests": 0, "tokens": 0, "prefix_hits": 0, "prefix_tokens_reused": 0, "prefill_s": 0.0,
         "snap_hits": 0, "snap_parks": 0, "snap_pins": 0, "snap_entries": 0, "snap_bytes": 0, "snap_s": 0.0,
         "steps": 0, "step_tokens": 0}
T = {k: tok.convert_tokens_to_ids(k) for k in ("<think>", "</think>", "<tool_call>", "</tool_call>", "<arg_key>", "</arg_key>",
                                                "<arg_value>", "</arg_value>", "<|user|>", "<|observation|>", "<|assistant|>",
                                                "<|image|>", "<|begin_of_image|>", "<|end_of_image|>")}
STOP_IDS = set(eos) | {T["<|user|>"], T["<|observation|>"]}

# ---- images
IMG_CACHE = collections.OrderedDict()                  # sha256 -> (n_tokens, embeddings f32 [n, D]); LRU
IMG_CACHE_MAX = 64


def embed_image(img_bytes):
    """Image bytes -> (n_tokens, embeddings [n, D] f32) through the vision tower; cached by content hash. The patch
    count is padded to a power of two with a dummy image segment (its rows are dropped) so few shapes compile."""
    from PIL import Image
    if VISION_FWD is None:
        raise ValueError("this server has no vision tower loaded (config: vision)")
    h = hashlib.sha256(img_bytes).hexdigest()
    if h in IMG_CACHE:
        IMG_CACHE.move_to_end(h)
        return IMG_CACHE[h]
    patches, grid = VIS.preprocess(Image.open(io.BytesIO(img_bytes)), max_tokens=VISION_MAX_TOKENS)
    n = patches.shape[0]
    bucket = max(256, 1 << (n - 1).bit_length())
    grids = (grid,) if bucket == n else (grid, (1, 2, (bucket - n) // 2))
    if bucket > n:
        patches = np.concatenate([patches, np.zeros((bucket - n, patches.shape[1]), patches.dtype)], 0)
    t = time.time()
    out = np.asarray(VISION_FWD(patches, grids), np.float32)[:VIS.n_tokens(grid)]
    STATE["vision_s"] = STATE.get("vision_s", 0.0) + time.time() - t
    STATE["images"] = STATE.get("images", 0) + 1
    IMG_CACHE[h] = (out.shape[0], out)
    while len(IMG_CACHE) > IMG_CACHE_MAX:
        IMG_CACHE.popitem(last=False)
    return IMG_CACHE[h]


def _image_bytes(block):
    """Anthropic image block or OpenAI image_url part -> raw bytes (base64 or data: URL inline; http(s) fetched)."""
    if block.get("type") == "image":
        src = block.get("source", {})
        if src.get("type") == "base64":
            return base64.b64decode(src["data"])
        url = src.get("url", "")
    else:
        u = block.get("image_url", "")
        url = u.get("url", "") if isinstance(u, dict) else u
    if url.startswith("data:"):
        return base64.b64decode(url.split(",", 1)[1])
    if url.startswith("http://") or url.startswith("https://"):
        return urllib.request.urlopen(url, timeout=30).read()
    raise ValueError("unsupported image source")


def _img_sig(img_bytes):
    """Negative pseudo token id identifying an image in prompt signatures (all of its tokens carry it)."""
    return -(1 + int(hashlib.sha256(img_bytes).hexdigest()[:8], 16) % (1 << 30))


def _dec(ids):
    return tok.decode([T["<|image|>"] if int(i) < 0 else int(i) for i in ids])


def _feed(sig, imgs):
    """Signature ids (image tokens negative) -> (real ids int32 [n], embeds (idx, vec) or None) for the engine."""
    sig_a, off = sig
    ids = np.where(sig_a < 0, T["<|image|>"], sig_a).astype(np.int32)
    idx = np.nonzero(sig_a < 0)[0]
    if len(idx) == 0:
        return ids, None
    vec = np.concatenate([imgs[int(sig_a[i])][1][off[i]:off[i] + 1] for i in idx], 0)
    return ids, (idx, vec)


def _run_offsets(sig):
    """Per position: index within its run of equal negative ids (0 for text)."""
    sig = np.asarray(sig)
    off = np.zeros(len(sig), np.int64)
    for i in range(1, len(sig)):
        if sig[i] < 0 and sig[i] == sig[i - 1]:
            off[i] = off[i - 1] + 1
    return off


# ---- sampling of the first token (the rest are sampled on the device)
def sample(logits, temperature, top_p, rng, n_cand=2048):
    z = np.asarray(logits, dtype=np.float32).reshape(-1)
    if temperature <= 0:
        return int(z.argmax())
    n_cand = min(n_cand, z.size - 1)
    cand = np.argpartition(-z, n_cand)[:n_cand]
    zc = z[cand] / temperature
    zc = zc - zc.max()
    p = np.exp(zc); p /= p.sum()
    if top_p < 1.0:
        order = np.argsort(-p); cum = np.cumsum(p[order])
        keep = order[: max(1, int((cum <= top_p).sum()) + 1)]
        q = np.zeros_like(p); q[keep] = p[keep]; p = q / q.sum()
    return int(cand[rng.choice(n_cand, p=p)])


BASE_MIN, SNAP_HOST_GB = CFG["base_min"], CFG["snap_host_gb"]
SNAP_ROWS, SNAP_MIN, SNAP_WARM = CFG["snap_rows"], CFG["snap_min"], CFG["snap_warm_tokens"]


def system_end(prompt):
    """Index of the first <|user|> token = end of the rendered system section (system prompt + tools)."""
    try:
        return prompt.index(T["<|user|>"])
    except ValueError:
        return 0


def _match_len(live_ids, pos, prompt, quiet=False):
    """Longest reuse of the live context (ids[:pos]) for `prompt`: (k, fed) where prompt[k:] must still be prefilled and
    `fed` = the ids the engine will have seen after that, or (0, None). Handles a dropped thinking block (the template
    renders `<think></think>` where the live context holds the reasoning) and re-tokenisation drift near the boundary."""
    THINK, END = T["<|assistant|>"], T["</think>"]
    i = j = 0
    while True:
        n = min(pos - i, len(prompt) - j)
        if n > 0:
            neq = np.asarray(live_ids[i:i + n]) != np.asarray(prompt[j:j + n])
            d = int(np.argmax(neq)) if neq.any() else n
            i += d; j += d
        if i >= pos:
            return (j, list(live_ids[:pos]) + list(prompt[j:])) if j <= len(prompt) else (0, None)
        if j >= len(prompt):
            return 0, None
        if i > 0 and live_ids[i - 1] == T["<think>"] and prompt[j] == END and END in live_ids[i:pos]:
            i = live_ids.index(END, i) + 1; j += 1
            continue
        if pos - i > 256:
            return 0, None
        a = max(0, i - 32); ja = j - (i - a)
        t_live = _dec(live_ids[a:pos])
        for k in range(max(ja + 1, j - 24), min(len(prompt), j + (pos - i) + 24) + 1):
            if _dec(prompt[ja:k]) == t_live:
                return k, list(live_ids[:pos]) + list(prompt[k:])
        return 0, None


RNG = np.random.default_rng()


def sched_feed(req, a, b):
    return _feed((req.sig[a:b], req.off[a:b]), req.imgs or {})


_drop = [k for k in eng._progs if isinstance(k, tuple) and k[0] == "group" and k[3] == 1 and k[5]]   # batch-1 loop programs: unused here
for k in _drop:
    del eng._progs[k]
SNAPS = SnapStore(eng, int(SNAP_HOST_GB * 1e9), SNAP_ROWS, match_len=_match_len, log=log, state=STATE)
SCHED = Scheduler(eng, STOP_IDS, MAX_STREAMS, MAX_SETS, feed=sched_feed, match_len=_match_len, system_end=system_end,
                  first_sample=lambda z, t, p: sample(z, t, p, RNG), snaps=SNAPS, log=log, state=STATE,
                  base_min=BASE_MIN, snap_min=SNAP_MIN, piece=SCHED_PIECE, min_free_gb=MIN_FREE_GB, max_wait_s=MAX_WAIT_S,
                  spec=DFlashPolicy(DFLASH_COST, {int(k): float(v) for k, v in CFG["dflash_prior"].items()
                                                  if int(k) in DFLASH_COST}, rows=DFLASH_ROWS)
                  if getattr(eng, "dflash", None) is not None else None)


class QueueFull(Exception):
    """Too many requests in flight (-> 429)."""


class ClientGone(Exception):
    """The client closed the connection mid-response."""


_IDLE = object()


def generate(prompt, max_new, temperature, top_p, on_token=None, imgs=None, rid=None, on_idle=None, budget=None):
    """One request through the scheduler -> (out ids, prefill_s, decode_s, reused, reason) with reason "stop" | "length"
    | "stop_sequence" (`on_token` returned True: cancelled there) | "cancelled". Tokens reach `on_token` on THIS thread;
    `on_idle` is called when no token arrived for KEEPALIVE_S (a queued request); `budget` = Request.budget."""
    prompt = [int(t) for t in prompt]
    if MAX_QUEUE and len(SCHED.pending) + len(SCHED.active) >= MAX_STREAMS + MAX_QUEUE:
        raise QueueFull(f"{len(SCHED.active)} requests running and {len(SCHED.pending)} waiting; retry later")
    q = queue.Queue()
    req = Request(prompt, max_new, temperature, top_p, imgs=imgs, rid=rid, budget=budget,
                  on_token=q.put if on_token else None, on_done=(lambda: q.put(None)) if on_token else None)
    req.sig = np.asarray(prompt, np.int64)
    req.off = _run_offsets(req.sig) if imgs else np.zeros(len(prompt), np.int64)
    SCHED.submit(req)
    stopped = False
    if on_token:
        while True:
            try:
                t = q.get(timeout=KEEPALIVE_S or None)
            except queue.Empty:
                t = _IDLE
            if t is None:
                break
            if stopped:
                continue
            try:
                if t is _IDLE:
                    if on_idle is not None:
                        on_idle()
                elif on_token(t):
                    stopped = True
                    req.cancel()
            except Exception:
                req.cancel()
                raise
    req.done.wait()
    if req.error is not None:
        raise req.error
    return req.out, req.prefill_s, req.decode_s, req.reused, ("stop_sequence" if stopped else req.stop_reason)


# ---- output parsing (token level)
class StopFilter:
    """`stop_sequences` on the TEXT events: `feed(text) -> (emit, hit)` holds back a tail that could begin a stop
    sequence; at a match returns the text before it and the sequence; `flush()` releases the held tail."""

    def __init__(self, stops):
        self.stops = [s for s in (stops or []) if s]
        self.buf, self.hit = "", None

    def feed(self, text):
        if not self.stops:
            return text, None
        self.buf += text
        best = None
        for s in self.stops:
            i = self.buf.find(s)
            if i >= 0 and (best is None or i < best[0]):
                best = (i, s)
        if best is not None:
            out, self.buf, self.hit = self.buf[:best[0]], "", best[1]
            return out, best[1]
        keep = 0
        for s in self.stops:
            for k in range(min(len(s) - 1, len(self.buf)), keep, -1):
                if self.buf.endswith(s[:k]):
                    keep = k; break
        cut = len(self.buf) - keep
        out, self.buf = self.buf[:cut], self.buf[cut:]
        return out, None

    def flush(self):
        out, self.buf = self.buf, ""
        return out


def parse_tool_call(body, tools):
    """body = tokens between <tool_call> and </tool_call>."""
    name_end = body.index(T["<arg_key>"]) if T["<arg_key>"] in body else len(body)
    name = tok.decode(body[:name_end]).strip()
    schema = {}
    for t in tools or []:
        f = t.get("function", t)
        if f.get("name") == name:
            schema = (f.get("parameters") or f.get("input_schema") or {}).get("properties", {}) or {}
    args, i = {}, name_end
    while i < len(body) and T["<arg_key>"] in body[i:]:
        a = body.index(T["<arg_key>"], i) + 1
        b = body.index(T["</arg_key>"], a) if T["</arg_key>"] in body[a:] else len(body)
        key = tok.decode(body[a:b]).strip()
        c = body.index(T["<arg_value>"], b) + 1 if T["<arg_value>"] in body[b:] else len(body)
        d = body.index(T["</arg_value>"], c) if T["</arg_value>"] in body[c:] else len(body)
        raw = tok.decode(body[c:d])
        typ = schema.get(key, {}).get("type")
        if typ == "string":
            val = raw
        else:
            try:
                val = json.loads(raw)
            except Exception:  # noqa: BLE001
                val = raw
        args[key] = val
        i = d + 1
    return {"id": "call_" + uuid.uuid4().hex[:12], "name": name, "input": args}


class TokenStream:
    """Incremental token -> (kind, text/tool) events: kind in {"thinking", "text", "tool"}; partial multibyte text
    is held back until it decodes cleanly; tool-call tokens are buffered until </tool_call>."""

    def __init__(self, tools, mode="thinking"):
        self.tools, self.mode, self.buf, self.tool_buf = tools, mode, [], []

    def feed(self, t):
        if t in STOP_IDS:
            return self.flush()
        if self.mode == "tool":
            if t == T["</tool_call>"]:
                self.mode = "text"; call = parse_tool_call(self.tool_buf, self.tools); self.tool_buf = []
                return [("tool", call)]
            self.tool_buf.append(t); return []
        if t == T["</think>"] and self.mode == "thinking":
            ev = self.flush(); self.mode = "text"; return ev
        if t == T["<tool_call>"] and self.mode == "text":
            ev = self.flush(); self.mode = "tool"; return ev
        self.buf.append(t)
        piece = tok.decode(self.buf)
        if piece.endswith("�"):
            return []
        self.buf = []
        return [(self.mode, piece)] if piece else []

    def flush(self):
        if not self.buf:
            return []
        piece = tok.decode(self.buf); self.buf = []
        return [(self.mode, piece)] if piece else []


THINK_WRAP = tok("\n\nMy thinking budget is used up, so I will give the answer directly now.\n", add_special_tokens=False).input_ids + [T["</think>"]]


def run_request(ids, max_new, temperature, top_p, imgs, rid, tools, stops=None, prefix=(), on_event=None, raw=False,
                think_budget=None):
    """One request end to end: `prefix` ids (a forced tool call) are fed to the parser first, then the generated tokens;
    events (kind, value) go to `on_event` as they are parsed (stop sequences applied to the text), plus ("ping", None)
    when nothing was emitted for KEEPALIVE_S (a tool call is buffered until it closes; a queued request waits): the
    tunnel drops a response that stays silent for ~100 s. `think_budget`: thinking tokens after which `</think>` is
    forced (a short wrap-up line first) so the answer always comes. Returns
    {reasoning, text, calls, out_len, stop, stop_sequence, tp, tg, reused}."""
    ts, sf = TokenStream(tools, "text" if raw else "thinking"), StopFilter(stops)
    res = {"reasoning": "", "text": "", "calls": [], "n": 0}
    last = [time.time()]

    def tick():
        if on_event is not None and KEEPALIVE_S and time.time() - last[0] >= KEEPALIVE_S:
            last[0] = time.time()
            on_event("ping", None)

    def emit(kind, val):
        last[0] = time.time()
        if kind == "thinking":
            res["reasoning"] += val
        elif kind == "text":
            res["text"] += val
        else:
            res["calls"].append(val)
        if on_event is not None:
            on_event(kind, val)

    def handle(t):
        for kind, val in ts.feed(t):
            if kind == "text":
                out, hit = sf.feed(val)
                if out:
                    emit("text", out)
                if hit is not None:
                    return True
            else:
                held = sf.flush()
                if held:
                    emit("text", held)
                emit(kind, val)
        return False

    for t in prefix:
        handle(t)

    def on_token(t):
        res["n"] += 1
        if handle(t):
            return True
        tick()
        return False
    full = list(ids) + list(prefix)
    budget = (int(think_budget), T["</think>"], THINK_WRAP) if think_budget and not raw and full and full[-1] == T["<think>"] else None
    try:
        out, tp, tg, reused, reason = generate(full, max_new, temperature, top_p, on_token, imgs, rid, on_idle=tick, budget=budget)
    except ClientGone as e:
        e.n = res["n"]
        raise
    if reason != "stop_sequence":
        held = sf.flush()
        if held:
            emit("text", held)
    return {"reasoning": res["reasoning"], "text": res["text"], "calls": res["calls"], "out_len": res["n"],
            "stop": reason, "stop_sequence": sf.hit, "tp": tp, "tg": tg, "reused": reused}


def forced_prefix(choice):
    """tool_choice -> generation prefix ids: "any" forces a tool call, a name forces that tool (the template's own
    rendering of a tool turn: `<think></think><tool_call>name...`, so the next turn's prompt matches the live context)."""
    if choice in (None, "none"):
        return []
    ids = [T["</think>"], T["<tool_call>"]]
    if choice != "any":
        ids += tok(choice, add_special_tokens=False).input_ids
    return ids


def tool_choice_of(req, api):
    """-> None (auto) | "none" | "any" | tool name, from an Anthropic or OpenAI request."""
    tc = req.get("tool_choice")
    if tc is None or not req.get("tools"):
        return None
    if api == "anthropic":
        kind = tc.get("type", "auto") if isinstance(tc, dict) else str(tc)
        return {"auto": None, "none": "none", "any": "any"}.get(kind, tc.get("name") if isinstance(tc, dict) else None)
    if isinstance(tc, str):
        return {"none": "none", "required": "any"}.get(tc)
    return (tc.get("function") or {}).get("name") or None


def stops_of(req):
    s = req.get("stop_sequences", req.get("stop"))
    if not s:
        return []
    return [s] if isinstance(s, str) else [str(x) for x in s if x]


# ---- request conversion
STRIP_PATTERNS = [re.compile(p) for p in [
    r"\s*<total_tokens>[^<]*</total_tokens>\s*",            # Claude Code's per-request token budget reminder
    r"x-anthropic-billing-header:[^\n]*\n?",                # Claude Code's per-session billing metadata (system role)
]]


def clean(text):
    for pat in STRIP_PATTERNS:
        text = pat.sub("", text)
    return text


def anthropic_to_messages(req):
    """Anthropic Messages request -> (chat-template messages, tools in OpenAI function form)."""
    msgs = []
    sysm = req.get("system")
    if sysm:
        text = sysm if isinstance(sysm, str) else "\n".join(b.get("text", "") for b in sysm if b.get("type") == "text")
        msgs.append({"role": "system", "content": clean(text)})
    for m in req.get("messages", []):
        c = m.get("content")
        if isinstance(c, str):
            c = clean(c)
            if c.strip() or m["role"] != "system":
                msgs.append({"role": m["role"], "content": c})
            continue
        if m["role"] == "system":
            text = clean("\n".join(b.get("text", "") for b in c if b.get("type") == "text"))
            if text.strip():
                msgs.append({"role": "system", "content": text})
            continue
        if m["role"] == "user":
            items, results = [], []
            for b in c:
                if b.get("type") == "text":
                    if clean(b["text"]).strip():
                        items.append({"type": "text", "text": clean(b["text"])})
                elif b.get("type") == "image":
                    items.append({"type": "image", "_img": _image_bytes(b)})
                elif b.get("type") == "tool_result":
                    rc = b.get("content", "")
                    if isinstance(rc, list):
                        parts = [{"type": "text", "text": x.get("text", "")} if x.get("type") == "text"
                                 else {"type": "image", "_img": _image_bytes(x)} for x in rc if x.get("type") in ("text", "image")]
                        rc = parts if any(p["type"] == "image" for p in parts) else "\n".join(p["text"] for p in parts)
                    results.append({"role": "tool", "content": rc if isinstance(rc, list) else str(rc),
                                    "tool_call_id": b.get("tool_use_id", "")})
            msgs.extend(results)
            if items:
                msgs.append({"role": "user", "content": _content(items)})
        else:
            texts, thinks, calls = [], [], []
            for b in c:
                if b.get("type") == "text":
                    texts.append(b["text"])
                elif b.get("type") == "thinking":
                    thinks.append(b.get("thinking", ""))
                elif b.get("type") == "tool_use":
                    calls.append({"id": b.get("id"), "type": "function",
                                  "function": {"name": b["name"], "arguments": b.get("input", {})}})
            am = {"role": "assistant", "content": "\n".join(texts)}
            if thinks:
                am["reasoning_content"] = "\n".join(thinks)
            if calls:
                am["tool_calls"] = calls
            msgs.append(am)
    tools = [{"type": "function", "function": {"name": t["name"], "description": t.get("description", ""),
                                               "parameters": t.get("input_schema", {"type": "object", "properties": {}})}}
             for t in req.get("tools", []) if t.get("name")]
    return msgs, tools


def openai_messages(req):
    msgs = []
    for m in req["messages"]:
        m = dict(m)
        if isinstance(m.get("content"), list):
            items = [{"type": "text", "text": p.get("text", "")} if p.get("type") == "text"
                     else {"type": "image", "_img": _image_bytes(p)} for p in m["content"] if p.get("type") in ("text", "image_url")]
            m["content"] = _content(items)
        if m.get("content") is None:
            m["content"] = ""
        for tc in m.get("tool_calls") or []:
            fn = tc.get("function", {})
            if isinstance(fn.get("arguments"), str):
                try:
                    fn["arguments"] = json.loads(fn["arguments"])
                except Exception:  # noqa: BLE001
                    fn["arguments"] = {"_raw": fn["arguments"]}
        msgs.append(m)
    return msgs, req.get("tools") or None


def _content(items):
    if any(it["type"] == "image" for it in items):
        return items
    return "\n".join(it["text"] for it in items)


def _images_of(msgs):
    out = []
    for m in msgs:
        if isinstance(m.get("content"), list):
            out += [it["_img"] for it in m["content"] if it.get("type") == "image"]
    return out


def render(msgs, tools, effort):
    """-> (signature ids, imgs): token ids with every <|image|> expanded to the image's token count, those tokens
    replaced by the image's negative signature id; imgs maps signature -> (n_tokens, embeddings)."""
    text = tok.apply_chat_template(msgs, tools=tools or None, tokenize=False, add_generation_prompt=True, reasoning_effort=effort)
    ids = tok(text, add_special_tokens=False).input_ids
    images = _images_of(msgs)
    if not images:
        return ids, {}
    imgs, out, n_img = {}, [], 0
    for t in ids:
        if t != T["<|image|>"]:
            out.append(t); continue
        if n_img >= len(images):
            raise ValueError("more <|image|> tokens than images")
        b = images[n_img]; n_img += 1
        v = _img_sig(b)
        imgs[v] = embed_image(b)
        out += [v] * imgs[v][0]
    if n_img != len(images):
        raise ValueError("fewer <|image|> tokens than images")
    return out, imgs


def think_budget_of(req):
    """Thinking tokens before `</think>` is forced: Anthropic `thinking.budget_tokens`, OpenAI-style `thinking_budget`
    or `reasoning.max_tokens`, else the server default; None = unlimited."""
    th = req.get("thinking") or {}
    b = th.get("budget_tokens") if th.get("type") == "enabled" else None
    if b is None:
        b = req.get("thinking_budget") or (req.get("reasoning") or {}).get("max_tokens")
    b = int(b or THINK_BUDGET_DEFAULT or 0)
    return b if b > 0 else None


def effort_of(req, default=DEFAULT_EFFORT):
    e = (req.get("chat_template_kwargs") or {}).get("reasoning_effort") or req.get("reasoning_effort")
    th = req.get("thinking") or {}
    if not e and th.get("type") == "enabled":
        e = "high" if int(th.get("budget_tokens", 0) or 0) >= 8192 else "low"
    if not e and th.get("type") == "adaptive":
        e = default
    if not e and th.get("type") == "disabled":
        e = "low"
    return e or default


# ---- HTTP
class H(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.0"      # one request per connection: SSE responses carry no framing for keep-alive

    def log_message(self, *a):
        pass

    def _json(self, code, obj, headers=()):
        body = json.dumps(obj).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        for k, v in headers:
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(body))); self.end_headers(); self._raw(body)

    def _raw(self, b):
        try:
            self.wfile.write(b if isinstance(b, bytes) else b.encode()); self.wfile.flush()
        except (BrokenPipeError, ConnectionResetError):
            raise ClientGone() from None

    def _auth(self):
        if not API_KEY:
            return True
        h = self.headers.get("Authorization", ""); k = self.headers.get("x-api-key", "")
        return h == f"Bearer {API_KEY}" or k == API_KEY

    def _sse(self, obj, event=None):
        self._raw((f"event: {event}\n" if event else "") + f"data: {json.dumps(obj)}\n\n")

    def do_GET(self):
        if self.path.startswith("/health"):
            return self._json(200, {"status": "ok", "model": MODEL, "layers": cfg.n_layers, "max_len": eng.max_len, "max_streams": MAX_STREAMS,
                                    "max_sets": MAX_SETS, "max_queue": MAX_QUEUE, "piece": SCHED.piece, "buckets": list(eng.PREFILL_BUCKETS),
                                    "active": len(SCHED.active), "pending": len(SCHED.pending),
                                    "live_contexts": len(SCHED.live), "hbm_free_gb": SCHED.free_gb(),
                                    "hbm_free_in_use_gb": SCHED.in_use_free_gb(), **STATE})
        if self.path.startswith("/v1/models"):
            return self._json(200, {"object": "list", "data": [{"id": MODEL, "object": "model"}]})
        self._json(404, {"error": "not found"})

    def do_POST(self):
        if not self._auth():
            return self._json(401, {"error": {"type": "authentication_error", "message": "bad api key"}})
        n = int(self.headers.get("Content-Length", 0))
        try:
            req = json.loads(self.rfile.read(n) or b"{}")
        except Exception as e:  # noqa: BLE001
            return self._json(400, {"error": str(e)})
        try:
            if self.path.startswith("/v1/messages/count_tokens"):
                msgs, tools = anthropic_to_messages(req)
                return self._json(200, {"input_tokens": len(render(msgs, tools, effort_of(req))[0])})
            if self.path.startswith("/v1/messages"):
                return self.anthropic(req)
            if self.path.startswith("/v1/chat/completions"):
                return self.openai_chat(req)
            if self.path.startswith("/v1/completions"):
                return self.openai_text(req)
        except Exception as e:  # noqa: BLE001
            if isinstance(e, ClientGone):
                return log(f"client disconnected: {getattr(self, 'rid', '?')} after {getattr(e, 'n', '?')} tokens (stream cancelled)")
            if isinstance(e, QueueFull):
                STATE["rejected"] = STATE.get("rejected", 0) + 1
                return self._json(429, {"error": {"type": "rate_limit_error", "message": str(e)}}, [("Retry-After", "5")])
            import traceback; log("request failed:", traceback.format_exc()[-1500:])
            code, kind = (503, "overloaded_error") if isinstance(e, TimeoutError) else \
                         (400, "invalid_request_error") if isinstance(e, ValueError) else (500, "api_error")
            try:
                return self._json(code, {"error": {"type": kind, "message": str(e)}})
            except Exception:  # noqa: BLE001
                return
        self._json(404, {"error": "not found"})

    # ---- OpenAI
    def openai_chat(self, req):
        msgs, tools = openai_messages(req)
        choice = tool_choice_of(req, "openai")
        ids, imgs = render(msgs, None if choice == "none" else tools, effort_of(req))
        max_new = int(req.get("max_tokens") or req.get("max_completion_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "chatcmpl-" + uuid.uuid4().hex[:12]
        STATE["requests"] += 1
        kw = dict(tools=tools, stops=stops_of(req), prefix=forced_prefix(choice), think_budget=think_budget_of(req))
        finish = lambda r: "tool_calls" if r["calls"] else ("length" if r["stop"] == "length" else "stop")
        if req.get("stream"):
            self.send_response(200); self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache"); self.end_headers()
            nt = [0]

            def chunk(delta, finish=None):
                self._sse({"id": rid, "object": "chat.completion.chunk", "model": MODEL,
                           "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]})

            def on_event(kind, val):
                if kind == "ping":
                    self._raw(": keep-alive\n\n")               # an SSE comment: OpenAI clients skip it
                elif kind == "thinking":
                    chunk({"reasoning_content": val})
                elif kind == "text":
                    chunk({"content": val})
                else:
                    chunk({"tool_calls": [{"index": nt[0], "id": val["id"], "type": "function",
                                           "function": {"name": val["name"], "arguments": json.dumps(val["input"])}}]})
                    nt[0] += 1
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, on_event=on_event, **kw)
            chunk({}, finish(r))
            self._raw(b"data: [DONE]\n\n")
        else:
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, **kw)
            msg = {"role": "assistant", "content": r["text"]}
            if r["reasoning"]:
                msg["reasoning_content"] = r["reasoning"]
            if r["calls"]:
                msg["tool_calls"] = [{"id": c["id"], "type": "function",
                                      "function": {"name": c["name"], "arguments": json.dumps(c["input"])}} for c in r["calls"]]
            self._json(200, {"id": rid, "object": "chat.completion", "model": MODEL,
                             "choices": [{"index": 0, "message": msg, "finish_reason": finish(r)}],
                             "usage": {"prompt_tokens": len(ids), "completion_tokens": r["out_len"],
                                       "prefill_s": round(r["tp"], 2), "decode_tok_s": round(r["out_len"] / max(r["tg"], 1e-6), 2)}})
        self._done(rid, ids, r)

    def openai_text(self, req):
        ids = tok(req.get("prompt", ""), add_special_tokens=False).input_ids
        max_new = int(req.get("max_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "cmpl-" + uuid.uuid4().hex[:12]
        STATE["requests"] += 1
        r = run_request(ids, max_new, temperature, top_p, None, rid, None, stops=stops_of(req), raw=True)
        self._json(200, {"id": rid, "object": "text_completion", "model": MODEL,
                         "choices": [{"index": 0, "text": r["text"], "finish_reason": "length" if r["stop"] == "length" else "stop"}],
                         "usage": {"prompt_tokens": len(ids), "completion_tokens": r["out_len"]}})
        self._done(rid, ids, r)

    # ---- Anthropic
    def anthropic(self, req):
        msgs, tools = anthropic_to_messages(req)
        choice = tool_choice_of(req, "anthropic")
        ids, imgs = render(msgs, None if choice == "none" else tools, effort_of(req))
        max_new = int(req.get("max_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "msg_" + uuid.uuid4().hex[:16]
        STATE["requests"] += 1
        kw = dict(tools=tools, stops=stops_of(req), prefix=forced_prefix(choice), think_budget=think_budget_of(req))
        stop_of = lambda r: "tool_use" if r["calls"] else {"length": "max_tokens", "stop_sequence": "stop_sequence"}.get(r["stop"], "end_turn")
        if req.get("stream"):
            self.send_response(200); self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache"); self.end_headers()
            self._sse({"type": "message_start", "message": {"id": rid, "type": "message", "role": "assistant", "model": MODEL,
                                                             "content": [], "stop_reason": None, "stop_sequence": None,
                                                             "usage": {"input_tokens": len(ids), "output_tokens": 0}}}, "message_start")
            st = {"idx": -1, "open": None}

            def open_block(kind):
                st["idx"] += 1; st["open"] = kind
                blk = {"type": "thinking", "thinking": ""} if kind == "thinking" else {"type": "text", "text": ""}
                self._sse({"type": "content_block_start", "index": st["idx"], "content_block": blk}, "content_block_start")

            def close_block():
                if st["open"] is not None:
                    self._sse({"type": "content_block_stop", "index": st["idx"]}, "content_block_stop"); st["open"] = None

            def on_event(kind, val):
                if kind == "ping":
                    self._sse({"type": "ping"}, "ping")            # the Anthropic API's own keep-alive event
                elif kind in ("thinking", "text"):
                    if st["open"] != kind:
                        close_block(); open_block(kind)
                    d = {"type": "thinking_delta", "thinking": val} if kind == "thinking" else {"type": "text_delta", "text": val}
                    self._sse({"type": "content_block_delta", "index": st["idx"], "delta": d}, "content_block_delta")
                else:
                    close_block(); st["idx"] += 1
                    self._sse({"type": "content_block_start", "index": st["idx"],
                               "content_block": {"type": "tool_use", "id": val["id"], "name": val["name"], "input": {}}}, "content_block_start")
                    self._sse({"type": "content_block_delta", "index": st["idx"],
                               "delta": {"type": "input_json_delta", "partial_json": json.dumps(val["input"])}}, "content_block_delta")
                    self._sse({"type": "content_block_stop", "index": st["idx"]}, "content_block_stop")
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, on_event=on_event, **kw)
            close_block()
            self._sse({"type": "message_delta", "delta": {"stop_reason": stop_of(r), "stop_sequence": r["stop_sequence"]},
                       "usage": {"output_tokens": r["out_len"]}}, "message_delta")
            self._sse({"type": "message_stop"}, "message_stop")
        else:
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, **kw)
            content = []
            if r["reasoning"]:
                content.append({"type": "thinking", "thinking": r["reasoning"], "signature": ""})
            if r["text"].strip() or not r["calls"]:
                content.append({"type": "text", "text": r["text"]})
            content += [{"type": "tool_use", "id": c["id"], "name": c["name"], "input": c["input"]} for c in r["calls"]]
            self._json(200, {"id": rid, "type": "message", "role": "assistant", "model": MODEL, "content": content,
                             "stop_reason": stop_of(r), "stop_sequence": r["stop_sequence"],
                             "usage": {"input_tokens": len(ids), "output_tokens": r["out_len"]}})
        self._done(rid, ids, r)

    def _done(self, rid, ids, r):
        log(f"req {rid}: {len(ids)} prompt tok ({r['reused']} reused), {r['out_len']} new, prefill {r['tp']:.2f}s, "
            f"{r['out_len'] / max(r['tg'], 1e-6):.1f} tok/s, {r['stop']}")


# ----------------------------------------------------------------------------- 5. warm-up, serve, tunnel
banner(4, "Warm-up", "prefill buckets, batched decode programs, snapshot programs")
t_warm = time.time()
_u = [hbm()[0]]


def grew():
    """HBM the last warm-up stage kept (its programs: compiled code is loaded onto the chips at a program's first run
    and stays for the session), in MB."""
    u = hbm()[0]
    d, _u[0] = u - _u[0], u
    return f"+{d * 1e3:.0f} MB"


for b in [b for b in eng.PREFILL_BUCKETS if b <= eng.prefill_piece]:
    t = time.time()
    seq = np.concatenate([ids] * (b // ids.shape[1] + 1), 1)[:, :b]
    lg, cc, _ = eng.prefill(seq); jax.block_until_ready(lg); del cc
    log(f"   prefill bucket {b}: {time.time() - t:.0f}s, {grew()}")
    publish("compiling", what=f"prefill {b}", secs=int(time.time() - t))
for B in range(1, MAX_STREAMS + 1):
    t = time.time()
    _sets = [eng.alloc_caches(1) for _ in range(B)]
    _samp = DeviceSampler([1.0] * B, [0.95] * B, seed=0)
    _ids, _lg, _sets, _pos = eng.decode_rows(np.full((B,), 0, np.int32), _sets, [0] * B, _samp)
    jax.block_until_ready(_ids); del _sets, _lg, _ids, _pos
    log(f"   batched decode B={B}: {time.time() - t:.0f}s, {grew()}")
    publish("compiling", what=f"decode B={B}", secs=int(time.time() - t))
if getattr(eng, "dflash", None) is not None:          # draft (every block-size pair), verify + capture, head, settle
    for _T in sorted(DFLASH_COST):
        t = time.time()
        _lg, _cc, _p = eng.prefill(ids)
        _st = eng.dflash_start(np.array([0]), _cc, _p, T=_T)
        for _nt in sorted(DFLASH_COST):
            _e, _st = eng.dflash_step(_st, DeviceSampler(1.0, 0.95, seed=0), tuple(sorted(STOP_IDS)), next_T=_nt)
        _st = eng.dflash_settle(_st)                  # the stream's end: the verify run ahead undone (its program)
        del _lg, _cc, _st
        eng.dflash_end()                              # (the last step's arrays, kept for a deferred release)
        log(f"   DFlash block {_T}: {time.time() - t:.0f}s, {grew()}")
        publish("compiling", what=f"dflash T={_T}", secs=int(time.time() - t))
    for B in sorted(DFLASH_ROWS):                     # B streams at once: batched verify, draft, head, rollback
        t = time.time()
        _sets, _tok, _pp = [], [], []
        for _ in range(B):
            _lg, _cc, _p = eng.prefill(ids); _sets.append(_cc); _tok.append(0); _pp.append(_p)
        _st = eng.dflash_start_rows(_tok, _sets, _pp, T=min(DFLASH_COST),
                                    sampler=DeviceSampler([1.0] * B, [0.95] * B, seed=0))
        _sm = DeviceSampler([1.0] * B, [0.95] * B, seed=0)
        for _ in range(2):
            _e, _st = eng.dflash_step_rows(_st, _sm, tuple(sorted(STOP_IDS)))
        _st = eng.dflash_settle_rows(_st)             # (the state's end: every row's undo)
        del _lg, _sets, _st
        eng.dflash_end()
        log(f"   DFlash {B} streams: {time.time() - t:.0f}s, {grew()}")
        publish("compiling", what=f"dflash B={B}", secs=int(time.time() - t))
n_shard = max(eng.lcfg.seq_shard, 1)
for n_tok in range(SNAP_ROWS * n_shard, SNAP_WARM + 1, SNAP_ROWS * n_shard):
    cc = eng.alloc_caches(1)
    hs = eng.snapshot_to_host(eng.snapshot_prefix(cc, n_tok, rows_bucket=SNAP_ROWS)); del cc
    cc = eng.restore_prefix(eng.snapshot_from_host(hs)); jax.block_until_ready(cc[0]["state"] if "state" in cc[0] else cc[0]["c"]); del cc, hs
log(f"   snapshot programs: {grew()}")
use, lim = hbm()
log(f"   warm-up done in {(time.time() - t_warm) / 60:.1f} min; HBM chip0 {use:.2f} / {lim:.2f} GB")
if CACHE_DIR.exists():
    n_files = sum(1 for _ in CACHE_DIR.iterdir()); size = sum(f.stat().st_size for f in CACHE_DIR.iterdir()) / 1e9
    log(f"   compile cache: {n_files} entries, {size:.2f} GB in {CACHE_DIR}")
if globals().get("CACHE_USE") is not None:
    _used = sorted(set(CACHE_USE["hit"]) | set(CACHE_USE["put"]))
    (WORK / "jax_cache_used.txt").write_text("\n".join(_used) + "\n")
    log(f"   compile cache use: {len(set(CACHE_USE['hit']))} programs loaded from it ({CACHE_USE['read_s']:.0f}s), "
        f"{len(set(CACHE_USE['put']))} compiled and stored, {len(set(CACHE_USE['miss']) - set(CACHE_USE['put']))} compiled only "
        f"(small); the {len(_used)} entries of this build are listed in {WORK / 'jax_cache_used.txt'}")
if CFG["aot"]:
    _fs = list(EXPORT_DIR.glob("*" + AOT.EXT))
    log(f"   {AOT.summary()}; {len(_fs)} files, {sum(f.stat().st_size for f in _fs) / 1e9:.2f} GB in {EXPORT_DIR}")
publish("warmed", minutes=round((time.time() - t_warm) / 60, 1), hbm_gb=round(use, 2))

SCHED.start()
srv = ThreadingHTTPServer(("0.0.0.0", PORT), H)
threading.Thread(target=srv.serve_forever, daemon=True).start()
log(f"   HTTP server on :{PORT}")

banner(5, "Tunnel", "a public cloudflared URL")
url = None
TUN = [None]


def start_tunnel(attempts=3, wait_s=90):
    """A cloudflared quick tunnel -> its public URL, or None. Registration sometimes fails or hangs (seen in our runs):
    an attempt that prints no URL within `wait_s` is killed and retried."""
    pat = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    for i in range(attempts):
        tun = subprocess.Popen([str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        q = queue.Queue()
        threading.Thread(target=lambda: [q.put(l) for l in iter(tun.stdout.readline, "")], daemon=True).start()
        t0 = time.time()
        while time.time() - t0 < wait_s:
            try:
                line = q.get(timeout=5)
            except queue.Empty:
                if tun.poll() is not None:
                    break
                continue
            m = pat.search(line)
            if m:
                TUN[0] = tun
                return m.group(0)
        tun.kill()
        log(f"   tunnel attempt {i + 1}/{attempts}: no URL within {wait_s} s (cloudflared rc {tun.poll()}); retrying")
    return None


def tunnel_probe(u, wait_s=180):
    """Background: confirm the public URL answers (a fresh hostname can take a minute to resolve); if it never does,
    open a new tunnel once and announce the new URL."""
    t0 = time.time()
    while time.time() - t0 < wait_s:
        try:
            urllib.request.urlopen(f"{u}/health", timeout=10).read()
            log(f"   tunnel reachable from outside: {u} ({time.time() - t0:.0f} s after it opened)")
            return
        except Exception:  # noqa: BLE001
            time.sleep(10)
    log(f"   tunnel URL {u} did not answer in {wait_s} s: opening a new one")
    if TUN[0] is not None:
        TUN[0].kill()
    new = start_tunnel()
    if new:
        STATE["url"] = new
        publish("tunnel-url", endpoint=new)
        log(f"#  NEW ENDPOINT: {new}   (the earlier URL never resolved; the API key is unchanged)")
    else:
        publish("tunnel-failed", note="server still reachable inside the kernel on :8000")


if CFG["tunnel"]:
    url = start_tunnel()
    if url:
        publish("tunnel-url", endpoint=url)
        threading.Thread(target=tunnel_probe, args=(url,), daemon=True).start()
    else:
        publish("tunnel-failed", note="server still reachable inside the kernel on :8000")
STATE["url"] = url or f"http://127.0.0.1:{PORT}"

# ----------------------------------------------------------------------------- 6. ready, self-test, keep alive
banner(6, "Ready", "self-test, then serving")


def _post(path, body):
    hdr = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
    r = urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", data=json.dumps(body).encode(), headers=hdr), timeout=900)
    return json.loads(r.read().decode())


try:
    r = _post("/v1/chat/completions", {"messages": [{"role": "user", "content": "Say hi in five words."}], "max_tokens": 48, "temperature": 0})
    log("   self-test openai:", json.dumps(r["choices"][0]["message"]["content"])[:120], r["usage"])
    tools = [{"name": "read_file", "description": "Read a text file from disk.",
              "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "max_lines": {"type": "integer"}}, "required": ["path"]}}]
    msgs = [{"role": "user", "content": "Use the read_file tool to read /etc/hostname (at most 5 lines)."}]
    r = _post("/v1/messages", {"model": MODEL, "max_tokens": 256, "messages": msgs, "tools": tools, "temperature": 0})
    log("   self-test anthropic:", json.dumps(r["content"])[:200], r["stop_reason"])
    outs = [None, None]

    def _one(i, prompt):
        outs[i] = _post("/v1/chat/completions", {"messages": [{"role": "user", "content": prompt}], "max_tokens": 48, "temperature": 0})
    ths = [threading.Thread(target=_one, args=(0, "Count from one to ten in words.")), threading.Thread(target=_one, args=(1, "Name three primary colors."))]
    t0 = time.time(); [t.start() for t in ths]; [t.join() for t in ths]
    log("   self-test concurrent:", [o["choices"][0]["message"]["content"][:50] for o in outs], f"{time.time() - t0:.1f}s")
except Exception as e:  # noqa: BLE001
    import traceback; log("   self-test failed:", traceback.format_exc()[-800:])

endpoint = STATE["url"]
log("")
log("#" * 70)
log(f"#  READY — the server is live ({elapsed()} after start)")
log(f"#  ENDPOINT : {endpoint}   (OpenAI at /v1, Anthropic at /v1/messages)")
log(f"#  API KEY  : {API_KEY}")
log(f"#  MODEL    : {MODEL}   (context {CFG['max_len']}, up to {MAX_STREAMS} streams)")
log("#" * 70)
log("#  Claude Code:")
log(f"#    ANTHROPIC_BASE_URL={endpoint} ANTHROPIC_AUTH_TOKEN={API_KEY} ANTHROPIC_MODEL={MODEL} \\")
log(f"#    ANTHROPIC_SMALL_FAST_MODEL={MODEL} CLAUDE_CODE_MAX_CONTEXT_TOKENS={CFG['max_len']} claude")
log("#  OpenAI-compatible clients: base URL " + endpoint + "/v1, model " + MODEL)
log(f"#  Serving for up to {CFG['keepalive_min']} min, then this cell exits on its own.")
log("#" * 70)
publish("ready", endpoint=endpoint, api_key=API_KEY, model=MODEL, max_model_len=CFG["max_len"],
        keepalive_min=CFG["keepalive_min"], startup_secs=int(time.time() - T0))

if globals().get("SERVE_FOREVER", True):
    t_serve = time.time()
    while time.time() - t_serve < CFG["keepalive_min"] * 60:
        time.sleep(120)
        if SCHED.thread is not None and not SCHED.thread.is_alive():
            publish("stopped", reason="scheduler-exit")
            sys.exit(1)
        up = int((time.time() - t_serve) / 60)
        if up % 10 < 2:
            publish("heartbeat", up_min=up, endpoint=endpoint, requests=STATE["requests"], tokens=STATE["tokens"])
    publish("auto-shutdown", served_min=CFG["keepalive_min"])
    SCHED.stop(); srv.shutdown()
    sys.exit(0)


### Launch (leave this cell running: it is the server)
What you will see:
1. **Runtime**: a pinned TPU runtime and a few packages (about a minute)
2. **Weights**: the 3-bit experts and the int8 non-expert weights go straight onto the eight chips (3–10 minutes,
   depending on how fast the dataset mount reads)
3. **Vision**: the vision tower (seconds), then the draft model from Hugging Face (about a minute)
4. **Warm-up**: every program is loaded in its exported and compiled form from the serve dataset (under a minute);
   without the matching dataset every program is traced and compiled (about 28 minutes)
5. **Tunnel**: a public URL
6. **READY banner** with `ENDPOINT`, `API KEY` and `MODEL`, then a short self-test

Use the endpoint from anywhere:
```bash
curl <ENDPOINT>/v1/chat/completions -H "Authorization: Bearer <API_KEY>" \
  -H "Content-Type: application/json" -d '{
    "model": "glm-5.3-flash",
    "messages": [{"role": "user", "content": "Hello!"}]
  }'
```
Claude Code (the banner prints this line filled in):
```bash
ANTHROPIC_BASE_URL=<ENDPOINT> ANTHROPIC_AUTH_TOKEN=<API_KEY> ANTHROPIC_MODEL=glm-5.3-flash \
ANTHROPIC_SMALL_FAST_MODEL=glm-5.3-flash CLAUDE_CODE_MAX_CONTEXT_TOKENS=262144 claude
```


In [ ]:
!python serve_glm53.py
